# Chapter 5, Unit B: Break, repair and transfer durable memory

> **Learn with Prof Rod** — *Build Your Always-On AI Agent From Scratch*.
> **Read the full book and get the latest learning materials:** [https://profrod.ai/book](https://profrod.ai/book).
> **Join the Prof Rod learner community:** [https://profrod.ai/community](https://profrod.ai/community)
> — bring your questions, compare experiments and share what you build.
> **Original source and updates:** [profrodai/sovereign-agent](https://github.com/profrodai/sovereign-agent).

**Student edition · 90 minutes of dedicated work · 2026-09-09**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/profrodai/sovereign-agent/blob/main/book/exercises/ch05/profrod-sovereign-agent-ch05-b-memory-repair-transfer-exercise.ipynb) Runs on Google Colab as it ships today, or on any local Python 3.12+ kernel.

This is one of two practical units for Chapter 5. Unit A constructs and connects the
mechanism; Unit B investigates a controlled failure, repairs it and transfers the invariant.
Each is a complete ninety-minute session, with its own setup and required conceptual introductions.
Basic Python variables, conditions, loops, functions, lists and dictionaries are the starting
knowledge. Libraries and specialized concepts used here are introduced below before the main task.

By the end you should be able to:

1. Explain the chapter's mechanism using a prediction and an observed intermediate result.
2. Repair the failure: Putting the new preference first does not remove contradictory old guidance. Remember, correct, close and reopen SQLite; invoke context and inspect the actual system message seen by the model.
3. Solve **retrieve the newest eligible memory under a limit** using changed inputs and an independent expectation.
4. Retain your implementation, failed/corrected observations, causal explanation and limits.

| Minutes | Dedicated activity | Evidence you produce |
|---|---|---|
| 0–5 | State the problem and make a prediction | Initial prediction in your own words |
| 5–25 | Foundations and library examples | Values, explanations, revised predictions |
| 25–35 | Trace setup and the main interface | Input → learner function → observation |
| 35–60 | Reproduce, diagnose and repair | Source, visible checks and runtime evidence |
| 60–80 | Implement and challenge the transfer task | Function and a new counterexample |
| 80–90 | Retrieve, explain and save | Exit ticket and retained submission |

Installation is preparation time. These are planning estimates, not measured completion times.
Use the reference primers when a term is unfamiliar; in Unit B retrieve an explanation before
re-reading it. Run All checks that the artifact executes. Unfinished student functions deliberately
produce NEEDS_WORK. Keep your first attempt before opening answers.


This notebook belongs to the nineteen-chapter edition. Its supplied teaching runtime is embedded, so it can run without the textbook or another notebook. Where code uses `REFERENCE_LESSON`, that is the frozen runtime exercise identifier; the reader-facing chapter and saved unit identifiers use the current edition. Building against a supplied runtime is not proof that you have constructed all of its dependencies.


## Run the self-contained setup

Open this notebook in **Google Colab** with the badge above, or use a local
**Python 3.12+** Jupyter kernel, with **Pydantic 2**. If needed, run
`%pip install "pydantic==2.13.4"` once in a separate cell and restart the kernel.
Package installation needs internet; the lesson itself needs no repository download, API key
or prior notebook. The closing extension can use an OpenAI key from Colab's Secrets pane to put a
live model behind the same tools; without a key it replays a recorded transcript and says so.

The collapsed cell contains 91 frozen teaching files. Base85 represents compressed bytes
as text; `zlib` decompresses them; SHA-256 checks that the decoded files match this edition.
These are supplied packaging operations, not learner algorithms. `tempfile` creates an isolated
working copy; `Path` handles file locations; `sys.path` tells Python where the supplied modules
live; a `.pth` file, the mechanism an editable install uses, tells reviewed local subprocesses the same. The code is available for inspection below and performs no package installation itself.
The subsequent lesson teaches the libraries used by the mechanisms you will implement.

Run setup on every fresh kernel. It writes scratch runtime files separately from your retained
`practical-work/ch05-b` folder. Rerunning setup restores the frozen support files and keeps
your saved work. Restarting a kernel clears variables, not saved submission files. Source basis:
Sovereign Agent `444c5f6`. Some tasks use reviewed local subprocesses; they are not an OS sandbox.

<details><summary>Supplied offline setup and teaching files</summary>


In [ ]:
import base64
import hashlib
import json
import os
import site
import sys
import sysconfig
import tempfile
import zlib
from pathlib import Path

minimum_python = (3, 12)
if sys.version_info[:2] < minimum_python:
    raise RuntimeError(
        "This edition needs Python 3.12 or newer, which is what Google Colab runs today."
    )
try:
    import pydantic
except ImportError as error:
    raise RuntimeError('Run %pip install "pydantic==2.13.4", then restart the kernel.') from error
if pydantic.__version__.split(".")[0] != "2":
    raise RuntimeError("Use Pydantic 2; the tested version is 2.13.4.")

# Frozen, reviewed course files: data until explicitly loaded by the lesson.
COURSE_ARCHIVE = (
    "c-ri}3v=5>)*$*<@Oi4P$%LXw>P;_auOcfG-#C_6Qj(pLR4kAHC5$P8AxO&_m;U=buWocV8Xzgl9(zjj?ZzU|=;!"
    "J4etdd1NYnQ(!trf*m-W-hi{W)Vd_PT-$vlI<TkYob?pZf@c0vDlcpXmX@hoU}gLD!H@yF>n87A}KCXM28FpK|M#"
    "M!(N+$QtuAe;p0Xf#fs-7L<g=_HGTQSx!Vn8i)*?P78<Np7a;Y#xNOt7$mPV*Oy4PVe-O>oB_>Cj<TaUs(#Dj%Mj"
    "iu$YbEO;d&rq`@y|XYWqpD4E5>`7hxl8ppFnaH6I+SR~`9pHAaRJi|Gh{&M^-=mnoHCIS5K;@NPK&C?rr2MfP=_G"
    "U4>yLi^fU(ROn1RvqcfBf{$csHG;(PB8y@ZC8-#Q%M|c$U3i@CT>A{=V_Y!Q0nw-W<S(@L&?&#Qf@yaFUG2AwNzh"
    "{cD&wzUkoaS)9(Ic-E(r!KZuojq(u=e>pxpesgek<c@K8oetA+IFAdXwEe^ER1I@_c5?9NPe&&w|8&PWozKGCK|G"
    "t?6$kQ;v#rLtgn#exR4yh_JPK~&EDNvVtd?D;(|VWN%wm9|iM_@8Pd<uSI+m$qciB9?(WjI*8K#qY%%fjC`*W7e;"
    "~)%Rm>I(JY?4f_0)T_z`(Q9jU?!t<79cD%1B6_=tKR{()5-WQm|w?1wwS`Th$H!W6aqvvgSW6ZfdY_V5RcPaY@N&"
    "k*^7WLov-^>X)sCW!E`Yj!c~B2PVVN{I6s)ua2zH#&{H_48%I6GACnAvvexXJW4=F&3?aQ?ZwTJ7X0*7O^2RrU47"
    "R2JKEBI(XR}3I*Qb@A)HJ*sr(q=R2_y6_p8a(jPc|=}IVWcv*Tpkxii6T04eOPhKE1Ud!D^PxXK{FgjeZWtS!}(7"
    "v!BMZ5YY}cXxaUTAN%w4eLSHNI@>$eTj&(F0rt5ckKptLQYL`;^v)WP3az6)fl4h(hl?B7Nww-^MCX@e$t0VHlVP"
    "k|G=eA@>UOk4VFD-oM>t-@N3&TvtKqraOvmv&Nhd)(`2dhJje{G&Q2<4B=7WC$nu9&&PKMWMGK@1oXljt=RXoRT_"
    "=Rp*oRzd_1Y>}oIzrbto~Wnw;D366_$z0MM8TLXVr9U=kux`#n)G?zx~!gy><<AtJ&?7Gl1Y+X_xXgNdz>4>K{|y"
    ")T6Hs{B%)EdGx&3M6OPAsdbR>_db%>UJjOfo>ArL?rIoP&c_O{c({$V)!gzQf#n-cVw8+A7esfBe69!X2028(AfO"
    "QyK-G&*GX8;Ui`;(D73Sj}+y)erXocw&*7V3u9SwmOF1E0qq=dgeFs2b0dZXSLHIA=bioxzv*qzMm_X>G}l;QhRr"
    "&hKO+3R4mkgdOIWmr!+|Q1uc@Q=6U(6})T&__ro)@Eh<+t;LTW3U?W7IIZjTyi=qt(*TYU(QM2(^xWhppl?9Fy>W"
    "Omh~UV(K|YIm0TUAoC#Xnv(pVA@9v~4t-ckBdlP+Mm!8pD_)NyX&!Sp5pl7$3ICIrudA886q@h^QVDJ&sx1P5;KM"
    "ex7;dj@aKVbpXPTqe8}y?j0V^$dw5ya#B2+cAVoVbJ9n6G<j{@>)?&c_?%?vTq;GT6axDcWpwg_OBUEfwzos^c1c"
    "NJPQ^R9Gwm~03d@4)0Iu69&ON6{TSdm38%23^L`Tfz*G3+W_BF{4XZ69-c8_Nfz;OO_2zZ_F-m}_&JhOGXv7~|<E"
    "Zfw6#`@4jN*8Te{ds*v_Dg=kzQIem%r<{$%8Z^9$K>v>ZL_eKP2(3Kr+q>>7}Ne%!$wY=0p>o{rudV#U<i5b0WSk"
    "ORxzxfeRH*u8^Ti067sXDR~3YEDiWlKMI2p$UE1Jeq2Q8V-m_aBfy69MaFE$+oPjbr~RJ~4$po+c+)>Q`s4M{pB9"
    "66@hqB!BieMj^yZt<IJ}~Mim1Wb3x})HIFU}$K#{(W<lKNC%`%X+fPBSD1j}k3flO^YA2h$}%O=*ccsy$0mNtSB0"
    "5F0pVAt%b5nRI##xvkiCh;u@e@l)}jTsO<jiZ+V{FhD;h|nmx0<JIurYa(g6L4GM02C08lMgWw>KcNq8t-qx0bHx"
    "$`*^;Q5`T#_aGu=6=>j$t@lzPg(|I@s5E+j*hGP^3t`^}8$YvZj10_jss_tNXLY2U|;q>04D2SK?1gS|zLM_fe7_"
    "f{1ptArm4yG0WT?H0T0TIJsPYJs<^FXiX^J(|R3z$kczD~1wx82^_+<Nh$4T63+e~}aO%mS<^oQFLFr_eTN9AY$-"
    "qv-((IAjvPadtNqF(X9eJEVPjLGt@EB}2#sAf<sRn%7()0(^yUYFf`A``z1Y)dRSQO3h$_#4X)Ae$mA1SKHa{Z*T"
    "7Aya)2Q(5@c*J)l=R)gYyn8S%$q97jT2hJ2pS0Ug|dAURHeX1Xi~?I9w&jK*USJO26SH?Q9w!HqtBcl`GB$WlZoj"
    "=ftc>pIMES98Jzkyo>>UAGrkYBrd`hOeb1Jd5T6=}^!K=8Hi_lsnT(0D9Bz7DQX_YmlP4Xcn&}74xTBpTQ&W;Ef<"
    "0q;NBkYq7ckDLiFDpF#T%IAA;pm<MPU7HUe&_nONvbrZmNYi^X(SIAc>RaWz0c0~yO9lfbVaW(|@6)!-q-v{j9@7"
    "K*wn_(37L-_{aY-0m44{{{TF!bnDXK{ZHG+cvL*|#E8HxRvsi}BnL#rZ>Mjf|drPyb*KnPLWsszC#022e$&UcmL2"
    "=SXImj$h{Q&b35;$-9v?`8U(S*`0B{6b*!i4w}V_N#El0AO@*EaBqrnONmdQi(bVr7>mHWJDmr$<5RlU4daTN6h("
    "F?{+O7U6#v&Q+^ye82Z}2QMqx4rhzd^UA>i#}xEePh^Rf$gOT1`2Cee&_9xpvQAEMOMRx4fMtN8+`Av{Fc8n!YAX"
    "yQaNq67e*CMav>J|WO}^ZJjY;J4#fM{feXk<UE4eeq28TQ>`Se*J&Xem^<NVUN1xJwMm%$7POL^()~&F3qi#`TQB"
    "Os*RewByHIu9IGQ$(pwinxzL@UP#CG-M-idl4|*uc_WO7>`+Y6Q@=d*%KqpJ|-5syKy8}EGP_l@mngB)jEC38M<F"
    "iD~UNYIZiEmOsb)`CPOQp>!Z22);Oz;&cw5Q=5d3mMDeg{7<l(Ktz7Xe02hU(={AUY7(HG)H8XI>>kfb*Y|cpT}E"
    "GMs3fTuu7SDG<c<i6BWgXRw|_fZLJ}NK+9d8O&^Wt-g6BzfrM%^f4YT=E$vJPliUF*42dn_Hk7;pa(TvPp#h{!36"
    ">Q*F>UEM@RZ8vNiyC)$4G`*asn&cPFn8k4}j$+Uj%E*tHr(m3KGTZ0YBwjv7AL-M0ADx2eL?SxxVz!T`Ke%qYU}`"
    "4-oJbDo8OQbF7tBoV?UswReWmaohK-m(0fk7A6drK|S~WUrBU(9zXy0C<5bxtd@1+N4y_zgjJ;_g8r3^1j`{)Bt}"
    "i5NzgmUF0(LC|C0y?9w>KUR$pC*C!yigg~vlhLf13w^{+wb!0+#S0_On{_tx8?k(O+y&yFdJ}7>$gv*Z>%v)*&dy"
    "qsitynmDC1;QfQ|nvZ6*N4=WBu?0%${r-#N&|ynw2}ebdP$6Ul<CWd%I8El72()3Tt}QBAdw^B>tMI6uPu#F3O7<"
    "qG?AfL6{rPLa9YmpN@KhH}2Q?PHo#_@_v%u>h>D1`^8Za3fcanhAdD)HBV*?$BW_J2FV*f`%9@MEpM}!By+fd*}M"
    ";o|B!f7MhE9=2A4LMmhTvmqr!IK?)UT#<-O!B_`p$Vx$b!v4@F%(glz8DdnOhz>u;iDG(t|5kTNSBKt4lOpI9dB&"
    "pNWd7U>+7)f!X^xCa*6Q!B17J?tNTpdJfnq9&-}Ks_k5Bmp=-IeSZf;0d354*9YOR)!|z)ycunXSQuiojX%zOMPh"
    "*r<&U_S~~iK7Cg{*0^%a=ywG&w;i4Egf*TaMkP-{D$f}Y!E6NcD#&_P(xF)OCSb5(0J@d~xR4@(_saAaA^l`J1`5"
    "KZM6K-8Vn_ir@3q9`KrPGH^pDt%#>0I5nz>EwRB-Mp{jafOMK?j&ZU=io3>^^Y8F*0LHZ|gY&n@~D0X$a3cE-mRj"
    "=Xh!aW+0c1Q41XP;DY`;o&a-BFPg~C5VFT#xXIMKUM~?G70jAs*N{}$h}`35&1CW+eIM5p$5u<Co+#>&3Bi$j4on"
    "A1EX;bJ?&}ukeqJ-lxWaUb;9c589J_<J9}rc+ONH&O(ll~sm+z!KPZ1FGJZ&H_CtBq&VWOWMWq-5QW$3DKdIkNr5"
    "@f@bXAVstb>29QWmAVSeKUV-foIbz@N9Mo?*VrJ$Zed0&_g#6DNuMvdP$`Q%g{m>lK37FrDwKz1s9)nMW8oF2Pbc"
    "jPF9jDV0P*FLtLw)W&_XWynPwGKztvAycy5b%2C*Wo>5x|7aDt{#r6Uj3?QCg(gn#Ms+J_6pmJNpF#3lmxz&em;d"
    "GNXr;h0786ca6hCizt{od(D3vkaamGsp=J3fAMYJH7tHRBf^?CA9L;2$Jd3Awav50wryj{fhLcswQF7=Yp!H{4-W"
    "=)z=8kUM7CoK++_j<QvI7tel%y^BMTG!-vTkSBEZ?*VXR)PZpcxkdajMBP2R!l=vUb}pVYOjNW$V~)@ZTFu~eadQ"
    "*Sl7F+Qfbcj#)M5;6Eu!i4eh>f|8j$v4nBKtBBen}fDO?jAm+lWx-r+2UH<K&so+mTEo~4VcYdv6L=Cp^TCWU!V("
    "<~g55u}9hDdlC!nTpd->50CpI7enk#RL?{8J^COT;tA!H|b(BccyY_8;1~!5HNR@svH0BU@^Z=(IAuKmN{B8aAq*"
    "|U<Bm$C2kP##;JG(WXkz4iib&twhIIEC{du_Pmsmzi&+WDxg`|f<0d_Y;|D<uC_FVW`%n%R(ueX!Y6QXz{LskFKP"
    "a)oAJfHLyo=1iYz?TvU`*nVZ1`%b(NzP{R-Vq&>Fc=|o4kTVj!#TOh2mc0kJsTMV}}qv#JA|nq}dWQuAsLQ4PPsF"
    "fzW8EM*b^3A3L(qUX3AX)J?82@7Z3Rj8n7#=#h)BNDk9%$HoV*A%g@yA654^eGrS{>8sxOoTX%6b}6n`r@5M~RaR"
    "3kkqL~`6?E1OK8WdPM=Upn&1j1ry%aNQXQ$O-tczHN6yFN%!HusM{SkQ+;t!|_8pbuo-*rF22`AYtiQ34uUniNmG"
    "B3rFdxiE$m;^dJhZ9EXF|xwpJe_HTqfQ{pZFmZJf&8ML5OrnzYqXe<RR^AxxzQ;1vt%N~r>Wf`t#(8Xp;5ViStB@"
    "C{JQ9*(g6^F4@B<H4cD3SA#IJ5JUv$=<&r(fa7@d2jFu1BJDGscpr4Wq0V5ms(7Z!Uh8%qgeI#p3-y;S~n5H6Q>_"
    "@2N3~F^kr$e3O1L}?HCoJ_4M{Ql9u18*k`O}mPnaFOXwpe=u74A)qIx++>^*g_D#zRY)Vm!~)Xy%R?l7;MsC>+(a"
    "Xq~2CoA2^(HG+1Vb?Ky^GjDUPfcSQx^|q(=<$sCEwCqE2(V&Sg8+w`1m06iypyh=c`%&jsRG(qg;HPR14VQi+t-F"
    "BEbMXMpx4G5<0H|iY6AO$6BSZ)+eUa?Lb88r#gZv{xMT0#cvK~uMwYp<6B;3MEy@p0LY;Hl6rJv<k%+~><ZGAdfM"
    "hMN1_-!jFUt!gT>D;@6(^EEuN>^eB{rTYa8`D>nc0WJ!f=$zq;T<H}0K|%hiC(8}-^~_VOO0J&EOdDM+q*YMXGgD"
    "KUSvNuCe~h67r?CHDM@ihXvyYFxRQ2Y%c~vO>H;O6p@RtS9=d1E&_f5nzCqIjQUW+u1RRjY*nbYF>Vo)CLh&Ex>q"
    "Y_j^ab(@o%~VsU%`eti$^iy%NUUR6)+$FCTgBF>9#h>;;Ftn=J2Ickefxk&p<}(qfvuhYWq3pm)K6}zpPWpC(fFK"
    "k`I*MO<#5?&|5KKhdtih28!CU61pxo@y(!o8oJlpboO3%iJ>EnAfEiSh!=2->_=461$u!E$oW+a!18-#D|4;`F+{"
    ";*p+=@K=#kiNOtY+(*c7%j+)MPk5vm&c>f%8oCF-m!&;9WJ)?;h)b)IC@yTc{;r(v5))r+DnCSPDv&$N7wPXu;F0"
    "!x}6g)N$AF@k0|yL&|*hUx6ChPNvD*b~zwyL4<Iw;5*i9W}Q+grLJ7p_rO}Unf-~Uxe(haf0D?hSSpkK2X=Eh8@&"
    "R;1vv0iv|`{@co9T4Y8{-(K75nsDR4eGaoPUYp~yNUMdI=zj`4+^rCIq4x>SHI8I?Y^%ds{U<!~%^wvN`h3VxJsO"
    "wlCIwzoGhzz9{<bhL==17zvD(23$=s0vRLQ*t~h2s$qA>E&1${)UoAr3G}QGZvyjU8(n?YZS-ktVnLKCL<Z5e6>J"
    "`S1DlERKW0Vhk7~^UQbCny;XK=`kHg8*|bw+x;tLVCeX?lazBiAq+#bMEW75w}F%fx6}-CAp%3K=sRru>iDfhsge"
    "RY%-&}uyWh0?A~INUl-8vZZ@Zo%qlNFP6rN4OIt62&X+alwhD+}O8s(Rvrokti3%Z_eoMT+=YXyBj!`7s8crB7Pd"
    "~9=GU0$<yt`#E{*NbQ)olF?Wn=#pZ(Ut4;=*`jL84S083H?wHem*(=EwGsoIH2Yz2HpuMXAOZK?|r+tm}BK#FKYE"
    "!q;EMbLp-kh5Iu=+D<OJS^|l2~itVs?7e~A=7vK;{*<qzU6;tYn@1~$<zhQS+hdVV<w}p~G%1Ke6Sl=B_S;$PHU`"
    "mn>4{Vbxi|OG@=crMvE2!p)W-bvTQ8(7MK`_6E`LIoS<Xyg+g$v6}w|pbI4Sz&sA^Oz9UuC@+%07V`8zUkddhp>&"
    "BDIG{l-tMbVUZa{bi?fKDtN`$%L*8(3em0Cr54<c1F>az)J#}nyhBW6({plC6ykq+bA0$K%F$Z2U?~qwjB_Z0*9@"
    "(Y65esc@<|cjsE=M*3We?(8il1=g@<Vt%DB33UE{Fr)i`XUgDizpi`qSAIp72Tg?>$xf<gHp2vdlr(qSB}mU85K&"
    "@WW?E_;L42^r$>;1rc{L)f7NB>1>byPf6R_{H@Wg7np4^)rlY6UDI7X((M7^S2G<6%7shDr{3dL?F<8)W4Fh;z_V"
    "xO><4rp%??v3k+Ic<EZuS1DPTZnzT70FE>;SEnHpdC-BO?WvTz+5+94%YVJAb=eSAn761*vAA(Fb2zP9JXgBBS&D"
    "dO<MlZ+{<~Nk4%>(*m<ROKx4q{kL9OS5rXeWgMSO@@j24-iwE#qa)#D-~XwaS6OUDONKLAv8TjRR{I@HQCp^ccy&"
    "+Ojz75gtA7Zn`EmnS!$M@z1)~8tLNTym6Iph>bTrMnu@-;zK<aIiprK_<)N1bmD2Fwx>t)^)yhyS$*E4TrwVAJo|"
    "*tX!QGT3*d+i)toya-^_;BU}?C#<eu?ezi?E_F?qSBPZ!T}KqGf@Ily~A17)#=8p!2LD2c<#(a*o1BDYj`g_|PlW"
    "dMUm(9Qd&?bO#mW+?nKzCuyF5m?WeO|xb6wRL*V%?v#XHt(H;DJU>{kfiv?A}a8FH`95!fY&Y)MBx0B3MCQu>(c~"
    "%vR$0+y_zB3x~m18F^M!vSmIHaYju80fW-oqR^2*wON4teUI4ejC*zY)JaTyTx{8wZO3TSTP_l<9xR!riA}=e<U8"
    "^sZoVBdEis-YJIQlq(paiG)9;#X6?+d!M;xykWmr<G_fA(1dX(?(Y9>w4@yRK`|{lkP;*-1PpwQD_i^YtXv9j~Oi"
    "(+xo0Bo}$uBP891Z4vf6<78P2vX0~AbrMabM+p`HI5Sjn&hOS0Thr`Gg8grhVEqPr#-4qx?h*q&Yr;seg$n6@?nv"
    "_$<YRxxcs!1;X5mdAJwGP#SVx-6^M(R;xK=Sxpma=<_B%jk@6LkWV>zJb()(hHuv6o(;1Pzc1w=D??apRZZc6@+J"
    "OjId@wF#Or6;Kh2x>4>e({X^pivPeCRu;LDYD2-@xB-ydmYaGpU!ns9l<bpE*&%P<;OXP%i^AT5`kKA@N$uVN)?_"
    "DheTrNEas>P1$0&ERL?tAbgP%=(2@M*yiZ3XV4*U@5mmp+Sdm_Mj#QzK-tv8f4%9Y?r>EiFfsWM5vtxdZL0|~<b="
    "Qp*UomO8is!$}1fTglIb@zn(!<^``j?&-T`&J8PI-vr;}7u!_7h{R$8m&t#PDOf0Ryyc#jpf8>@mjp$C)TFqgVyC"
    "BmMWv5a#xJcFvq>z39W@Y@%<;rTbhAJUHXf2c02uh|(KHa@?e}z`=r}U+!RiSxKn5x=C`~_FJvsN7e+WHJQob5c-"
    "mEocvql809@wCgFH=ERuNAE7WwqC7ns77NDR3sFePfN&Z^umt2!9R}{4u<@$n}hbvfJqvGlW9#zR;dtF8YBh?@<2"
    "f4*gQ&`LKY&D6tMBb7vo5BL-4^|exiqel`lWzEg??zD)G#+t2K$%KUDQay#qh`q>dgA83D$WDZuI{DCj@WzKsyCb"
    "n*hf<Xd>JwCQ7w)t*C&0Wni#QIe228fVNDgb@Lg3pY(kU<5?!v$Hg4*Ga)IVtL+xBRtafWtu~z&0`*p<^!1K2Gqf"
    "<BUJ!eo%N$@qaD!`lFo0Kae3q&e#1XF0_UlvsqNI2jVigW1rlw3lhl$0h}1#`t4er)OJS$1QNi22yAVvV{mR};Hi"
    "ntOfaS5w?8fp9}JmTM{XphbyZ@0CuI1HZ%fxrQ&pJY{;_WjpNAYC(&ffdfvh)d<?HtxNaJoa8!~XP|XiOpYU|21-"
    ";Pp6q|33`ruNgHiD3FGnXw0bl*z^OK{4SO0t-9G|>8IthOICx_s|&4WRa-6&N=n?YF=zHEDTOi{<8eLEV4jL?I%!"
    "!nDT#@%@RG_n|09`*C(S05~T2FIjGP?IWjKt6HwuzZqj&m{A%#k<+-Cjr-gNR8Cxpji+k=?>}@%PYdNjSq#`<r_|"
    "W*YNm06<1*BhrH@WM+;T1b>vyEckaF|m42!mbnM0Q2j10sb$W+72RzHkbPgRGYkQb_)E9f42nkA+PO8h+Wi7LN@k"
    "6nbQ(p1|d-)t(cd!74JH!k<$c9X!HHS862yPn59u6UqCZ8Cat$@H3y3%?+zgOfvj#E%1{QmaWx5t0J^(4zVJ$hTn"
    "hLn?`VVq3o6$<5v7o*zhWfWkc55}kOlS$Nju4mm8PJ_1Sxrs-Ys-REEn-h~N!ujZ_iXS)s_y#rO+H+m)Y%gfLO6Y"
    "PBRtoH|<X2*7sWvtg(_#UO*XP5^={%nHC|^Wy2U02OC5VNHIQIhbfw*{*RL}-yFjS&K(%r2*(OH@ji-dZv9<rqkH"
    "4PngHg_k2WIleG8^`cEp6LazIe|)FoQAn4=o`oi8&rvkYp^+Xz_)t&1ilS79?^)f?ME@Weg_=aAS9`BSp(!@)_7a"
    "h31vYJkg%Z`YOj6$RBo{>cfH)8>PlG9WR~0zfi7j(<cetl%8uJa*zyEy*|Rq;OFrMw067B=6}YNA|CsLsb~w$N@#"
    "I4?0~EzE>!-(m0Q!9WkGK7Ue*nhrp8>l1$H~EO{j=j=kKQu-pC8juq@ktV9KSm{IXF8$In_NhGuS(9+Hec3@Z6-J"
    "3X0=2Of$)%<O;s1(@jPsNY61JdW%Cb@Fx;Bo>skd-`pH^@tjT&kp!^Y#38{=A)!0X5^^$_02I%FN-HXyV-g&YP>y"
    "VXd+a(LHFAmSy47fn^2#q(8VmF-&d?$n!f_5erDE|a;;EuBZ{u5~@2QkKRb`wg^JGqbi}~;JeiYgz(^WWwU&sh9W"
    "-86l0FW_KTbyAY(142SFw!PV#8zmIMW+JgY=Dys!ssTM&$$2r<`zt_&<Z-&W*No!s@^d1JSIw3Ze2v4kx&O`*JhS"
    "voP}cYg(JpEQt(DoWvzw3qZL?})dWFdRC@nAIC%TY6$|ZLOMW3Sc=P(V*JnYeWf91lp$(uI0pu5j^fk1Gi<u6*bC"
    "xnMbMkKp1Ia5l@vfFGP$F`~XMe~63n;nCx*(*eKW*E9>dYV0TFm94f?>Yh_?K#d($Vy~z9!n<^XcLlQ*z4;7!-Ae"
    "T^aHI%ewcA5Rt?#2=z&}#d!N#r)8pQEq3&l$wAh{^=E#F7{F=1FARctP?h;fzONUcTz1$_&U<dgLSC*bw^WH%f(D"
    "YP_rLKjNbXBM)XPSFiQpngS2r@w%E=t<t*gOH&7EH|Z&8hOF(0HK`<$XBccjTT)%#U8OqD%Y;dt1w4ER1)ApyX@g"
    "#FI27#ty4onjy5|5T_f;fu-|7z7XkhDDo-CY)Y@$JnHn1zt6U2-BIdPE+1u=|YaHprNPt9=Pdg$=CyOp6(?lk@K0"
    "3pY%>mxf;pxzr4rMuGiyeH}^Q=GDH7LbpaOq8BdceMHfM)W-6mDUWwc5go=}e7%du65tPSqH_8?R;STagcbUf<Nj"
    "(6r{>!>1t=5CCY-solr3h)Oe9aZIP~(-*W7SDE3@44?7VU9sx%*@oezq9Lm|1WLY}sO>o_xi)7nLun&pO3CWkvT("
    "COmt*OlRUURGQ~()dXj~Ji!^=t;b0vwtgzKs&r_cRA{T`E6X=ak?zmQPgeIPALIV({|W7v=yV>W`7(&6tL!m_C2W"
    "@CQV=+jakP^5>$}X2avQ#ntL)1pEf->>9>?o{#}k-7us!~B;hpUGm7@E1A;@qzbwua_$z>D&Q>zCX!REf9CeIT)<"
    "{$iMHBm&w8;&oTW9)<p#<sFMAzggke^38~w_%RF>$uOxtE|^<c~?Wg2Fzk|a}!4iIt36Q<*a^D@+_+Se}DJt;Or="
    "{J$Erv@afSRz&RF7?Adv4M0=0)lq8=?dfT%zA{K)cYQ@N&3i|_@)^O7u5d@hsZ_W$7c_rx^#43TS!Gn<F#Sx%OU%"
    "<y#+uD}3wW2E_==ANb*SIDh)pIk-YDYsRDVuJtbV}t;1X71~1&bGKdsEVkM9&L~=rZ$?H_<>i28xDN;1aS`0|sO@"
    "AdOEPfVA*tU-7lVjoLI|;TqT%D$^>OkCdv<4~qH@=`;V}Jg>SU+z)#?6+Uq}M5_x)l7Ft9J2S-?IZh!}`A1!!z|z"
    "#CW<y^Hl}d|M9;0LINX~B89Nf4fLIJ#slXqd41Oe$%(jRY((_1%3jkUsh(F+Bo3OkO@xL4H3xK*mLI~UR&L@UAuH"
    "-0c&=+7%)qWokp%Ul#JA-C)-9}_1g6TEk*#70~w^`Cls%HQbyqW9cb!*d_FlpV*HpJscVX|6~4!Wm53C3~(+ye#e"
    "0688_o78De`B;Ln&T^4eRzGd*Pk*jkIvzpXhD{yS^0k-^vz;yPDT*-f)o*n&ma(sNozt4Vq_v-bDWG_2-`w#i+uz"
    "&F8&Be2OuS-US3DsDf`1N}TmTLf{Ql9i*O0>_0Yq!yxH@|Vio?DOAv+sYS-0B=et3(Y%Y=#Xl<D4t-OxKoDMYOw8"
    "4jE-1qsyBGdJ`pp160{eWkV%xB|`^w3Rfc_YcH4Qh;<S+Gr*VZPCY}-)O<&!q3uPTXLp&vDXPcta5iovPn@j_k%V"
    "aZpM<*j&)gva1rz+c?iMJ6lj|9$S19upVU=ogvajDA6&uB~*^)+R+oMWAg^$D8yg!L=`{Ib_HY)X?S}qtBa}q>x7"
    "@@?X1d*F`lFmU88UnWZF{tggJUOcO>IENAos`vl&Kk)=xBn|2lzp~&i^$qqFELHz!>q~-I>y9?B&ZhCk<X^!t4ht"
    "iFq%zg$yGAxJ?EMj8qm82-S2ju`+70aWbrK~Z-S-e%I};cGt$b_Dr!IU`>ubkjtNJfJDx!{j^k;q)ogW~Nhmic)T"
    "`$Xjot|vJVGPzWw5}Cs#G*oRwWMfh;cM#Fx_Z8K<xxxoEJwiBn3qIwr8$MDe&z&ZpaqUD8-@!E4h+J$S*FZkEW>H"
    "Az2)eOUiYK{e$$14PW^XXEm4J<0@6=mM~wtz<lj)Tg;bz_~**DB?s-<pXU-0p8-H&s9N{pVzLnwO8M?e0g#JDs}*"
    "jxcbnTSD)a%Z&rq}2ExdVN2@Q%#{5@C5i39EuR?(8WHH~7CfkR8>OJ_cXPW(*NI>M<1T3myKF2hnDs~VjYobT~}w"
    "sWgzLzl3w{y+oSYl3&$wwBZD7kBGi!?n}AC_ZP-HS^B+&$HBUyu^rX!-N(^yTNjCyRob{Rzb;GVV~8a+S2}VCAit"
    "CC;ZI+IR>wd!#ns|Qb>40tJm8Wj=Wkv^<~?!UO)bAaIof*p6)B&SH1HX>MQPcLDf77$Ay?+jBs0(?@|r9R4xg!4k"
    "ugSDsxV`A<QrrYNmd$oZezWIHe|K)_U7cmy$Bd;Gk7h^2aIq1XoOxqH61pJRhI@s$x^58@7%O5kn%X0rsf)Mvd-E"
    "!gG3d{O%n^GE#|FoNQ4|@+SqUXwSUiI1VS{J6CgTd!~O`KZZT8AH!akYd8ks6m|h*bvVk|2aq5#6{6M<@c`JE6a!"
    "s?kP3KVNnoH#6v_?Pt?0yH`7E%Av-udV3KHLIcq_=0n{Wb{ZuYny4LX2*HD?4Rn9J9v^7|0}148aHKLjCw>SMGA1"
    "7dxb0$smr$O&OKR;JSpsYw~5Q4A~T{wL~?;Q#%<?g5~YS#3pM&Y5L+o%JJ(U`3MAQH#U$m_kw16V;28+i@jjgd>k"
    "QY6;hwZR5rh=Cj&qdp_LQP+L}biTVtRpYnW!o9pSV<sCuoR<V0}W2xnhzl(~IS4V%m{r$}wyN#n4vD+xWdpowdQb"
    "S>yBpIP_ZC02m918$wCcf~GqJ}7!qs6=19@tyg^8}b*=tia27z)PJcYGAWDlj7)Yl19M<SvQFk#qf_kL&q-+I{h&"
    "-Pvuzw3=;pUgo*rfBTfVpkC*zp#$I@zMzuM7iqj|ej5V8@X^gRX?PX=E*(p!e27PA#;mc=gf7Q!O`6}Su4Q)baIv"
    "Ijb2C#45zQki`S=cqUaR($ABE(&#RMZ-u++l%F0ZJfi5eZFnr$BYVxcdx82Aw!S`KRJRl{gxWJ`G*g<whg+G0Hh0"
    "o=W=*Ug|T>ywu8991Eb5Zmcp^}j)7?VN1;EN-GcE*vu}hZKvx<gf>y?Fp{aRg5S~kUu+r&+n|Xtt#tKlz@0MucfM"
    "FMct>7RJlhUpaS6j{UWgI-mE9lsIQ<6aD-qTJlk_fzh{Pa8caIM`+6fV+`)PsstT!l<r%AYtyZgAkwXI)XR_kBe$"
    "9@@58#lB)J1BO1v_-T8aahJ{O6_9s4$mQA?H&i905s$zuQ^Lmx42B7#=vZ=H68Jub8l5)-(NpehhY6s|$OLRg`C*"
    "T_O)9#cj(1zxW=Nzto#o@x5G1X<4sms&-vD<CV9x&!fCu*+DfP>!ix}&%19G`?bP8IZ)}5r+HiSP51So-udiUYi!"
    "wy`IgruZq?9#5~D>jG7uz1-(d#a)ewdPdxh|G;*+C+oMc;2R$l@Me+v%TMGHAW@*xe9$-l&HfGn;oU0==6znynE6"
    "8@Ki0f14$5CEIVuIZ6b=9N%F*a~iDLd?)*K|nj|BlA1%wF(qY?O1)OHKhDRg|C!d`EqBdA*TL4Ug%Oh_pr<Qx8Oi"
    "(!h#HSeU~NZQ!ORe#g7o<t<mcg-Qp&|UZR2vVK@tKVgi>d_LKNB(E1oYGz>Dlf|a<<-AZoLI`nb0hE-d$kJ_A#8k"
    "%TDHnQZBm7AxPN8AEf_!`W#&UV3PCr5uAOFp#r=UU3tij+Gn^Lqumm@te>tV3e`YOQ2WJi!yCsJeI@hj_MH8Of8$"
    "YgjmLxmNO!N1z6kJr`HF8pBn}YzZWnyj_vUT8eH~Wv$j=tLhKOwtCTOee%2hNc{P?JeP0bMxyIM<t)=V=i)3wt_5"
    "e}74d6@khr*6j6=#?fqH3L&sP%_E5x9=o$0geeqys#1fa?b`c26{tp8$@jsnb@t?EJ!fb$T0sK3UW_<ot&L5Z`0h"
    "kED)Xi7sX)K$`Ah>GvulKjf+86g>_uuf5dJZNN4Mjc9s1NB&R&N?_eJbFhNQN7<iV0`RXhAVJHz<h8CU;^4|^!3Q"
    "6JxgyX-mK)bbDbGu6yQpI=N=Z7&4%kej$wG-<4?>j$Y3HH=l$65P*WeZFY9)=CHArFWMBx#B!BjV<$5G{BO8%75Y"
    "x?<A@y~<82uZl7XvI~OPtRfXYM=+uR`^?uOp_UOvnvAqpa)VW=XDP<lJZYY`sMHWu_og@e68}I)b_Pm@}vPuCWGz"
    "QU+3jLntf*cQu&mD3l*${Y!l@Om&AhmUhixPGxde0iBU**EMZ3+5$n*nYvAV<>UuE#Z_Dc_Ps6w`#>o%Q<J3brNi"
    "K*hLl$xp>~>1vRG1CMKQu=JjwI*6`cc~`~kk8Kfq`B0X&}l-++U@+J>SJfNA|#t(0L~DhQZ_#@GD6%r(F!tY6VbK"
    "+g_<XoI}*Hq^zoI(YIW_<Uai6G>_u5hW3IH;9MnO&rM4J*HEE1zZ1(-UW&)LX)^cGb;t7d<F-WEAQGm<!FHc69Y6"
    "eix_sXnvX#_68(1l4ED?X3{I4Pf#ML?a4y)_S-;(&K{H9;XL}cTnB&jv6M3?Hjt)P`M(+nGmDlk}Ff!-WR}7o~Tn"
    "~U&!2>|4T5uQ5RDQgW2dmRuVBqv4Ug_N&bu9s2IxPzDIy<;u=8*3w1!S>{5<z8F5L8G)G4a|rs*7nCYjGjdMfM|="
    "DUB*|%~ZIdGAyOH6X^6hnexTQser_iVc>A$T1NjCh+{R7aTGvE5jY)(!<d|w(2Xesj5Q6h#FtKP+-@0Iw)y33*j3"
    "u22gBdvDUeo~La7c(Rr7hM1?<ZrE4e<ZwhByen2eK<bDuSQd)Sb9eKQwPmHzvzn^=0krT4-;HP@Wk`W7xI0-u${q"
    "#0YGRY$h(*Vj>_XcplcQKPh5UNuU))m69(a1^q)s7yiJ*Fao22}Z<aVj%ulUQm)}luEKWG!H_I?0&p#ks#g!dHMn"
    "xng58Y<ukKLes7AJya26)7e7dIgI5oQDfafM5ge-$I9YE`*Wjy^#-+L3-DRl+j7zEVFw(F!Tdt;eU2h3aC)tQ!JT"
    "LJ9*ZJ5*OJrpIrH<g@E#{UBI9$d)%O1o8C|2KJ`Ldc#@Q63iyHU?*J|Fkg0#s@!f&jOAkp|bEs}7LanYsA6qSd3w"
    "M$06YRpgTrn)NvHhAC&{hNVXmGvxL1<*SDNT0_3o>s*bdhLZt<b$HGB%~UaZx^eS%2JprDorh-0o^svSl+vogk6P"
    "WIC0lJKHP$=pYpPPV<G%nv-Zpt$y>6L5@_r$Eip}LuH*0oTcI<q+g#6b0J4|F#-_;x|){OWxazENgF~w+IG=4e0Y"
    "%8_b05d|92bv1j$C%_Bt_fw8x;7JucC}-IpmgJ|eo{GcG2ys|DAud!jJ5a<o@1=$Vm?JYc3-!OW)Hbmne4h6t<t#"
    "1{h1%yUFM4JyhD#?RelxA1doQ5Kp4%2R7Y2Wfv8r9?4imY)veN6*)c>b#Ss}Nr;_pF6r9}SUiEH)&d4zk6PGQ9Lt"
    "tf4mcZ~~z$iwyg(3u><l{b(st~iE_r;wB_gE%~C~!1hWY;~z;mu{Nxytz4^lf~5qOzuTjdI;mxlT%U|9~kL;GZAf"
    "-{N29_LeNtl)Nd`l&VxAj?Vz3s2^1>#Uz)_qN)HZ7`{QQajv~h=gph<z^<#K)Do2#ogR|Oc28xpJ<kDHg>mT=&32"
    "HNsw(_Nm9HhLR!hfe>s%kCYW-VqjH$kK$}sKyOHW7#LmB#2M7a3RkX-@~bTJ7(gvl6HI?ey4EOERU%M+Y=QWjJh5"
    "S7Kvfk~iX=@^iF-A`~ga=y^t9YJErr={jc^9D6l_Bf}_<yX(7R-VbN(lSx7AbZHUqoc{GdR{QwvKnH-djtc+B&b;"
    "LGWDGKeM7$XrK;!@RZZ#4PN!Qm+LhQO;LE&@ifv;M!zQ;d`7EEBR%wR}kz}5x8*w-r-${P^V!AlIf>@v4qr;-bqg"
    "zg&)eANLr+NDNAHRZ_Q7`+%w_ls}a%~oUWGC|2u!kz<{O&I4&Nc$Odgo*2$y7>u;z93T0?LH;Llm&1G|t2vOl&!V"
    "(kRU48-U}0qDEY{kTl@UrBG=~Ab;1Vsd?Z_U?M=+-7e^#jdDwrSa!vL7Mn-#iU7pR^em!ZiCwe|Ssvi>KuNE$icc"
    "GFMOjKbF&sz}?=^CAzUp#cYpcHvxrViUetoz%*WPOJ9x^%B!gd;gdq{fOPA55s85GlsiEl&6MPK>i;QvYmsCl{=U"
    "e^eB_S5&hpF_B%t0V*}#~It};f$8*wD^^^IR=X6$<)uQ*z|HWyxA*eOnhcnlCVXzh9zHJepS0)t~x4p#`WYA*C(z"
    "pmZ3Jd^o0ywR)F>Q;!|5<{P?Vhj<U>qCb)cle3elEU8f;0kb)vcuNTqk>sN1%+-(zetyk+bB=<K|Sg;`klW8$mp%"
    "Rgj$)g_@D#+x0h*iBW8Y~5JQ%=|ERbr`3oPcsjU*h6~6?9dCY+c@}R1?;xBFXlbxe?M^T25VJb?Lf6+utw;Wom>8"
    "AH40V^a=W@_fyPn%oVGo)>AiVmFwL~Q`q_FL@E2|^ZZ&J<1e)?K6b$)Q&P*TXD|68N9#)(=2iDTyT&x7(B!EadvD"
    "k`j0O#Et#7=WZRcUdi&#9XFw>gli*gffp*|?GD~{%f*b9YASck-_u}0u)NmoKQWX8iTI{EBZw$SpVmF1Sh4P+V^E"
    "Mi~sOQ+=U8;n_FU7+qqhplncf^^lstmY`{PS0*^L^28DaCi-uD;-ls7{ss~-oUAsJ$U8PO1-SgG<vEg-$?!uzyPY"
    "aG3xTL?jsG6EpF5x1%y#Kh=R(zDu|Ovu~41Wsml7;ru(+vbl>g<*C|pYR8i?7pIJlKcmD109cSlH32#uLPxudN4P"
    "k6{J9@C)KDV_$rg{Bss_-6}FutCxJ^lz>Odrek<rZ_;x<UG}p|_7;uVo*9XavKGRT?P=&lx;u1ZOa}LxAp*Z!=yD"
    "Ey3KN>D~M~os@h;RSYo)FV?A1Uma<cA`Xd%{LAs_+3Pn){a43_zyEgh_AIZRW}t+@VvHihq^m=*xLgYm-f?n6`9^"
    "}*R}<9fqE159Abu{dv!>3lfAQ>h*fJY&V8Xk>;ts3qaEO=A`!K%4g5ycX)zINzr9np6pIAnTLlARd8!#-7HnywA!"
    "~gp^oeeSHDD+BfIt-|8ANKq*xB>PQvsT2D5jIt|GUkG)08<cr3{7X5601jE?3p0WZ~riB10(IZ;d+*4B}=Mb>91U"
    "QX&33tiuLXkK-H26XeDqQ3njTDTI(68+kB_^gsk`J-c^d&Jp#iNj?*inLwVA_8BSO3%2N;lJ6CahUM}68zZB&3Wk"
    "t!SAhj>MMGpltEVGLc=w@TC`*p`~I7n>Gb3~sMDu#wfQulkB=sVA?di>9Mp(_w=3mlU1ah?NR*)ZeCE)GqoLbP{c"
    "Z2=;~<}=bTQrYOHHl6V$I*9?}W%Cot*KHW~`h66Q`h5ZMEQQfybBXx8$&b<8L4SNRcSICVW`E00$0JGbdHGcX_T}"
    "Jo=f>S&Jl)Tq+n~w2g$c^@UA(Wal?0=(0CTbD8^jA*b!fkqAS3*WIDdY{x~9yw!e~{+O>J|kQ{(hOmR0e{vTdqDY"
    "q}<`wT-JGEAum#;95hf&!K;tj@R6nJ0Es;+t8pPW-legfWgnj(VeeM4$pqWW`>}T8U783&ZqK1b^4;si$=nWO<V@"
    "#-j8sXSt1Hn2$vx!*dPq9u~$<Y1*pj87`c+(aM~9F$1-3ERd7+3J)oT#ATP#H4Bjwm-LGvO_>9^$S^SrmlGe!Qro"
    "v2e>9}Ch;>}{%Hu}jCxwT;{A@h~2F@@$lD;BR+-RyG(gY>Bjh9-+e2X8CwrNg$<VZKNnO9m(={r@!61-bdOaW`b7"
    "dz{ZF537O?4u3g5JbrU<c9cId^W)r?ExmIoy{6<!M+yq;AV$|4zN#cK0Hy3xHbhhs+!xCb{33kw)L}qKY}aF*>y9"
    "sd6;owL=hw4zadpj-EeO2IiB{_}WJBu3{d?RxGmc{0UudRPjMF7mLr&tl{>YExQ&Vj325*ngf|K9hz6`Jyut_2%2"
    "f~C@0wF1jrpW}8NhddiCsxdsnHPc3^TqE0f^%OOI3c&Z;OW8JS3e#9-~Q|0;J0<@0CA8MDT=g}dO1BjIXFA~#fE@"
    "*a>ZWXhEvW}OzJ16n^#!w?4*y^(x$sC3!_^v|5c;4HPjp%GR9Y}8rtq`u9QsBW($;Nt(!B3ih10S0)X22inKQ9Oe"
    "JPxOAfa;NO9o>uE0<VHPY<jVuA@ydK#(K$TR8ri)1og%wM2-gFIfI*X!q8`0n{7eB;_=fV(X?97^-<g#6pipUXBq"
    "H&7j=@G16%rQ=|<_lxJ3J$oPPjpvIbdfsKX;sv@D-veHF_E2WQmUU_((ljitK$0I8`2hjKp8Q+)mwG1|%<;w%s--"
    "Isd1?^DSLckTFFEA^KQO3_L-P)ge|z`l$cfu4W+N*0qnvuVYA^YT5;TDz+}T9HTd*N)XM1~d%R>te1sGWwq_8=uO"
    "wdD~RZrqSq_G^~Qt01Ux#*DOyjWT)KrhFEAotY;ApzYg9e(7KE4mY6=rMY5e9IVD^c2o;paUIZXMN<_dyGun&&55"
    "v)!g>;fhvy_ud8@sXRm)d>c4sY+v_v&9LZZ!U7y8NHyx^6Nao?H19&|_S8GaI$r3x#&ElX9Z~tGJsQQT2SFBdl?)"
    "Vk)oo*20mB8p;9Gl{GFkMDQb{?2;Fc>EkvYE~^afOS;b4&Jp1x-A#5vXGKYvzlgZ|TjLT0f?G{&+OPaH-Slbov`t"
    "4qipI&U-@+^W4qh!RgUyLvPwjy(Bl}uzh33;)Ne#r4d=Clvj?~O1F<Dww2ZFznq=D`;AeOSL2>*zkUY9IGO(m+yd"
    "qju?<LTy0;e<c1LL2nqQ;-Ey>WX9XErY7I(--;zL&x_a~MRVEj6~tGP#4f6%OGPUwt0jc2ubQ-A1wQn3#F*%Tp12"
    "fqLdKw5RGo=GyrVwNI4Xaw9IRcG=ylN=zW@8d~UT=4-_=KOZn>v$XcSC4KIfwKks^gr{dD?S@r<9T82lTBm{fK6w"
    "EBmyuMUZ%T(!nI<*lu$FdtCh&%MGfq#dTv`oI#Lril%x{WQ8XSb(S&1_b0wz*H(?Yvt$2_xt>9F(+N)HXW#6g-6?"
    "kHp=^wuS>o%TjF8xq3ZiahEBe;&oQ_?;opT)w00D+JxPQIQPjuI;?UumXfXYwM+hK;Wp1;~_cu)x)V10K&60jUYU"
    "i0%adC`4X>dw^qSXd?q0LL5!u(u!q8fMN$}Tpzk>;fqKsRrIpe!1coC`cqCv7?Nj<hsi!U`u~6jq9To}bgiDUE`-"
    "*ml|sarWIW0w?Q7`OBDZ4ic&{4))sxv6%I*>mqHoZ~jCWWi-p{10aXw!iLWo>UPZRE%fpC4hZ!L#Dw)Ph%;X`w{C"
    "91W(Y%TM6dtSQzf4n~Wvww2*e|~>`l1n<eI3}(I)kmn1q9w7^KTKx`>TJEComo1biV9O=v=d~5g{>y!Uc+ub$cw;"
    "d^aBOx1WyVNc;iF6iF}++J4n8WJ?377Poab~<a@>ppYjN%v8|{LeSzrS+S0xD@z<;l->Dv{r{sFglmliz9s%$mu9"
    "={3^|QLI)}n6+saZja9`4bQdE_=2z$FuH@Uj82=-);iCo+vgK<yok<1((TfIdI07k6$kE5h|fRZ!)+6%q4JffIbe"
    "=(xEnybv#P2PwBBr>6rA8seL^3`5v=MYl(z67E@_0uJQDJFZN*jAY^@+65C$=&o)^q|W-Bvo;30pl;PVd-aoa7pQ"
    "JFxSK$_?=Ib|HLY>E7_kADG#ynL<)%c<bYcKQP+)&TCVm(U;~4DJQ56MSk-nPQb=cY7p<<bQy9Rgj7>HcGc^!Y0?"
    "CPWo;lHc2DS01qLtOyjyhlFJDa!GuAxE>)Es~XkyntaNk*^4ax~IGq-Le$LQi;$|4oq`LQYCw1XCC@Vm%jZq1oKT"
    "xFn@<kAXX-e2r~v(3-}kQ3-B!pyF@wxJ1||6YlD+?kH>>>_`WEQf3M=VM{wqfJN>Le2riYJ>7#x&38&e0O7Sn@lC"
    "IYDa>I1pbM1u3G4FkPxV~GQE$N_1DmzqQQ7R$w$sYhm&DlmVVtlAlfe4Glo?W4K`btN^81{4lGh&uJ_-g@#d49K!"
    "v)2RU(DIM1o38c$F{%(0dxP$|C}uaA<x+hKhpEkw>pQwUu86<PV`Z)nx*)6>MLuoz$l!cFpE;|WNZev-)^#|^uWA"
    ")_Gab@%()I8Ne@5MzUG$nB<$$Yzm#%va3!YHo4BPRT#BjCeD4D=-B=|0Mv65zwE}nhDhW8tvq}zLo>2>HQ-a8_kD"
    "!RGYw?6}o5kJD(g5s+m%``PmnvJ{-e0edEsV?tE8kYBDZsGFo%+}?7qCiJUgO>JZoR<2i=u2YsT!;pB%o-x07*v6"
    "*T?W%oVN&%)1RD5%-AYwPQko`Br78KQqSA^spUMl^C~)rbBAb*6rmDG|eEx;HHB}#JuBFe6CVuU(p5>qG6A+q+N8"
    "HE529gm|ELp8y{<X!yR!(vh#}P6~ebY-zDB}ynWG@D@Bpz+FD=P;v#37GN-~SrNQ=v>0IRbQoII^4YeJn{p(hPOq"
    "S&W2bh`OoCFriu<m7`=sz}q~=v3Qkl5PU{}T{D$TIUK*tqvg2<BpF|I4es(`?w9*n3aJt{$;?BK==9<lIkXvf(Y)"
    "qfJ6p{SMawz5npsOY$HGW*X&9FgQZi3wg;3i4OK#V**MA&1M&f#LF`eqgC{(h9B{4c4gb@W;ue8Qa^%@I137uD|>"
    "I&^`1{hs8(S@sZsjfG7(Dv9`3*M7gGj!1akX8~(v2d6(yMhZQi7#=WaZSU6k1HnIs65Ec)ee$w?vfj7^nR75YaHk"
    "1n#YN~`_dA7Kv(-{g?tJc_mZ6Jc4RV4Tw1sHIM=SBh0dNMqbqF?4d(&U<2P@PUiE)EIQ-SOtfO#7uDBxM!i43}St"
    "V8I)B^xptKMX?7l^(S7tXPNAA$3qh%jn97@25KP`~0r4Xm#0I{b>O1sa~RQu9x9y1+~bpJ8}jN@UMOyq>tWd(Axz"
    "m-SFJX?yJDX!f|i5z#S@xsz$iU|m0TDQgaMLvSgnuUQ)7@1+=0fX&+J&!QKr(T*hr{}j<&ZhW;MYaFkBGYP=m9j?"
    "GG)P3<Lo}L}NQ5jrR6RijHI6a|5&ecM4S@2YK5NodI<S<ru#NwpRYZ-H+L8kCJ8@}ikdP%^gCr?JL6nVMaV9f_8W"
    "+P+GZ0X9l2zgKgdd8?#VIWyyu|mzQ5^=T1oAPnItXrB_ubAlY%+OsyL7V%33#Wa#v=|@sQfX)61aJjIlWk19U(Oh"
    "a)>Uy7=;>Hw4Ib!fr8F9jtCgP<C@6vrXCApETG2mFs2D9hY+jAi0ets^N?=;ATfH*Jy=vzEk~N8@kiBN#CzGi6yi"
    "DMGZn3OoIZCYRDm1-bu<hX&^m{RsD|sj&_02?^eRM}I)hfEN`CZ#k;Y65z4Q<Xtkb)ZN>!OC-SVlbT(87w@V)g-^"
    "X0i)D6NZ|#om=BEBk#8?yTSoySsQ9{L1}!#+3;ErKf17TZrLHiz<8Y76;w>Vkp!z~VknEX|IeqrcXj2D7MzbdP5{"
    "B_>XG^|dVFQM6AHF?Jwg4IszEF>J_LpZ$turUDdDLHrM>xUk?OzIv>Fr|zD^>1I`BpzA-?!|8tEq<oD-kY&D6{!w"
    "2DGvrcdC6KAd+HcVJ~_)vq_@^5NnO-)^aI#W?D|-@dG`mtCG)i)UV#%+)E>d`Dx_|8O#a+cNaC-|Ms9!gf<+tnS*"
    "wY}*Y=*=yW9&fS|WMx*2-Ah4z?i)U$`*-jszEIf;&LgWnxb41IF2^NbA(iBQc`x&b)^!xXKYSWf->TH5lmGC5?GU"
    ">=9N1mjT9L9i|{VOUKKWLYz!dB(#i-8!(&RYOgH_Hh1@;@zrHqWMz!KGf3t@?Azz~sp4YZ`P)0*l4sU7+q#0MHje"
    "`k?U9?ERu4@j4A%vWPggJoG6K7{vg_RiZ+1K3AwNCL`eh{>%16lqwt^q}>SM|7{o3a@N|d@>lS@)4==<#dlly4ZQ"
    "Y!*J(kYofdy-y)HVo3M3Q7Autx2Rl}ICLbN8Yw#WaJWZliBBis|>%kn3Al2R3*Q!{qX!{#4|FAvX9<zc&H*FIvw%"
    "*<wZB24%CoTVC%LZU*6RqMI|-h&v8LuBO`qT4E^VKaM>JbMZ$6w26zlJL}yg5$?rVw484W8>e=u=bi1cOxc^%oD{"
    "L-)>16U2S8t<<|70V}o|99N<>Fsy?LRM#N4wBX><1prNPKQ-1(&k-@xBpGWM3B%$41J;~!28BY{FWvObN*2T2h*3"
    "8zl^T{IoFIPICPMN){xE=m>{ykc57n1r9e_%kJuk?fRZd@sGbV)@7r7uK$oQ_7iXnFzsg)J2NQse82D&D4$|H2oV"
    "RrGvgRW31-su%O{Q!f5)@S*sWGVwQp53A(hS4qGhFAKkuNK!Nx%L{o|<m6u|FaJYw^NaNuk!Y!%vMNEpjg~4pKp?"
    "4Cyl0T`weTetl_dL(vi!<Ra`&&C<i9G>|4O*2lvCec%BYvg3d<SzVkK~u_J1i8?_dGdlbNivUazzrc*>>0Ca{q}v"
    ">r^eRf%vCeaqDQ)iS|ylf^n&mThLVG_m>WWj>cG4h!avli%OIeT}GPDFrZ%3rY_{q}f%uuz&E9TqWPkW(rOI?2Rm"
    "2VEHZ<FO5F|JoCIc!RiWc7(r{f!Wz&O+Q*^@nMV_8W)(Kr*^*nzO$`34UZ!O*wF#<@kyq^Y-R~9B&#TXz1oS>roQ"
    "N)lHAKYf^1>ji@5qvjJO{s@{c?Qr`s|<m)3f7uWhiF3&2wG2|6+(_<vXD7!QsK%!y`2JuVFv^3Y)S#CA^y|s?u4D"
    "1Zs2p8jrJot#4`19t(-C4@70iv|=yvd`pHwLuD_>v?f+rKllcBD^O&K?DBxTTsD4^=&_=)c*$52wL%IH$>_z36f*"
    "v)V(!I8mTm>xqH<`pvGCBZD3MctA<5e~c}fKVcJkeKDm@+zX*A8uWc*uY9*nC^T!oM=2}3LE#?;^XYyjnY+@;&2K"
    "&4w!_F>6FHIx%WOUKOGF3K`U@W9$X(l+6CzLX%uoJZhVISoAlSDorDLr}`-S1|div9Bb-I=n!&GL6eRMQ)x+(wtd"
    "~^DDE8$&!?+X%&S`yKZQrY*k#V+4g5jG6jtGw#SR^bHUF8WcSPba@W_LKLVFgNfgSt36DPbh=CwjOh~ws$^#M}Zg"
    "jFTR1Fr<RXl$*pz>*lorf!en_m&!bJf63I|Bvw+;S*hm0(G-r<6)Ox;3~JY7uu$_JqtId1007b;AuPUH}}VP_$c^"
    "d&fr29&_W!*7uiRF|+8Zy$+}7*Q(3#Twh}dV-<=(Yp06501kpWtigQpqgiUuSG&=>5k~j+hOAV*H+P}R7cfLKGt-"
    "WFkh22;WzE#?Lk<VEdZAxmk35*qQ%IQwzf-&5*GN1@Y}&fcY6fy>_&73mDp8n>?^F&*&fVC1j!3f9d99fEk4}tc%"
    "jQ0`cXoCt#4>;L=tQe*bIlbQyYcrH9CP5?lE1gAstx*RS-UEE-si6?9LE{3447BKSU|nes{d%c^eqNk1wKaf;@R?"
    "({~yoMud*MlN_u__2s675k)#iRd!$0xLpnW9sYv`S*MopdN=)xg!xSh2ktOOCnQjedIKJg7U%rz7R~wz-yJQZhf`"
    "QagJWMD?Auy5<je!3k{EfJW(hrT2kCdrnJY0;W+F6M+<!Ua0;#;L^GAO4fbeU=MdXyVfTP0OwQwv+HSNAB~_NNfs"
    "?yBNNL5L292pNdJ$0@8aP=1<0cokv@hU|@*Zyh!T@#mUp3#dqW?o-Hx8^5$)Ap=}r6>D<5_>>nYTqUPitJJ3GO9#"
    "qhkGnA(_8!NrvBB#2JO$~a6d`iF{&bcDhaRD$kWi)-zat3&z9K1fnd2f}%4Dcr4bgRZF~#g(;TZF#z^%o_G=S|-="
    "A_#FQuT}M6pH9<<<vzp>)5kctMjmNK8Bv`Q*)KNy=GLt+b>&2aSLm@1bJYWF6}<tBTI@rqBLXk-bE&~#CeN-DHGw"
    "ai<g8rRW);6MZNBeT$-{!jO2)F5MLIk8aCd*B~uzh2y2?co$AKta)YdMJV~-^ad`HqE3Sb@&pXa#2i5H|`9N>6E)"
    "J_<LcIT}J08YaEbv8u)Z=-Ncecp`%<nwi>tsR?s`n~idznvHu-6OmDSnqKd}Zm93bEXkCf_fWD2Wmfrn6ptj#3hv"
    "|7iqL>Dv*C+nv@*<~i~EROgTv;dLNXg&H(;g7Th*SX4UBCOWjhk>+>YWCV(A_5{VTG^Z}BbmhCQ=GT3spwNXAQKV"
    "j%CxKeUt-zR<>d@~Y^fS=mBXSt{Qf@*#l>l+}?F_yDsCI3T*ZTB(nCX2$%~h=q|B(0`{69e}ZSC%Q)WYTF6@guKr"
    "_8WD4Gj&!f(~@neKEg>NAEllhprD34=P^GG^ifx;u%n^1i6%r1If@?enH&)7KqL;w{~{ztVEeISU*zYOpR&%5nat"
    "^;q4%vk+Wfb4Z;z;oun$ezgxni5p2~zW77*`Kl-M-n<FEdI}_fri+7-opy+*;6ID+Pu<fr&rtLu98n6m)3OXQUey"
    "Xlyu5l=KMoy48nm;O$I4p|6M3tzNQAK@LG1fpnaj2rBQ7iFp@FN@Qio5Mg1XE4zp+VE+H2ip|TG#a$cnE-eLk__x"
    "qmhd|Iw40HMtXcrQ2Qa{%;FE}d!^ud$S@-BFh}DEKLiWvPOgKMR9OKKa`XPr1rJT<nQ|2GI7-ZMXv~5?!DoGx5yx"
    "C+*gFt5qQn8B;A7LFMYySq1giz*G86$?y3w`LL$0}Gx@p@V{4LPF?9=PJEI~2asuYFXDHiQTu>qOtbVRBKfxIYJN"
    "jAMRF&E_bXmwUeRn7D=yTP9Fq?>)<qUWvqMp-9D!6vtxc4v)#OcGJ(XlXz9-i;9KoWxn_P6#=r5p#OiC7j2jib&0"
    "-HIwAtdmTO%St|6XOHw&q7CZ7Uh9y+W{-c*8*NVo{_V;Z|D`r^OTdn(hUnIK`afD-)<zv!P#*+B;RnWa?Ml4hT{o"
    "u@77tewpf}O4PaL3nlR@d4g;;e()m{N6?IIcf{y5>=3WxKSJP~@M8>bwLmn092!i*e+2ih`T40MUL(*j2%{ubCn#"
    "ezy=9Z;Gxa8b&)itycAIC}$QwbGOc|HI*Dq2~|&qNcAd7ZW1;8_H#{i%REUmiUJPw=*?@aBlgOBC?@j3vPo}$etq"
    "(rr>(Q2tPbzFIz@!lCXf<f6g-=47d(){N;)YHrd#OihgQ9VRrd2-QbgyJ7e!-vA{CMKqe&b`beZaAif9_#C<o=s9"
    "KfM<K%`oz{Pz`x1j4gb_HNJr-Q_CiPgFXz6abMCp}fsbBPip0IPa;-*-*@aDCqC_^h@DBG|+2>emqUl!#-WW!@Qc"
    "cPmHo1El4ulQ0UY3v`~7vusRT~!ewTCfAoq_w1=qD#M05%l_?%%MqOE-R@7gtlyGLbdI)>AbWEP~m|~GpyHvCF(W"
    "tDR#W&=vHYxSp!}rUipbDX+<Bi_-(>1iPvGA+7J1@B!qm@)0dmmUerqmUmD6YIu_OO0V@Y*HKDT&w^KKOjv$nAtc"
    "^eqnFPuvxO=;w&E8*F(Sk6>ZfefAve=GqTmQy1*3Sprt(^%oQ2r%HC$pi+MS#AK9fS*pX2az%o#oaoZH`rJ$k<Z)"
    "K4nW2N<Btf&71Ua=s&H4HT*=JjO&1qSnm4CR43gfNDk*?v0GZRejnpAj@QfAC&)iWZ*>2#w7=ruD8;4aQN=a5P<p"
    "lkDVJ>@J70~@VbH?DhZZTvl&0pUw+4Ky&cY}<=$_JI?>B0Y#HsdwR(1d(Ph0DxXJL45x3VvtN;Xe;3cS<m!7+Z8d"
    ";e+v4Z&3ox&Ln`BN+H#s5my(gp6{qubdKwRV`>ni%UA*E{Sq79=>1CcOJU42R(op%jM^eZ``KBAV&Ml2`3?v&vLQ"
    "VA*`4HH&T@at$Zy9oQ#+c++{SCPYD7!IIf2G6#sLL^-oj=TQllNlQOMlw}T2<|w9XMh31M`J_B)cs7b~qboBHl~w"
    "SDn|bDlGI5F$ckJA_|7p8Om|7{+6t=fW5uz^P+D53XN&~j5u8T#d^9mk-RzX#j}^f%#e19v&FEdnL_d<Jr-#}$vJ"
    "Q}jVQo6q`Ac{)Agf4x@LbGw8mpJvVj<N3AD#dsI}vFf7$6`$!e7E$2doHYymbCuHqe|LsL<wZxO-(?kfB#_$tmmw"
    "~VHdbj$IL0R#m?L7A(}e`AI&b=x^1mg~u~Zj!Fod4>u0=3c2Ef!WDbUsX$!b8?=q)ES99h+P36kq%g$)G(Yhf?E{"
    "HSL;VXpeiORzYj^`T-9So)m7m#;^kg8U)M(n{|mGLE}wk<2Fw0tkK+I|$D^NxH+|$vCs>n@KNF2FfLBZ!Bk~7&cl"
    "_q{;XhY%@M(88(Z8n=7-3nxO-#pgF**F@^_y4yqql$L%IWv<T{mE@#YCr>i0>NwtSZ?ACYXrtJ~PDx3F7}e`{np8"
    "_W)e@ea;r6*%KJ|c`{}uvY~55@8va2?L0MhMgnOr+$04udt&)jA9nFh8nNcD$XOBy&{9>{%KQo>^C2R0!vaH{P^D"
    "z2Z0*k>au|fe9#%M)rXgCI8d_@p1<Vu-#RL`xq>wvn%(v;=`1VAVxBHA?3#%vDgZymCAqSI&-Yl}Y_VyHXGXx;d+"
    ">PLZ3O}W)I+xh~D1#F0=7v;Uu&WF8`&iDq-=`Uy6QuNZiz#MsrsA2G&Z~@F%ka6eepElIIz{tqOxufh-YNOYub4T"
    "wis!%cAeudJbO+`{JacGr$&VV1Qz-TW*G}pb^|`P-x$CNL>&88jW;H7|2C%HsQDgo1u2skElgXR$1Z5C!W)wXWHV"
    "#c((LTKqs}c7>qvF+1vi=~QYx(`YkkcQq<J|u#mTF~90dRO9NYW~PPNW1H(PHzLs^PtxQIh>EbEijh!!*yRQO_EV"
    "k7b^H486>{Q32ZdxYwz6^ixVU?y(TX1f1~{=yRM?i=r~obOJPd-)!P-_<pL8NsL7lD+ZG!pLnV!P}GS2{`S|m$A7"
    "+6x<~S%yIaAHP;2VM>+WG8fnN0{X5UAYWYl{iNZ6p%G|l<VR2LlOmL;hZKOuU<<sh{2Uk$F3C7ylo*w|^?@TK=fy"
    "XC6ERKYOs<gBbx1|90Os;Q8KC4yLi3U#Vdgyw#cA6RV02UARyoXsib@%<#dogl#gDiB*0p@-u{uFVhH@Zum7BOei"
    "P&c?x>JNchrfg3l>6gd2n_(7>Z^P%XQ4V(aHh?lfZrf^Aw+|7101aS)pMxQfxxNm*=k@O`n0~axAljf>&!BcGxTy"
    "EJMTM4E%NNbrC2fT&?Z9i9_88V@Eg0_`t?m&i%w+=0MJQr^r2;Ngr^?APDJZHEKH}TCtOs=LWN7ZuXUq@U_Pc?bL"
    "@uN~+>GQhQ1^cLDDTkbRlDi2mRNYUt6qQwsP-nB7vLQ0kq#+#*^1@BaxZyfX5t+#VaH7RH_L`f#y7Jtm*!NcE?0Z"
    "#aO!G7zC9@0wc@g&_uZaV7xWE{X`<mC^*(z8fTnA-YX{0FZJOSRl07!<%m;Qo`Pi?J1(O;4Q*D->Ph#z<J?;2b>#"
    "?6mOO6;+)So4NHcS<sta{oob`O}-@!(WSzq&e!PI`wSg)P0DSt?q_t(+Rt4C|Wc>YaueqYDUokxuxeUzH2an_a?J"
    "mgc!x%wkyrLjW6&ad7P0n&b7Pd*W!@ReD(hR{t|n=0g;sO0?%TACu+#_z?}nQQ1`e+`YI(n1Yes@iq*65-HHkqna"
    "2UK>e@P`a8=?wlPg%3tP|PJw`Wqy22Th9kwh!cQBXOF_X|$J>_%f0T@HPrMWl5T@<4xc7GH|e$q5Ghtio#H1d$i}"
    "t+J&>9(nN@OuA1M&Z!bqr`}yReV#tyssvFL33@z0QQlbbCp{+`(wiB^6jLhNtptH3->*>gsS162VAQMz<MAsn>+g"
    "^=lIfv?lfgs*mtA%cgBEpv8d>pKTH-<WL#gN3Dh_j>U@l#8nc{+mNCV`SG}ILIdX;MNzSL9D@gQz*mv|=>oqOBu6"
    "`lN0!NW6lsn*HC&xi<&&C6#_>{?dH`1h?itrR1UR6dMeRhDV*t<A}L(S)~7l9{cx?@E<qn%VK4v;U%c8giB$)nEx"
    "5X~->KIH$1)x@9km(r;;5-zD$Wk~SnKS7}-@fr|3^Dj|FQgvyuy(VWI*VtNS-tj=w=060<d{@B36-7h4}H28!l<a"
    "rW3zr2^f+06}pdxL%vsYo}1TjI&$hHSsJrC8?k*vZZpV+U|x+mcXob%za-j%Vo@z8K$ez0Va1>WZ0g&7fJ?$3nnO"
    "QEcWqLEmCZ^Nn0l{IIO%*hQ`%D(gfV=)n0U^C|5}wqK|PnGY^2`VuFTrJa@a^Ic&X(%_b9omsK#KxS?V9fvoADC}"
    "|{+M^G600Cc5j`XtF46}oV`rMZ+vEz?7jqEhY++leZ<Zp#1YWr?Iigr<-om*=BPxJKkKYj&7QTMj@?bBwxG;$Tbu"
    "yadq*n=!RZ<8(hyFP*Q-12ZqIU!ryWv6XAv2akYR{b(cUeSx46?3i%sfg~2@|sNnN}}kg_m!1ke+$kqEfQ;w0@Q|"
    "NoG1y^jJY)oX<HluYd48+b4yIKV3nQS^}}^iTWH`5KZMB`QGTB-C@a{g^hy;sy-xDVWdPFCtNG&KKuESBIFu6)6V"
    "K_rA?;RapcsGTBw$zlE-OgOvsiJ;T+^xN|Gp$$vJ6R|F3FPYYqLU<<nr_jn(T?eSjUCRdov*eIRI7Pw68Xhy3!Ro"
    "DE7)(Q7c#NcV7%T`dos;FZx(?nBdx`JJrECxw%(ypC(<(vpX6#;?kI$u$bOh=~?nzKZxQC8#8nA`2xyRE3EOLQ?F"
    "KVJTOiYa{~Ies#s21B_qPQ{NQ;{e)nb~?-D72BpC|FbT(n7^gMVucUr9_44h<Hcik^btxh_1xONzqT*^k|q{N;Za"
    "hLGqzPh_ioB_*Ir>ipEW^{IGbdU1!&zrW&BW?SH*0TN$IgBn1tnp1>X0M#W8AaH+<~iCBy-sz5k~q%HiQ>%M7|yI"
    "1!I?3LsNLi!(*IZhr@K9hEcS_(WZKU?lC5aFU^$5>*r4ZF2brW{Y8E7O-Z9D641m#WNwA3Sj6+KBZL9FIq^m6zhG"
    "6GCtNK=@S1QXPkquLJ`6|V{2Zq;t1e-Lw!L&N_WEcz<Su(*0B_L`qk%D3M>Pq+dMo^YG|KgeAHqXR~H=r6HxW!GN"
    "O#za!V0uk1ydVsXb<iq$DF=%wvO=60y6MIF{`k0XF`kpd(0SRkY_HBzQCo+d2ij?`wC!RdIcENiUZYE!1M4-WI3e"
    "+N2*gDqj;cOV$Do;Y;Nr8259Fwn-1KZkWq{vL#t4f<wV6RH>Z|v$9Ic|86(CifdLLwN@=Gkpg(WDNZ*N!7g5Px`o"
    "GiG^OyBSq;t*W?)CtP<?`=URl?UwlcoQRpw;Ji^HfUZ!bN+Ge)QDC%p<ec-LJ!LCt;7>gHEWPkb%Lr3s%dp`QS{Q"
    "`*dR4BIezvqX2#phx9P7v!8iZ}!s9gv90Dnl!4RPEjCl4%FH6!Q=a*Em;DN;Fvzi0_#c?w#jNbk*RwQ{UToaAeJf"
    "DV^zv`7`IaVmVw{8s&v3pk5@i5!0QrBx~j4j1Mp{%N6qM8TdQTUGD)S{R#6Nb$9`%yZC`(S?B45O&8ybx=aZZ<&I"
    "<lehH^;+2%xhi&ru9r3vCTCeI@^|#j6216QRn8yMKlrH`?}0y?yiO~C7+t50cMWcss}uA)`IDwvaA^N>(a-x|B>G"
    "ux2ydy;r}4CT^B%gTgz3Fjx{FnFc-k2&kpr4iKYiaL2Y54acX(GbJJRbe_>zh@wVhvM-Qw&Z>sM4|R}*r|vU{fT*"
    "-?^lw1BIf&R)z=xpfn7<Wbt$#)mdxe+-)>77>}k>-Z-816`Hb%bL?5*bdfWzgKT4o9-c?5hlVH&-OdfU^Cj<+=+%"
    "e`-APB{jE424mQImj0e$PXLq+X+TQPMw{#cAoD7akbFCD&fZ?2WvITU#o?r8j&Z@_NdnfS>;H=dlU^PC<Y52vn?N"
    "MiU5bx~An|s?^gJIkm?YDMDoxNyhCm!vE;nvn(d$8AzTH*HgaC>*OJJ{W6?+&A3)Y{pNqsZ!*t6gx%2KFYXQ<8Y_"
    "KfQnjrmpoZDLD(D5)j)BxkwhA%Z{}XvveRM8?mPi_I_??z~#_`hK@0<h!b5mEN}g=(maTCX;yL4_R43Hlcpg>VV8"
    "a@`XLoiX@&YVL@fioiMbj2%Rqb?q#w~Gc!m3ZAvc~*V1sfLY~aH>_B7|R*L2OL+a_LQd}%usa#O&s7mi<{>dJEGW"
    "(rKn<k&If!nVU+AVE?f*W1y#sPTmg?Yc0Ojbzj6UNAgejne@LSwti?X2$0H1OOCC1MoWBknq3h!EdAtx)8|hDo1D"
    "<RP9;*>qN=4YPqIq$S(V%x>Wpu@qv{~+)U8Tq|%?BoBvT=EK9|Gv3A<+i$eH}YlaYZL9%M&e9bT^C;OSTbyfy6l!"
    "Yq-S&a|fXjasm*S$Hof81%Qosrm+d}=5FTug3?Ar?5q5_BwwrZb1!oYO}OV0`-XG#zL7^RmW|Pk%XnN56GGGk&f!"
    "{8L*@WQH(0OD9(wTJL$0XElRAqZy!C(Oz9Rsm<^K`uklVQT1IX{IA(>cNfEpE+#C^Sg}ibD4kaDf~8GPvH(9)x=5"
    "GYE3UZDFU^MBJz)yXeQWFlPhyZzr}WGs5J*c=L(tDlDt7JoLuYRAdW3XFHM#e8HiM0ytkBimNV;eC#5xEhP;USX1"
    "F@B4=pG(_h$kEK#UP;Gg4^qOqKmws^Bc6vl8?>c^g71!R+LAa*>l8989aqSfe{Vn*WsKyX}Vi74ktQr3=WsL9E{0"
    "MB3AZibBx>Kg<zQ(3Y)ls5omTzGK$(NFip{kXG5V3&7>4NP4l=3-;3!~E6Zk!aUA5%EH6ou{sEPG$y_cNZb*A~yS"
    "2e}oM<r(Lb#z;d)bR4UD-%64~LYLDWBXt1-bPD*D{;q#v+^C4Bml!o(yrXli_tR4(~{W#)L2#n{|^3GReU~05^Gz"
    "^Tg^fn7J0W44v~Q!V;V7<DH;I#gRQr&M$5p0oDsr@!Rk2=GUni*M_ljaHvT*ewp*n!Rh~b0~C?!7{cC7Qs7$>9eX"
    "_sROCN;Fp}O-R0D7_tSivMy@E?WgSojTgSlD7U~cY>cH^DFaM*6|w|3j%XnS*S*xBFR-3v!stuT&S9ppVbaVv_p!"
    "qKSJYDb%+&gSNB+}hpQAC)qg0MluPO4A>E;&k2gIdPUTndav$KZDuXdSWo27|bUI^NGRyk209Owb;u|gS|A9ekce"
    "kX5Qe$8%Ev@H}M8@n-oq%A>Yhz8j#NR6~)u`xINAnlihG@o<yzQ;FpxEH)nLKxL;q>oFkV0#FE5K8WMxUuO!J1_2"
    "{n~TfxNwQk?>4iauz>na-N525QitMv-oQX_lqbjIG%qe?afuEa%+FSMczQ@xq=&>)UoB4!b&f+j?p457EZoj$a+U"
    ">HmE2+A4s&4oB^oQ0t3M$YIQLkH5UWt7L~kMitE+61E%8k$hpwfGm}g<?vJzC}z-6`aT%V;&=%!j-0i0C8n5}<~O"
    "klucR4*R%&a2D^8~wvbO(@XDRaL2^muI(q(~SZ-jM5cK$<(b?MkzLlqf-N2jz5fE5#zB?fFzB4<ZlA#suw%=XD*6"
    "5OSWS%AqLs0i;Jp$TD)Z&Thw=?ON9IPj2tDL3$0kfW6*^q|X9ibWmoi!u{aR<7F^rId^}D*J9(>@6>goy~?X95(A"
    "kA*opvv%R;q7jK8daJ0X<vopm1H`>|Xk9MNY?tUw5Z|(qx-D<ZtH+Ne5d)uSk&dz9SZxrryI@|5N;nsdBvyGU^PW"
    "$}7gM+uP@R&fj>a~elUO1LkY1*ha%9wCZx9jwz5ZZq0iBW%I)SnpjCr150$*3>6d@3W4P65gZVj6~bQ${M`JaLS*"
    "L{h4%a&78?2}}kb{Gy#}nAhnpFMa2B*wG!Pw^`3gKVx=W&P!wZ<^01w@1m%N!^vH3M(jE#t3htP*K=0L-o)hZ$KF"
    "&IoTem+KS(#{pWb(8=kM9?_&w)Nfai#3^osNV^own`H1T`8|I07Na+hty=jfp;`dm>VEuM8*?jOVia<E`Fae}}rX"
    "^*&AHpN>Kn~LwV1?Klc!>$3*OIY7So$yGqg^T8y*$Oj*Gn@$Vhj`eS&>8bQ0zE=rIhntNYd0hAoF{rqKIa^;6Qwb"
    "G7_nBCV?;yL{Q~I<uq?>cB8kGnTpg`4Pm~u5&}6zKhM4Ag7SVQ*nt72?(iaYD64s>|yk>N9Cn{&yM;ITlIkrx+bW"
    "8!lq<SWnHKc;$-zRsw{he{QXheyTeSk$amC2>vOfuL{d?7+iz0_j?IzY2RiKD{)&J%n3#GXE}r%&wZ_rsoI!qB)+"
    "7_q@pK4$Ed-zuj%aC$wu4lmew0MX|>y&;DLd}(@AKi5tOnEpT@4F3Pc<OBQK(Yv;O3cLU?E_A3Fy*eb@s^sP!194"
    "O>6~gYFGsy8&#m5~PNow_6uDMzj9^Vw7*^5)`tTD-ko>&8Uj|H^tafEC@pL9Ptm$i$u;JzXALR~X30X^r4jrPoFD"
    "lJz}Wz_U>>_1@+xj?vCAxm!0M)5k`xt}8k0yR#q5{xlcOi-U{vP|aM4TP4cUlURRgrX)(2`1qg&6&TtG<D+&P@ps"
    "Q$rod*t5)+lto4$9S7fgNb)jnmv1bw5Oi2F<_iq6k6D%e=+#55=hJyr1N`fj+U?Fe9JFQ+G-)SBkZz^10{DcxsuA"
    "pURP)Bx~Jr%C8^e1q4*g6>|bLbswg{bPj0;A2~h&BM{beA*KMAY(<(Oq_mAb$wjNQEZyFSuY9HOcEVvWggiwu%8i"
    "iW26&hv_sS5d$a-YQX7AlfF3xK}D@_a;0?5a=Qb$tgM+vUHhGKt|9-Y+#^KII%9SlP>o6?f0bYr8xHgTPC4_Pl{s"
    "^_o<tR6-fr*g#=FD4t^LhtchuP$Z0>FDM6GtX+1cA}@9YmdVJB{H4mW{Q2PP_RZS6*TyTBLk?REB}_OO&ON9f0l?"
    "M=8nMA9iqJ2DdF2cDyQA_gVQId(#oQ2kCGn@`O76LbE=oIf$=-yCyZR?>&6`1mTBSfxy8k<@0ErZ>Wp7i*75=@F?"
    "sQm6@XPD{=Ub|5F$ZnaI_Fg;_QcE$toFVz|`?1e7Pfsa&ejti;BQTB<#_QZ%@&C&%qlkkU{uXNes`+J0iB+7a!W1"
    "X1^x>3jhQpz3EfPHv_eylVU#`8kQta%0|j<WNp>)c%m>vlRR8!giXv?2P7G0OJB3jP(=!tm`wnC8M5?aDFQ%Q@(a"
    "i$$l^-UpVLs9zYNTMw{x0)@<^g*m4rY}_@);1iRufxsrW9V<?c7g3z8z&6ubFd17s2C<Oz<AYdOL#%p|1sO&B28v"
    "w?UcUmK`Y(21P|98Lwcwc`WHS$rT5V{jBma3Y>|l{CFx(bU*9e$_m#lkcrLy+8HjDQG;+r$@DSv79gTXlfuzZM(G"
    "*;+kf|WQ{@TA6MiUD6<;i>5fIcY5L_nbm!BSx*ToW+FI*kmKfm}^H48D(K)-|t>xHlAzN;j`QMAXXJSzgy0F_W?R"
    "@65A^H>cQsre!RIE?{CN9*6#LTYrnI<xjow5+8jo^(N4U*-`VU$ox$FAYj-%@8npKJT07yOy}jGn-R{KY3U9RK^t"
    "tK-W$U@fACU+j{1~7%d^&|~{xN9RgCBG|x0(WLSV~y$6~<Y;Uxmv$JMAZy{fT9NV%eWq_HS69ZeCnf8wQaOYGo(7"
    "kdEkhT{n($oo`<329wYiO53z)*6bYFJ4b$1%Fi?=>20sQxnHllS(&%$m!&K)<=LH5W;=H9?sC@+{gQrv5%#y?Qj6"
    "syXmaC>@UL``eN6`2-ZnSGQrk*x)2jNF|FDZ&8(Uw-JTGI=*O_DGuhr(bRn6cZ1jHiWg0v6}XEFIgXpd@60!8I>k"
    "mCiuHH?CEM9$G%>}t+$$ugBKZqO6N5*f7Hit^}Q!u9`z{HfTbGSX|$nHQf)%GZx_2e(?y;8?N48IFP;Auu-)a2Sp"
    "ArD30J6%rW^-;dL)m(0p5*A>dJL*=;GnPiMhyh$*9Sddie$KR133G;Y_WvDJHhzhU@bUFdPH;g{u(ox;r3=Y#9t2"
    "mLa+#JMH5LhBssiQnK#@7nbkQ8!99xg1F;NHWg%xKRsyz~3@$=-Va2QP_FD|q;5(B6-BHuu8a?O|syinha@L8r6X"
    ">THc#yL)?s{Z0r>e%#(2wW6J9u(Q7xZ@1&!sJ+?V+1ZUdRZc5hi-MzMrkFoWN0A_BZM=1IFiL|kk|X~+z}Tc#;oM"
    "TyGSA1c|HPI*vE@%}`4e0IpJdBr3M&s>7#I3rIJOppkEzC}fF!k0P;Qim!fn`U6h6(f+*6t^c)~eogLjxvfjmSh2"
    "C8efB}F+qUH6s(fQ{rE21RgzLTC7Oc;fOfNSaPxK!TO@{H)<8FI`mY2oLX&CVJKWa?H0oSLOO)F{j0(#7_M$d)g1"
    "xf_vsNw-XxsrsCxMTx}|4=|!^SW-*3SkJCjacN)vV-Co0%BTfS7L%(ByMqtQLSxy&XWls4t`KZg{VA^^Gg|we4Q{"
    "@Ol1krcmb`a0du^bx$PeGN`Bz!T?fR_7<2#^Ocu%a+C!e!0ih@K)havdUDoKf^4b33?hWT+`5GfLU(1aUR`XlKFA"
    "V$SX_%vsYt$Y{SSqZ~;P$$h`$kAVC1UJ{oLQVBu=&4zb?{X{Tr%0&6S=$s_<g>na1F6)B7fmfzefKQ+&Oe}ImQVm"
    "kz@0GLO|GqeDl^3;_moB2&m_)NsI`P6No=B1D^)i0CwEM*(iJJJIts14mU3(CP4>z9zBA;0CCszE475}C+<o4y0Z"
    "A5TZNzf@j{|ebpj9j`eRB~QCda)}O)2oYD_=KyuSi|vUqrA1cu1iC5xKnu-1<#KmXV`$*<2x&`r)dK>0I2i}6_kz"
    "yo!=E40+0|)8-Su0Vt2ucBTj33p{li5^%@lbdduFJyJ8l-F>)c3ACg<;=Ayd3T2W80mWm00lZbW7gqCs?q_*LFqk"
    "XAdLk|uQkKUafy%H~>iCYu7XHeIrSK2PQv+Pn=-DwkrmGc*BJPYAH>=cT7_gE+57Z%6q5G_)2A}lM)50(e5LJ5Wi"
    "iT>gyGF^=B1x{QfWTUCIo-Zd#`es`2WpdW*no=&F{r>jXx5t0Jb?QnO0Q#)VhEll?54HT1t*Y}ltJ2ROx~R!wJYF"
    "SUIYgoFE*HzJx-h}~7PH-O?i_MAq6O)kJ(6K3SO7krVdU`&eqa*TfX-f+P6xvVf{<jl*RM$6#Ed_SX+X(!Fw;(QM"
    "fRM$q87{<`J)jKB_-7AoR-0FoC#Ap;i7*bNt0AfW-*SKxtgQjX27+9RaFP<d_g{FB<JRYP7GkUK^w1MA+-T=I|nL"
    "sCy~&@xsi7#)ES6C#>E_Afu!Rx_M=KtJ{WQGB6Rkm7>tN`;<1Wfq3G_2UB--ZIm$f8(yl(xeBV%DRv7VRE^HglZy"
    "2#J(l*T%=6!n&lJ+1YjY0y%onqQmpoqB<?1QFLt^RWS=G6+KHb<Zs681O*V7iVuX_JZ`5wQE^?Cf2>UiAg7EQqJq"
    "F_vc@<IzviFc@&<pEHiCS9`?q@glN|5Xk{ToDRw^Pzk$1R!m)gh%=yS(VvA1GyxKVjH6&NB^3Y{8xp7_#*xe7tSI"
    "ZZTi=r-oYc5&#?y3ojj8Fyp4_JgWfDAm#hO{K@V)h92YV8}pM>ux;rn}WPm{{eXq3ufgwj(Y666cC+%*m7gT3}NO"
    "ejv?fe7lg$95yCS%v)C1|wXKQXxSH4m!EHiK7JTGSxZfpE`F1*Cmkf`@2^MXGeh@+&)Xk!RgUiz{882Mn4*LzV=N"
    "-w@n<ppDjjk9jUXXj&Nl`$cSmDuN}*vY^D~&>40JEJdj0COK+nQJ>8W|Nr5%$eLauup+xhmSrK(>Wk)PaavF|frg"
    "(i?9j9cHk2%@~**q==M^#Th%i%BY>}xvZ6<6uC47w|Qu<6e?T*3I~8Lhp{Q%;O99Pfg$7Bj9nsQ`u{oa7_GT_`;0"
    "LYVW^_KQ>Ab1TJ<bAnO4M`L!%o1=K}FUjA;Hm;0iEuH~fZw`}ZRzrj_jF>?-KfRqG8U?&$a`|w+8C748qh@e0K@|"
    "kZP*ki=;|e_%mL@>H8L%Ek;e=XF!b;>ARTGl)9F8V?!Vx!A;2oq)quP`i?ww3HG)QV~3bP)?h#gjpmoZkA;KEXp^"
    "+vEA*KCEeEraxpYzSdy5Y=!sEPi7bTArX}5@-a?g&gGAxfZE1oWitjDIV!ggwYJGkC;9dS-$VpjM#3k9=(ovmQ{{"
    "{D#>El8gzC#VS6;%iQ1dnt$2HHu)n`G*c*+uwxX~dc7}VcRyf+(Z%12EC*JAoY(_hq`&)ZEdwctvtt$6CN?t3q3T"
    "Yp3{m5l1J`Dx=reyJ`$}MGf<j*m+z566{JjonSGRKq5@$E&io2Xc03Z4F?A^tq*Esq7Q%DUl%LOd+@k(a+(8{521"
    "Rdv>kuhXP=ekm78LUe1oG*&%#%8MsWW~u1#5udX&oU<>V%tTaXVkp!6VWg0xbHL&CuJrN^s-4>#RP$%KYVg$wBej"
    "6QyxYct8}hz>=}90}&#RL4^4X}eV`|4$tudf>L*9Uf&hF)znNeh{D&Agw3EV<7M#F@zn}BglxcT*luUgm_E*9pW)"
    "<eaD(%zP^{qN*Zu^L8zbeH&h4(kpjHxbsH4y0pXSF<THWn?QsnO5=LQ!Oi^I6&D)msJZp2>7Ilg9$KlJOk8D&xjw"
    "V#7`I|5L~6HV*FG!t?Lx^>!dG7D+lu4oYf!0zbWur&mB|E0@SIR0aomMk}2-}08JgnSJWHx{cbe5SW(cTpj`i|IQ"
    "X*}<-f9eN5Gc2cUAmL!!sGoQOuGFJCicG^n^-^<@O3%i?TvtnK8v+tK_e;7A%#T8Qec-+?8FI`sLp(Ki?_k=Sy(;"
    "3&Q0uBn=;)W(8|M+8u1S2Rp;gu(jFQ8SU(Z+nb|!6z=ZFt=;WGyR+5lY!3#l!QR$zFWTPk?CwRo@ZYV?z2W9?cc+"
    "xKuMzVwzeZ^eRW%zDL-nI}!7lavEBVy##<%<A#rMS2KQZ-BO#OeJsjt5DIX+gG@bKXD=oAK&e5nt>9mk^);=xnkc"
    "c?@VhPA^r>S63{ZAg@Nh#*QMA^Z;M-t~G`K#ER-%6u*M+*6aPxkEHKQELtVX;d((idF325PK}Oy{L0eh8l|B)1^+"
    "2O4Qycb>{TYS1C99thsaZs0vIc(2~Abf5g0pyl;&BmS7&Eul(tk)5RBd(5D+OS-P*O%()^P>oUIFH>09);6YO=O+"
    "4#yXbPirikX{6A&4}<1CnU~Ib)d~n!!+$jJOE6#0YrRF|u(6Yl%5rRRULrCghw^R7FkmQeOW=RpeE7NSK?!EDqmi"
    ";@BiQ`O#v+%751Et7?GEDCWDQe66IygarHm19Y2_{#u=lo-T<7NP>Wq7fm|u!JS0T%T^3xoF<US2yF%LP`pq>aIs"
    ";jp%&)s^9*?A_;v*)zRGSzyNI_7g%PF3Hv)Wu@CJ=socSvp4;MPqm#E-j(kPMH3VV<f#6EzQ3AK7EOp>u(hceyLR"
    "3c-SNydTiG7*SS68nC+|IP2PN(3dt&L-g$pcgZABY8)+AqA9GT6{9XmXwIteJVBl#Ogn>`cJI>e@lr78&hK6up+X"
    "3s-j$Og#~6oRaCkTvw_<tH708K`H$q1ql!$_jaJ2C6VHD%ICr|KKR#cl;>7QO8UOLqJ71*<qo#yJ;aGJgFc?GbHM"
    "%0Omj%}ss^`u*zPc0EE>u;Xc%-TmuGON@q3zxDpkf*M7CdBjU(n?VIPNH+%*JY6QPK5@YmxO4epu2w&)A5M5Y4_B"
    "MXsQI`K2A1d~^58EY|#sVH`QYnE@LQ_joFqS3y2nGA#vCZgk1&({-dPH5<DK2S*&B7gesppfHFiM%imXZu98Vmtw"
    "tiU0~T`N$^IDAdgHA{pRR-;w-L6F~6Rri>qtNlZ#$Ud6|xbcdw25&w6i>*=F%aT{4g`-ojRLrGtP?Vp+f$z9N|V|"
    "IgmLaJO+}X@Y+xTUC1m$VrHlWZ8xauAwQ)=2#**B<-47iUTEq1eqng8VN{ZY53pwKI4^<36PR^d-k}xk4ylOkr6i"
    "{Zrta05ugj$crmaH35X&Gj`4y`l-O)!RX8m9v91Di<_t_w!=^MAI)gHpF6WV0GGDDwe<f9qy)~el8UVBFoKfFc2B"
    "U)Y*`mNhoV+R+)r>q?PbpiGQX?3FSP4fiGD%(uLcLxPN(1qQ+=N)o2u&JWOV|JB>Q&F~=~Wpb1oc|4N<RAG2jGw2"
    "fB)^}aPTa9{%m{j^t-{cXP4hU89g2Suzi^!>cS7h@4tVZ4W9n+^x4zFcTb)@8Eg-~&z^&xQ>$JPC`b>at0`U)H=U"
    "~$`5C`SerP9;A16E8b*hy-7(7>f#G)hoscZeIYyGKf{i$pH7k8^zhOCG1fO#CQ#*Dlw5~@QVkpj}7c5gFt?CtC{x"
    "HpiWI)OFk?spdwfn#V;Y3TnX2w-SI?yEsLe5CrTS3=H20d4<b5;e`Pk^${bg%TSBR4#XX_YDWqH}5pP`P+NExTba"
    "nR%*b^T;4z&Kg0>Ev@>iSHA+EUFXZkfgS;4HhefFwv_p_s^JyaYFp8jDjRJAV3d||Sko0^iZU74Pm*H@*%fIu*Ov"
    "ECZBgYJ(w06ag04vIwc|m{1{Xh?C5I-DlbmvIjwiG4RK|UVagabFPfJC-ZFm83=Tyzd-Ly`12VzI}sizM1fF&Q>_"
    "vb{|&f`KYcRN|M@N<R|?I?1WJDi9)klCfxvDGurtSfJ3UV7ve43aR%g5seI@801aT=?~jK?0mod?7OFf(bFe8+0G"
    "A_+oNaOmzUo?{Wb#ubmxcR<zTd(W#4_5J$*I=S#{@0KKgz*91XsM2HQKe(&?sYW$7G&n1{U`dzI}p6{ovvzuQgyD"
    "s%h(`#&YzKPB8hCEPzH-2cMf1mWg-`p`_anx|+jLVl1Y*LgN3W)mbOGgsc?_M4?8t7*2!4`ENXY&CxuoGBRvwUmc"
    "RU}h%FLnA{kN6f4=Vy0H8mJWw~+qlbrI<+)#E2FGPGwf%pVIjasZbr2uuy7^Rf^+`xp#EVJaya~$MYnzqUs8QJ^5"
    "&|IyYbbV6=-&9Meqv~U3e6R79h0|!w~}|hIuS3I13pb71?!ueN!^N4F%3zV~GA`o}---ZLh^Rta27L9kVd!4g*gT"
    "`MB6lNjNR$&gDRNvSAuojwsei4g?{W37Cw=*_9^pmQrzoPD3X-!n|{ZwO!;NW`ZoF067#-Lr6zNYcicD!v(xsHz7"
    "QWg*iroi)6r!nj&tb5(Ul)0dzR%g-<0S{9Y@~k+zut46)4M1=VP#FNo;yBP{@ntt%=ID9}rk8-RGa<my9m5SOwjI"
    "}EFHAN(JXr(v8ENgHO|9}%ZzHmq;Z*eNa8S*xK+54c`b<b<>s5Ws~`B4@mu&6e~X=2WCo8~i=x`~ta`Tx$d+^9_a"
    "Y=l8S}3oz64O0s;KB=YANZ}U9+Hv8_$U}v<QU+#Q=dHLjro#ALWeE#k8XWu=8V>fv6!_%jO=ilypxAXkjW&Yi0`w"
    "9Fn-@eSgt(C|r^bQxim$V2VJGJ5W2m!cUf#A@h%dD)8$9FoZEfM|0pB1A2l&1fbrvH?t|Et@JL$+-2eS+^zg^Ttd"
    "fwiN>CCGZ=7dPnbJp18?AR0d$oE*Ej6wQd0%+uqM@NQ`fj(m?l!zN}-HoRKoBuvZuMK-4ms51eH?Mlk~8ecPY%d1"
    "-^+7E>au}j*Fh~rgone2OG1v8rF5UJnvhl%<TdPQaKIPJb51V8@bikeaT-c94|O=%1+kX)`79PZAS9;ULQ4gBhw%"
    ";Iy6h%n*srZNVVpTN<~=Q&C0bVN}E=l!ygkdHDX2Cc6M%Tae4g@QW}pcoV)ayeVWWwaa@k%#gBa4~=e9PCj#=oH%"
    "`!(;n9%c&?4W9lm5f>Lp5miaBB3~EyX1q!2@I`Ewip3X44(V-!6Wat8kHSA*%4PbY`Cnob3$TD(N{4oXx<>wJ_-j"
    "YL6E^pv+w?LsEf#zFWh>oJa8?=ejkJ%il0qWRr<zcUKq`smM@i`W_(L$@9NWjP(-3Y3!75q5xyv3l94CX13jqs0b"
    "N^eDp@ZwL9H02<lW{YA*oJ3(tv4w^N4Wx{DWl{bs8~X7-7jLKVU$`klZ1#M(0z%Cek3m4jweOF!%d-Dr2UG?a`Gd"
    "Fge|xj}4R4ht)>nj8(dIydA*ntDP|F*M?>tt2DEou<8A8rFAWj|NdsEurm^eO$y#|^ooKCn$GuHh~+!N-93|54kr"
    "4lLSbuTW9C6S6L3O!=56G1g`9tjYe7fSa=iHY1W{_&6tCc-)`nP&K&qf<A18wbeKhk?pPjoL1#2j(+gf9AuiG=G-"
    "YD>$9@BS_<!^U<ul{p!a4t+IB5N&+t>xk?1;ejnTS`|S>}n%Nj+r*;R~&UE>9y0?3_`}*i-XzHM;x9I7|XuaWO+_"
    "Keu$}L!Yn%D#EFB@{iZo7TXH-xl8TiO)=kJb*0%40vTv&*i?k06%!`*4Dn{eBB^lwgz0utL4)x9zTjvW_rI2pJ5c"
    "r1ZhX>?18P{+Ch+c@7|bQ<W8~BNLXq_84|JTP+4g>A;t_Van!R=c-a)1kV{*=$_k;OA7k{j7auGhjr?f*F^_d1Dp"
    ">a9+rz?0T#r!|LA$cP;LC8>tn(%GT5@o?|9Wv2yKg^`RLTi+?XvnDM|BUin5PAPj<ynsPiXkdCf)&-bnPj!S!dMc"
    "z65B@OEne%wenAI}+SzeBSFs@iWb7rE_%oevs&Mj|^n+2;xl1)=*nyVtr$fz7|v5K<gS(1gPI(J6I+@>>VV-62zW"
    "h^!Pt-1L{oEYCB7}#%91xK%3~vd-)UA7r<-3giX@#&u<36rhy}nTKV7+Fms?_JLs#jA4mbzZ7<_ke_mKd@}Ghus#"
    "~y6zmo+j_Mn>;;Ju6OI2V;|1-JfUi1CEy#DJ97@QLW7Fu#~{SMEZgBwgwtd7|!#hPRH3B{g!7HyM`UsY&*!wVisj"
    "VA1>{*-D-|4~T9^AgOO3bp8d~u?6SmbCG<L?A(GpO|LjFE^a{$!F^xYY992ml<K!4o({DHWt7K&QI?=kI1{FP&pu"
    "DPGF?goshzED$M(8ZBw3#=sV(oqpST$3E<7ekQXj*K@?*s0c2H2Ep(w{w#<-r-{e;tl$>(zm{O*DF!+ApynhEei$"
    "dn0$BM31Oe#eAxHXIFG{536)7HP8MlbZz?y~(VL$E7;U3#ZGKD{ht&pn_=q%iIfR!3{vrHyMa-LtFd~M4&4cK|}z"
    "#>#~COb3F^!GrDkOz$(d{4Z>~986EDtMH^QT+c-aT;QInJS6y6LcYY_Y`}*ZQ23@JZ=$|PD@l3Jg!%dVnt7!q+l9"
    "qN*%G`Nzd34^&RhM`*mGzyYPb5tqr6&1T4TFw#??bVeG0;!X)syjm@4Lwd{~Mgn`!xB0$9-0I<mnC!cH(gySr#nl"
    "4V^wX-aY$;dHCr+PtW$>oE#mU@#nKQ$1e{~`0v-dhd;|-d;Q(luMw=w^mrQ`Gwg@sf1dqvbjWYGz`e5M?QMBD-Mh"
    "}<{*oLbpevO<%t;4C=?0j{@*}#>2uivv{%yn)-O#Mt;>@Y=0+!tD_}4#OS3?82i6iJN(5f0<XEm4aVnSChchhzY9"
    "8D&uhQMtBBDUDY5U34_we=Gc%}kqaoc4uRhltd(o+Gh$jxq5ixZ~w5!Sy=$IP96(1i!&b^qkcQel3^7*$OL6*|#1"
    "X?^he;i^ZBocm^mhtMV{#%l<U~*axMEQtf7vG=;U99?WND@yQo@943U>W|8rxG9)%R(}<8TaDY)0Z=u|ZvqD2^K)"
    "qq^Ji}Kge(7vnPAPNddd9u&lcjbTjOQ<6f)O3i=>4ksU4uwWQGTHKT`gv-c_;am6Mb_2oQC}&3nayoGE+#kA~$PB"
    "_LAWihGdckuWn2Qels?h$`OwdyjvCapc#sw^Q)FleO?SxQ7Jo8`XiudPI>H=uCxQn3~YbF5`G(BL8|!WoE|g~Yrp"
    "w)*u%Tb*PZxn!BEYE@RNNoO;A1Q;VI^yoSq!BLuDA%-Rie?v|=8sUT>aAh-J&#mHpVVS6Z4JOo#a=`a9%Nh&1BRg"
    ">XmEDH<bU&qfRDRX%htXlL-~5g9Do)#e7U!U;U7wnCUlr?(E=o~`%!4Y_97yh(Rfev`__VrqKUd#fE~5G<lsZ$;S"
    "Vc(0;Gj4#;mXW^sRAkI?J&~CT`wHj7?;bp|1W%ETE!2Xd2xXHo>4q333OY^X)QUu%S-;ozm2|vb(+&w)-7sN<60v"
    "=ZQ{?+cmYpgU@eV7#+b^k~7vARdpdk|4u>K?Iy0Wq_ND{|uPUj~MaO0RUKuIZGuz)kwRjjMu+_{tvx1`{{nY^INh"
    "u|-AJBh@_6fL<188c{(<gJOp4s$kV*i%xWn0Y(%>9<wWiV#Tjq#89~JB!xPamM&`%>D@L8C!ty6-u^p0I_sbQ_S2"
    "h#Gb(I%TgOHDeZ&=l13`lSXXil%-K;P)J22_R+sVXd8jTCesOtLRtz79xcK8PUIJ4>lpQBJc&xXBmANjCPJkncY{"
    "2z=h<gptJBf`@!5m>nLsOPgvN&jr?{Z+@CWm7b;sME8P-}cUaJK695zWe&%W&dn{ckh?|lm5y6?+5$;fdBlD-wsZ"
    "2^WYZb#b3+B%*cr$*?nDUh&5it|AwvxklH8%8F^(Xs&qyP7tY*YBqmHqDovnUVM;Y(21QVqo%o17Z|s^4?I(f3(t"
    "2y7E)?}HJm#Ibwam)*#*B_$6e2fH4pT;KI+0KImnXZg&Sap*#O<cu$AiRNj8@Hjk)muX5O@0sbX)okIS^}<#y4{m"
    "kP)-E+D;2nXeN;fB+Lwy7O^Jw(P${4*M$k>(@;YB=-WgHAb3P^9XDOrqFA!TR8Ax(>=2)kLc2v?rUrL7Gj<?~<<w"
    "YAQrr=5jKAiiD_Q_e)CT9tv@{$h$uX(k6kfrMC&Vse=Q1jbAaf<hH$!YaFi69E3~bWU&p#b=LR7xkVh?o&D#!ICh"
    "le^Lg)nNv54H4ji&Eo)-NKwAlXAqf8ladyT8-T`=_GsHZQ|?!L`XBG8!dc!LP4j3-L}X3DtSt!4*ii1i|lF&7oXO"
    "-nd%HJ?Wp9Z4u4C)n5PQ+jDcxe<ns1AzAmORe*|>FGCL@C@}Nv{Wjd%Uv;tx`1e%b~aq(|b@p>bA0tUS5l8~!Z78"
    "(J4^8m<R(4ZGtr07ZncG_I>;Z8$v3Nl~8X3fSkEUKwncj)X{7P=Y!K?#PHJxNw({AmL%t<c=a0Xsp=<D?K`9nS2Y"
    "rV^w?0(EMae5}e1LrtI6auJJyU&}>at`<ntWRg$d1UK>!!T-Z^LVSH-;r!6^r)h-@WYok;e!`7yeCk-Lwm|F(sk&"
    "r>ot)8Q?%o*B1S#@dhLw#UyflP&LRN#4{nxMu-V?Uq7ThKI6huC3@DbMw_ux32r)-5!XnkR~PV<i#3THPh*F_cPL"
    "1Asvtbz#cv4E2tsCbD6XdxBv{9J;Y(E22y0a#}hQU8@%tSU<BRYEu%ERmIg$_zIdEKW&r)`-)u$DOF%c*>7g%ca("
    "QrGRlmPbp_=0#LAK^@m)F)}m&L@-aeaU#);MRwnHq`V5UKz?U%j_1~0knqwAqnJ?K<U&@{lS0SwQ6eF&E;1uVS2|"
    "GrkdiJT9tR~F??nG#=NSxw3&<%#R%J=z=P<2UnP;MY5b>~RXj&N0nI%wM=(8~`w1&JeA3JT8XV<@hsqEjzO#wm3P"
    "YtQ;C8WXL=XvImiiuJ|RrwEChuqW}qvyp_fm-tT5G%4~QE$*h6@1~sL+gi>LJEaa+_?>ZDkYk2U)Q~UyEL%jaX9J"
    ")MmC~ALmjxo+Y$jBgg8;(f<Pr`!GXp-@`54qU<Wf8uk`RDP7fWZ=N;zfB0=A4)sglZ2k{5t+?{j$tMm)X26oS<dw"
    "~dWKbk4iUkQfg2FsXn6eBLZc16w#n05i*{SoGoGmMTb*xeIrX4@_wU<_k(9h5^EQY+7%5>Ewr#KxV1GP7V+XThyi"
    "rg_%)jXd=bl+&N3Ak}ftUL_~Q5)k{&%41&Q6)2)fyGTuR87B94*Ubv)sPnZ>zb%Y0oaL5v}XYs`#t_tk`zn>l*x{"
    "JF>mJA`y4Y#l>0o}{DNIck*fpU!(<af&gW`4u<K9&rxJES?e-|=ph0u<1XVDFM226iwDHy8<G<T;9e92aFhH`%A7"
    "KugQ-l*D`oo=M7I3}>1(^g$AuLg1-QSjh8Q7Sf;3#^XK~)*IHb5iP93_m>YjR+TSC*#sj3St7zJD|$VVsGqTUUn}"
    "r5ag_r+@eDeEUS}L3MwyQjTK0@1ugyKfY(O^IYEEBWQavS8GclE|Mw`s65weukB%7L|Kypn>IH=jLJe|h_3=g&91"
    "-#8@No>jzc{dx<4XLND#Be!s@ef5*gA)b>X<#_NmAM5xqlp`sV|;-`CraM}DyIABa<+V-vI=cZlLV)D4pPR2lmSX"
    "6TZV#ez-9JuKT=gn7T$7QV4`GcR#v4PnBR<m<fJMOU)ObP=D?AfxWJ__Y)<D(?K(U1{3QZoXM?3tgO;F7r&O@VW5"
    "Rt&1WB&*@f-y^5(@EhEcuPs#T6o8z@<Gxfyb7_`FK4W=P}|$A+fUoIpQTjBGqk3G^Vh-W%3@O*+}sfG9l*p@*24d"
    "Yu&mQ7UA3SDH89T4TWx7zNf3zNgZY*ApWxBn3ceh%nOpE&JaQeBtHW4=5S)#V^b7cFf#=9IP#svro866T3!Qvzpr"
    "C5r|36cjhng4n}Q7XA7L6OSb_G(rA$GxD+l>{WrcPvzlZ{UV|=8T&R2Lnz|=`_Tgj;bIpT6G$)WHw2aXS6Q~hgpS"
    "t@H$2E<uDB83d`CS0HuL74(7T77($<ihFfqnu09MB%=f&vS{7g<+Lzc!W-f)$9U|EmxR@x>Oe7Y9+j1hFU@u1`fj"
    "~7b;Kk0~Kw`28$WN!_T=jIZDcAA%v!erwvZ!jCUIiU)W8FeJ?l%)hwa1L}2ce=~KfT_6%%%4k{l#PUq54DAiOidh"
    "=``=#5&HO34l*Q{v9YtI`{$2}LJ0!r2;4=N`$=90(Va^BG4AevV@0&G@o}W<Q>xedMxagB0^(CNYI>&^q7}mJZB~"
    "Y;V#$D9V)L@Oio@Ar30?p>y^rzS~^E2q+0Uc~0@*t0@KdUlVW-+H-VGmSj+&X}M_yyg-|9fbVNLfMzz<FoAt~spf"
    "GsFoM$wmRXYww*d`zkfZzttZ}lEKvuzB+DvmXN!^&7JvGZpJ}`$naIyq4cS=!1tC%BFi?so>{WN9#g|$8aCGrJF>"
    "93Zc(Bvdc^KUGZkb=+FP|n48p+)4lXtX3fi=ZmDr=i|EI^iaUam=ZRl66A#Gx?^t{U;u`FsWQlJ%<uaW`<hkmePr"
    "k3NKrmjil6sqcQM&IY-MTvo}i~3~lO)9%QMK!V(;QTf*%)MPtDjy^J?D83`eX=Ya>pWiG#>6%%)ptq#sg9?_I^Pn"
    "N`ylRHU(FGwd{Em^0Os6osYSQN8`uKhrD5*z_D9WyZcWS&*iFm7zr%y=mqZK5c#&^Fzo124eNY?lFAh}jzrWh0Q#"
    "QtKm?G)^|9W0_4(+VuF5O{U_AGpcJS*-AE~riSb6PVIRrN^>h33Y)BwARlPcto-eI216NBeL0kMeC8jl{^Es;Y`I"
    "aNDOZaR1(kczXf)KV39@30tkX=YiO-x^Kn1HeixSR5W+5bB6r&r3q2N4hjwg|XGP_49VwG%^yu&;RpK_EL7YmLzs"
    "W6VZE{vbWY*l4M#$jgTz$y;mG;vuSWNj38<srm1`BI2SUkYCcA)HCDr`#I3r6n+wMR%%2&v|I?;ee{mhAwAssv0H"
    "LD4SR<PH0YXUq&%>Ytf5I1WxGy^9!eBKvhHv8djsFguQBvO|}rZPM1pDYFKg}q#znn%63-Y8ON(<7Dfz3z$`f{+G"
    "w*xHBLoDrjjiN*IT+mwA(UF#b{z7%=;u_n_ipYO$IANDyrrIju-=+EvQ5T&CoVuQD_mKD{n$!H$aLU-C-+&4}{lK"
    "#U}PLvHZu9A*N4E=F4VLh7E3`+%Z+cDJI@DwZ}xPdFC=8;ha<}Rw~_C>J3@WPA$t}A3&{tmkZO+#u_6~$7xO+Uzo"
    "v)F$n(absJMZTGHUDH<UPMh^wN<&zL=w{o7KlsT@g09d~VZWQAimQoQ^LiCid2lNW&a)l}Hiznvnx*bkoIWRZPze"
    "YC7*p(KP7SB?=rO>)ZtOFrx~$>A+SmDh=xIL=gjp)wsE1ddYFTDsLq>Olhki?P))A_I-C6$e0tk-VQGPLNn<r?#3"
    "d%>6@FjOp0WRbG^=xTE5-5VljR-_RLmuTyzl%#E?S>f25ot1d4T1#iQKz%4&TLxH;2fzJnc@$Q-^tLOZ<Aua?gNP"
    "@xR>ogu(YI0%`9k~3jczm2fzu7P)C-JoK>;CF5{-EUyvtzHMEyP2r<p=JX54B#iO(ey%2?hMIIi)Z30~v488I4ax"
    ">Y&NlccRDcQ?fS*2Qh#GHR_Ln{%_zDXOx1I4ba<5aXBLVvt!OJNVj<$h?}FmQ2Z0xd8U+{`26?abC)b^2E%JoOkJ"
    "^-UY%NSSyF7+4?4^-8VEU!_0PC*B_er$nBhKh@NFCT!YkDJjAXFKb>EoPY263DK*LIwd_{@yWa`+Nz+I;_9~@^=z"
    "HEU(l9T12c^TJ@%;Ny{6k9e>@-i5mie>=9FsNojpjTO-D6p`v{J22Ffh}{v;6)w>S`=5;wMxz_*c|BSl+X#3Ow8O"
    "3-{RlIpDN{-UpO`UFcL>G91O`Vp<bx30uMNO?K_5?K*PE5-%8=X4cJT&)xkQs85?dRuA(9Ph&@ec%ncOQ-30xKFv"
    "l$_R48{MZ8ic9XFo$GyZ~<l()+wA&|w<^t?IQNvXgNP+?0?56wIJp6<m?T2Y@r_^J9>dR%i`|{+HJvrB7L;y>pSk"
    "KpFB%5!G>>Jr-BdssN-wL%<|dokE=mt15~>ASf1I*%uQ=vtDX(99{&8$Id$#$b`d$*!$Ksx1Ox`OhQ4ZMiNIM43<"
    "uf70r9H`jLcV6d!^THAVzARaa<REN2sBZev>&M;PTYCy^*x0>$G@`^tXc8Y?l;j{FHEVL2v{l&tHQIYTlyu4RpKA"
    "lEY7P1=Lj%L%;7&h#`cYK<hLd({-+L2jZoeKsxW<JxHF?7{T;B%DIH-RU~Ft0N+b(tLVloyvT~DMX8leZKHk&>%e"
    "uMO=f7c3XbT=A<w>Uh8$SWvaf&e&VV;5_dwLrP(x3qx;LmFtB!Hb@t4uYvycoCzO#&>5$L_r*@`0Yfu4$bXkloxX"
    "P<m8f1+`>j2Sd4iMVW)N%<&c!~BZloZAsX?3-bC>pJ7be{@3KGWg43pKpvC($UZfuVuV?2id*ND$$tbklzX*_sLh"
    "tYs`=O(!4L-<FW&%cT;vB*62a@8FOX>%UA^^N%2Pr@r4AHvE_^MArC8QQoAK$_8P(E?ut_jx-a!T|?8Glpl&Rb0W"
    "%k5+r}P!J|S}G1#I(fDA02O*fRe1b>a%Sa4AqULyy21-IxTK@_2qO}WE-LBlG^!XaKuT&=YR6H(XgMJ}!|hsuWI>"
    "w&bSrn*@2t(ID_vA&$f4BN=8AYuK4qpGb<u=AaiCfJsZm^y)|I$S##&@|~pYu$np&I|@Vx-hI<&%qfYLSaOTI@HS"
    "<cAa@ssV`SJODVex9hgQXFP#YS`;;K?Pq|pc>dyI*Le@EeuFbC96EU%on8jXKW2`Ax6SduKrqigsj-$^Ur4Fx>CB"
    ">j|aGo)_a_tONK{$LmRfJ(<A`{yeCL=7!Fo;npq_U?328;nNV)v#1qZOLQ=Bvvws8deCY-_(2Eo6Sf;SyC7XEB8h"
    "(ZCvh!kc?FXMKcxg4X%WG+=RtOl!Dv$+qYkV_}m_GGmO!YI+qgVV8^fm|u@+aggy0imE&mMzUsXN2G8bv{;!*`ne"
    "(mXF&#0&MPo?{t}Zyu=Pq{#_KrCv8)@P(=tR7E+!ykzAzCWfxuU_L-Sg_AWq}M6`NHvu_hJ+@`dB{5?MJm>Y!L4E"
    "5;7Cb580qdSxu|yRWxW&IH-|IRpTOb8;sW?*|^B(v1Y9qCSL*OwLhyV^kdF+^J|XmN@J8O|kFfIcCUZd`kuOF2sj"
    "JlNhQOa$Ol~uECAF2PU)6l`1fvHSB5Ec*+{p90&W5??@DpuTkza=5F*kZf<8W<5LJgVo>OACY@P#dN%@Acq7MJeY"
    "T8Hn5wyj5xjBvHCo5Io0m5)^nETeq}=#q(JUP|HOBnl4#GSz&kQ`oz1d`p+2dfO0FE(L^hWwY9LkCpI^lpRc4er6"
    "SA-lHvn#xiA;y5y4Q)YAnt4%TqzRdE1W<InN%z%kF)#~E#vXQ;;W8U|M#mWpFBu?rsUQvpa*czx;d;C|?;G>hD2k"
    "rA-!k+Q%bh9qGy~YjvLkH(Df-ESc0&vp$Xz^1gc&Jvoo)qdHYZr}b^c{zgfX#gtf3i)nj^uwUT5pz5@)7^GD+)cy"
    "j?tG6V-i&d&tAXWR%R6mls;gmR?t|I>qYZx`)(cN{F4Hfy1swD-+z&uwzFBaWf0xMo48`NH)}wnsFV2q{KcYC5FI"
    "`2$(2XXj{e%sa%b2F1iLwuB|2R4kyS(-~$QF@x;emv{RrwXtcz3l2e1=k-L@0LVU*#LPI*&nnf2xnWuEIGp!k&%="
    "eO`y=4-!F?hm*awMkL?{gYOzpuE8<jrG?odfuomCMb*9|>OGfpGk^{J6;GUkAd`nSGC6)Pa8~kGh5EHQ0|dCfJ?^"
    "NKZ^IJ~uFeuMs5+cF@`G>941C2$8Xz)(AAD%~$nsAy4V)GkCaVU#gk5vIj-t$*d2W?QFUaQ{^yJ#%?L~8wM1rQc8"
    "Xyh*np5$FB07At+bCA%Nzlz27p(3|9Ay!Je#uQI+p&uq>I@8zWhw08U_cexmf$gU3-6CH$78&2riKn2q1J@DJD+l"
    "HhF64%$_~DF<R?N{Fub=qp)(m-(nobS8B`QESXh&s{<2B2x96-hh~PG@DH!!i+3KHz{KilAI{W>k@uF=73K-f`EF"
    "s4)|IGnD<vnAL;Xq?CgSe1-qsYrw9N45jPyn#(j?1u0a_zAfOJA4Emx$uqp|YDcn^GZbolX^gsp-q&MQ9(^;~W>?"
    "B<t19a8rZ!0*7Hds#)aqOXx-xOtnj%Wl~!r2qt#)pIxSPg+2Sihc>^}lqFxydL%*W9$IEFW!MoPgcO%2_)g66OdK"
    "#Dn)yKLQ{M+<Dwp=W;f@;r$n20xiNBI}gClsh)?*^Zo;TN)%?q+iCSkqV+^Gp@*RAjtR@WLc{=G#3JeS%ylF?xsH"
    "ei*zX$!uuR<p5J6TkTbv;Py<Pkz$DE`l3RlMbbV301rpKub!^%lhv_Eek70^EiC(-OH>2W=(U`<2!xNS^c|4d?dR"
    "v$HEkV`6n6iRXX9-tAP>-7XKoZyw;;FN*>9XK6LGz|_xZ6q=hA8An`mH%G@78NE_g%({uZ<^~*K6)q%76xe3B;4D"
    "ZWRP!yEomTyHh~Tm@2`t3*#mX;=<slVk3e`T$Rl?}#ua-%qav+<+cAEn0k3o)j}(}=Xa@*Hq=)Hv2^Mkf2uX_Nww"
    "`EMKS<E=PIeDZU+oi$r5WpIgaRt1k{n)>x8CT=?1P~sMQ=udk@(AV3$Sv5o5rATVTklSPg6EE>Y)d|VGM6xp=Cue"
    "#n>kh7QC%BoDEiS_|Pg+jE9nDjZv;JT7^mz>Ax9%s~M`i(y7lm%oq$I*F3OhFs7RgM-u?GnN|0`zi7l``~Beb;HT"
    "I7ecTxVbddvb#k(8+X}f#gseYr2iQ?2^i8xLy<E05-p1mgy-}g2&>RZl+b~m!KLmbu28B6@2RV5(pNCC8DchyKjL"
    "2I+$ENogu>xn<9bZ+zE3rW51@o)oEH+SE|>m<G4t>F%Dv(&q8ur>QUc&<7D?dH7t=8JXn*+702#$#jLVB`d|r{Pi"
    ">o{v>H%tFV7o5rzL8NIe<bAW@<jyhv2s)@^!j?o}g!y}e1v0ptNTyH<|j9V{_R#7HyMM<nEAqxDH7fy&}DUq0Jq?"
    "EUIDrG>>W!7j0BCF2;qng-|*2yjhYW=~y1dJ_`R_KNzQ%P<r^O>)t3FZ}>AK!3z$?;<L=?2leY#XwUaHHMnjWKo2"
    "6{|{q`w6ania|^fBx+P#y(|!9<`n{`rMP({)|Yk=_Hz2m(Q(!A)Kq@~mqM&fuvPGRV--3+IoR7jr7O>?(}zH2-)<"
    "AoHGI4G%hBG^>)o?`{Ql{-`Ti6nhJXCDe{%9qdf>ZnZ@E_!fd~DvU8Q`#!h#yKR2_th!avX!({_z9I5d{1=y9yt<"
    "r=RJXS*7n-ZEs<y-L3FK1V0Pdo3+nz>gOhNO<s~%VH=P)9`~T{q_aN{9^kS=Z*Rvm<a-vq7Y4M)S492KId)sb}+r"
    ">llip0?e+c=z4Ny3@MZb}1*1a0&(gf*YT4UHKM(e*-O_bcl*>N3eOmqC*g7_<&yOB0FzM^<0}k87<&q)_ygrMZtB"
    "KKu;W`TZ1Mtpk!hpSK-})Hg{ZyB&q>^GSUA6#?_QGK};tOnn0B`17VC>^n*#W|8{YLG{;it??f6Z@{lZvlO+Zrxe"
    "MO2jC*EO&4%A@I+Sp1-hDr7MCL=U)iIf?j20CIjh4t`&unQ}0L#o&^2(2UR4xxQ89o?A-~ir3)Mj2=mlu02Bq+XM"
    "e9k+;3Oub@wG0~7ZjJ?XPTZ}09h?JH$&{3vnk%Pa@SCiU#nSI!ug3Dp_4YZvj+BkIYcxemyzt*N2uwsHzeRvIRcc"
    "!1}T1-$ULNewcjS<nAwt_i7bsqBRhzmIh%ghut3n2Cy?oDD_Uf96}0lGCM|=iG2U*$$G&(KR>O`8nJ-pv>dHrWB="
    "3uILhYDKZwW|L5p(B!^aP9d#I`I2tC1wRu%!d)FoZxr%`f3<*tes<%+~NUW7Z8q?VMF8YjtFe}X?hL#<^E>KwL%d"
    "|sv<~D>VeLH2`Mq~Z=*R<ts7t59z2$sCS7W~qL(GMeM2i=+OU6(pcp5LKE4F}*bGxJg>ua?gu@K=PxoQv}^t=xL4"
    "d7|z;Zgok^!uX{Yo<T~TzYugJ4L74uw%{^MT~=u4=((=&)wspK=GD)|c{jYzpKNQX1aodT5&}q(L}zmkNc6wXv#o"
    "#cZvEfeTR-%-F24DzVU*4>N3_0k=v|MnBSM%gr~0yPlJNB?)$FTOtbct{m8o(H50T@L80NdT^SuA``)5_qUD@)Xo"
    "ZI@0zjT-q3>Z6OELRfO@u}N(#o=VTB;^sm;!o=}CzD-Pve)S{k6EI{RkHKs`{ZxE$|Ui-o&NT-@B80=_k0seEr)L"
    "|8J(1Zwg`SqLJUH^%+Q-YxkN{wNo+BK4l3iVPNB-L%Q}XAPAjC`jXjc-ou{iLa+_TgBe@YB(PN?&U}W?czKI-T9A"
    "(#```4>@#dnO;H|ph-zZI05!}vZ<izSmEg<EJd6$sri)WWLI-Xou<TMZczWswTh5cJ=NuKLo2hqGR-NMVj5IuDSB"
    ";kBWG$UGk>XST{+obnWbE_aW4$cF5jH+mBhLU#OAO3u3o_hXHHb%Qamz0JfL_SLq!hNjWMb!%$wxkDuy2WA@v>y*"
    "V2`KM)w5>{W4oK`W~v3-zw5DwX&X44)`wd#jQ4kaB}>re7y{J2J@vk66|3z$rQjGhgTlCe(Mkooq3VBB|^9=FZ)P"
    ">^ZMpQ=T{)o)VDwUyNI;ZH}W(E=(PA32UIrTM7J;{af{5a--cdG2g3z~?LG0%zf{Pjk;y8@HR)om#;2o4{fmXy0)"
    "S{EyJWYc+97a;nyZ@whh$gPzt|ql{x`(Yta3WxTP(L9J4L2gQ67<@~M+dZi`-$)*)jv67C|2^I9Op`>rDsDDjmUC"
    "KR?OPTk_M%sG)DnEKe-BlK26U}`i?VY+`r@gP!;JpQRunD{{uZpg<xr!H8>Zqx!bZ(-}8C{N-X#-u3*0J$5UsZ+E"
    "H(hv!9ILmqEPaiF(qn^32?C-~Dm-<<KGR<7b9j1JpTpCy>~ok<IB7f%wU}(?XLvis>#DEH*E(B&;Bt}s>FrciW)#"
    "Ftf9DKCgWm_ip`=tZ1959D`Qf3v7!(W?#Ux(NPon?@F1b!gXLA(sG+xD_+*%tMz5fjMOV^mKmarzOY1`VXlo1DNC"
    "*08axlSfcI-L#{&QzBiBZk1F-^1QgYx=g!*adbqqCk<ZM_v&GEHY?fT*ZJ-iHy$r{rU54HAsku8_H1ligOW^s0Ek"
    "(Jg*9o5xbc<^SL$~Dj~%%t!uS`ovzMaXRLEZBg2_H>POcVs(o#!eH(enZS|{>D11iGV`6yiE+Un>kbb<axbi{I?j"
    "IUS*=re-QGrDZiQ#VT;OgN_dLaupBkz5_ZKrKHLk0l4SE>6$Q5|M`a9xMcZD2Ra+Xuho<9YHL1R0Y`=-<V}K+8_@"
    "isnN!u_Npxu7}<M@lffMbBS5(_%YdLmE5(faSIKtS|R|kEqvm4{*s;1oKI@JdF~tFBw_l%WRhVzDPuWWGfC;+OhN"
    "Cvm}hxOIXzfcFrNwvlvBHcL(mDIRs-O8FmAFUT^8ZUiH4NRpwOF$?o_?LYLd(RHN|QGO)Glbu#PNoY0kf_;VEV6A"
    "Z0k&DHzJdMtt%$nx9JF_{kiRdEEE>i{&i(!_IT@VN%Us_<A&+W%$GMZHME2Qgv3IditdQ-Sh7oc<N4n`^htXCyHy"
    ")R?qs|Kgjzrv2KE*5e<(;HitpmX)(^f-(Y8Z+s`PNquA$|Ks|kiQ^CtfIXu$fE%DOG&z6F*Up->Q8hB)g_g|q_T0"
    "2UY7V%2Js4q-!TFCF&z|q!xC!`$k3$~m54Twx!*VGYJDs5nf^;7Q`*_e%?HpVVyjzy8EQ<MyZBf*or1=}X}Prjhh"
    "-C-C4t}|2sm%3pPQRqdn%@-1?wqf$OO-47x=)<ebrxOcH+NnR#O%DBRHa{Wes(}ck&$y040#b6m(5tIU*GQaQL|H"
    "}MS+AU4qj1U3YUM#u6)TptoR_9fQH<1#jT{xcg~T&K0Mk86W1GxiO=i<#kP5mxUWN1CIGbD!v#vTYb=S&9?CBq{j"
    "?5`lAaw91dx2?5htpQZA~>f#9u7r`bGN!teb>{91N(`L<^$3V{-Dx#h=yaB8^PtFah$@(EHA^&Tw)R#u52m`%5|E"
    "n#wEKUf(KG(qNzK=)EQtZ<4eu(%t#nm%$?cabnfo%@#S2*JgpjYp-c|zSsooz)yn--4!8DtQhNW`Gpn$szq5lk`~"
    "BAkZw}6UmkfJHfX;T`b<|~2bxk>P?72%ndE%n&*?)x|{(adz%)O(-v;F^h78xn4oG9n?5I?Uao%{EKzA|!WBwMnW"
    "?I7q{yN&VGkcdaW?jJ_R7PaJ?9c>Oplvn=fZyWD2uQipM5xqHjx&OMqcXS#f8C$t^6uqWl_z$_;`kP9$EsdZjb=W"
    "ker&S%R&phmQX-8BZ*+7wBw0^tcHe0&;BaPA7;H`SktZ&ddQW)K8_IES!2aBQgWdEQOtJgF(_IH1HKjWX<xUH*A@"
    "moWgHRCuq<chk#o$mg;?{9nV>zmjcoB<C>EffXZ(`4x}VwBai%VOxbCEyOg&msur(~etUkA7M+iIe@~{as+B{k`4"
    "Sulom;<#8^hZ|v)TwESdN<>+w^ar8RIW^v4%g+)EPtV555mINoFsL`cel@l$|;$w-NM2Ee;IexuQ7fLD}v8G1OZ1"
    ")S3_(fHcO$YmLj?exH_tNp}e^yvc+Yf!LE^?JeHr<GKycIV-XLmSjR_d!^g~kTv+@9j`k9cIOYkdbZmesd$L=DC%"
    "vse3@S-u=y-MQX*W>x3!dUKT@ofzI!c>FA(tyz^4>TiezH$?3~k7YIb2ET<h02OD^;wE0PY`vkl4=YpK7$CW0((3"
    ");MQc=Ud70zXWV`9xgRA8f_V>Y3&^)4=kz-XRU$>Ztr8Zgq={j2>+8gGRki|r^qgj){qGbr-L$7buM>vxW&{;ui3"
    "-re%mci;$8(U#~Gm!<63_DvNYImrZ6@7<Fys2YsG_KGf1V0TNWzS;<)!44xd@AK1I~6xSAe_y!O=jaJ*TffB41O2"
    "`F)ha9EO}0lkQ9oOeZ0&Ui<{&bef2_Mo1Q%1-e$Wb+Tl8|-&`a-$A4u{@Goa)$L{%MpT^NN7i^Uycrn!n=gs}X)P"
    "qD%8-YfnABPdk)`Mb@^a-^8&yV@^DH2W)5lKI;un5tuIgL{)9rRS=mM{Jwo}q4^p7(ZN_%My)5l3|5ON0~0D!hF{"
    "vhG|jm-D{!q`1=jxx|K18Noce!6dy{N+(gsIB%F<l{P{g3e-?SRXx<RF|}X<R^85vJe$}snaVqW#v)g8I7p*C5MG"
    "y02t0{$!Mn+JHP1B9;I2cSnU6*UdxpnxI`fd`y{j$8{e?~UEPY$C7;9Xr?&HUF3=jdXyR-A`>9fZlcBnMwW7%wk;"
    "uB*ZX!7bgSF@~_)wLtfT!ZWb0x~k2J-*+!&WwjEn{&_(q7gDO&TWI@r5nAL=@?C#$<bj$3(D?Kwzp%#SM|nYvmbI"
    "0Q;0;vqAG!EAM!E6ta<6B=G}yq5nN{1vtoc}mtnJdrVB!{q2DO2-}#zadRhH(gb~v<nhr@hLDfONji&t9%pXHKZ7"
    "~~qK5nG76CxZRdXWbb<Cru}i*s{GV{<ZXSXArt(Lg_|#5R2<ui%_-C)X2uOGt^_v!=gt1vB3k*D!Gh)r|b&BSJpY"
    "<*ZEI!ru0R7rWz1lk<!Ay0tT11*>PiNA>UNlg*}~cz-yGEw0P4SlD{fku}7TBI7{s{JjJ%j9fcs@AgHLcp~i_u%m"
    "PJSYND_Rh%Oh?Z<I)U&Po@JMPKGzBpt>Xx+$LL_w)6xlvqd+oKFYTU@@;=xNev_3=R46MHnV0OSycIhzZpdW0gY6"
    "mklFSu8DEQnT5pQ!(U3+gi(xvF23n#DJum9!C!5SAxO2gcr7>nvWAqpc7<IVAa;DRyMJvrHLDNYlCeM;2eFbM&s@"
    "-my6;O<ixlkuI@&ON8LCzmCP*@Q>$36R}4c@iQ7(ETItJ7ICVXeUHue2c)q2?)_>NLQAVbUT2waTd?LCS+>ajo#9"
    "uYP(sx<Nx94hq5J(a>L3FF<u9&VITe60rJ8Z;uXpV0X%}Ec*+%R<7oCfst#T9Vzak<PMSBE;WFZ2x~7j49Y3zWB@"
    "dyb~)@sQ3<EP22Hl*IK$ofdUuLbfFe@Z8n)oG-m-PUm|Yz<bedGPy{6vkr^q=@<c8F}3qqn(XXw&N@W~yUe^*QMs"
    "Q(>u)2iFQRdWRFI>5EJlhg%DG2S#^r1>#vnvNvLh^F63M9rYJXBPgnrz?FAOdc&ZrVx>W6}Q3kxGFEM!-yfpXC0u"
    ">mZf%owyZDCe{&@(~qq9Xf$VY)a`QVSo`3g245q`U0ZqVCq|!!75B~Jw{V4{R&hD<&`T3QZ<1oy@0@pr9VDXSA-j"
    "kI6c{vVDRfAnJ9vZ!kOMqVHN2mHlV2{gy|aV-FojU7;K;F0#R5JuEke~OL|2zXL5Oi{ZYLEM2)4gOBSby9)4JwaW"
    "iJOJtY!Hg@fbw6u>wcq}3QLk^`)jBhgnVWtJMZn@aU1HwH=3#e}6%Sy=<eid(!~3?47ANIAYA=zWz<i+?i`>tnT4"
    "crmue+2B1gTn0w(Y}yy0m&bcNRf7PU7h$Xjo=H3BeGUQ(tjOEdlkJ^n38i-?>{m>aV?ow{o`(qUN)WgA1e*0Cddv"
    "SDgIM=;!^qvJNyDPs^Aj)Sb4_!ufT)d6MDwU=%T?NSI>e~^^kwY>0+X$yCm@@?=-xFCp{q;UPxFs(&4ALm9Q3ENk"
    "2MdFlwzJag&%MRexOT$yAdF4M04bAp#OlAWsM+75%J!)8${LYW##q$)EYpnWp5&-uMldPfZbEZo-Sb~$kC+S^faM"
    "2cGN6|9U@D~6x5s^=dMbvbPP;}O3)Jn?Ieh;tX~x(7`;3F?M;hc!gFfBLo6%ofRMJ4+LRwx(%ILVE&4NJ)5Cp|FU"
    "L0$9D{t1=Fv$`v4lK_NJ<#h3mY5BFwhdcWTy2FgP*UcXd<lrJ4WvZ?v7k#<}f3A@g<h6=Q?j(4M8EHkz-4&lC2F|"
    "ER(IR<Q*bz_q9eqUcO6+k$KUo<6`<AJLM|3RQ!_=_c^!+TTG~+0AVNz!k~ZSQKOF;<^oK(2yDk)cb%=Igv(H!MR2"
    "wo+Y+311{mglvnHlEuY`i3a4wPGVmmt4&Rc=#o4DP7!?@RcvAl6Ib<4gk5JyNRTR##CL^lox=uR0U-s;Zn!4%odY"
    ";ki;`^Y$e{1Ys>O&;x7F7Uo`GNXcMvJ$TA*6^}xxF(I?u_kK>bWb@6r75v&cecOlM11dp#5_rsd4}Hwl<)04!CZa"
    "IMtaXo#y<%h!^@6?-tT#J;U^0@*b&2jD>Tg>9G>o<oFzvm3A&kg_x6*6!?PnJo#3bqfMW9d?(5(7Pm|W))8y~%iu"
    "F{{b|`hu^=|6-{XW77^!x30JXrLLMqKE^w?f!BQUp?6^VBSL>aTEon&$u4%@<^PSgNPq#I5tmX?pvIOrLla5CG)o"
    "vV~Wndthp(7MT=iKG4;cXa~3_ceeZ6sDAhpQ^RG~suFJ>)XvS70pQ$_{(y2+r<AnQG^ruhhIb|vNsC{baO6Oo#av"
    "Jsr`$2fD&4}eSmv2zf39qZ@;j$o2?s&NNvg4Rl6SJ5-q9WR&f7`v_#(>}Teu)Mc$erUSWNY5fa{8CZ=tOOUh@tK("
    "U<Pu!F@HJeT4fOV@AysSXjoQb0Ecx^Gu*usK$@fV|-W!&OD)ISJ_ZK|BExbzmO<EK)E|Ri4mNW$!feTwva|zpssa"
    "=0s!lDD=KpK%WOQv|2}DV6TD^rLdDpZR3vXmbsK;|LKhhEiAo4@GzydbIQ3S>nK+Ugdy?@cG~P}kY}M#|gT;)cVd"
    ">3lu2rM3pC?<$a$#}DdD4>nyE56{+S!&pBT)gVtv#1YO<P--_CrB^(Hg?yUjiZ8FKA4AN3UM(@B1}0%VoCE)RioG"
    "c>pa=_7Bf=QFMmxPu}oJ=pv;K=!oQdXiLtPT6F%B%VMH|A3IZ&S=z+Y6&CWwB1Z%0Z=gj(rZ4wj?f&-qtiQK=w)^"
    "_%XU%4F50}koQ`yq~)^am;X|wgd{@v%W7ok`0S?Y=m&Nl98<yn1d!_fx$4rc7Un9lHj&8IgR7C;$|@;o+$Cp*Cuo"
    "`1U?j~~#<>}=Q0=D9PQwl9h(<B*}df(g3X=vHW4&S2@`1t`z(140&V>v7jpEP0tFpj@pmR<yzs@Ypw>c}J94bMVv"
    "f5a3ZU<rrrljb5-j9C98;hUoBAh1ETnnt7FZ2>9$L=}OCoqVZq}t#Q8fPB!j4E?$b|g$TiUj2Z=Egx|=;p+!GTl`"
    "~X_mbnZ}tQ^WqU2gLoPlqb-!ZF4~v3v^~Zb3t?mxdLZ+bJ^#_DcdY;&LaL7Qq&EDsqY4OmKu+6Aw|#S4%F?L&fO|"
    "wC_;;EYUTB+oxbfMb;G_KivO^tgcd=Z#HJ9IuY*cdmOXQrYL7)wpf7%v?xBI3T0LbFK4}ysk)KXIPq@8eD{nPxkD"
    "Ip+hhrs4+ITD?=;l$FzFWgAAXwiE#8hQb6e%|W~}9Z^xiOf6pvnufU%N|djZo(rd+CyhAO<cD6K{ABV5e7<aYYF!"
    "LsQMO94dO@V4ZlVq@dREcJFb$D!*<0KtL4_#(1sz#kuB1g7Mn#%mm}&=1Ez$Ym-uyj)Ebb`<<VvI`nFQGNmHz2K!"
    "$BY=04QOP(Uv@90ZCE_9=sv~E31Di`3wOqM~yz|8zqs#-v*TcIZ7S;yM?@2R6N@jKlT2#vh24uW!z~ITmO<TI-2%"
    "j`q-sm8q$rvuYfnlsPKHs*{)A*@rj3XDv1qVVF8)cZE@%dXbwslHgY=og(WdpV1tLuWKN>m8y=@9;}QQa`&p8n$5"
    ")a!|Vq^`)SRn-4qrBYKn9a+51cH^`D6yhzT+G}Nqx9z({O5RxojJ)yx|BABMyEsb4qd@eWJ2D`Tx{q2+%NPFC?v?"
    "r=&LXOY25OS}*f8!1dT&;gByP@*z{%jfFtvJxS4UE9oj5QqPKYLWz536trKBhbCB84{n)XQvi*<VNa-W5W$s`{ZR"
    "8P{rft`d%%&7^sreUGXg)bbF%Z^lN<o!;u`;v^!T&gqAft=(fGdMXSt?|zo1F(!-Ov;$e*jnv`jJUclE3ix(Y}wQ"
    "O*ZX^CYq?MI>g4E+vl#!NN_==>^^(8iMkG${1}b+(`Er1DQ-dr+!su$KYa(u*D1ZG;0Jh)?>qPk-ma$!;&q!2K9<"
    "UvE-BWgAlp`E(M5+hD;fO`1($1f$@n+U0b2rTu-4nJ81^ZN2MeX$2tRT}zwSgqb>tYDf92ZOWYuLxIp`t(jR-y~t"
    "WSkopa}yD)xuS<hy;VW^O17Nub?a3EMhq2xv_0OBo*ywAi6D@$Lbt6aOXm>y-PO!-`o!sQW`%V1GxOuXHzx%KH)8"
    "PF$LoBNYa|zQB)`b@#&kc)-FSp1YO67)p4KKBrMg0Wt%;?*tX~<r)`}sOltf>sTu**p95MEw6?_?K1TFX)2$ruH_"
    "DnCq<llp)(sNF4N6cjvDzTD)=r`Foul~Baq%vo)p1rCtRHC8{M1!Y4a@AXoISfhnx1L8aAL35Z>~O53S!DoDY2Tj"
    "slPf{r;+XEUpE@$mzp$GBgDPcc^t&v^bp5hsNw!S=m2tx_b9Kjtoxy%xnJDt)eSSmERtg$tU+vNBm)t1F!a}5Ji4"
    "GQa<*ZN_m5tl(XW3=9cy%zK&WyCb*+7DzI9wVoi77fp!yfT=wWdnGT*<4+S2=rKZK-(@74#?sU9q?$oc(n7%|6bR"
    "b=7wjdG1}(A~Sg#FL|p}+TIuzfthd#N<Y;qALbPQO!ezXX274t)+?*Q)+E0#S9`W*@X{{f<qaWzAb%}}FYtS=gtx"
    "$Ogn(d39P~=Fh=MiGfl%2(%r_Z-k0RP4XShC^f)<LdOfwtde7H!s9UEdHjYh>MH8}7%Wu^tXq~GUd*q7L`)D>+X!"
    "d+$UK@4+E1{S+XDn~%f)blZ-*U9p+V*@A8W;iT?Qw;Nf-gYbZMhzPlJ}Wc2_0F-&7$aiWgWrx{?w;*CBGl>rnZvU"
    "G9ucXGP!^y}Zmx-ZUG+m4zLo2fM-`dA#u-T}?%N0d$ysmue68;eHovoi%%LiWvCCCkb=&Zfqk}=)_M_V!Yl2%C43"
    "*kK?G(6Hl!IbUzyq!;4%otMv~>wb#K`+{lK8w$#RR7jO=@sLfuNJTv%Be?<l&deZ-)o}<F|dcuhyiv62zXJHcM)1"
    "*v1pK#F#*e3n`Y){skVA8?T-+G1O27h!p;j&pJ*G56~c6;JO=Ax6fqVJ`BH2lgkyw=d7-{uSI^fz~bl>rE-}KDEu"
    "Uob3_qo=v8$4VmXV2GKm}O91sK$tUB4G5)zPL^4A*VsO-}1hwbI~3LrL~niO$a2G?pX#^~;?0Y>!<VIdb%>yTCk*"
    "pS;3Rc*%{#N=v~fl9Z`T?TN+6D3wpI!tz@?{bv52h7@*&N^;C@)e_W;8CIy9j%Q#-J$KOnu-SprDM)bWoVykEL3`"
    "mV?PJSI@-oaqTQt4pyZ00-5LQj`dA>05lF88RrE38Xq-jQ%q=>7W%)z|vvl>9Qi6DGC+v@sPhEywiReKM^xQ<`T?"
    "AF>GgU#$YsXhGoDp0r@4wPJR*UQ-!83PUQ-a`(ud`|y^FZCL!Daz>KER(}?w=$-{S)7T1QQ1$n*%W6NoZqSgu#4+"
    "JO+XE4&b-t?S6`&>=gXPS5)Lx@$h}~=7`pas>gC6Jr{Mm&m~OH?pgQinkU&#$M2wVn<P87-=&FaEU0kQca2Dws$h"
    "U-6~*6qzv_AKpd%e?35VHlNvAv3GITwAPX#~5d+?(9{O7%&s!zA;4x22tgSC(1Utbimn&Sp5aM8Wr(LjMMY8+G9j"
    "*xgfZLsc0D4v6h{vRHKi-3)YQ;>;VogGHjxYg7)p4Q&c?(6;2z5P}t0yj;z+sW>!gxgjaKrlVW04US5+n9RLLeFa"
    "huEY>>EO@3KU;GD(DH%gJf>3fr3M7bd7~{R5KxD?)#`}Nw^j9RRb>7>CBg|uA@u8S@l7poYpFlk|#Xh<sC{o-qyX"
    "86OMbjKXFMEe6*jN^X#{4eHu22MAx-MSv%^MFbbR@oeL}OT@_gUkFaRQeVW-KB9OOl?rAU$Kt<6n5%QVD?O8-f?E"
    "lkCk9N_U0{ZRoVufG}IsfsbA)Va!SoF(bk%aiFoAy1+4D^X$gR#}k|m%w4VFAVC&aI8zS(iN+ZFF|wY&fH%dP-T&"
    "D?**`ry+WWN+0`wVH>HK!v9)dm^&vLPf)HovOB*1^YE%TC|kkD*2l#Vh?jmadg8l`!1Q0Sq?^D`f>N+v9AUAu**>"
    "}yCU>&FrfP5qe&w#mCt1AK>e6x1PjBy77XZFR!wkdim>Z>k71qQO$SjvXg^>M>+bJBWA=?s)dUsxzR<2bj>kXmQH"
    "0E~O(iQjoXrK;SzsYVGuTafQkax~Y*iD^|S?I}0p<K8`YS&VizcM>%>s_3}Nnj<J`n(juBV-nqU>EkOK>+(ei;&L"
    "X3j?V0b=NO*m!=21DPbeL>yHs|J)JSbouZ0oXpnM4Oxx-m|O2F1)XLr_!#J>>wAasDB5{C0S_xHUlspz1lGOk?1j"
    "RB8%Y4J)`oeujx!WR9y8K-ILoRz>;r=*_-6RD~-TN<fh|@d8)0=A;-K87ou+ZgVuef>?1aM#Yf;ne*_`EJj8`IQ<"
    "`Ccni&^75=kWkO^CcUZA;Z`Y%Z_#yp}iUPmRca(>weGh@+2pmt8-s}$!e9#Pm3NInyZ<tW389Fs*1txr}{y!t4Dj"
    "n$0t7$bb+{!l|-U0%KznE7b{S|TauWK_J&mmd-Jm5WN`QY3^ztuTux9*sj5NN-<;p)PU15Bq9l?{GFZUgM^xPNY5"
    "#Y_`=j(_HkS@Knv=`8Zp^-nq_vc>rbwOEDy;ZM8>}5w=AcVM&d%2XwpDG(0x6`^-r$-J{cq86P+B`Whymbwr5*QO"
    ">>+mel#a#5zp`!HTj2dU4=O=Tm^ikGF#D3h$`@espk%y}*u5j^M8jIfAkde1*^R#gXy1fA?n8@>V=mCRn{qYF)}r"
    "TVq6Vz)I%N-n(usID^AHVoSSED<R=)HkIr0hM1a53btUv53mg>Z*7`?(OyAkEQKjhG($G6ZW<HWmrbKv#d#p2McZ"
    "Hfg30#mh!Tmy7iRDiXn+Gg`L)*JbMVTC^u~wnTc)UaV~m>rm@VF`SBO`X0dVkGi+@}fjl`+SN&liHlEmlWL`S^1Z"
    "Ci6`kSi+pCMVd)><q_5$4~kw^=P>SHPTsdw3<$q4C!m0E@7Y2bqy1?*^=rV)S-;?(GowK{9RgMxYQ~;{jU5s-&#~"
    "yaGcG?tWP|=hy@B!P}#sE4Riz-a%_ym@%90un7CFNX<Kc&`C4$`6H^bZ3E_fpUOt>P9pXcN>%)uN=;{cjinB)5<1"
    "~$4Q+GZ>pNy4%MXEB2hqy{LCn&?<pF5r(J~tVM4e_ao9u?$@*od)38cz)m!>Qz|^+JKGyf5b&!hYxt<7^>%%-cX}"
    "3R6i#@L%q53~40Jf}cOsNx<qR<BS5)jUYo=#mokqc;K#iz?u{|^bW)2$w9{KnSQv=K5)qSM#qzB7mdPFu0r=6HDe"
    "qLJN`Kq@?46Xgv74dCnCb!Yp2Mwk-Z{1s1c7Ex5+RcjI%`^a1nVpwh2TFBe<NAnE@%AHai-AF&*P+F&}_?s2X<%z"
    "F`$Lu`MK@rI%ZF5|U9!hC)UvO#1k&Z@sN!?%XHEv`ti7v}p=qhhM-$@CcwGj?Fy$TG-@uBlHcGpZ&$`Ce|0=EpnT"
    "xff?Sw?dlju+o=L;Dk!U+u#ec*`EWyBT`N0vL({02N=1>k@K5MS!q7E5Nuco|ww>t>gIpPKxfPi3Q-Nf$-?W|JTC"
    "|z6x(O;RA@~tyUbeU)L+XMx%jEb54(pU5SuE?}A~9&+PPK~5&_ruYxt5lgE5%Zaxh)D^Batn$+52QQHvqFSRnknd"
    "E;e(`kzfjQ-U&tJnSMT3x;A=5eavS4X<$uLG!X0A#;K08Vp4w5y(+T+0-E8Y7NKb?JJCZCrQVN8#XkE5#Dqb_$cV"
    "slm{)_OpglzmD=&_6o1X-?(J|gT_U%b_Ba_ORpHVUu2%N~qs4~Ep@g75HTv?!OWYA=gDWep(cn){Jr3}<l)i!G=+"
    "-Rib<>rFVcNPAfFl}c+F^(#F1Y!}}nR)wqrI}e{(lP7u*!5F%hL_1h;0SoeD8UM45+%>W#?yi07kDkQmuWTBSopU"
    "kTWd`qn5@>$dMmoWQZ*6vrE~q?K{7m?1-=c^4Aw4|Mk8m#f2~R~@hGr?K+hl-(^x1X+ZS=FFyy{QfA*Zw694Q-Rk"
    "GTiRCONlP^-qqp3R=q<)bR^6Z@v5kMcY36j1cF9IfH`bsVd*E6mp9o_^F(Z@yb}*^w9^1X4e@#4*W3^;tT+)x=F3"
    "d=p##v(S~Of%G7nmFUi5L?gWIOo}UL5&2iT^vVfYwQEusd>PPe0TF(z@D|E$f$y<aI9n^49j`9O#lXcdz0Sv=*>L"
    "tzX#<+sVGw$n)frvtvA~NC<;oWc0~aEdF-;o-hO$KPoK1>)pbloF!aJTsCQGg4>iHNg6&%l$H{2+AKrQKN+*2By6"
    "-9#z+*LEuUVMW8E^|qdv{MNwwcIhKO>MK~b4e*mjt=*|>@byu_c1y^)fwQ4m=!(D5p^-=uq-00`fT8*$$Yj%+)tM"
    "6QX5xp_dsf+zSwU9PX}phS@Q0u{htpGlY=*J_Fo?Cp6$QGO?yQzbu!3|x25cmZ!KF;DRqV7Eb$`F-Kiqq#mQSY4y"
    "Adx00Wv*Ps(@bAo4TeU<Xk|1tq)9vr?R5;<iFT9^Xa3lH(Ndu{4zCBiXlS$<8yG%12wZ!bExLG8{OB1&;KN!*2uQ"
    "C#8@W7iaq&;o`7FUcU1(BXDS(_+Vmw4cjt4Jzl{V#ueyz=a&TvF%FtShF31ZJ7S#fB+oDr9uD>8g7UP7o>S5wte_"
    "ujhyo=R{i*6Wei4JaWDGKdGi9q^ZIU|ZU1C~cf-Wf`U!4g$**$zoWH%p#`V}8gjW38C@vbxWf#n$}xKu*LqJont;"
    "wMuUWi67I=$Zr`J#yp<eHLz9g^d(zfH~BdRD^Yp&@6z_Ot)d-j=v<<n)d48<n%0c{jbF5$0Tr-6*s8~qVv43y@>C"
    "BQ0L(f2<P`3-hJ409pA5E`aUB5`gF04L?>~9HRn_4U*AMBD<V6rBHpdpQq^1?7dY~8|L5JagWvbN8m9}wOIeMe0#"
    "0_hx-m$=_aMR_k>=5W8&Dr3jsM%CU9vvOKi;6BHd<KKg%jJA@z+*lsstB#`qa8-B&NN(G1fg;Q7gL6Nm}=C5cS#("
    "j%)qnxAC9u0h)mRDwDO@%oR7<rQUIs%n3TIo*S)?uqNu!eE8RnW?epnPYCid{*VtJwv!luhIo5@b9}s6%Vv^pP|a"
    "e)B>GtF++CIXO>eB)Qcic-=y%%Ocizy7Z|^@*#~Zv5zjcv(cTU{6^KlAI{@&eM@$PJxD^h*cZ6$Sg1L2g@Cn@pD;"
    "!2R%8TLj}D7j53&B_xgT*M4kPiAsAp~H+M_F@@!xR_yJG^KtCa2kukuYNUyiVCh_Kdb2$3?*N15}l-}8F|_`48Hg"
    "-6cLoK!3LV9dZ1E9v!@j7G>OCX*tM_0;JL?*YD6IzaXdw$k|t4G=Q9+F<Lx%M1S6%$YFe6P^-fXFK%JRn%U10q?s"
    "$~$j_u<Sz)Nw{WsVoQF_`a88Jn7{MND$XKl~ypw0yYXtY05}fO%~xn-HjFGkBptT-o{pMeDwJFBRx*U&Z3v;11f="
    "ru7yuUoNP(@b2B5oRi7cB&m74Lyx~hGeJ2QpB(NpxT?zg7Foy^SMzLvl+Z#8H>Gqj&z9F%!$E#HhL5KE#u4fhUhY"
    "#Gi7&4yTi_>pa9>Bxch;PTbZo&CK6Gs+m2ZO!@{nk*7^I`~?(9m9v19rL1*5AT6`HG`pHUjQw@x+$#x30dGu_%+%"
    "x1J*vFKJ0>4u5Aqii)^(kEzxRZbO+Z*>Xdak_#p_*2;$BG?86UQI|INlnQ&%}$aY=s|wUjoYgE-AIO}2@JoqqoDu"
    "blu9D5v;j&$6L%r-*Rq4Nb;6JED;Y4CK+@54T>x4XwnlCSLho}V{{UXu8D6$;JvH4t?Cr1KXFt_0`ly`pZ7<o5TX"
    "ve<Wg$jheL4!fu98_-pGj(_@)=ppLwF4aMkzEde<`2v2QyXr!5aqrY-Va_Ne3N^+oItPSOKkvvLCY|?0z}DDVO<V"
    "|5LGS(Xj$X(81L|-|1f9UfqnhN$nv59z{#E$Ky_NLO#ma=Z=v<<@IdNUL1j_PDa@EP-_^WuYj3XSVD(UYvfqollg"
    "%Hj~p&uab{t8BErs?535z71zO;Ilktv!*qz>_$!R_wnQ_de8jiMjg*kgtTv&}+v&CjiwZk4J>FIE`Yic8I^03yfG"
    "R)jKHBRN1J@}`#Tn*mLSZYveKiQHpC<Ortj%VXyaPyVw?ljcWvE`soaTDy}7HK)OP;tkFdxsNFbZ=edz3ns^=e?b"
    "6!Yyn^3Sl%gmeL-<>b3AQ@My{WZ`+C=Qnj<sFJOjgDu3WU@x?f)-z=bDKk{dtNDObxgMl><&aZFEVn6^)cud^z6+"
    "DxxprWNya>OO?%Y0nm@jbGqEJkOF*(KUxB@_KlHX?h(l%QBa5`?*{<#q4Ly~s&<l6~jcZ$L~Q9mXHx_*(L<;ss&)"
    "*1XkjZ$e$^X~=Y;q+G@VH{HER<S<kkaoy+jM5hXqKO;OyjMh|xv3`>(ut5pTid4xtuR&e=?Y8^s6JgiKaQntyZrr"
    "l{d_Ego`%4RgA}rPvR_%YCXIrD~tsgEvKYeohS1VmGGr!|g4lY?Y8G#7yOk{}@p|t9LF&)A6HtkWBMwLShfRf{Vw"
    "JhVpLOb(1xy>S5|3XuYajRb80(no^b#wFMXVXz}1(y&Cbj)H~Ax!;Hice^ku+NKi-RT43(|NzP^zrt+vTPqECSi&"
    "hyAHB{S<o(4rEUo_K)&p;*^tN5!Q2Nj*tMJUqN=5hYUYsY7U@X_&_Fm97S6^EC*RZGUUI`4!Hv3UJ!-F=t&!U}ie"
    ">wv8Fia%S%~M)07M|fV2_l@fVu|<3YtsTOlRY9&pBT+7&W2sV`<cNV?D-D^b$Sl&ix<~7XjOb@ov-_r=mj^y4WF4"
    "M>8}>^9E2UZ{SF|@|`Rg#UzbpnnPd+kh)*Xaz>-qf(pl`cfZ21!MN;<R^u^2YeBy*Kf!$~5BtwiMPwR5wIFi4dG)"
    "o9u?ne^e3_w?W2Un1%-a3#Q?1A25w&fa&0%XTYsmigfePD?wPg({^=DJqn$viK+P5BYf=4v5OHzT(FPa+L#m2V!s"
    "o??L2H2ecu33cYw#LGh1%eUsb~U90C<kaVU*6Oo!n=egREu!mhv~5FTsge~8s{SUdW^xbkVS=@DnS|Pj08v#^@XI"
    "856*y)m|rS;KV%el12j82Ne+HKJUS7q@}}#^8olBFV)%f;-4rmS8uYSyFWcrfWlZOHN+v82u@koT4sjjAI>*~=n="
    "|m%j(fv}3puwW%?uHwlZ4WuWDzek6AVQ0Q^AFI7|VH~#PSKvj`>gpIl_DL;Y}LF5G5(~oi=V|li<|b@ppCIU7@yG"
    "DggDsXHYF2I4ALWq69G=`Psi9U=nBo?I5FoMMr2+wA?xQaP1C;8uyp^r)7&4)9-q7HfAs(g(l{<w&9S4If=t?1#M"
    "=KBOmE<imkXWyDAp*s8&MWK;7WdlAj-?PL?AFKZ+5QbpLK(3rN+U!IbsY2HZDkB|36<YVhVv8l(6#_#GXrsfkKrx"
    "dBPFtwE7RWdV)E4*gV83ATJ13|m_3!TQ6cY;CleoUOxhGCt)iPmty$Jn2{!vh`#%Y-L!CMtB|AbW5{GYi<Iv<R#U"
    "&iumGC%Lp<fu1fE)Y80z_pPh+xEB%qDNUVEQN(0wS@epk`3nkt3b;;>X8;GH9J72H~hfx(S{g+myOlcd8?QhA>y@"
    "XJqT~&Cjp?^$#R53IdxrY#dCi8mna0B`C4rFW)6#XGKJ++Cv8)zFkp_JT8{<T0?y8MpVhiFULXu)(GH74!~cP|mE"
    "1_%U`a2*K>iZC?`(kQ6LXO<G*ZC6B3iG#u>MCH%QNDD=TR2Qt^LyB+bw$fAlz2{zk7oj@0xk?xBui90-9r?)feA>"
    "|5;3ViB1f<yA+uJ`Tgb|$k$^L%_9*>`H*U8z@gNq`Dsf(qXK?dN7(`u~mc(ziPrgl-7q_DQGgHU%9dI4LFl&ju^9"
    "s`ifWQTPJhC#!BJ)E&7=Iyo%&NRn}Io||uXv%cD<|O#Njk77pW^m{TCZ-z(t!*^;yG%d=b*$M^LE0*99Cu+i7E8z"
    "aL6?r7k3pA40OcalmyXMEy=yZJ7c?NcjM7>UG|ZE+X4Jab1bdC<r6zTMVq#0@7~2Y4F*9?yUj#UJM6a|bnB1QB@U"
    "EMXYv-jYxlL{@Af8){>H|8~?<ra@vX6b-tli}DW|`|IW{f>o(!RS0{{Ojzttzn05bw(;&!6;no;^eMC;rz?d7VA^"
    "_PK4<xz0ZcrY7A&LW;)QRUOXO1Z2TA#S;fVeJFASV#T*Gv282|e%13FApyT-7<4m81Cv1N>RO%m(MC3#nmk@?80f"
    "4xhEB-9zEFLwzG-t6mgr~7x@dXs#J4+JYA*`H%DKNLrR&`%YzNk&Sa_8P=Lmh)B%BvvWF38VE;k8pLZQB(rn0e4X"
    "l85=8(M74j#V|!q%I>!l|#VQIO?<g?1a`3f5W5?7EO#?p|NSHe~IbDta^BB=|F@NaU>$3tAMtR=oM6u2mfhdg(rH"
    "{x&N@m)92gUYvf{y>u@7F6mOOtDf(v2ZmEjNR?~^wI;0YTnMlf<uSfr;!ufjqpHTJh)gQB}R`j`g(JkzKrQ<wzMH"
    "i(zgNhO$&u2`?!>72YgL~C=3J_<?R(tIYXIB#idYXg1rPVA?FKk{KEvT%n(i?w~s7NDyZN|?9dgBW<XP-#9nm8-7"
    "@Wog7WOCeLUXLQb+l+uY%L*5Md03xhj946og{~J5?%^e<2+y(2;}7Ki2dQ4U5Gl@EDOgBc3pEBS>+uVraarKcEPT"
    "2m<lW>)==3%S#*jT|ny}`N@8c@bfzDc8bAhAR@Me5+lcnB38#7utp@wWJ)I$v&qZ^xw!4c+&P*2J=bPcq{26TK>>"
    "n*zVw>RGwBVuIIFTGruLpWuF>uBhNW)a&<H33WT@yR3^S`PKDie;%rqmo>&b1@x^S3^fSsCCZO+cVrbIr(9c>JD&"
    "mIJpFM%JOWn#IiSKkFUuxpJyxq_gZL(r0cY6qjlZ?8H2ymV~{nl8JP?jv?Jfzs%6&3Gjz*gm4OJQjEkctIcx9e@Y"
    "TV|8-hpm*j+S_=;i)v5GGDGY;7$eUGGzvcSJi+>Pr|H%1BbQ<E&`~AfR<d8rO@n8PQd!AOS8zC8-4ZgN86$mE#*q"
    "a~iKoL<Uo^PmkvJJ=U(a9FEjsHHk?0wz)$*j{7Z!QSwF3i`%aU*%Pt5Ic-e<U9_>^P!xB2CLCS<eup9^a?~xB);N"
    "`b#Lyqjv7~U5Vv-p`>#T}l2mkptAQt^_xcg@R^mun~zyI6GYe#V2(pwYQLGsNS1&nmqdzbjK+XY9AiDc||Ps^NCh"
    "!o4c&X=jAjir1JQq~nxrCc>(_3Z=Ki=I3TXZA3Cc))SDpcCc`*Uztq;2}Kt@a56rKK*gJe|YvVRx7+}lyTP}o-Vm"
    "DnIn%o@>5MATk14-qOx>GFPxwT7eAb9o`}Dz-~`0gye2)5%j_j91|N%5%8iE~_!qjl%e$3R?KBc0C}RD;e`a(gz#"
    "CJfc~|uwp6vgA^y~i1hp@js%--k2-cFj$(J~9G2@6(?d)phNM)w*LqaB%MO=fgO0;;-0Y6Ky3xAu~#ED&6$)4^<L"
    "0a7I&(L~MPS8Rx`WNgXrxQub;p(146Q^;8GIAXe;nH!rnQpRsE9EPRArWANDK1ctrUBBk_XadgqHZ)qpPlAPPXc1"
    "T;tL}H;LNM*OQ{qR`U%SJ9wA!Avvf(vK;EIOp{7_YG8#-)ec}A<N1QXl=tKqKVNv&2PDDl0d)hs{8jrkFIk978?="
    "Lnm)<w>#xUF^((Y8*wx9W7?(h|KNAY~Z_r#x(KTzo#-;`s+^Ib}grEiIfo0iZCqKF5Oy$j!lU8RrpzLc%YMFQQ<p"
    "(9M6f0ybbNy!gyDf(DXoH7rNhB#ygZU8d<VK?~iAlskuPE0yGlsY92lOStm)MEZ+-ZSswmXlc7lP{HsZof{$&Ba8"
    "kGTX{sHQ!pR?Pg|Rh1H-j+UTevSZV@UV#4oqzu=W}$JAoqEFuy@u%#D|whiO?0PSrzMbsC;kqwo@{;%BTDjnO%k*"
    "=gYL(2SE<YBNNQ4u8gzeG)Yab&X*BQKpcyV%EC{wCAhtL`SR%+K1ynxE3xkF9`5bGevLBg0K-qlW8$M%`BaSaJ^S"
    "Z3I}uudK$b6hhrhjkowDh)qz}5!_%=H@Oj;lu@4oye=uIcT9UdMW{tSQp^!jKI#pSk#k@{sb)G8S`NmcK94}UxS_"
    "3-E)hYuqgf(L4JXd3GA{^3g=;;H;~mmxM@U-fs@+Q9H?zl`mM&@u&EUGWXsVv=((uv=IgVPLF6$AMI3c2gP*L@o+"
    "Ij$F>5L(mhgPD~yq(&{>Hr+Zk^n*r0XjyP#{)E7FqGtwBXZ<+j_94i32VqRX;Fv``!1<Dh(y{sDN*w`LG-Zq<->f"
    "xW{H5HRnR*Y{D<OjHlw2ZUf!7Ll+<sffOSL1OpYF!rapXpU!tqu<#w$%brNp7Gs*d&+8ga6U1S{=g2oF%@kxJ!&T"
    "!41=BqlZMPyx!=~`$1NXP@^$HUX1uMYtx;#<%Zdfz%9ETo5zon@1H+2Ntou6mGz)#v*_4p2r5jw6C!b1>F#ER(k9"
    "02#~{<Qx7}`k335@rsD-=73<c)z;t!FqmDqve&wq%h%zGFa0TQ53cTSH9Milv`PMjcZikn-CSrrSO<CK@1zIuW{z"
    "V>!(kD+T|YIVW|Em?=gk@Avug4IbAk3@HqC92f;7s!A_jA{M}XtRO~JNwkq54tL@g4q7T3#b~nj<1)1i)s*iAOtw"
    "R?=Z&iBCV>Z_ZJP~a+V(}vVp=7j~<bzkJJm1NR#HtIg3y;Ee}h|AII>r8j7g1(gAc9YTB$!<8$21ikK6KE?1+SMB"
    "^eiG{Xk5GF>HEgT2@G9u26hiwtd~dK5rSjA30rf;$eQX9Z<?*F3Kom+Krh6JfT53U<XZbDMsFyrr>bc=9;bz_6;q"
    "6^-r~A4G}sX#<672(mHWRW|&9QqvH$_Q}ixggKh#RPr`MFud7l1otk;bAx=EEegbm&liKD6wxawi%*Gml}vKXZ!5"
    "~l-N9ejrwWtDQ>kr!JKIammwakr5trEpSXdRXl+|)J@xZJ8fkEsqbDRs_0Blq3+jc1yC_>I%(<3fC@b&-`Wdb(}{"
    "GKWW`xHeB%Ya8g9;xjR*1DK_gCy8SBkDmvR~X8<=wHKKS<_81G&n5`C0N(4oQ=JA9q1R0z%dipE*KuCcszK);=th"
    "hO9Yj_CwN%&vE^(wHj_Dn4=Me-$KfDOtS>B-Zo(EKUv+6SM2oziXnG!b?_E~$0_#N%3AKk%XpxYpt6ic_nvW5c8R"
    "X3CVh9(aR{^1l!PUVO=$3=QHRTLU*~+aa+fSaS$q#ArE$r%lWAc3u_PRzlX-PnIia;01W2_cv>jVa^I+ORqV!_Dc"
    "{n>jr#emg(=q6ZNH<hXY<$^$_`uMK0*D@V@UY}{cB&P&!ys`(Y_3wO&LjPwcx(V21hSJ*G2e<B{aI#2*$i{?t6+C"
    "@#`1?Kx1}FdQpB??We;7PV+8r<{+6n%H6ejD$H<{-&derQ1v`Qnt?EGt1ObOTvZD}367V_I3e}IjL4(!{v57?Up-"
    "#HQ3QXHezHV`=)lGJl|q(j$p|BAszt?HOLzCIfYF3=dnjd4k0B3={l75v=}uPf;bwkF8MVQ>7c^oPJ^O}+WmU`#S"
    "H=3YxdBmJ)BjuX3N;e97~lzelhA3ncElXmJM+ZUKGTec4k|7>@rRK46hE}S`ljK?1=$V14+TW<U{V#=j{PgHINdB"
    "C{O$hl?j`L;m^?cw&O_*FY~Dmc_9p0n)1!S;wao%KhUnBUXeqxyblcY6b|-;cA&<uFV76rk1hZ>pA&40w&N^K87l"
    "rq%ZoQ@-~Ol6Hxs!b$5zRmp{c3E`ydS<<{Wx`G1I%eOYWl#C=}y8b943eS*30epm*4R6+8^L)-l4G^s53Ri!cV|`"
    "I~<nRngRSSM%k1Cx2r!U6iOkwiF=0-T<XAU$3*)JSbeGa!PKk~c00As%4KDu5e%hhGsG%%XV1S6CdTA2@Ka|w$(S"
    "vx?j#*4w!(-I9~c+-^02Uu{fmsC5@6MhW8kf3y@wd*VJ#gmpE1-q=Nzi=j#!bmYq8`<fmy5%UY9b>o|{Ggju6vRC"
    "f-zn?#!YpxR1vGm_yKAG@@ril|J=3XwAE><63#73KEzGXR+peFb{NGD==aT@2$T{IRD(yRhnt6kEr)Ml#E-lyD*<"
    "SzbrFk~_#)`u&?c#|sHxZ8_j2cINjIR1-W0G1n8+W+TKy|t-I-1T5MOoA&D%&F|0GkQpd=@2YAe18weGJhE(rfK>"
    "fPrs!3^B|b8g_8xrEbE@-t}yN;)G+^2KB|yil^gRj6oCci9+9{fLgDSGt6xqCX6-Mr!)Zcq4q?&j@)2xAAIo1!GK"
    "C!QWoAmUJd9*_|Fv@SYo6x>*KwfF6=%jd|p(*h<OeHtm!S?`)vQFi~82N;d`I$-?yFx8oPo^h+yGu7a%L>m$yN%z"
    "Z}!}sN_dberK!2An$!Aha3AwE+N0SD`O~ElSzhT?tRA82a*BdIf^uJ1whk5Fe>)_Fl+_}p|EV}a_O*Oo{>kp$!GK"
    "TZN)rfm!Mia)XEFche#S7HYePYx=J=^4(py{@h(pLODoo1L(u9wTrCuV3J2LjRMO8}Y0>%WcI(H_e5Ac^V|7pYSa"
    "tPI<JHlv4$D*K6GCO}ebz5QZtUo{eVJseET08J4giA(=rL;f<`-VJjcp{Pg}Szp*Kn{NqFT7b%&{pJdm6Cx!n!I2"
    "UWmzrEKX^%Dz%C=h^Aw>yM|Z!;=bUx42B!WxvfRG{V6N+H$0d<0ziCOplOFt8*2?FcLa!K^BU1u9(bdc5PJ0){G!"
    "_Xc-)}x_kktMQQ$Q%md_YixKDOC)ZEVq+Vt&oZ})8X_0i9kF<_MUufJ{mzI%A^`t`0S&f$kAY4Sb%`>o-+(thukq"
    "rIcoyJ!0<X9~Z^k2~-XsOwL+!-r1KPImwC)BefHKl!QC<s$nC7Y!?8JEYz}`R-eM$^-UvusmC4aa39p{|PM=Ov|n"
    "qh&<3gaChKOmjwZj`=|l|5U+hl1Us~RzY3sWi>X4v!oVG1*qs2C^yMNOEp6ZatF(KKoC%;#mkM>-w^*lWz$tUk@$"
    "u5~u&u~`<5UUYGjqV`a&WUGGjjVba^QUA3udPw;Z#VC0>!1Kl+AoeaQB|=<}RsZCPW8syOg`eDvao0o{cF&r6i3<"
    "Hf8rKy8g60af=2JtK4;`A|o4B`e>@krFDg9IBy@2q7EOKGSLD-Ts96J1BEFy0AGK*z5VqHReceO9ZO2=p;m}DuW?"
    "8o<a=FJwe7}AFw}K*KRIjRQT(?3RRW)a`EOV)u0DO?A@;B(g+Ctx>w`nS9i4Q`pP5_3O&%wEUf`QE+yd@fb4^qbd"
    ">@%oJ{3efdzEx8gf)UAaCA$n-*s6tJwg>Me6R)&*_Epm1lB%mPY8pxAtakrck!Y~3M$T1m}t1d2EyT=?hZ5oBdx}"
    "`5)^wWo{z2-81SrP!p$H4)ztrb*sg)hOlIQdig(}8HC9w;_h?4s#ILEX)Nv!ac8`xwj(*43F%u`1noz6sp@|BH;i"
    "g<88(z<vV!ZnRUZs%9wEo_E_z+J94X=k$e+s-p-_)F6tsSL>x$2MSMKoJ2C_$k?NWfU(W+eOq*ku?_P<{D!YLdI0"
    "U4Tz)C1K?>P{y^bqSbiKM3=m%9leUtXjq=wGi1QDvCv325I^eiPBax+>JOptOqEJ-V^Dr8ALGrC8{~#WlfZ1Xl;U"
    "RafnuOgoddaDdh~%C?p|(597ME*^kNR8U6Yp1#I59a-;zkluQ^wnGQGx-C&D`DvM9_~QM%fqWYt*7biP{5ki*|AD"
    "G=SHQjv*?g4vJfk^BFm(kCNK3~qRMKmT;>jqBFkUPBZ)WpDZ7XY<}-;jc#lIjvTZ_wDd~R~~P-ZML-}xc7R7GVPQ"
    "L82S5`UgPA#Qfp<Pw3D%gdz#lr!_l0L9L=I_4j;#C!xnfB;He-Df5;QBDi>8a@HY8>u4YRY_AwCKQ2^<gIj-GXj~"
    "-Efl_j<D@?D<>m6lHAaiwwch=;pL>Wv>w@$zK%)meNSv3l;^)V?y=szzSx{)ily6XOg+|IuKEg}fD;8QGX`AmC-U"
    "i%Ae28K}6zJgS+A&W;RzvBOHMrtS@<Ww}CSQm<#T_sMFWWI8V5m<k2}v%=*tN)DXZgjS_GdVg~TKN`fBdkvqCh2B"
    "$@8SIhw4<^8;hm<=*bvZpVT@ErqSaNofLxfE+)#J<RynNmh{7A2iOzOfjKv@yAMO!je7q{cw#vjO{N*jfIJH?cDa"
    "N7}snG|f%YKdc~l$vW|e7VpN2&umoL_(mYG(jr|tZ2C!4B(;BYAnRlV9D}@L~2j7#lobk6Smj_=sU?_hkGhDIr;s"
    "dYBXb801o*yT1^Ln$#bq6WFQ^MtZ@1|LB9cZ<@_q(qGU&+$WOe9&ov)k^h}<xTPuX{zt2f~gMThi*7|&FJ(`hG_y"
    "UZ4i<jUG!ygFwK;3kd!;d7P!P%x}kb)V&&v^U0YCu>PQx7E7Oa61NWcwQN=lQ4ve!7@YZT3C@V_o}t!gj%8ZMLs-"
    "bWOmw-cpi!Ys;v!UJ1`pOL;rP8m8WN_C5)I@UD;c9gtQ<(XN3v-NAr$G)&_yh=1=Wifo7?tRsu#2TM!Cd7!77ewY"
    "-Ey$H4b7U{wzi4~`0Cy_*TC)6E2_wJq=XM^`;qF8&*4F)eq{?B6z(mzhnYc*J*g9C#Eg`LO@DL#X~7ql5=w;f`Q<"
    "Lrhi&$<1fydz9Xd6K<BPk;UGg}tOu)*p>gCucX1)JIU8yW^V1?#^ME@vOJPt!eS+aLB?hDngM(>_Brr3l|Sw`n<D"
    "i&Vex=R^LXc2$VP7;&*ATpSxz`>@pwwneWscTp8#$$jU)cc;%MNz&h;v$H1qp<TLei+f6>}FmB1So+fWs+uPaG&$"
    "8VfKgP{}+tJVe7k=z+KN;TQ!gtJ)(@7NkFlmZ2?VD4Mk~U9Dk)YZR(VVfJVNYEh2KQ@5!5`}eAq(QU;;4JiaDBvn"
    "nbYgpoDA;XWw?tHG)X<^o{}4nO34m#yzNuI@uZISYy^{<Uajy}KorXbvKLCmQ+e|Mu~0bt4tBl-HlS3~Wv8qPk9;"
    "`FYgm4~R|x73L(PIxiqt3NRiRj5ZC@mojb~Wu2=#f^ct^!2^d-@6ba+rf_~q>En5Wfo=k2Olnrq>rgy-XtBh%DK8"
    "BPmv6W9Ix$Zc9oa?sSJjF|hD>{B0pn6H-o<?KCdaPlIkmmDP;<5KFnjN%AOR{F?bRKF5wfC*+L<9XwMu`DTi<z`G"
    "V+&*q**fJyEE*e}~1^a{3g#5wz6ofyF2Q?BG>5CKR?x!qxjtpws8nbR1k~c<R8}7yJU)d~wxRS>?4X%3;eJe{G7n"
    "SrZ_l4bP!D~ZN$z^J(tFA=>1t$0nC7E8J93@(3x9oks^|M+eSjT-o1?=pEDeC0zb>xt42vxwEeDFk6+E9TLNz0X8"
    "szx(2ySVEC<~)-7aO-n%TbsKs)M!#lOaPXpQN3TqSYllJL)z5dRk`D4!1|}F_QCqL5yRm|9jisLW)ipS0WJ){j6v"
    "43XOOl<JC08gnm)H8rq6Gx0|h~A6FOG$7q}E;3>PnY!06xhnzI;n2+&cOk5{FsZRs0-D4?cf@yIxnUeuVur?Vs*a"
    "!PMDc0@H&kvk1JpwyJO*UUMEv{JJqxWBMkG3gxc39oV~?s6S}!hp`fY&9OzGxG(4{BUA0J^w;4Tl$KW=N#qRvg3)"
    "VMr6Zq@S#?D2C)eKHm;A0TQYgTt&CqgN(cTze<UPj)2xT&*P$s~RAfbW_$S_K(Gy(-vWbW=`t9M&nkL|88s<Eq4I"
    "^1hf|?u6az>WK35n_;B4BV!LXj9<DwwDchcp8(j0Mic7G~qK?D2<Gdaf()JeZBgY^4X`@zTV&AT!6o+DxL1q`ur1"
    "EiO^vUmHAghkDO}Imk1}q@kBG9`a%b=jM~;pRgEV;BQNVyzaV0C;YchBBAvC8bkF<PG`dJ|M-f~n??w6Ow!j63S#"
    "@1(5=1#08+6VP_?s%0)C}LK)4v4Q{RfIVE@&29{RtS;lk1Z64p21ip>}<Loxm`h-+kaC~VEUW}Ba%U!>F+^;ZtG?"
    "wY6Sog#ctUlkM@LlxdXUguMRi#8F17L|=K&t23_TnBPF%)%5XGJSsucD!`KZZrWv>;2ujUlsI>BcCWusZ$IC^mwP"
    "(AKgUc0LI&hIuG2Amar9AolAJxdmnb*D02}fUDY&(r5towS=Pl7RME1;WkF3yQkQ0e-BEQF;Wa94)cSIP1}4c=GB"
    "53m^g;!vBhZYC7+Nsx#&8P@ct}GyP$eVUw^Q+G$;!DwI|J#z4#TM@Hj}VQ2G-bYf@|`i<r^^S1Y*X8l=y}Kzncc8"
    "<Yx%Nn!u(AFxn@CY7f2IH7oR&<X}p*BL-B^HoLl7<X7bPP%IQsy0Q;aa8=RbO(Ar=r8@DRtNU0Tjd%2rOXx<^5f@"
    "0bq~&jr8gIFW$AZ$fj7w@jNpy&WVRC>7e~=iFw-0Ws2dRkzM7Tixxv88PB@YJc6Bc4{44k?L-hQm$M{l?-$m$Agx"
    ")>LN$(v!Y^OE6D2l)d3!eSO=5JBcLVAFqcEc#GVEMLU>#Vo4~gg~4{zTk?N{}L$ZEA0OACK>UCf<~2(ohZbdv#TU"
    "z>c7rKp$DT!p!We1wrnu<EGHZ8wJ^3va~Tx(>MjJZ55>@Zg2NP_t$&0#K>%+bLB-;%Dy4&hG#b1v{@D&ga;*?=*y"
    "vP_3H(HHpZWHkt(Jq?Bs7J`<DNpB@DcSu@cj>qAuDL~f!7DVNaYK@3yogZ3Ee{BvbvXVH}tUkgV?3}FB_X01Ks4Y"
    "+==ec<Zr&d;IlV^d>k1AZ>`O^l{cuUSwMXtLT@<saI$CP5136?T1x@qW|by-wNqmjIJzX(;mLl+%cAhU6cPYifi)"
    "R<+ZL^QljXofeOZ7R5bBql{y22%23r}({9ra=ga?M}c9Ku?kDOkLMsrgGh6LX`uA$^*_ToMU{|cZZ$Sl#}|8(e5>"
    "8Jq{MA~u2KYpPn8o^oUVG5X`zixd91Pq6-*LFz%>Wb3kD<A}{*LC6qV4{wmayRA1e|7^ED*7DdmzP~pUgKTgSuxV"
    "fXaO1}Z-R)vMKP;=@ndjgfUjYuNvG2xzhS#k+2SeSF!&GPdOVw!1N4Bmj=yur!xU6Wq6NYZ3`qX77C{_Sx@M1G(2"
    "?v)kWzQHQ*WX@_k)j;Fp>btie3{3tT}l3df${X@`a&$1X4b_$CXweM~UX11u+i~IB+(ZWJng`iezE{*~qDjmt@0i"
    "zC=-MN&!F_AC;{#A4}3W#^Xe86HGv!=IC(gkW;3_cq5EXfCsM@*#x1+iF45mxH^E}@^RJEQBgG$zAZVzvX~d_8QP"
    "#k;JScWQz~%46f$8wCUdg_u{_7I!y<bVf<wG)O|nn)NAF3M)1wE7o+Q`LQz8j{#)?LC5i2jQ+{O1$NNbf*e&x<Z<"
    "Ihj(erA=kx)61}ZJ=eUhAF#=#a4DpG^m-%QG1a<x&{sNE{RU}!iU}z23D|HvkzH89J^0CNkg>)53f=>Q8?E%4`Fr"
    "z@DWXf5M^PlT3QcQ%{X+rk=mOp_>KUvD`2b$ce)h7QWljd+>b2mD5a6`JYokm@PRG3?iR3&(;+HZ!L&)f+~AxUoT"
    "lc3CPwIv$(&a?remO}3=!<XQD+(z?x>c*3CM74>(i7Ol7n_AUX&Vd$2r}b%=JlZu0Tzym9^1KxpBihH=Ya@z+qfj"
    "V6P>L*uM~#WicQXf+2%(xD0a3<;$d0#r-GCg7KQkvV1`;xfI?;bK7H?n`LJ$>hl6Fe`o`8hbb^v!s#r=WAYtRJ8O"
    "=(_q13A*z8*1w%)|jdbZtYckw7ti-+$hhNP%;Q2I0VLM!xf$WyMPPlSu0*4@3&FjW8ol}FDrBkPjWHcjN9nB2{}e"
    "vHDzvr*hlS)(Mb_eCbMPE%6LLZB=)%cc6rrh1dTgE2&(tUq^>Tz59KO5qK}<NS}_L$WtKOkN+nIe;FjS&>H4O*cp"
    "xG9$-hV{~(;`d-|Tu3j-mGYw3UsjFG_oWI*am=}iqRZ=lc4qT7<s3gP%314eRYeGmVvYB3|p=wsNKW2{=hnad#tL"
    "Y3cQOtSCM<dNG6KHApZEl3&z+8!{w03llgrh(ciOn-&9z{$m0Yx*D#=;NC&EkjA56;r~+$}QgJ5gGGT#^%lMEN!@"
    "B5Hdh8B*djWWeVf$Z=64I7R}D%@R>I(}WzlZZ=Z@t3iP-l~g7hqh(4q^{STvzp6n;kj-yW-hzZB#Z9ijAa6t{=8("
    "OO(|qq>@HT3(Tb?U2s3_YA_9yR*x`?IEb>nQ#7QA~pSZ!KZ#nD;&U~@8IUB+0&CtLfBCS<HHNn;(D-HlwDw;@||f"
    "{w|cP9aPqNg_%8bRb4pbSqHP#a_)Mxu}0fy-~R_6>GDUEZgc>0=38`X;mI&2`|L>(vW?%Q5Kfbzct@{?RjlS@Rxw"
    "@PNJ4eEt{{!q?b?O)oe|OYkRK_@cfaxxEeN}CXbwcV_PXYvgCg!>oUunZW|KUH2~S;wyHiUmR?nVa!f_*X%{h4bQ"
    "^_7W@LT2Kw2KZ<#&g|XFaZat?|gkch?Eyizje0hS6<W%Q{2hw62}4gkxIUuj)A)70bgXg7SIw#tbK&L>D+Lq-d+^"
    ")mS^$)Y`kP-&5<e>mq9|rM4UiR-W_nmJth6P*^eOEg-?f@q&0~Gx*NXB=p9i!Cma`X{;b$-V^vsgR{EvEJC<8l(u"
    "w-_td3v_j3@Y;l^eBG`uKil$57f_ik-t3xC89mP}Ke6;9m`%z@-Pf0OF0;<4zss1X8vss#<nYH{@D`1L;5zYga_I"
    "Usdax!qjVcqF>BW97xw<kWj7sYas8>vCmzqTclz6gG%tDATkt(4qReb@`)Vo+vVhFQ-s6f7Z!np#`o@o8?!?f;L|"
    "4nnJR%%3802`RbQA+_&hUZk8jn&Nm);CskPz7zt9lYKw(yt9WLNG;D>fIGe`8&HR{b8?U=;oc8*X;f~qxbTFHfA<"
    "&I>M;0>|TQAlCV{12uRa8>YqsFB5)*}<tQe!~cNX2nXOSZo+z12fIRBb~F)5c~cLrKvYYR9z)J0%7_MYYpK1_%xy"
    "!<#hnOAheyjTL5L&1)D_54(}yW(&cx=6DcUYNR0M*|=}?GVLT3i4Az04$fJXwU(v3-w+@@^*4@azPD-6ngOFe?q)"
    "3>=4wY7Ifxns)C<Dk>+j{N888TD{eRed7xuP|WKH<5V4Sm`$ONP%$8j=rI5UdGIGV_kjwC0OV{0LZgd~h9l0%SIG"
    "#>r;uj|$~G(b?6XU}tX=j_HJE{(oaS65e8y>FGt(8wbq6Yk3|xn^;tnBO}m`fCT(JVs#6%F3+_%={wSb^|9hHYKD"
    "t+|JdO9nZR~y(OTn;B+uE%W}Au;!eYT({M{u!OvZ<?A|@u7Ots?z}H76+e&<RDIbayI`V;{SBmR=R^HRwA#zP(UB"
    "k}*lhf_}u=oFdRXmHeSX8~t6>Aw+W*i-!;2#dceuMESjYVJ_s>cAb)GwxxZ$7f3*)G$3Y!3u2sT_id{ZIN&NstZ("
    "AK!IDP|6SPCLM*wr&`&HShPn^tnmWc9zSOW0iSdvHJ>UJ(M6`rUCb_r6!r9dCVni{54Z#6U?PfnWyU)ffVL(G3=~"
    "~C;!6|fVPuy@_p@#+*YRNI@Rz~yZ^^IQd#6=FU5|BYv~qg?_t0=&7(Lsjg;8%>0|K6hCb<QwyC4+Se|H7fEPWJMg"
    "|6sa%4dp$!|o2Ik^y@E9M0ls#d}HQA#+*ha5|ua4@rCxWL~Kx+1Y*?B0B%LK?=D4!x^ML{JM*?zD9nj1p?PJdzAD"
    "&fnFxG3KahX5=9*-`Ol0fFK{t?6PWsUj{%+z?_7b4qVVw0GkAQQ^`ALt=!kykp&I8@o1Ro_)p~WBL2T%pAVy{Yg0"
    "j4Tg@FkSNR252K~Gu+jS-w1TD8PU=d4)G;(XTwOXcR-c&X*RPbE%j;HY^yA>L@+KK`^mN=6uR7c`A%zuRv;jPf)6"
    "b%gA~dp2ShtOQgV;C$6rE1%$Dy{3;VGNQG{_YJ7h@&b0Ax>rbc)GqpuZGuGd(g!A`vst@8?b}q6{RCZc9um1pl-#"
    "I!cma=ZZ90!Y4dF6EM>#j#6I^V`m()Dxx%va!0IX;7pKlJ4oxyzyNa+otha2i|8K%5|)tIEPQcValoUiy(&a*K`Y"
    "T<(f5EsA5V!xKT{#+6Z0v>O9Mtb*RIl9Ug?!7Gx1@_1+T_+((BCLJgyZ|c+v<KJDMjayepq<b9y>11_ZaLG{c^Re"
    "OYF`+&&W6&n9)_Ram5U4YNoBdAhh1DeDc0`B)z!40X4JpUGJ{%RHp^zrm`9X#0ms|o)zp)k4$($PR8LdS1mGl~g>"
    "+iT8j;!*A`78IA+q9ivMfmc<921dnP+(OmsuH?pb6==kBAArci*XNKdjTcN^WcWDKCrfc<@)%O4k+0uhjJlkv02W"
    "!-+C#)T)w?+YRx^vvGcfSaL`YMBS^U;-w=qXxYL-xtteS8MhgCT`V;1VCZ7=xW!TT%lMYh<NdaqDizTv`s?A|fiE"
    "++0rrQ(gXmW6CHkYTgx8Z~jk?t4ghk9Jb7y&5C)Bam<e)?zCJu7d%y8U|^_b|ul@D6_EWu;Um2U)3Iaj@8>!b3+2"
    "=CcqZ4zq*9^2YB4(iI-(RA&~R3SHJj|5kunh6nPlwaa526iAfWIm&)Tup8=4-eEbRMhI#vYN@_;wB0Vuk-PUGmK;"
    "d(@ioZ{MphMYU_x*1QV@AxrfIOEN%W{U^%4o;dRYJ($QY5fLSWzkY04M!>XRBNh4xCk)$KmDj_@xP_2E_h|(Op$k"
    "DhIw#L5xuy+GAsiR-@<*<veIl~O{$^;cmi$_1!N+Q;hT$*NaS?dJl+g<hz0gwK_022)gTr6Y%3DYUuS5CYUQ8$`k"
    "BFn9!v%iqWmuds{tketSNmUeyl;Ds1vdfkp&?YZsmo*^NqWlhiexxm!0$_nA1Q0Zt$}t8yCW%P*2GNPdzlmPx2UB"
    "Y?C`9vh43?arS##?zT<)r-Rx7<w^|6q$Y=u_78N4~8bxw<H1p$syZ4P%0XS{73zd1M{;u=X`Yh!*uYt$97`yPQT)"
    "Q7Ys1RUEXp|#adU<I|nE$Z<W8YlQQN$k^j5gV~p`#20RV4~Uj=FhLTV)7IAH+GGAB!UUsY+8(T@!gFI`~t_g4G{L"
    "wRzF@rC)RZGat(R%ij$STaUa4U-eIqVjYIr75O_V}s(%j<uL<ZW-gdUm!l?MYwUI9wmv%)$M0KPi#_FIw-Px+m@8"
    "s|u@f;=iPVE~ewx=9I1!-t*7*)Im-Q6l8CJR-RKG3HS3}KIf3>bv$QMTJlP~(~pQ39=_Thz%nl&MCzDUSjwVIt*i"
    "9=6YLK&KZv<0h;EcB(pEIXpUo{o}oJB<Gl*wFhz~JIrc)-~9O)kPF(s@F@sN_LAsDrK|!G(oN76Y2=QTzM+<LhL+"
    "he=+gAofw#M=nR|}6L~@<TV`2rD^LrqhpkWii@W!R`W~w&W(@8#Td5@_Fs<zmsB!<-}mlbuq?=hT;jkX<B#E{o<="
    "0LA==2zCzx}<@tUw{VX6;QC^wQT03APqs=xq`qk%#MzWa<+3q`BCSXaG>isy9D1W>o1Bvl#Vd4_7xct9?#;+1eHc"
    "w94uh&eM+`Ba=B4h=bM&J4JW6#e&SFzV^+fpXx%DP0(~sE<lf_rM*&3tVp!z;bA5mq(B31Qe6A}*3Abfty6`9ZW$"
    "(nJPSi#vi_WFtdqBj#5g6~r+ci=$$T$H-Tig|k?4~BpQ!nB@JE6&(F_DB2QnEW--I~1W-=<6B0wgRxCWp9K-bsu>"
    "0SJQn7$Yoy_rXjQtARvH5W6k0bf{mzSvc*1QH;tTv$o8AOQ;&U%}hXi|9RjxtKt4lK54xG(n}G-kl_^CuiI5>?w6"
    "y=+dF-`oE5IRMyVW*ow0r*M=(XhDRa#Pjn&;$-^!{47^hB+(_dESsrb676zB`~U2;75&o_gUP_*wuka6n9Y()mzT"
    "A(^>0P0|rW&hyK{(daLLhvJ6u@$Za9;n!ZtT3VLLzHIFRiiRk#fp$`L`#7c-sKS~QHZGBCcG>*Sw(y!#(?z8e5q7"
    "9a=O7^0eVYZczflKx_oMd37{{PL37Z=8#^KNrKcO^0Vpjb4OWkSwCX9><H%9MaBCB;u}ZZc6>ImtOtfhp4%oqLeF"
    "Plw!@S6a-20diX(0ei;=UO4h-iy2AVp&!2=Sm=oC@5=E+<#>lz1shA2LQ;vIIQ>AH+@zsk#Sb<&V0yL-^LtU(@`g"
    "^#^Lz7C`u@t@>ci-mN`kMptGPv|Z{{J4*xRrIV(?bGbSl<BMpAjzGX{$q{0H3DI<E3Fkp|Bklqb?JBeQ=4_I}KrV"
    "_mW`QBR1moVe&eQ}EQGdGu(J6rXiBT#bhacUjn0t=e3Zvp&V%&;Tp3M*vA$M7?`|R?*t8@W-UNrFPYJgPMM<oy-#"
    "O9So>5cU$7eW+>b5fnC?qBA}*@rMFCPK9x)QhWAuh+Fx9_UMjEunQ363Esd6;9d{{<6<PoJt1sMV7@H2_ws*B4ea"
    "fhlWwkzO$+(R|Mn88Cn~&Nb#5^^-@OAF;$x=Qr{(Bgs|mzcS4;Fw0!pab-FCHuW+sCW)SIB?yl`Or#~GY@16dZfK"
    "l<OamzPTCt`Wm)jT<J-(*PTthRFSc+rb)(#evr+DztRjMm1{Qq4c)_&bMfs($}m4Fs&!K*tDpq4-ay0|^6^C!;<E"
    "efI_kl)r&f#$q_nFVqBnH%W(T5~)Ivj3Zs3KXx%+W_0qF(*;NvQX96OxJn9;md>A|d;;@n3USmJWrYyzoC3lC8$e"
    "%$9wcV-fn_os%P6QDR}eD`Tt1RB1}!}~{1xt5E;`k!bP{RMPy|-9N`b7O96oq-e0X$tGB{C^D)CWDNulynj~(49p"
    "_;(i>>k*)C#r!~A*MNBDS20N{}moWF4#dGsW0CEj-DY1g@!p}ihC?&fzhfC_wh;x>!Q^lSQ9wrBdZ|QJV0FpJw>r"
    "PAM)&$;PTZU1v~;hd!!WaFrO(2QA4L>9;49@rpktac<@Pd`qTC)aWdYf?;(180cSYTa@JMz{Car<$o$I*Tn7<3r<"
    "32LnBJbpAXnh3r(ZY_=QIT82onfRhj=rkLWYa+T@2v+WIM4nMd7;_s(aGO9n}a5B!ny)c@$q1#6QN&{q;fZ&QPFd"
    "Am5f)vpivSq3I{vuLsn$#r1r;yuw)icS^t~OK_&1`{CDE-nrV^!8FJVATm^%BgMIa<tZtHMq05W6Kp4)hC}nt53P"
    "C4$K+g3&<5vBX#i?bjR^F32RtNjfI*h0!1}7PQkmLzr$4p2{HgA3%-N?vf7QS5Cd&9*j<as`!xWZD&=YEB3RNZU9"
    "M!6sW}8%#xTRsoPt-)8F{utt5H-m9=DAuG$9D<Ok8_$Z=^Wsv<PluQ@1BwjJ|&5R?7Mu@hLQY~qzTGaNEd(<BF`U"
    "T21S!x1$l$5u48J$k&C<Iv`=DOL8cPcPqBnD!}64XT9Xr^40jXlMrjiy56FSSwZiKOh(ki+5YA0NW;l80IJ;C^hj"
    "Cq6f_`>eOY8~dl^##WRbR1dEbrS=UGW+=N}e~fbBaJ2?<OZdzo9()pWk#&2iqhCbZ*DsXVUKp6mQbO;hzMCy@#V6"
    ">vRS@Xz<3doCF7wDYC4VJpo$kQuo2?!HVZ59V?sq1`LIXA+Xi!L|%}>T|fx@6$R8_?tx`_bfY~8Y*sV3c#j5pswt"
    "?u!m}HVrWpnJ0*Iwrr4ZkUY6(&2(JWU>4{DpA-+;k11=LD(rVdp_)H0h7ulaC|){a|%>HIr*_Aqq`jSi){b$i5z0"
    "4)H)1fenD;jm>WJdRxpJkRR?ZeApp`Pj9z$$ieESc-medhm7*mOVlpLYduQ!f2sB0P%aUT=T&jblHTu2-{YA4+8b"
    "9dX^my$5}@SB$i-QcQB>YHAiRZ9UP8(X%|+HfGSQ9Nebiclo<8jJy|X;JKrf|)6FEr68?H}c(9v6gdUok;J3>>yR"
    ")uCFTEZ$0}H)GsT%MJwQIy;@56muk_?E30@Hko_`iE%%wJ^4gd##>ATJ&;1V84#J5!Oe##avuEo#^rQN0EZhsTG3"
    "URaHcv>rh+Omjp_0<9L=3GCvASxXlMr-F<20|!Aq8-3K}{b8<|IB~jasCxBN>D_3u1XSD5;6TX?0V<0t5#O=xW&?"
    "-j7+FMG-~l7@Q~DuuhlE=qdhSYHGU_WZS$IR_sbJuF?;}K|yxi9v;{|JI%dIt^P8X!#y~Gi<7^#7Fj4H?&!57+Q>"
    "20a!x_Xyt&?Mk88&7<(Wd{;1FcRYDESnMiYL-TSPv_+O@UYU<9taKr6LkOdOPcO*(z!+gb5#T=T1>M_FQ(kb&i01"
    "PB^}YC9xsRF>D2U5tclTGY#M}FC&Khh>3$c>*dA_LCi-vvPw3rCl0uq#g0)RN9^sPMz&<C@wJQQoIMpN%N`i6J?!"
    ")%!6CDO%A{*^hI=Jz0o)^#~wJf?P$uP!hy~sa+7l=oh-5xiolDc_kIMta7EHb#YKzJB!$gv2;u<pjuvp9OLlOb7k"
    "IF_JDTv=-@C*W|Fb7Wz+$1O0O2&w(Tv?$7QVOJ277A@&j7c7~xT4{9<AQ%A=_-ouXKnQ>wIQzIpWcqV2eB_N5I4b"
    "N8$2cLd6T62;gX8Vf!(;o*9~S5YgVelEj%`oTDeU%O+k`$M^%7N#HFP&pi{Us$&!Ey~H(9DSZlvkZcSDkEmxsHR)"
    "1UTEc*hJTD!3AJHL27_|C%;56M7CeCD*W-y+(Il0p`|4W|w!lRJM~^=ODQT{JH4MMss7VS{2=9+54{T0itx6gj96"
    "$UhTj8p1|u4u%uJ(*jnoUtA4gPnAlrEV@((MA&_5jUzpAf3U#nBP_X7=%UWUck(RSBqTpKA6~_k|onjwa>xt{m25"
    "lTYRdws?UBSC2cmej}_^0)K=wv0BYICGU&V8$+BwOAjD-rj8Pn<5*D`3Cr)&VDVJFv&olU@U`Move?k)9k@pZ$WH"
    "+~~)><CD`K8rAH2L9ev3aE;$)0tSpeE4abv#^r@t{X1$Qxq+Pgu8|YkJnNOp)n)7<FAY2i7*xO#9M!#Zm{y^OX@Y"
    "sBmo%@t#>0E$^(1rGyzDLMo=c{BVK<cC=r<_FkddtHVb4<uQ6&p-(rxdY?)@_8LGOA#C@9VGL#dV5b&RMz-mOv%I"
    "2?}usf*&zZk^IRvKW{=cTHrxHI`>7V#kt}Qpn&24TXMK<5r3ZQW&f{S_R0Z30hE91%tTQdq5?fp#MUZH@!3nAFYB"
    ";FQRYT9<}lRGnoFL!SsKf!KC9PT+b7HH1@q~!fwi5+kW&G5XsSDgMT1KCoVpSR1U3yKf2TjuxR!BW?eXQnk<FF2^"
    "jtPcJoXkMKGa2r%w6LsIJOKmGQ|=FDWCB+-=-H*RYq9PooWk$HT_310qtZNrowszuGU?#8-!<$DCXG>5Kth%u6LU"
    "iIS-jwo}x*ZCGt5(|@GB=w$!L&CM<F#DhCKrd5O7+|gxvlgk+k1Q?qYoDk|%zk3eQu8!Af*3lj5T&wPKge_Sxs*n"
    "9I-<Krx|90c)k<6Ur|A3j(><pI!0(|VLJ?o^d7-kc->6!AmJ{<5zg`B&XASenWqa#Viw=rcye;z?^E29BT8diWvc"
    "|@ub9LpnK%-B~wzsMJJ@F`7H8|SJQ#_8m03E@YR%T(E}YCALm|6a5Y47zxZ28Tv?gnviJu_*|9@sd!nC_PNTcqny"
    "emZccu{_x`-MuQJyX3Y&qmeOQ1a7cR)T91L66`Nai2Tv=YwblU7@?f4H8w9IFfgtanywAsD3Zhk49|uA!fQSWGXr"
    "u5K-LWmo7Z6B5$~fFfHEkHkFCO&pN(FqK*6%OK?wDQQ706Y`LFf{(B)^q?o~h1-q-k);GV~Epky3ooM1yS$eJ=eM"
    "lULC8bpCTjbh<i~3n^E_Z#s`s#JWi>GvWBIkxDW7B*Kj0$F34ww;~APB7T{oD`#k5D4|0|f|FHaWohubArO?vi4U"
    "i*(rNWV!yjK{Aj1-`IN8~TkO;j@!KgT?T2s|~5DtP4iIFOJJe{%kBWful?ict&nS`5h^qu<So6U9%abPEKc?&Opq"
    "b`5@O}lM9|H69cxg4K}VNO+eqAYJcqV-?@&iOhN-S)~7Q8B3JwWWRSmL_+;RzJOhm0nuhB@CKH<<7IKx!@E)Rm#<"
    "w*pxhsS7DW5-**Z#H=s(dq4;mGlmDpx`|KO)(;c|HiT_VE<7;K8w}UTQPk*f+sjTPq6gUFGfI$C#^I8VPqaAwWg?"
    "k+j2nED)0;h>ZdIwp)R}<VJ3lKCD{~<x6-ZV;}AjF~YVwo+~zQWCQdH3WLEInF4;R`9m@h;{RLKexi#WR`m7L#VL"
    "QZ8zmnj;fw0y*0*^Le2rb^bfeOZ7A##sLPa`jBx<s9G;>r_n8VF47U00TQy2Nr9@N!)yd=FN9r@LM6)qYRNEC*<A"
    "GuPI>iXiigCx`II>KSa$!J&8PkpyUxZVGA>E(PmMHfT0(YUE<ab|_eYuB2Q;DMYJndEd)b_k(U?aj^x?muV164%-"
    "@<Ow`t)xoEhtl3zReq3AYW!IESHs7TP+xi9##zfRG0<nsq?e4=PCzdXpXLqtBKO2DfBC!`5E1czV9rB<q%*hfW8Q"
    "i^<9`kvOq1TLkkbl^iW{bI)tTtrPnwQGy$LYmO*C$Oh7cDrCri^4ir6wyqWd$d-EnecxHihaMWwdMv=S3Gl4rVK^"
    "m9-GcVa1C0|y}9Yl;|l6qWtB&G&zlOCBU(Z2{Bzi;O$-~+lGHf3zKd){}c39aGnwql#^zQ+bssrc$FR@vy|<h60x"
    "_(SmLH*f>^Vm}TIf7MWWzqm(GB)E@1SHQN75Q1nZpTsm0h=-UeRid<<iCUaxi=3kOOK5J?TF6tUKU8%%El&FcnLW"
    "NAeMDymSzq`kXA(?jcdb#D&B$PL+$A&+_qv0iX;PwYD?PNLv8Y&)1pypX7*7T(2x5X_0TeF-K{gjzWJO<F+4T5Wn"
    "5$DN^%ft(MuM==uV6cM9{Xi}wVZ2)Pmnm>#J5zAu-;*hOd-F-bV6H$krtdyZRmE2OiM=3gF(?_icTqr&26!mwL@u"
    "PvLouAW?*=$o!&0SUaD!?nyKAL{yPx8EQfa;vz6=sJ&syDM)G7Yl!5{&&TRF!VDHn?kGIj`Kw9M=%0$nC^p_9ib8"
    "n>Av`uexTdx}aVl9gJGCq`m!lHU6cc=s2RL{n46uU)b_7MPJ^97~mFpO)DT6N!U*Ob=&Y^~J3ZmjHdl7O>5*1hjI"
    "y>&0tvD?!pN+4HFd#i5v-#z&Vj<)st>5?!Pab}}+<j(UN8Tvvka+49J{X}|4t<0)|46SZ7;Q6ICRgAEYk@Xlv$f}"
    "K$&Sx7zv*jnn+d@K4)~7LNRL6v(7E_fTGJ;zTMzH?2tzT8wVKvu!L)x^~OC>bZ2CsWGg>|KA62et<gR3xgYDCxpp"
    "eXZ%1>UeoF>MM=wPV%G0oco^Mtu@YQ+9v}Q)Bo!_#y`0K`q~xO1&*%Gm=b-R^6+(iGnIR#(C|>G~`Ju{IwGD3792"
    "Mqpf6fb5s6*AccPgL`5m@KI*%sa4X0x<O4?#RD_KVVR#%}>c&8E**KjQ!8)2O*9vedutUP~M%e_l#CV^^6c`|Rqu"
    "3zJHYzD<{uqH(hbz8517WWLOqLt}8~lKN)`bTRH&Q@m40sOr9HODOE%H6SZkVatA?kCE1MgRZhR6}s$jafa$X;JX"
    "kDX_fZAkX5WyHW<pv1-M>=LN;72N|zst+g=Rh9T*+tmcMbJ`^lXVzV7ST7<r7oxp)8<#1Et(0!kiizer!j7s(d8@"
    "8RK8ea2*Q-l&WHzedWQw%O3FVQjuAL1J_YiJPd-~oqV%M}m`NI5~>$X{t`;&2ckr9{x<u!CE6S=1OKv24TwwVt9_"
    "3qsT`U?N7{(v8EPImdP(R4gc=Xe9Oor{kEU)iguaQHH8?rUm<hj^qYRhJ^hglWhe6TJV0=NkW-fxH)g{@1(Z=4Se"
    "JMb+Bv2S|{nJV(faHLwXBd;(mZTVJ-r<b##NT!nJ(X_C##252+j24_~0$Z?4Ua>07YGe}nIt(B_iK41+@Z)+IsYk"
    ";6Ecn*TmSYfuOqja{Q<{4syb~i~t-y})A>U~m6JxE$g7mK+VExHOZdRp2m%b91XQYrv6m0ezPN_wfx@<XeOzmMxW"
    "3dRP(h+&LY>@Xm#V4N@X<M^xgdE=(x414}nUnL4|MucQMqOW_{fW3gmrWd-1Mpv|IJp=^Gl9-HjXZ5e|1%sf1t7n"
    "UCP4wljluCo$B1+%FBdL4aHXrrN=3?!;4*IM0x^um@gb+@ORZ!@5eR-@8v^2?u`gU9xiWt)OwU4`iIxA)%X)^~zh"
    "c2z0yCqDXRWH;4_N;yvvKa_s!NS-<I#M|U97*8GOGvUCFGp%y0?`Gk%l@&<3!ab<))|YrS`siU$ChH*nbp173@@o"
    ">S!jn*fFMCn_LLf%3KRlhHyb5!=0j+=YY3Q>F1bmes?VH*R+EJh5GtRS@t3{9uZUQ-cZ>j7CAb~Kc~S~~6_6w->n"
    "+ZgaXMi{4~lc3qrl|^MjcmL0ukZs7l2==1_i)+IDDy$Y^P*5ODSNXr?n~Atr#9Lo%~(Uou-!@HpOJZUI5i>c`?q1"
    "RJ(Ervg&kg`)jxb=t4*~hR8KOF$Pd6_yh-SFexWC3jBqpA#T>)y)!_1LLrJz5gsKo|BB@eONmhyXK$@X%0g2C239"
    "%$*PF9N(@E<#yT}&cR4pu+G4_qPkC~cmled*1Y1m~;M?~(S4%Si}xWs}ZfqJU0Zj8Vf{kIg{5Y2R$my(j}yg)z@J"
    "f)klBp}AQK9N^d%Y9HV*!$<)ubxxpV+aCP%(2yeQ;et-NV7&xQ{X)U|0cZf*d9+$*R62ChhyKFwJjv_@3T82>gq?"
    "8CbYk=9)r6<I9@$Fg5BiIhu{;AD=hSYXRrjH37aI<(7f)18Ol}-n(y-FRRahjwH3`3xbISC*+;DI9n|(EEhMsPj$"
    "BKM!xB+?DlDBfl4+mdt|OFgXOSu)y|F`HH6$;Fd>^&VM|d*vg2hJJ)1kb08Y1#E_&S4@8uG(M_A&=CRO4upWurpy"
    "c5t=mM*C9;Y=lu~_z8>PvP>GNhb*Bbi(|NZUKoQH*?<7{PGk#a_Yh7D2+vdN5c-2s&=3W@n99lt>|=oO!}%oekR+"
    "~$!dpIGzI*Z!Cg%ney>WKmqhID5>YA`oR9Q6GthZXU&l=-vVdxFIzzQcg`}B5dag9ex>0dybAWNuVd}Ix91H0#ue"
    "e{wR96Q6{6b}k{E|wP<$_3%Za_O$9j#6$Eo%|UIe&dD#%j-rC4`<16oLk>~y>qmK7aGSJub|j1)GWD}{*F@Mpw(c"
    ")>pHx=)m*&CD51Kuf}Ff~?%I@=_eJvcchB360;o@@eG;s6gILBUh<Z4OCiGZBaAN_fd-DWhIjK5MD3nnzV$IO}32"
    "|&hU#eQe2vX2$4|NN)4mmL*?)jcn(f^PMymNb2v=doT=70!%#Mal|P3+vmOW~|-CA}!n|4v+r7P5d5@K()3c5Fh0"
    "V9U|Dk(l1PsRMPxUHm+Z)Ln&|Pw~F@L74E#hkQQ8^qT(1`x4O}#M59iLR5K7x`p?<`Q(GLI9k^C;^^aj+wMZrl0n"
    "yGGadsdH&LBX$y}d~B$fvET!oWLJI19?4UCNAOSGNI@Wz1nV7a!63LSf5QPrlC6-1_|kt<c49+l3oik?5uec9q{I"
    "4-K32WQTiVfhFi;p>qqwMA11AqGP+NoVSH^pr9okbQ<_9Vzxd-nUmlxk5GZ&9`*KbJmP|ky!zt**9P%aFjLZt5Qt"
    "ZA{b}JQfl;Zg;N~5zssR3Rc`Yq-1eSs3bA)6d!gbTRF-`nt}gznc#_qqewKe%JZ))M{f2dIHHbMwNEsucdrBNw17"
    "fDA93FX021wa!;Cq;h<!1Pl4aWr~pNQ^%9Cbj0nFtDPSRpJYI+m5uCm4I%-XY5VTsNg#L8*rNg=Q8&Vo|G}<`cEu"
    "dI}PWxsjA2Fo(!s4(3<&ds$K9WU7qx1!mV;gj7XX5qYX14#_$^u~HZDl)(OPYUe5g1ri`I4b!%B4+T!dfN0T9&V>"
    "KOwk4k;1SyBu0v^oxA{{>1#;r~3n;OFx$g0-L;(PqmJkuZglNgs?+GEAKe;$~4Cwcf3q!-h8BwC$6idBt>0w%%HL"
    "biTL0w1h`MT)=&lqswcJ|wRuxZ|W4eMWFx%w<V|x4Sp*NBNv7mMO?1CW7H)`ksG6mHM00Z#BdK=TUR=bjznXi@t)"
    "?Xv%+<nUn&ik{w>cQxIWvl}Yuijc>h!#8u-t5Y{U?3YdbqJ}tVC0N0}IX!Ywb**P9;pDK$Me+>S%voC-DdVF}W|6"
    "3ewPG9J{qtU7EUXHLBQ#OJ}UvNOS>ZL5bv4hv~LKnc5vzBU2yL9VvyezKms5AITaW@&Z<N<(JoVMDoa-nv39u@e2"
    "?KH6*4H(Kw5?jr+r_(q2H@kG~OO5kJI2+v(H8-E6BF95CmbVkJKaRc{la}daG^1*-P6e4GN>^iIJC2|6LmtTU%WM"
    "f~y!{z@R=LBN<PQC2a;=W(4!tBzNbc0vuLx`f;b<)PIG;%HDuHOsdQq@5EgTR7>S~+zfZ!3Dgox-a+M>>3jYVC8C"
    "|er_eV<%?qButzfs+{MKA7hTdVGe979&N*8~=e0$c;Alan(cI&-mQ_-s?S8vofu%J0!Kt@GG)jr1uQG58ymzAM)w"
    "4K=4OyYor~eZ7Tyx$>53Q{&YD9$&qG-Zn<=Uu|%Wyy`{$l5Nl`^o%Q9+TVAvJ_5xP)snl3zW?!;a>>?p<5+6(W(+"
    "l)BpG7_=j(*&lv-F&O!F^OaXX!!H$8<iadz?=wZ2>n63`FIE_JiKBmpr46WX)?kki7jWXKdSmm?BAVKQh~YaE<Et"
    "6mX1Um)%PQBBu#aX0{{;rHgW6kRjL;78G;*YP5v(lXkK8>PQ(}GbPN&RlZnpBvpO%BoZcMTuDuEj0WRFH*jBmhDz"
    "m-3x=Z7qF=iWw!7~TXrhxJws(FG3lCzGZJ)Y)GL0A}tgoD$ZUb>@B^i0If~DQ#7`zBl4cVI0rYu?GsIkfUV%?#lI"
    "c_S;UhoCOS%O&0mESjKLNr;!h3VPf)58~h&?thW#I53;WmY2Mx)B!Dr8Mr8%28O+`*tS+Ah3@FT-$j7cm}!S5U{Y"
    "&oLB-nLtzpc%7zu6GCX7tuPJ=Ib1;;1d!d_Tx7G$!j1Myor38Hod4|$1G^F}XmKIAy)j&}PniB@N>Clt^SgOU*hB"
    "Iz_c4XYS&XyVw%O_Xp>=NQta9ZN&0#SuQhVJzJOV$6Q>2e{C3q+z{Ow|S#u>`uG#EeinF?tk8nw@PmQirdM-46to"
    "EwqL$;}XCHbJeN;#MKN9Y_|WitLY+Vgf;6rE?p^}<f(nkyd@lv;AdBTlux=BOMt&X)KpCW!Q&4|^uK^UYP)R0kXZ"
    "~{mFQ0q9g_s9+FcA+TG4GxcV{zolP#{NBLus{XF1rNE<2t~<c_C@`{*G8=IkNMjny-yJQKIqOjmgXoJv;mi5*!_@"
    "nrr6+doOMs^`<vTD3KG@C|4%-QBjUy?IIzS0C@&25%h&_!n)HQn}AIpMBHWe9_tZ&H-5TdcYBna{7QylM1d$rBC;"
    "~@5bRniCKD)k97{qM+(rOzwhtEiwjQqy~*X2-w;L2dDITb%LltUYQu0}9sQy#kV{7jpwI2m!J0h<Cw@tI>ib4F>F"
    "omo%1!G+$TNXpQg&Tfl!?r_vB=WlH8Tt2p^G`c(YeLl6{VxM_C!Z0t_*F3vw5n<6rnFyW~LHpKsMJBaUZV^q#xyU"
    "5PIaiYP%NJi8Z}nnH}d#O{-FslxJ~Sq97Wn`FzJI{V^;Tk(2g@Q>c2n*UPNM@B9#{HRFyTX7xfvkM!*P`?nuC_m1"
    "LDHqT;fD!`SkvwJ%Ng;&P2_8Mav)Gm2}n}RC&xpx%*Of4!I_TUq!`+PQJz1Q8O(a715v<DKdweJIz!vP<W{T&F!r"
    "K+9LC&3vC#dp4b_G0rHW`0m8G*#k_6`l*}c@ttI(a2EuZj@N{SE+XNS^nGbvA{`<>%gV=lx={nPYH!U3+XbF>=u&"
    "$Bmn~^NwhvCd#0%Y$PR_c6?*|ZGti@TBkZ;g;g8C>l4gcw@*PNt#6&32lXRT<D2AYgg9t7N_$hC{Io*l%#eYA5;U"
    "^;rn&=)Am*S2~%l1WA`))0YC@`H){&PQvz+m=%LBuwPX;ZMNj@})mIT9GCC2Yx2#l<y@>}Bo(IvJ;@?<3Yu7|SoJ"
    "1QYWOQXdQ@<Son?36r-_2<US=!7WH+A^`*om2*iXMI8R4Pc_mzbr~j=e(LGd=(`sN$vNb@ee%QL)!u>rfX|t<q`Z"
    "WS@?mhWF!ucNd!BDx&Z#vk_!b<^R3rg<ysFUcWk%(?I`9U)DmM`eeRX{J<|z8%w@CK&VSZZ5V|?@~aIp<5!J^xbn"
    "mgc!({!wCx~v6J)ZlA`u0Iez)!sqW+SoZf_;K&}^<Z}+-q;=N@BN~NW_P0<aKoR%tCYnN4fa)8o6+E4w;gY)g|qZ"
    "<gD-C%9UUM3f*(Fn1??TYQh&WU`1#=Q*Ml`H5O{cHCTf_7`U05O7Hf5~2A-t+v0?^Lk3mXqxP^3>_5@f}BLk9yCB"
    "va7pz^gr<#26z@h1l+fmzWV=S*?Uup~DmiL_KlM<A^emgH`+SA`>mLXm9jR7f@+j~J02wWKp)kbnK*lf=pbK*@G)"
    "hGO?C<X_niEzTZEBu>16n9_R0RO{(ofMmU;F30u9Q>eIE(n_cxsKJ!w8_1ntd$>^wxbD4t6tk9NhJv7;4aeY6J8h"
    "kCJlNj-O{sz7HwOo#5SSo(qg_$P;qmU^7?dYty2r{+r8B-~n>94&XZ8de(!BcoJ%w!8rpHK^#Tsy(pia|qqKuuqh"
    "?QBQrsb%j;o-HUStmuigOi>0drD1>91z{JHI~Ba{@~K?DwjvnIYq7TK=i^S8>@!=Jcu?HKp?L6y1oCkVOCZ7PE8v"
    "<b&H(#>#2|$8-pr}{97AR`Uaj?lNj_jb`K8*8&xZTHi|Cu3D9j>>e8y9oPx9*o=VJSabm2=FzfjTTNIPU@qkZ|0}"
    "u{0^|31~F>KS(VI>iu<P2YLMG;`$1npKWZuiZ0?I+=|6kdF*r@PvsW9I=fOsi<4!G1ADUYeJHcME5L;F-$jk+93_"
    "S}c`Xo`b!QgcL9>xI%1ZP=%te+?#B9&`sSOUDnF_>nHCGs<DtFs0Pe4*r1y70@Y1KQek%Ksa8?<(gQ2%uM2P95dF"
    "$MG=w<Jbrl;m4PiNG8ET>RVle;E7e%%T8MZFjjJ&x*Qcq-^BN8C?eeqt8xMkRERwhq+HP13fSkYK``wSszHtRp-^"
    "KIhVj!^Y<7DaXkpaxE4y?@D~sj8yyLfgQhN>p_S8Tce{@W?S5f#mMQa!YSH=X<j0OH%ZRiCdrxdvW*!Da!QLFb3y"
    "=GX07v09L1N9-WnPTWa|Z8)FpHFB{aCI9+4eDZw~sYnPJl0Psr>4Vy;YfJOA$>P~!<^Sb%7s@7l`b(O^n;LF*Zle"
    "2MGkS<600>QHZG>-Y7D|n&6ExObxlS@Bk5}E_Iuvtzw^G>42ZFRei+Z;^brWA77DBvkB&WSREl)a%~-~k%6a*{gW"
    "c9Mt_2o5|hlczfGx+<97J8cHTC@~GD3xVE9rZ@S5aWQVv-(iCS_A={NjcgKi=ztH4X?ue~9hXSI^Y6-M{7<?)!xh"
    "krQGr?$;b$p5_z7B?<Y{GNlYZz$OIMe_+O85KS<{n$ET@aCNPKuUyy?6_n~bISTT6nXRIAzv+%=!YBaA7jn9QC~>"
    "5P9;ew$@<1+&f30!Ytpm7n$ZIFexOXjf-{ik>97Rh_(RR4^22ewmYg_y<muJ?O)CPxhC?yN$xQ5~Dz-f?DVfxeqm"
    ")*4qtBg<|Ha&7Ud}9jqDMsp6M?A{Ky7EQn<G_mK7DBW>oNK8?0EH}AXr^MApw-sUrpyHypq4f$nAqpBg@L7GLy(o"
    "Ex`^%J^ueWm&Xhorc9eaBw0%p=P+1T*R0&8oOGx{cThZqpn)UfUdQVV2Jc3JX>xV&G6+mTZpviXCtNcv?{@V+i^d"
    "nd&ObVDl^RD_EX-8mmXO9wAemzsB~cb6@P&aD|32f{<bDytA`?urq+5v~YJE>sS=MHz7^qR8j8Tdq@=C&mG&9($h"
    "CB1asco`YVHF9U&@u-#&7quv9#BJBy=i0%1EzS`72thF#YaHf&U>rgox_6fPN2A~_>f!)`$BK@hY$k`!~2Xt-rOW"
    "WF__^V3y9C^Yc6%o&I5L&}B3eJa%?@`ENX*=W=dut1!KlxQYP4xCFdaR|0mPtC<tseBMiR?A9ytW+t{$@+mUYKb3"
    "_B|%jPU(Gc2nh+TLeEBd7QPl@74a^pj4kz$fiMC#Ns^QRCI<$07=g_#qMoyw%1@6;mb=UYe*L#g52R@+8L2YeYn_"
    "uCG^{v%%pJ`)KR0K-^AI$J(6HaDyEL=b=V2e%iRax~XM=XH2!QI;};KUVkRbtF?ndTDbo%>)3Vv0E3T<Q|l;RB&8"
    "9)ZOTI@2<|?#1Or8zkO8mz0%R6&)K7C*v4iWLuxFQ%)Xz50Y>Y4G2+S$UN<O)P;IMr6YU_>&9#y>H=(>8;Qy5c*7"
    "&!Sw5b+=eo#_4OcserEEBSqt+W>6WiK~E%x5|at#0cq#6tVZncr754iLSkdZ?5aCGjSah`9`;T^f(W3vH1Cl*s6E"
    "LBzF$KFk5<8+9beLWIW;8%yrcVL!5^jS2T;^ulhy;46Y&jXH)qpvqRaNdVct8$=K5a4mK%*P94#+}Xp|7u8N!+?Z"
    ">6>pu(Vd{Ato!g<u=WV7Hd^aIhF+e3Xy$Ty$-Cd<-!>J`uvlH&gVj8`lOmE*!(hJp5*lj<kdW{Os(A1=cx<s{|!$"
    "WjBiGV{;&A&sboGDR-jJT;z)qqA6DdGWEB`Sc^Np*7^L2l_h))&Ti9fAj?2(6(Kq*jq-WzObE$e5pFeU<UWuSv=b"
    "EO85^6UKL15McD)O@Q}-B>>5UU}QP!b>|u(JEN^<{6O}5HXr7g20(2_0U!?4cN1Lir3+F-D47-nSs*WV&=OTM4AR"
    "{&3|95?p)#{a%Q><;roeodF4X*lrREMMWu(lP=-drOKabqA>UWc^ZnRCzRvm<)h?T*itT-(69Q(gGN0rpN$*<T!x"
    "b>n9<3L1Iuw>!P9Ztu~n+fBssot54RP9HY!)r|r7y;G7VLqcUU{mNfQ94F4a4ZcrKNn-QVzc#2Ox%DrnqMrLNb{M"
    "h`-Efa$!ek+cNsqYoYIl7v-U<ntb$owM2d;yeW?^!2hsV!mvp%VdR1^g{&0BuQ>30!y?9R44CiWPOowQkzMBx?R|"
    "+|6%s9qT2!fyk-NT_Gh%z?LudWx-Ik}L}BVdu5u$M!XrbY}iyWy;>80Ry9bz<slM6R*=OBmEi#lWhf^$V*VBuf^S"
    "b2WC%EWv#X7<B~>d`E`2i~l6K>Mq9B+u1%ra4qQfz1_j<qr=m|!D)0nINIO-4N~<09YF^3Wc&31ModbH7Pr9Z0tv"
    "u5K*Xt`6*=0#$#8VM_j>#Ix9I1=Z^YtGSX~iy0~7@{Eh5a*fG?>1!^tV%<_QG?$^JjJSbz>u8ZOsEoX`+MSrnprZ"
    "Vw25Pn8k-*F;`a=Wv1(R$Wv*MT{?|P9vgLI=)Ts3bkqkIuPn`Joxd=$=<=MX!}472SB!OpYHvzKZuU@_76{c5%67"
    "OEQ{{L>2SE5<)|o>fE8m%S4yF%-Mt@wR9iDO8uBbx&8@GDx{QJ_tLKQ^Ex6@${#r!m$0YjLOT>W&XAIU5{QA?Os;"
    ";FwpqQ4gT(~M2`k@|c0{}d}?kEz}eQF{D*P<#iC-njJb|_cKMN80B_<2mI$rs=RT%ZO~8Xm$X;W-?<adrvE!|LAX"
    ";mox}fb?}E+@?p$?M5hfd6-AwvGTz4k$xH+Ot?TdfFL@8VLC<|(FwNNc1$TXv^>(X`ruT;ASN6#O?DVTrlA!Who{"
    "vnw>GvBsp}+NI2VEXU9J7@v-CPfZH+Vcme}eTJBUo%*3FV~pwwa$jg~<DY5KHWa3rV37?)P=r!^CbrO4Q7d+anhF"
    "10icG|)@1!6QkUYkU-`HA+8n;|p543Ysi*-JE2W+49z2?a#<^qwNME0ub@YNBQg@=J%n!>9(xSNWG{w%+v7)M6Cl"
    "1z29@nX|}{JR0hxG$ndK_wN1$G=px!6dbj9H+R0FtZ3N1Ks2Ca}5)HD`NGYZ2<qSdyxk+wa-0h4dF;;)cn|(;}V&"
    "FW!1PQ;f9nvFWVm)gvmYz0(<UDUKdvEx{R;rP?<-J)JO6%S7P4Df}v?SCBy?fNS6=n#kfeQpwz?radg*u6A^hsQ+"
    "Py|_^N?V@rwhfDhV=`sMU1sRhW5bi}cyLpw{Te&~uXII4;>3at9GIsOnWt^FFlWlx^(_G02(fKPCIE2VD#4#v`KY"
    "j`GMl4FuT(}^q4(r`p{(xWDf~B)9W~elz6Rxn#C-m#W|f+a9hvf|6xiUvz-q9mac2TVI1@sF$|A@zu^cV%QKj?7O"
    "W4<`HEapi?&5aZv7&Lc(LB5ZW86whyO=U#HrUIZoLtkIj2SKr3Gv4oCQf!k#m$RkD;OBNE(2m1A*(UlhDTb6s;Sj"
    "v!KmWiC{N+gQW8tRkSe)dQ<kBTi=6*g%_djFDx#ZJ$DDFobJ*pfv6<dj^-ZfYsviyX(x+Cm-5fNGma*>MWRYFX)!"
    "e}lNszoOkdMymHIH@i?g?{9WA-q<(5_(2!o|FN>x<|F_MX~+5GBKlFT~k%!q_mK0CAa`=5w$!%LUN}4v}_2U66xS"
    "`!YI8zvt+WxdbpxfmII`glyil@dUL8szQvpl=nKgLy$V(UQfeKvqj~pFAz@yb{6T9P*IeKah=&#33;cCrk2!I+|7"
    "Yy6D-Imd)b#30Q$4Ql>@DaVtSLw;pzsVuXjVJ32Z0zJ)_o`vhASoR7_MUuWR=ja05H`MDSH5uljAjo1uh~ZLzYHz"
    "}WnuADF0tSym_i6~?S**y4)s*syw9u#HP=g)BUlzpg&DZQIB!igCelc;p&H`$4P+Gz03`K3zL6w2vTkxZ1r$AbnC"
    "9!sx{-2|I!ijqhGYY7x%nx6qs7;7(Un)wUpLX^Q}aV2(2zV-TUjbhbquu&OH`&-!=EeFN@C&*DepPR1o?6J$Eiuh"
    "8h>S-(-N=mP?AT3032_rp-F<X6-AUEk~|ND?72=W?9aq}Vc!Bqb460e#ARw2>ws-S)9vTCGC5T<b8+-Mq!-hrOx&"
    "%&V+jwZ~X%w@Y@tC>F~VnA!a5&>!7UQOD{Vp#9Zc%v?WSUX&eqbf6NfllI!NV=ldze#k1^bLi@uOhd&?=d<e+0s("
    "o0K+B`=x{EIEaEf2zq?i3r2|iXJuhx(#&*j2#+t&E8tEAALlC98u4d-6C6{FWd3h1}Dp9j(57FdwNf<^TGasnr5l"
    "Rt&N-ei$73zV)6tWf0*Vhva5AuKrB6|EHh9c12crBl$t5SaE-d(7*09cCGHyoc=h1#_o}<m@GdrK6@SWjRfZIfP)"
    "k8An5z%adtmI)jYZx)!d#KG?FdQB(4pf(%>kEqb0$2wD5$f&o!Z92)_it!y%?u5U??^H3)Ak>Go-0`~v|ssX4f!N"
    "B;_a;g^m=EwT2CndPA*zNfas$P>FUC2MxYr4k^JE(fi_kFGpuGe4>cpnBY<JE{KxbJFR$CVi9WXpM2T|v)h80`3x"
    "m$0Uv1Af8UZY_HPZD0k<6=+XK^~rM9xS>LD+;Tc11hn)FVzE$Z;QD$er<vXY8>7mec#J4@zW#?TH|51ec6>p{u;r"
    "O|R}n%HY8W$PN2$1hGZ<;YsteAOqbm$YGmbeWqgjUxU8R1nCg@(o5UPu83{HS~h7m(>@IQmVg{2zui~MRyu$W5gS"
    "shIQ8IZ(5(O7H8-oP32Y&o9+MnkI<;sK}KPf|bD23kbSVc-;1_RLt>HwB(sQXq1hEF;>-*pWFL>H;`9l>2a<jXz|"
    "C>{2A4h5{C>s29HtT^xY(cev`~K;rQx6@K#bn*xs1C~Rvs&5^o=Vaa585XMspLb1r^FV$+op(Iq^H5|20iF`ec&V"
    "N<I_)4|;oT7Gz3oBX#`tKHlY8G;${_vAj!n0@sLm}m$$Rym>Iy?z|rg=z$a<ahKpW`Wn%ABAWR=n<n5J@4{wTjnp"
    "gyebX!Cj84fChd+!3*Fdvj@T%A&}9VbON-`O1(|t$XxvcZuPbkk<o?XNMdxpwo9vFpBNzZ2DIcH(hsjD=TFapc!7"
    "RE9MU-yhdA3Q6H`qYKt8&KD3e%tpiJ!xjF6a8B{&3G^{cAb(f;<q!QeRA-Z?!yPWE;&c<&~KwswZo<-9-?vR@C6f"
    "A$}ek6MqJ^Bh=UFjfU-VoMEB&l!=J^-7Ni7B#y)1R)jwqj8~uOcNCrDIR5fG4dgE6F53TAH8f7(BLI42ui_SA-7<"
    "K>N_`pSOl#-I_GNh;S2a5BT1H4#q7K}hArOyZzS{x0OCyYsHKz?LNaEM_kN37MGhsFg^s<%!NV^}{6_(;RjoXikR"
    "$?+p%f`V9#>1NKFgu(da&-VQ!wmV6fd%DVq&FXAC^PfI{{LqGmQ2d{7sFqYJkS8Lk!h{C~xRwT1?Z0v^;R^0z@rX"
    "{Vsq+g9{IQL0)I7jWwG0_#C0^(33C*4!9bxmIg$xc)x_57{5x@P%7rqd|f@jyLLzafF!CtKs#JGI-!Z!g22S6oh&"
    "FBh1l?C0hV3&ET3eLM9zTI4^vj*W3`3^pP{1}A>qo}U;xa9a{lGpa)>{C&uJ1C>Ov`^nL&39+_Z+Cg3FwK%Kgb8W"
    "6+Zni)BATh|u_et*yHxEtDtbIZ31i66H(<npgw~YnEr)AP8I4gY?gAJ_X^?RtKv$?YCTxQPtA@6f!Jsrg$&`%F-~"
    "XaF3gfy;R`s=9xroyB3?Rqb&Q}6x26=+U7R|c=llnj|y0?0wXtffn#e}!alNq=U$EvCZNNz*gZTN9B(U?Y#Z@oNT"
    "Z)aO?=fJyEr`bkf+&W7m*WzoHMo2SMv+;-Vr~<R9M%mF<ckwrbN#$m+;SydDThdkO8XHXOS;7$y5iTlcQ{=_BtbF"
    "9H5sfo2UsfRF{=;Mg9Fy%&r?GDseEoJ{Rtm2Qa|z?oAd-%=RJF8~v4Wlw^~CEHfb7&?xfCc&KDI`S%l%l<~iAq2b"
    "lh9Axoa=Gokd;~?%{=hFM7{Zpbq7%sw4L||TsmPe&s@)>fj=?4sHgbkC68$fF&<El|Wur_3PwNR7A6dI-!4&22{Y"
    "RKpLtc4drbKyk<yWQNhaxc_fyo0H#rpd816ma>M&A12zzu7EK#_c4O#~hT8+-)*_b(Xv<IW6wM*mPC9663rwAcE6"
    ";)+UnB#z2%ue?8ngkXjefEIL%bx~g0HIs8`;mhcKuZm7TP699X6-2j0|cb^4yjh<QEbL8}v8v4M(Yq!-xjq-G#_z"
    "GyjW$FuNIqh%ZuYB}}5Off9fi;DyP=r=@hwTwzOeX0cRJ+_I=w(J;M5jB13u7&al`k81i&~s$ysdfDUWJwfSzBDG"
    "UWq?1;1ZRGk#fBtIED}=sQsMSM=PG=3AXY%J^#v2q8-$-bwwNn=50e7z>?xHI|H?}!*MPi@zUanXP>ylJP8vKS}k"
    "Ca4HFWsahCJuXm|T`P$wBDgHtm>;uK&Z)I@RQr6Nkn7NYtfod6DMSa#~{`YIeDzeP2$iSoZc_1#6KD%v41<QnYR&"
    "0&fd6BYvXm0?^g-~;~rFG>U75ERNe<E?CX3&68juz5zsBHBi>D-fehlp6nEI6SR##SPA)vg=+<8Q4iEk@Et_JqyO"
    "SEY@5`9|pj+-i+C7rddh5stqFwv3ni%Wi@EhWfA0KPhYxFxU{=ym`++H%3}3eFS=B_*M*Ctj}*-L&GJUmifwKhj$"
    "(_$j)Ffea8*)Ww=RmsM55XZUeR?oFCb`5b-KEXHXcA(xA?nl@yzF#d5eE;Mc?<OUhpS?Gt`=MR6e7?(yT<64w!Ia"
    ">OsGh?YNAybd4uqO;P(SaIZmn2VFxkhOISd@)=A9V>@Y_ynGaZV?)PEuPNJatSvPCLB|Ye%m%Dq?^whBnW%w9Fl*"
    "2sn%N54Qny7_O@QSj{gCFw_Zct|{O^)SML-xzI9;F3S_s+*QOh1lbn`TbDGF16xZ5CID1NunA_|m2CrgY=ZI@(jK"
    ")-!u>)+~_ECyN@zd=POn%<5pgHFidqAS$<XzI3+1BO8Fqp9J2b*O_qb?Pi_iFKK07)zXb9x?=zp=2YgB0L!Z(SQh"
    "QZE%!sP2>r>EfH*oAUuoeU=8rsuWnnd)$c!CWBO~gRI7R_Ug9+)Nk`0aX6N0kj+*oRPc<?O-q@XVmO}Q2vXcnS7e"
    "2c{5queqGdO}k$2t4!Xi9cJqrF3Oels?5srJ?@HVG~Ba2%DiL%W`o?qP8^76F{wimpC!ST7pKvaWjNHdF_>h&Z_~"
    "t&}oXHT5hFR{25&7T=2S87-b<`N#<Z6jEMzz#GR8wqfri#$x$v7i#nS-tq49dyReBj78=6u=kYd2Gv9Lqrt<7AD$"
    "AI#5oH}Y^-))9ZezvjEM6b7k$Qn7~iQ!rZWu2{OR=c2v%Or=m81#IU#*rWCV->=^D5@XS6Z`+#ogh;6s09i6~tR?"
    "&A=v#oSyk07h$3PP=w|>18_c>&0T$B}ONy$PcRS(B5N;vi?bR+!%IoaP0)2d<e-nUSH0~K&Ol?(rMK5aiK<Pp}S^"
    "wH%j3|-mVXVJ;l)uVma;RfXVSA<g`@gj6=JKJ_5_o{YBg>xc%YGY{rdl>CcFD$|&aeNwuDvZ?}*gR~d~;iY{P&qm"
    "x?QuyMM?cMeQ%P}9~;K1tLBxmsNJw{Rb&e`#ike=JjU4$1C{Uc6c;eX8abJh-XhZu5nOpI-e!V&0J%;Z|z~VWabj"
    "YR&&f_Ad10|A#g!B`vV>CAEJ4hu%?AmkMs*&KOStN=c4FZYDJ1z)AyKvLvw-I3VEwM9c_J@97u<wx%G+)8#_8r~T"
    ";bSddGd^9Hr#O^4UgG3>+f9R<YBF(-Bo5yX~!v|Z;Vpx3&HC(DQDAalOCGo8zm*J`v7^o<Y4KfZgi_3Yd3rux4vL"
    "aplc_zxTdc-|oia*I=Y#X0gZRon;8%v(5cuDFk2G*ve*=jj!mF;*6?cD<$@RO_csZ5#L}3bi<l5lLMuYwKXZ)x9z"
    "%&VC1Eq?^dCn18c`06R0^>CWlM2<zC^GE_IbEsciDoZa)R3vk0XbhP&5ZQ8lq?ELZU<JZsbamPrHx7(#IL-z#=qw"
    "TyXJ%&$2-$$Ekb)k?-=?;`?gIgIPbp7t)2e`B7tNln{#S8Y-<U5FX>Q7I55-qOH^n1!LIVRMX*59xW*E2TS@vdef"
    "haRcsGty}7j<8ylv#R>^l2BTmYlb-JzWGqd2v}^}?I39xFGil=(Xg=Ksx2pSZhM>xs`vVW4+}qBubKrZ2{+rSG+-"
    "(#1x;UPDL~KlgY<JdL_gO#)jV=3NEi`$K1CxW%qaP{&WS9=oo(sqo-2E+-W#A0GC{8;r>eR9huvP9TQy9?xK*bq-"
    "^bGUN9;$*ptYJ@+%2-Azm;r0|1SCF+ZS;a2a614m$^688t`m$GbkvBCtBzE6BQt+rUV8*k3a}9UpLLi-g%#93$z1"
    "^qy`t$(OsD!#yU%<w_}_qpQ2Xei(b_jSdggc=*C}|M)J<65|L^gQVA$8M;f_Wy{0@KMpGP`1XTrUrC(W|oXxGMAJ"
    "d%TBK;Qxoeb9)4V#eKG^&Bk8sRQ!LSh?GcW%I@B>*edEM1K?B*G%I;NJQ)^`^8fAz(hmp47$HsXXbfO0_2s{LfK+"
    "kFVa47`jromAaVX5M71W??=x!!{pLs*5yyrmXhcH-%c?C2y8xEtk#F14U|dyemQG}lt80t?dUVqEM^MZ-=TQh^>j"
    "_W8KpZp^lejUdkt3Az<xmEi}NvCqt4}XOmIxFt|4Re$QVwr_eTTL3uGIqY9|QMnTW&?c$v_%>xi~=bVnryP0;J<c"
    "%(~6D1WmsJJd~yo15NH!)l6}XW;lN)gll(2ZFo_=bgc?(?MwcIjHV$<NE@KmSBWkWa|GdZXt3VgMJBjXu|oaCPo3"
    "e8(caVo%nu~&s+4PAkQ~tv?tT|ws*u~0+md0z6joh>pHH@qvP#YueYPWPnQ5Soq(UF|LgX?@mbU(#qAsqz>$5r4M"
    "355KSl?Kr_tbVdnczS4909EJ31Zw?bL>(j9pkbttRm~DCpqL{(j7I>t1bp(9I98C9fqdrHA$oP6w|B#|~=d&QF7#"
    "pW)PVtL;@+2OqiBC7W7`TBp7eaB_GY?Y%lUJRS(V09lI$M&Qr!CgBKx>#+;}dERPwue0Cd_9u78T->J6A*BIZyKy"
    "v2TUw!#Pm8^vt*~>{UO7k`Em+ToJWlM4Q>C@L^kh3=Bw1Bb4~eb&P1{a9rbXo{Vcuz0`ILhyF0*ugrF4<f=T%h!P"
    "Z69YA^1Bjvk)#Y&rn)`_9+K=G#$_csuqA~)rY%&W#GW@j;jHd7irz#e!r?Z^Yqr`MQ}H<fu1Vh07w*I3m&U{iVVM"
    "&Z{)>6Bd>WW`{>DKM{o;K;?vu9yB)TD`TfFgO!zn%-@1h~;4UN^Wmt6Z?6CF2eEL3{9Obi&W7l`I3>{}mhz3;%8X"
    "^MQI{sydzFSOWl1N2NXet&{Ogm*lk&*yt)#+@a%BGA?OTnFjwN06u9YM?9u%P<DzOa@D&nUnW-fo@2@<@eSxw~#X"
    "MThEIR+>I0F}W^yM+bzHzTw<tQiD~YJHa-5L>d^`J3#8t<{5}*CL05|v~_mxH;9kj6T4yPFdt`E>G00AD6qwFp%j"
    "P%NAi7V%i`BlFCzRH3p1xD1@}ORW#LaJaXqdmgf~P`#Ac}sIXk4HD31u+@K$%&870{Tj)qM``jNb(yB>phTGV&i2"
    "W*0OC!@&!4u;jV1A&b8D2=ActHG(aGQiB6O#h)Y<iU5F&&uIEBJ}VD+kaW5EJz8f-Bs)jWV;G=sOdK?k7yXb2g9C"
    "l_435A3b)6q@bk?-#vuL#uR$Acc0QwXHqpt8_wB0Ki*jSEoWS*=ZN!kH96@60(BW;yN+bf<t<B9kd>bpHFmP-RRp"
    "cC&%}?qoWy`DGPO_Lzld)R*v-Nw+`cC*okgV{XkJY$YUw-MzVl5<?O@vkvK4Tu{K|+eNw_e)i=2_cM)~pJtxQ`n}"
    "p@n+QNCK-fvkHDD<=IG48;>So3YJtHDn|Vbq2;<d?GJsIb#^^jG|*^uGdI+u)eG)qt;tGDuO1GLKs26<Qs{Nn>g+"
    "4lvYct`5VSd*4tnl)kW!P$C0JV2Bc(Os=<wum7701l3JhV`+Jyg6T!QL<@zr}`TDE#=ZLZKWyewdtHW&Rpcn<&gq"
    "E|hwYqIcUzgG^=E=3tlM|ZAckcI_~X)tPARG*rfx9VVN9Zw>BjQxn!@4y^WeTZ&Jm|V8C2i+)bD;@lMg+KxBT>gW"
    "v44m7wg#{Y&)~?DKC1I{CW5sbpx!j5(+v~}N_+(AN73E^AekHiHn<56-;cHk7C1qr71GIm_4}=bMbWJDM!{j->d#"
    "HTa>)I%fN+LaZ`+{jK(40R0_p2>;LsS0&$I7>5Q<j+Z>sR|iIS<Nozm>Ywt{MP7O|Jh)&Fw%2TUD3fI0e(rLJN+&"
    "FD%V}XN!n`gqB)%6qxDL#MIpMS1+D$iKVO_-B1dzPLH@U<(KwZCZBTI8AXkjsycPVv4)`;qW#8w9e>D){W?<|7Oh"
    "s@=#&;QCkipN4WTc9Q!S)H$iX&peQ#1xKp|DTiuc)F;ggXzm8)hlB_C`qQ+2ye+P_ncA8f@o*nB>g39GyNv)Zy`r"
    "#A2&ct-A)F>;mK0z*?|#y5GxavSL50C@8KgSTPQY5gd8u5Sfo^(q4m+;bg7yp%BsJxGVy)RyDHU)2}_>p2~MA~Y3"
    "4&~ESS436-ge~|C~Ty3H}5b>F1$jRPqdjcfG89S`R)T!;&Ia-+BC?}kuit{~kV<*5^>YQKu@eJK!oMp3C@OZQQ>C"
    "jnj2s9a?<>cV=8dEi;UJ=Ip6LOf6V$40B&9Zb}bSvhHeMWP=T#Tl-lNJTycTawLbDA9Q{(9VA{VoElwHivfKS^bP"
    "WE;uB{Cz84zt!?$jlo5v)(bhkvR~|)@Y!z+Srs0D+lTP#w>^#&t%C2<KM8&!$ogVGmjEFJ)R3RN1bp{#)zkB&)Kj"
    "5el$ZcOmTdXQOc=oo-6NS5-EB_xJi<GzQC1A+IWiyjlVmg<CP~|VwVRGci6n@sj_q_Ls|0!$qJ8@qdZ4$b_JUbe1"
    "XA+K>vUw6U8c+NqW|5uFJ7#8nYcaekFTn*nkyZmCN%+laYP86eL)LefEn>*MNu$2bs-%c8o+hI2nf@b%@u{|skly"
    "KNYSpUKb(Ss%}s^iKTwQ5gR|pD<av$d6nocUwCrf+vg+y6`{f(4H;L#${L{J>)Oqx1@K$V32<}6AT>o@FjA@PhWr"
    "Tlluz+(&$bj5}Bq487lE73-675H#nPqadyzHB&lF#8}XKCMNw0(u>P{=$7OBjNVfgZV-PT!-y_P?_Ev@^Jwji+~6"
    "=D=4{cGe!|^Jv1FSvq{Lz6`iIaQF>T_7T{}#a*$;Zeqy+L!8%1w!jQsgx+Ds-!*fyKACpXVv(F=w~5Zist0G?1~}"
    "1JQ2*ix6KG)`9A5qk$$bw+Xnm)KR=oqssxabez_+^3yPMP_Num@oUVbZJlUNUJ$-H~wlxAI~dRkYcQ_K~=3FeMfC"
    "F?5p7F;;16F-tyNBp({L@7EypWQ94ft{h$Q|ISCYMtMe!*L#Ipt>A;jy8ZNC})G;^F@oMM$OzHP*}HWgmRl*<!T*"
    ";tiyC9aHMAbDw&>uf>ky_bb9!D|Dh0ji|Gv{;nn5vRWu)Gl%{P})?or=#gVZC))8YsQG`$J4u0Hzvwuo3h$lwQKD"
    "v=hWMj|f`DB>S(s2jTcO7#Y96966d(L|TXw!25L&m2<WiY!r#5MO+Rh}S+R+db)Fw@R3o3Y3CYM$b$8(d*X2~`oG"
    "AJ@sda<UX`k%#wN(H?rLwlQS2czMg^_5esFw*pa(>F*p#S`b+j>rcLmM70Bxh#Iw|1WH`d7J2S5S_2CwRhfhkG&_"
    "?G&^SyI#QZI!U1VrAYL6;suEgkSReR*Q$88Tld}X(5ixWT<1E|odO3QP#l^$ipGYO1D!&Nmebv0njCc;eMR{zWbc"
    "p}@7^K4sQEY1}3;3XW#gvempWhzSqRAGxICg@i9>1vgG3mX&VguiA*3P*IQSN^ygr(43BGMgYw)iR2)#i_$ewfl{"
    "1Te}p{_6b$os*gccLUk!UfZ1Ms(c$cy!2n$3qr+<8E?&u_GK80l%g%Sk|H*jd9*%K}N8?C&M_yFv#aKOOyUo6c^z"
    ">5OyKz3e1IH=DcLUfxYCs4rmjG^D4Pdc~T`Ht6D7jk5uBmq-L`{P=BAwqQBeiuJG6-aZv3Xh1Rb5jM1o`;RsTWYU"
    "G4n($oHvL#%nb4~dXL$K&#|H1ZdY$j+UIt02tAE>?h&HhE&$$`q63MDMrtoLM@Fc{bZYqt2)VxRA0Xmr27{6MY9M"
    "OxA~f=Fi=LyYeAW&E9MUA_63v_rm8<7*AEXTm7b!3k90eitC0`!+i~i*Rx?;(H6M73nHd2rI5<P1FbGoq|Ou=Czk"
    "<m8rmkWl?bF|vZg#RXxLtWw^nd6a3N6kDX+9=6SmFSM2+8|T-B18leu=3?x?FI=Zc>=T{54Amq)7f1Z$sz#!%=Yh"
    "i%tXrF^g?4o0I(OrU(r*byf9ANvQOnwH6%r2gyq*1UL8jaLxKMchy^NG1oWo{GSf)V79utL-(PeUTdI(MW)pD~QN"
    "%f702MA4g=%#nkOtqeZw)@>+}L^{E)6da<;A8Bc87(6GQs4nub(B~zW7dKv(J_gL-szKm|~O7XV2xEi*%d>pL(8b"
    "{*k}049NTjd5@ubE>Oj$-7MmoaUVd~O&cI$24EMA5>XXcuha-;+f$%*;PXCwVC935h`%~vX#&p2tCV=>$bueIn{?"
    "yDK`Kr~%bgIcX8AEy$*t9u)bQ<I4VmGms(xiWMvi|c|NH9iGRh)o5(Jj74%xwqWoVd$vPCV~j{~>O1c<{~6h_uV4"
    ">(dBF25w~m1&cJXO08QnB>+lI(X#L;zk*VJZjjHG-yTTsCoUTZ1+~h@k4d5uI^bY8^U)#HtzgJ1JG~~0v5&h{WXy"
    "^fXxZJc#L{ir@+bd(HJ+Y)V3<)7?Trw!UzLLgqV%PR`{g2WfUIPg0#G-XVBp1)#fa$`GZ=&N^O@Ev5OdFHX*QkgV"
    "wjKkM?-EV=b&#ub#D%#kDQ)(k-cHd%Us*(;tW9>F|9|FYj;Pp2f6$oe7-m%~Z`OU`6LQ$ZE;#cl776<S?)ZZdDVG"
    "7w|q<{TA#n;M59wzVS=NHS8n+E^p?&0~r|5uyJ~GF-m)Gwr5F@$maSv%VrVLf)v8=Dr56fPMw+I87a#6h#iP-iT_"
    "<~fG*%^ox5|&{6|OR8NH@cL7oC?yK_?@4;v<gQVeD{+MZwJ3)rY(nin$tOs)z(Fq|-(sS?Nz<JXQ6<>x{>;)&9Yz"
    "qq&tn-n&1iwt3Fh+j>KDbqdWi!L7Aj$s|6<trVfvxOvJe_$5hc?RxRDBxB#y@S$3==o4sfGvq9dtd%gN}8mzrH?M"
    "vbVc6{26N2rx+66J@JCC8zAa`-IvVFV`1svcJNk-?6+P+ViY?L5p_#0Iz|;lgqTiL?Q3ka!m5MBb6z+^*mN9<HN>"
    "YS53=3{?-LmveZ0i~Z#tO@pwX-nyxF{(lwmKuWTM=80h`9~1Zh})s-q3K@HAxACI4z#yUMv}1Z3PF#4p%U>?z*4X"
    "HM_me$C)!O-^<9^>es21HLJ^H57d)9An%@>oE{z}$AkZTGdRH*5RcV@rR7h3Eki5HJKcLdNHFQd#dkud7>JKi)?L"
    "2n5j8-{i52AW)*e}Y<ILn-LShQ{hr4&EujbC-0i@#wYHH?bA8iya++E-0lYzBA>advkfCEzO_hpFVQYRk%JU9r{t"
    "sHp>W8^Fsf#4SVejwgT&u6U?g1e4Cxnk#Rj4B!9b&Fmf?hf{oox_t*mm5q5vx0?+G`Cx&AU|nCzpfFU1!_&Iulgk"
    "bWvG9j9_mWcPmu1xp5sPXLv9-u-@qLKNzt0RBXckCFaR_<GS}DARc(1nBCc@|#L+2zm47j|$tsOOL;2(O-u`FQ7q"
    "Sb5huygsg4su`iSnaxRrkz_dh8wivc12zo4h&Mel>6>9hLP}=nFQJF*>h7*1GwKQ0>vgnb2#frq4=>PQ8Z77VtN$"
    "n((_#`|51cg$>rcARiOnTG5E(!4W_a>?S+g`}@gWX?Uy!{VVhNKN)Q{Ew3kG{dBk=Qu0ve<qTqkv1H3<NQ)cxga8"
    "7x%bR-BQ{f5(rk|zJUmxucD2|Z}qKIpZ3$-t3k}h=D>H-F@k4}G6OX1P}Zzb*2_V@})BU@I>b&K$t3vuJgwZ;T{j"
    "K#wir;`5Eusqg+&x8ojV?n6w)*7hR=DZN9Q3=mZw`mQagGipDaSdGdBBN9WC)Sj}n9yQRCh3Hj%7bk0LD{KT5n-a"
    "5(W!Kq8_tb!?nW-%fGk2N1}1(l=a=b_Fr<;PbcrygInuqtX;TNbG-3Lm?q`94o)!l8nMrcx=bgDlyDT`UUIhxh7O"
    "->J1+JUp5Sb{;F}5yIP_li6u|@1lx7DFd=8!`^N1M0Dp=K<f13*NreFRYD{m;hWvb#x`kU$eP^7R;D!y%j%DRpL3"
    "o4b*A-z`FI(J`tAsnY0hHilcIT&s~L)R>&c00;7jGpWyF;4FFxETa++-MmbDhz~C>K~0`qPiNdYYK<Bt0Fv0eKpA"
    "d>)UE0_RNFGuM<5U2pqgsyvO$T6(G6+vCh+J)b?wFU_gEiP)9uDZmIx2zsEIhLVk8ZUiG9=15l=xs!xU;!z@@s-?"
    "lNn!D0{fL8as<N-Tlq!w;I!XbfF3tId0*{o<bZ1$|2!u=@m(jN^5k%mn0Mi%qntbf_-F}48Q>kdJg|ki}6(1{F4v"
    ")Txl{o-aI+{WpF&$dv%Z~lkVU&5n9|6MzRJWc+&0?ur@3pPta6=@*Dsrjwk4sv_tn~Mn0h<e8MLFmCDC_Acflx3T"
    "w44#Beq{RlAgYNJZMunOoN0rCoM{s*3qy-ZK2jSQPrER9CI+B3hK-zf9u{>_Z>iB<&misMa{%s=e|f`}?nxAGS{h"
    "$(!SS8VW!s?>&90vBi5^ThG6K{`3RnaK;ZP9h*T#|Km2BeEq;;wvYCbp9jBboP*f6a8k9>so@7IWHs6PF0iCM0RE"
    "%YwQ_9V;aCkHp#$yH;QQrM9*tYtr#B2>VS`~md2@8Mzc)Dc5FK<G&f!-;IU^fZ7@FQ7&uUYT5o@?%MV8Ko*9lQiy"
    "HC+H7*Zs#Ig#^iLI@XpUorrNRitzoy$FHp$^=#85xN6_<sH2cmG9@1Vx~4}oe;FR#sPr^F8IW~%Q0!=>3WV}cK=$"
    ">2Yei)JVJNkz?n8ZUB2#AprFOc!TJ@+=6eZ+FWSj5`Xyf5)>RWH+XuTp9R4lYdyPBt@EpX&t8q~$J8A*g`ANXg^q"
    "RzVg=n<36~*r14lt$vba-;=R!=6T>QorK#ZnpDEq8gxd2=(IsX<a}!t{zq!(51^8~glkTkj^&Lk?yx<`g>FetQRd"
    "r+eG`d;dk7Vc0887XsXii~PL~x}${W(oBQl<(vKT4Dj5|q9z)nsiskYUqNfIyy6U5?<l?I{&4u_fE&61SRrbvknk"
    "8fY$r=y-{))GBg**IEhkL=^5hQ9XB&k76AWMeSfV)(_yHS-<v~3Tn?XDvdr;4N(opJ>Ud7hmYYvkp(pO`O6YGoUO"
    ">wCbvn*z-t55&b^YCKmq%mVDzyb32C#vQ13yjh%K=`xtCP;J$+0b;Gveuuyc$REE|E68hEFP3FrI*z}RdCGNTDej"
    "Ji?x~@bRyW|i9<1heH+HcuP1=~o|SjAH-Uk_!wn!6ISEd^Np+U-f4&*K3FHwQelIDyDH)m>&cIEsFp!>8k0OKb6("
    "k36J{tWUa?yzYjC0|{=`M?!xj?RFgXC`0#(ouj9Yh;jb5#y7kTqDKeMf)UpeV_T*jgPPj!zUAZxDG8vZI;@83%TJ"
    "hVMargs;h-8z#22%l%qFc;!K_tdi)_4hOb^26^a)+y{Eo!`|ait4n7#hEKU4l<2C-jb^RA*6MPc%_t0KH_Ye>`Up"
    "P>^ayjIZ|WhRK?&}d4LTvjHdPIdqP-f+{I-d;!>ZxF<nyPALA6gHc)n^eks+^8yF5US4mHZSZ?rO*(|!5}+!)#!("
    "??sanc^+YA#Xg?zWu#%X3*gu(piVPu%c<7sacgx0_RYNiH0l1&Xy*8mZT{pEw_ZWRcezhCRH-0x!5^ZxJ1k@Il@A"
    "fx=KcEY5Wy-0ec6p$e45ZF+CAB9GIqRRr+-~sp>()6mK`NK}C1>P5@PWH&9W#tKL09XB_wkWGaHXU!+Pkjr61oEC"
    "?8%<WyDHq~z@4r>is!SB2M4^da9V`}XpIMo*QH!V&ribIz*go;<Tk&pWbwGo4TTOga6K2x#CI1<qm-LQYI+I9QxV"
    "Q+zScv&%A<Fy-Nomb}{QpV}a?uGCm0{;~t7$Etd6lD=H^9uCg75{6B1>cy+(UX96kvh&km_sxFLYNX@3MuVg_II0"
    "K!3El}T_RZ-|;2B^&R<@VYw*2+sG#zKfFbnx(0@J&R?3t%Nb+-R~dhuxUoKj2Iv-lYQMtzIk7fZ~)Zkt+=+pLju^"
    "b@e^y@dq2-xn^jBA<zFR?T(kn_Bh6=W+nIHdb4}zzPdFSBvbXKqPF*;WoN?xBv(#Z?k71^NudM>p^AZqnN`Jxw-_"
    "o?g)y(Y96XBCv3-e2PZoW*ZgeLcLiIuI}0?ARP?{GeY~M1c64bidr(J-x|Rb_7@YQx{MIwvQwWaw-IGu4m9qcFxO"
    "FLWSyjn(*GM*Gc!8>Pg6MKRd$qJ~@`I1E=e3;)fxEQOB8EXAXV5VrLja63pf~NiTU_cDY=T)yLZNLXi?lWe0$nyr"
    "@ba6RY?Omb0DUx%1us@2gvTidXrUyk{f*<n_U>;Rarz<6G3Y%>FV(u(-wetn6-pp?Qos-U;P~C$sCi-gU>CAJTRd"
    "%`4oO}$9}EC%yE2%?@^8iMdQ9+un=D|_VK`37gtNq`(hV10!)StZl?VWC7R)&OR5SaT51u>ninWn&9ee^S*389Mc"
    "JICTCi&*;Z_81~z+MvlDZ<+?R{K;1(+?smvr!Sfcpm+n{}6Ur2l}2)*_)WCA#G5t32bCU)gYgZ4?X04=Zw&ML~s$"
    "<XT`t#@7Edi9*Iwd;%QgP!2(k2{qbKzTQDGIbBxPJ))%5HU^Yjf1$?CuoZ)q|yWLfbHF?`tnvqylLVA>rV5rA82R"
    "lD)AH1?0F{RGu)Rr0xm!4SPy4~46*ct5a*L{4R-DEcx-sG>|NB|CrU~H6vxw=wVg}*4^czB=P#h$&5{<4mQbC9c}"
    "6_)zykkP8<DjSSWt3rJ(&7g-`)m3iL^*<gSzdAfU9UN45J<ypiXH^|<ZsQqZ+4!dWSWEvPrroYl_^{Kc=m062p9+"
    "_~?tb%V`^`ykvq=#aJb~zo%%fw7-_>vcvHe&_faT<(CIZq)cB@{|m<ruSo5~E@@qqIie$R4{>yp+fM!gHR>5UaKS"
    "EL`Zz|IbWxWLm*P=+5eXH64XE#Bv2Wh62YkG~be;`_RyElDk87bm`4^r?7o^v%=`pb%dJ{_wn<rP2|099~bjCJXF"
    "Z*cD0rU0!$~MSvv~3<T3|6BuB+88K(*b|ax9R2L-0lB&t2vvTMOf_9}uhhewHTt0hi1o<o=y}ZN6>4_|4<+zvY^w"
    "~Er@EFcF8>*xed$FTFd*uo!UJpwEJWIX5#)NfULZ{oW#0LUMkJu_GunWIp0*Df|PdE5N<rzM;14TpQh|x7j{{-E&"
    "_6dGq=FnnGjSrqb*$;f0;l~a3Zn+c6hx`tTwC~{y%sIat>ac!UPE22h&eIZQHf7^J1r*{r8eJztW%Ws$+Dgz_TXd"
    "QK4sq$uXDZ#rw1_nIA@RShhGPMp8VX1%?uxE~4{dkBenr$$V7ApXN*D<n3M-^j%6=$7u3?kC>AINCAOcB(?7M>xU"
    "N#4KgNpnLrb~;$^t&f}ul5d3<H-Cx9UQ;*Gg<g9{R6MYEQQgRNe@5V#bjVVpGs9{Wn|T<+BUyB-hQ26I%I%t>H_1"
    "JC3L-o1yi;UexfeW&=0kP|4o)R**qUQ@y6iJB#0yIBQp9R*jPs=Q}0jb-MmnGHDCC7O;AZBvx2I~CsZM@;pCq>15"
    "B6NZh#6Q?Kbv_lwK3~Y>&Cx$t>p>n`XgLP`)v;7Qo=;MS|nkYeIoxqFtT7uHdJ5j4*&T0uWY!_J58Vtq4-^YE;4V"
    "1kvr4p$jX$iJvQ^!O;OG)2(j_&3b)LCkNFZu~M-ddja~#F^#NnBe=)9g)X)4+kn}zEUqhvNa}<mI9;56fVKnqY-_"
    "8c46u^&N$A5%H6U|1$AN^ohW>cRCr<%(U$$$#i_}Ze-a*vbcysXc!Qrn58}Y`;;9z&};MGREh9MHEI)a2iN~Uz>d"
    "$lrLH&dLH8ZS;z2R(%4te$bo%^wQ=4~4$uLfHWw`de}^80?-{@$vU-$;=ng_C)m#`}(YiON1{W7_0YmUM#AA15^b"
    "bQdE=(YYDkvB<ruhPo&DaRK`X(I-ILD9?(zVYz=&(>0NZ4jYr)|A~{*2R%w0DsYe#RdO4Prr+z5bDc)9}4QRD$b)"
    "TaH8$7q_pk&>nkI?_zXq^V%SWCyc$2wjhP=j|E7ze|#`YT9IE5yHsY>XuCkz@;|cte@GP?O|+`1Z=+Ef>YX59A6h"
    "*sFzi^@1JO_E94tT}L>Z3g!C3*i;C^WN7Q^Co6@ui5Ptm?N6@&Fc*M3E;2l)BiEmL3m9MkZuMqH5KNLcJDg`2DVB"
    "~y3_N7bd=8r>Q2%PnL@+-X*(0_#5TU-a3IgOC{qrk8u>PjKZZlySJymPyuiKE_j`<f008vIil@wmvI%82meO_$F("
    "X%hVls9pwr|XET_*NtL(&5pObqG;vgJk;W6BywBGs6)BOrvf#|CVP`%ezRm%6?(E!XqjkK*TUX`~a%=riFHgObgk"
    "(1&CzG2vcB|512oxlDS?LIZ!2;kSPcWYUuBlgO^T@h&Yl2-?#KFC78-^nDyU1xmf1o5w?fylq}j`4)2t$5<nH;(%"
    "cl<5J8SzgP}vvL1%c=e3+@al(X&fP>S7JIe;c@BH&7OqU6gZ+A*G`yIu@vC=rlCF3av&9b6VDwC1MBcBM~vjYLNY"
    "j|p{(eKa4qk4Ltp>vEbHctKA3O1I=xtel3)XHcQq(K*t;IdjwGljg&0jZ9N>=u>{h_PAHvwfxxO2ZD~+vv~E4d_n"
    "=-N)t@Lk)tL)5;=<gv)pf0j8}X*>AvNhkFHV$z?x;vKyTQBJX$t873nMN`3PNgsxQZ5++r0Fol6IL>8Jy?Z8T&jN"
    "928KRV&{w-4>qw{HF8E_QBr%{`Sf)qO7DaquTRTG-Q&Q<kOgOyVDCbPnW8RMCAMCbSJVrR_5i#W5MQ*H=b}P4wv|"
    "=1c?>Rr|@NTC9}FJaLGVjNXY<MvF5@6(jXE3o+O&j^VS(`vqiMGOP*To7mn=eHTk^yDDpV1BlqvKyEWg?hgfU9#f"
    "LIs>DgNGa?0db>wJ*T3fFy!<NsB4FV}7$234*3mZ5T4>otjMERh2`-TL*5%~jJ9F+DmnsY1DoXPe39%2G7qM_oy)"
    "3X%B!3d+V8&sSE1*cV4jPsyy(iDs2uYvd^N3)S&Mi5>}}<YDjL3}+oqwtr|5oSNU!2G@g1(Yc`=$PN7z2|(9Dhmu"
    "9>%Gs)y1WyYy07-`cNSsC24EF>xFoIUR+{2(MCXa$>zU2*Lz~ooVihh&+PJfi~QO?RsefB%B-{gzva6C?LQnL>!v"
    "IXQIyo~%}kOGK6nJ@EfJgQjr)%yv*QM4F%x=;T$Lu%uY2pg?!AeEizV<uW7aY>DMO#n|Z2E)Y0X2G*5JGkgUSZ&v"
    "7ag79ftDxG{t6?ZMm!hv31<FG(+h%to)rq+-@`tp@7{Y4@0cdvB7CVl9!~vW<4ysxP$Ko0~iy#RTyh!_VLeW3L@C"
    "{v`wo8LTyjp#$k}Y(8^>Jf!L52kYP!1Qi#{+@`1PUA51VKCyZPr%`^?$!jJO4a`|7>>tm~_rQZpJUZzW)!cQdn&2"
    "-{0>1mwLMkuW!XKo?EYjjXYn{D7t0b<dbBaO|BN#{jC^bXZhE6TYr3}Jt1`Z%=r3@08Z}<x7a0G9>hQ7l(Y4sZM!"
    "}=U~_cQ(?bPQpwO+0O-IE<S;2o7)8%|fKRg&Znkk1u07%qAS$Q75<TFy?R4?F^1w4*J+CSt4;$3!ah<2CN0%Hx7X"
    "tZ>QCIu{UFez2hjt<}`0KD%2T0hrBHP&PNzPkO**0a)G_!QhjavJ|lst??LCbw<>Um&=E!St>EK{bVrg9&Wv%RnL"
    "P1ZhheTayTFeZ&b(^p7`fJ<4aC)L0I@rf|SMGK=eRv3afNc+-QF=6nFDB|@&MjwL5F)PnGbqmT3H?c2>WJDEz}xn"
    "CW?!PT+*c$m&oC6;#)p66wwm#TZ_KsE<MNsbYuCZ3?XCCoKm)r+MVcf0Q()SdfkGuV;P?EasfZp3S0j<Df!USLvH"
    "gL>^9oD7anqrHRE!*J#F2XQ*lY@*2!bCy`U6ovop$uHacZw4n(>(4R&UpcL&cB001W^Fs1;|lIv2>ip=cGc>zP*M"
    "O;rTPJD4OtcON35sV`mSkxmj%G1=Y;=#)FOnDkkS?tH7W6ucAUIl*ZZpMUG#64TPDg{hov`A)xK+*ZJ{}=%$D#=H"
    "OV5%ejkpPqb!<^;oSV@CY=LK2(H^?>j+nv<_aMlkMLR+u|n~b{D!L3x#I%@4$$HyX?D)4R@*=bETp(PRXz9$!aai"
    "*b4*T@>1CAiQo&TT#L1YBkttnBKSpR@l4>t=d#s}?8%&|y0U%$ms~gvx(hr@<YEhpe6abH|{+L{8M`o^Wg{Jc1L9"
    "}yt@Z<j8&Z$|;b`K-gXrQ~KtiJs5)mHU%ubR#FWQB9XiWv!_KUKyR`feIc!h6fgu@##DrB=@&C1CSAn<FKTzC;Tu"
    "zggZ0A>Ff0=H)>P%;hZ+c2<tR*aA#^ejWZ)t6DQd06hFTIIaT=KTITMeZl&6hpHu)YT~v8T~!~OO>YY~MLCe3yk="
    "^$ot9j=Le$NTvcRpdB$|tr&1Fk1f*!DYd<y)X#So7#kZ(!Z9n_)!A9iSqo*-#Di@s8|i!cj`uCv4btR3oe_pDMWa"
    "@7$~p4wNw&y+Pzfx7BKFKTt%n#2)bK(f+y`)!Yf=FGO&(-8-=d*r}20}H9WV<1ri%nBn<SIcp#jGBC`Oz)$^lfA!"
    "1m*eyb!k;deN(zg+n{hsQPm95idp{mVa1dPh_V<i<026+^)ov=z^m*EMQ$7P!r7Yc4DGXOWOpbRC5B7hH{uQaUPw"
    "lVft!V*s{V+K={PA#q|L|9spquUB0|%J=M-K>rlO07%@dq<XMps|M<W@>j!8)|Pd`R`k<p>`#Z{liSn+?m<L?imf"
    "C2aItoEr=6pG=+jWhZhgU35>9y_4g?tCp0~EtGy&Yavor*cXfZpFkP2^&<KDyXQg7y+;_D>I`4}u0$+w6a)uBK+k"
    "0GN4=mxG2p^zR2?!)GGbjVClr@@A+#C5f+b(`BD<R^v4?p?Q8~KJJeiStSSF&IbYm|j-n>NDg%Flwr+IT&0AD%32"
    "cKu~Ii+cGWcF|gzJX16l=0TJZIQU*%2ZedAo(~roLUdqFoG4h&R$Wf;HiCbtFbji_75s|_Mo6O<ZE{;^~lLKa@4&"
    "QCF}BoN!Y}A1RKRa^rdDEwud+f{4ug#RZme$$x9k@Wr<F*+ejZJJ;+=;JdXBW9UQ6-@eDWWtQM`QoN&aokHs0`?X"
    "ucy^j|Dp5sO>0ODSqNZ2*yu&Ypt1FIm$h*NP{CpMPlP0?I#P@RDsrf1-fLQMcRGh7$XWCUi_3K?Fe|IgeLMH3b5`"
    "ki`^t<t8UnvJIWps%-oaCh+4}KQ+TtI}60FF@Q8=YgE}XI$_cqG&_PyHPm8gvxm|dJwZjKZU`wG(idQfJUeUqW|U"
    "=AOGt#1Tj!`k&&?W_J*OTcs03-)yqH+Uv)G~rM4HUA*l~pdx@Q3nQJqGFUQsFc_Ze>IDLIu)d!L#Uh1L5=PYY3l6"
    "oJRkgHemUBCPJ%Z&=cFz}yCdLQup)$>sG${n`h*jL$p>_&tKdqMI2y!mj4*Co7hDh_t$Hu`=?rDY#QIHDz^j(v?9"
    "+6D6KCM!IPjtLNUc(YOmcxf#ngJyfv}!h%4xtL>q1{1Ehs7z{}Eqq+`TjUIh8xG$qcMvm4T4>oxDFe^rBmb10z4x"
    "cixi{%AuN4TYE?*XGU(?*l~1kN*DGHHICk|&E+dxdpwTA=>HA|YZDoa(wlwm@Lr9^`-i0x4Tr?zl6c71jbPoviw9"
    "x?B|bi1IYhvBc??qp1z6`Gux1Q}PRxvqHCSrRq;$rHJHMs&fma-p+}OeE7cWf_w+1^A9GKJ!nE4WpjYJQ(x0y>$G"
    ";Ufyv|zuJnwS7zAZ%y3&*=dy`qE1RTkhp9wWe`xE`v<u7eLM3dEdf65q_wl9b#v#dEIb_4ldS1U_hTRb4*D6>788"
    "uwME4c|9w9J@OkIE^S=oqv4EoMjL?>@7&RzhG2p@g>n-rLjLrt^D9dvI;`4QnLB{yX2d1U(`?ce1dvfTE@87=3Ej"
    "m`@mo-dr{V1|32LiY6svC8m>4ppOyL~$&n+|w$ZCRM1E90c0qT2a=iD-YNc>@(jsEG*3jeypcbkgt-@buLa9eD*h"
    "kK*-bxH`g=cb>%L*Hvm0uvKus*;}FuYvG*|4B8{8baj3GiXmIDNGH-#8l4cl)LK!o4lYwmTTp9>s%9lUQ`+bq=9K"
    "ck7sb`6#9`>QE>{o=yR)+C4lNxbSuaX(v3(+UG7Xsx3jflRN$={i6V{aIZ4$c(Wb&Jhx`=Bsw@ejSk-I?}xE8V?4"
    "340VJP0;%^Q;<V+!UOkC?s*xq<RLQs*7T0A25jIEiiA44EvmXxFgvk8VKsF6sK(0mk@PVigdnUr~yWB>qUdf#k4t"
    "9l6z!vGo=`_{ZHztzjnlu8f34gR$`W%UGd1<MPZvR=@F?giHuPvHx@8}_YzDyB_E_57DoWQ@<9)$xhz7i|u4cM0<"
    "`F1T=TK*`JW#23+DPYw^%vYOt2{RH+Pu8P@2Rb4GNGi{6&-Dp?ypyD2f%szB_LbC2?+I5zH%S%wkJ=4^u#a#hxb%"
    "DeSqjBFo`E9zyXgN&UiEh#fxQ?J6%CMYuqc?yejbwxvyN%J>M8Bmh>YHql54(Xgl4Cj$0_#zkAr$D1>mn}Y;H{EH"
    "M`zLL-cX$}#f}<|)BFZ}Czx0g_{y!a;AcwqmF8Dp0ux^u<I03aJ9d79`vD9X?nXOAQX7ixDIg6Jr)?Ja#&itdRwc"
    "sBCMr}2-@<m_rowo7dhe7q2B$BUK#6!b6dbWVeAa+13yP3+Mu?`UymQthxR6mK-Pl59!E_{iDU#UcScucr_%K~zj"
    "H6{B2Q;o(c6q6qQhb)^!W)eGTfy!MqJ&X)nPIS$OI0C*adb4FDxsZ@E8_sYMNgs>LHJ@0;t-DGe@`UgUv#TlWRpu"
    "QAcq(_yplbS5YxBg`7p`QN}6+F2s&q~uMp9sJh&+#6tK@o5+y*31k&u11f=`YkKeFQh1Bh?0mA{>VW4ft%T|qi(s"
    "utoDB#W2E^+fAbihuFoVK>yU6IlRV*=d|lUR9+vZ-pYckD`ghiDe<eD(YKuGLU$F>FASCZ*C24qeTN9C4X1$Qc0z"
    "ORKT5R+y@AGs^6c59Xsxi6x#vpl4CW{x6_9)J(WDq9EL9MhcX?^}t<T^AgN$rKHU}m_6XU1&@uVQ$UP_r((gB=N+"
    "u){sgmtU8G!!l!2WcEF#u%oIS6kK;gP>5}^TaX2Fd%=#QRgj^h1NhiRnPQ8=M+&rLYHCUOIE;5wW*#k*_hOTC#!S"
    "(IY@>7pYy<N*%dSP{FSCkW((T+NlJ$Qi3H=aZs1UCr4t&rm<Pa}R+PU?Bt-7N8MMXJz8J3Ky27>y9)I609|<Tf{c"
    "NykiND_h&tEf)JAnnO^pVKlE`Nn^i=wuA^Knm2-|qgQ-Xhi!#PGz}R5cn6}M~FdeB)YQ-Vu6TNW)L!7ov{DzL*dc"
    "Txlt@q2gU3uatA8JeuxSYJ7Om8RhqUZX<qaF)Vnx6vi##?wdheQXpTnu5sQyT@RE-2J++ZSjox0%itCy@*<vDr&2"
    "p&-AXX@o*Ru#mP1E{J8xUX}>~_vP<E9=5cWvub+y{c;wrA<Fb-e%VgPxih?@M_>JLR7nksdzFz42AQ7H*cgL*lK0"
    "u2kaKgKqgpEaL~RWwXAcNY?a-_6NnVR|7seWiz004kJ=LQf5uBVBbjpy^*%Rz4Fv>Cm3)PH}vOMuJSZ7xgNIL39+"
    "X&uqffxgJsg$6nw*m(9E>%(mTNyq7Vv57(%X2l~VTP)84)dy_z1?mFvz6vU`=*6|9&EoJoE&ZMAWt=EG~l86iaJv"
    "LORdL_aHPKZneEm}E&!bH7AI9Z9CJ|OOv62-)jZXO$Cx^bnsS113EE3T+cVhfW2kX~hecMdku|y0E7N7MRw#yHLm"
    "{@1hI}C64r0~>_DzBCCzwz(Tp01+_zxg|+-Qg5H+bA|wGM=Alq3J)Sj&g`ytUz47*v*F(`afjni~CpVAZ|j*8*a@"
    "JJ{a?Kt{kE@mzn{o2zWX(lQkE!Ei!_C`Q`Kk-2yQ^MxSDUShzAo<<WSE`Uqqe5!;n4^+EX3KR4DNwOqV7aVW@$QU"
    "r&M@PqpzhGyCIX(gid4aMK>eQ6=$aEokM?ta0w0~woPWNZ$&t|r6RTFnm^=Sft>uv>mN8|bj>dy1YB`^5p5NHR4l"
    "1CqyX%%C;72CDUEj8z4p|diqO;@{L<giTBg4iD>(2{XFL|A2)jqkp)QjZYnB=kxCdrI@htbbyyY+ULi_BhA1W;RM"
    "hsR)#^nFqDf;&nIHu`=3n|B`GEhyRUt9qrnj4)Yx=e#YS_3IxuWm+V*>s#GX?1{4158w6C=^IRJyCWcZ73C+UuSk"
    "N{Vk_rJaiSLSy)zcf@QWOp4<lS<q;UF8!M4n(|jm>_9K8{;A{6k%E_iVbF+Qx<2)^E(jHwQ=g>LpRq;<+>@pqC61"
    "i80QX;F8H&+*xhe7(x}lmo5aO8ZLYlVWxk>YG4JZ1l#}uz_DuZD7~|*Lg2@&)M-48>!<Xk^5#k7bZ|eRQv;3w9=Q"
    "D^pYb}T$d`|0Z`FB~s9G#ECBG5gRT@ulH87%se^fo&ctwq)DzoIivT$A!oUL2CX=sTEkUsgiffEi4h8*WtIcx%4q"
    "u<%9B_a_NOXXrJqCvk_$S-ASm2*9;gU<h-y*FWR+sG0{{|ef9_lZnMMsjw!nOsGd6HRQ%Ey<J4wf#{L2}u}}gqk4"
    "js6GDg@2s_<0D`jI)A##cW|9`M6sk_0I{RX_SXVF9Jz#saTaAyu)UsAnf?7*Z+1b%_B)9kq&F?6FDI`PNMHK~_gw"
    "GFLX*^)qN}wtzV9L?fAuCyU<)MMK8`cgqP=g^NwN9OR(p~T?BJEhG1(h5V0oJ&D3_>Ct2p7p$U}kR10oRMVNa?KM"
    "6=Sz$Ax3&jepK=ox2m_ph(H_Wl^7ZsMTq+d>K6Wsq77vhsI6PYl%1$>BI|W5*>zcT5(U(3=c*85BXGOIbo>}f(ya"
    "sw*~=5|jhqZ48x}oDq(TTCHFX%B(#?u26WMJ&v?b=rET^}!vuOR3GGo~PhwH?tQQi;f<2mCbZZNP#tVo#EW$yUe1"
    ">ZDF`;qE$`dee(bas=3>FSnkPT<F@*=m~BIaw%0{iD;++v{3i1iDtzQONY}oU__OecInUJ-`nidG*t^YI@I9QjZD"
    "$XU=QYy9ZBU<SpgjTL}zlo2Xvg4u1+-s=|fpAf`Dd36sO0d{!5nW9jNB(jrL@FV2pi?VTO&b0Zc@`lRcMHjhAfTf"
    "wMJ3hf5@?yM#1Vq=-^Mq%_Dw5u3pni_F2(PAOpC^;q0vOv75KZ4YEo7N~@F<rk;sQ*w{rPXn2)SFPn_%{@yIiKKv"
    "T7(V~2sHReCDW*R)1}pi^Zq3WoY%`Q)tkYTgF_UsvJ8q0SOq_!gT(b9Rs0d-7?p&Y4Z?|SV0OBz$>u)IO>0d}pWe"
    "*W4jJy3RvXJ!uZC(5w}{nm)e&hsd!ouhvw<Qv(o=V7cc?1Bbk87quYuC}RL!j@P*tOXv^R$WNoD`~IMUj4u5UIAX"
    "zc#L2dUxRiW}bHnTGFCt~T|;Dp!}6`JeE$f$O!U+}#lcxpy^Hl1HL)ramF4b<xsCiHfi-vC_s-MKmexduswG2$)z"
    "lZW@w*%8{p7BMCQa6IEa!E<g@}9$MCtp=%9|RhZIPH?AgbA|5aU8NlTl_8jHR@fU_<)E&*y&^K@=N^-$yp3aA4o{"
    "OG=zYJ|Ks(j>it1?s$iRNC*fuRA(IO|clMAJD*Y2Nabgm)bq%HIC|!SgdUoSYo|2Oxgvm}r_*v09FcnMr@b7B<sW"
    "NyR9=+_;GTUhg8RpNd_RH{f49|GctJsiI0=dh=|Hq~j3abOuwzY_6E7p~AZOpxw3Pb&jQ?e$AZfp<_3v#=_A?L?Q"
    "#+yHo_E38+OFIznz}%-)j6&CI#YCi&K5@MV^7P$fLo(GdI=-14WMe5_t#?|68^(xXgOgma51y~Jx_zzx!45_|V<3"
    ")n<N>G?tXj<P#+{{Es?l&Od?NpPVGitS)|HHt_b|CR1jYgn*tBN4%q58n4p5*OpVWiNbwIt@`}6;)$nmh)+1xb9z"
    "5AdkNB?}wtII>FZJ4`L8nsAtB3zvbqHLk5<fx<|tQ$Gg62wkdFoIdOp|(VS~nyVu&g&+4%tD13?KKv7F{*i;fydy"
    "Ujj=Sg}|qP>)(G)>^9(i`>|<qMQt$LR%D(w`P&3(HFziJ7EJPtn4jDp=GFgH<JlbDG{#R39a+l!N4q>TE7x^fGLV"
    "rf`w~AI`3Y>y$$fsj>DiQntlMr?VSiZP%MlS8H$Tb*kIBy{%StuV;+Fo|?wAD_>+bPKW1&jkyRz%DgcP<D()x8$C"
    "4t#-@Xacg@~34hg&|=dqnRlxW3@bHBp?P=>s9srd3J9eo^cPN9h(DHKM!95qn7AY@;svAbzpN6aixWu+Wv6g5rMr"
    "mM#IZQZkhb+7I*+Yw;bQjWTZ6(e>99cmw}m>VMrqMydxLS94P*s}NdkF8$%HqEIz$tb;qogHo`u`Zq(OlP>RYssd"
    "%WEdT^wZC_?fAI7v{M~J`ExAt9`qg@Q?3Um9tKLKos)l@!88*%u*K@`_H2`F8E|kQq0!5kRrQW{CKzTqKdy7dYWz"
    "KcMTxG&=NE3}>oQ~sL9F1tRdMPI<NUjUft+0$ab2QDRfJp#tL7L5JkVs;vVLJ6S;X-6i9g0Engjv8uCi6%=Rj@#e"
    "c})}I<-%nnta8JmGWRUmtt%<hTIIXQu<S<3<(Pu7V5{x<0<X7dPv$k+smCuWuI?7IGfZJZ#aPlf15xisHCEzYtq3"
    "2lqqo)1t4+dHE^)vpX+VyiS`Gb_w?qhwWHq=FzfrJwGkcy6S7nMDv`CaM1pNfP$X~}K7b=hB-#7A<CQE;)X4F;VE"
    "&g@$_%xcMJn}y+EZS<_3onS}lSeF&-)Xbzg;G$VT+KRc^`K%1n?=sMJl^=2dojngR|l%D;deFCO#oLHJ=74>@K=I"
    "K{Tmy^NBDzqgv16h5#7op1V3JyMv3T(cWjfP9=Ub^b<G=NqB*LSS0l3E+R+>be)VnY7H&sq8NpO5)3uKq;S+BTiE"
    ";C=W5)Pqsp!_6+(F|!RG2iTb(5o`?=;5%l6+!9@`l?)OEl=Zjn=83Y09!|8w4n@`KOli1YvDb0|)tQ&AF_`3>&5*"
    "$+_M75I%W=vhylW=}h`wob+FZmDe{%86zZNnP-5{YDzr-vdkcGg1uW7<#m3uCa>Efe9W<Mw{TfIW-MS<k8gx_P`r"
    "WoK(`524(RTFvdG-P{a9_(V>Iu*UHP_%J6{<^6VJU{0X4Q2sEMS><bIXS9Z7~xsJ&DzE#r%v5pk0<#@%mJWMIcf*"
    "{!Oxo5+_fWy?*3uFV9mdyel=c_E1!6D-2SZ3i`>5^<}+9faEtu1%kMa71kk5%-EziP<EwYrT2lgj2c6ENPlgY4P>"
    "T6{zLfCG(NB8nRQ13O#5`k|h;}{2#Mx+qM^hVcXo0wo+?SO-$(lU9}w<XYo;1>YAKe6$vCFWW#tCzb5M%t1z~CoW"
    "N!v?JE|yHr7O4i@DT=W<wI=r`GKFsO;K`ULx?#P16^a>+Wh^ug~_>JH!%AjXN@u*dx`d@_#DE5aeqHJ<w)mTk3Gk"
    "fypEnZqaR{7zb97x<Jwu8t161c11J)H+FP*VIZDCsBH?5Q%cKZE}3GxYw*>ST4-dV_KnreB$YyHt)u<yp+Z(v%tz"
    "RQb147&MJyni(Pc=f;#{*Be`AukytUOcuXls$8e!UlEaRnHA>y?yF4J(GicP$(2?~Vm6j%%ZUL1}F8@RJRK*yX9P"
    "4t#RC@$^6<=!!GB*+$Vh)J7%YQ88WG@z$(*mXc#-)HT9;`p}=C1@F;dT)X4+7m1i*gn6jH_W5DyAHUcZn||&Grs~"
    "#%}xb{9c#NXy!WDiYDlsBO!JC2oLfN4-K60UZA$B5^ZHrpmDYQfUr9v6&JVjF{iF*}L8O%^t{5BjYRmhcQOCGS*&"
    ">@2%Pdk_?{xpygJ*lv)(bD~pBx|xpY8qp^dKUrbTk|V59z@#2Pet%lf!3wCx0Zr9sH5NVEexT9%BjTZZDyL+2rit"
    "?`Nh;sz#EqG^Asg8mf;Ep6tDNdWK4W_tC3)oF2$rFnNr=;HHfwSiyMFxz7IVaMhKIX>rln`L+xHJgQwC8A9yRXvW"
    "UEdcc(OE0A)%r6ezo4*&Mzz*KIHFO;e(2Dj8G>JO)DSEniuU=_^<X!Y1CK?9n+ngqttX=q7WyyVP79QXVh)?uG#)"
    "4Exa$8XDq8Ra)<Dyw^Xt7nC%nn#|iqBeJS=J4q8!QX?O8L<F%d=y;oNJST2UBd?3PK>(EW%iFK3Zo(O@Cb%^c5rm"
    "oQ}<!8hLeLQKtPW60q24#<)#8>!^ZWJ8K4Z-KlO(701>X>zd&tkMA0}F#t~3kE-%pnIvsIcqq2%%a-PHIezcK}bF"
    "9HUS!E$+WpfM%XgJ|JikYpjt_8w1LeH(j+DyWN{ck$k)Py$xH#eOKk2}|WkBw1tFr{5WJ@mZQG9jv*9lK~-SJXR1"
    "vG%syeABB&kJP*#v{Jsl(*P4`Z4q47t}rpML?N~oD%<k5v!9iw3+utg#p>0>=>g1XMzsluL<(#=y?3WGMga<dB>`"
    "t42)X)I%}JNL&iQWURGiGzOB^4SFcaVM?6M-UTsLwvwV4WIik*?h<8C}Y8;L89pRP)>O^+SN@z=Np$9uR|&Z!f+m"
    "^O~2k#)zTmA-s(c?n8p<vTdGmGxBG8$H}!FYry$e0pn{Wt6u}(%U*fiHugRKddG-q6wJX-U8!@T8xPCEi=Rzr%Yn"
    "w=q|lbItXj)w%QyJ#7oOn8n(ztXfB&{M7035_nw}?bZbb--s8u~{_)cn&yFNmu$F(Cx{q76W0LgijJP5ET8|q<3H"
    "pn^!+6$_c@L9Q1*dT+MTL{YU+|e=TSgj<H8LHXUd?|#c!C8m#4>RFBw1h0OLo-bJW?DChWq94C=nCi$=>1VL1*vh"
    ";}f8gTVgoV;wNq+sjcoKOd_qJ20IQlXa_7UTU#fKHGzs2F9TRWB^5a7H!5O(BM@k&`f3~mbvy%o9H?7zF^4+|+i|"
    "!!TwKfPcAPqpDVaSR1m%`Gpb)?Hg7-aZdaYzCAGjlyniLHJu{7L?jy$F!mn?DEAaOY+L)k_>p_R>y3t4>LNX&A4L"
    "@`lS&1x$9IOt%?<kg5<=Ds6vnc>o9j4Q;>@57DHFIB*<*-xyA-JVxmfItj_)+l-)HlYYOr6zKK_j|$D0=%KQt)mU"
    "JbTo;{Sbj^}DwT-O7F6Q&P2+soV$@S{LK@)QVZ!{W&1jjFDtF14r(mo_|4_$BQ5UM}Cvw``3^B>l33a>-G+jrs8<"
    "7BBCs8_52|QdUrb~=ovlQP>n)Ic=3LaP(H?n)f>lK7Og+#Bi^j>U@J$FR`0=fI#fJe5rwTbrMKs@+rEqP&G!IR8j"
    "_VL}cd}zB~II9kbg6R4IHyBYmX91g;-AuD(ldYveW{{AH5jJSfG3vkLCRj!U2QB*$>H%aa52w(me`w)|(PCv!xnp"
    "tnK6^7w=c5{nKy*gNwt!hni&1=jXv1OXZNjd;Uax#sTC~OVP;Y^zjQG0q++G`SyUX2CEuh06j@869!e{ToFtm}Sn"
    "S1-A{>|E>Ue!PSfZ#d-K%=$*SZrzX-=gqveWtgi<-Xug6Ii^}twX49*CW&-!yZVR*?uE@b0foC%yforv?sT5OK?m"
    "muQhxduG-GW;+xbOJuU*Ok($sv1eBX<iLy=}3~8(P*y(G^48?JSu;}SfW}!VDvkO}sR_`}S3GBM+ZT*fYpHWbkZO"
    "q2s8*FtOMt?PLXhrEHEl>l>UNk?kx24%lTaBUuPp7R_pb<FH?P%19`59{$pkRD4y>BY}>O$s}!?O1TDTLPgtg7i$"
    "_v+~2_rV>&DNAB3zh|VvA=P7y(}Q>IEG>rNwU$`s$0Xp3IsqO;CxO4rL;%q3z>VSa<+Xs@F_7?l#fN*(@TZVkq$d"
    "Y|KRi7<Rl-@<GDPXCiD+@cQ{<gl`X>^{<~KE4lea}OGu`lBXE}HkH+y)89eXvucMSWVL@dN5e4svITd{6vSc2Uvo"
    "@#4!G(0CnNOL1jqYOGCdfXK$<Zs=*HY;`yoxf-qOvzqc;vvVoBkxQyPK$092~{3x5$TgH-sa;h=%Oh}+Sf%X&2lE"
    "m>EE7WvsNr3x#UKmj}jC0Yg3zGR0JJI*xa>=W>K0y7q&-c`Hv~g;4AxkbpcY`I4kYb<*ogDnayr4^J%uW;U<>ko8"
    "}ifrRUV7Gu~&jw!3D1Z%mEHzs}O>^151>Fd9Bjl}1Xxe;4;t=WZc8ucK{g)@84I5xRyhfcn+>yc^9H`{m^L#dB;b"
    "tr2XMa5o7N#{RL?N2_%LJ?1EC<V?~|7Lv%Mt&<l=N1VN>e8gmLMl&~KA$eMtksE7N!WjpjB$(cTo)@ggjfZw++x@"
    "UW0<5?#H~<Ep+@^Bg40NwI2a>dHFP=2%%Y*(mO}Pit`DJ%A{Ha46FVNfFKC579l1x+5ZOvq@Y5m?N?2~&{gq|M$c"
    "93i>(Zsb#XS@F^Rx}l507KjFzD6f&=OP_rDF8oR$0?bxOXP_N2mWDeDm%{%s1a7lX)5mZyd0IcvuQqm)2Y^2dIF^"
    "KfL=6(mS`?s!3NDn;4hf?YWE{-X&lVnO!7sCe`p0wdTUg?(MG>qxA@>sEyYFT06#NJ-vITS6e}t+f(9m(N-kH_=8"
    "rRSOpBzXxU_|*%ycN&uE27lMTu!a^Nuse9$q{ik))CQH~%>J`~Fk?`}dRMqo;rLlI`N#?QNMS+@SJa9Xoh5-;Q#2"
    "gi|SQy5FV@o^qbvl-EUA)WFAPK?QxbS`$L=I0WkX$=)x|_N3P-x`M~o+4xO`4soJH@g~|cELL)hF~JO*(6Gs6%Jl"
    "-fVIH4@4g~vG4;B|#b`+PsgiW3Wp9SDvmbde9hwl!MZO@BN*UusR84kk%{pqJZxu09j3B4)7V}2FjnpvI47M$0;i"
    "g+m51P77h!kjbc<UO;f^Bh?NmonkVtcgv!<0hV1h8H^zUQ^5UvWE}Pfi~UTVoP^~E7msCt<{n`zh&7C*O9{rH!E$"
    "-i%GUC#{oqZd<5kgNgggS@^nJqvzR5>`gokN*3TqETMKt{c!$kJ@zCiP1}gUn3dq6Z_K2d#%XpaDwvflSN-WXyro"
    "?8JBBL4FVIpppg<pbvmZJEWPp2BgBc^eV9-rQ-you}N8hupuj%wXVm7jbWFM`J#r<J*YG7g8P7_+^o-Cp=%pq7Y-"
    "o$k7<J}C^GC8hzPo|n3uzK}j#zIyOJoZ=uYN2^8N>3;YLJvGBcfms@Gg)zE@jC>lh5Lvby5Dk^g5gQhZ)s0`hKms"
    "30l4beT6;~rO6;`b=o5ZO(8DTWj1O((ee$5#3nZMAsrrfe*q?bG0_UkcOi{`NL=k*(p5}pSZDxgDqnJ<F&yPqY`7"
    "l;y>6bshO$Qg!QOE%dAFlDv4Amxe)1<8JcfRd^XGdROKIE!>~`#8t;q{ZU4gW<FJpTk!Vj4slbZ}o-NF7>4_agk;"
    "+Br?`94|S6-6Pw66U>pr)9`?jIjbgv<1kGP-k(|Ct^SWX1Od`z3poDNgeY9JBRBvtG*-|BS)YEA}!=HLA)4NR}xw"
    "{J8?{(L9s{FVa8MA?nwHew!zu4(f_}-9A?-I1fKVtPx+?=CG3jo!g5-s_UN>y0!KQ?Y;#h)d|z*=mw&p4fPr&TQ`"
    "j2`i2Q<0BB8!)!)dJr^<*{&qVdZPn>0Y=n`JhEvqg3YQ-xY!rPy2y-X>EaC)$8XU=AXXO4O3OnTi$}mv@{U}f*m>"
    ";iJm7BEl-pDT9A`w8@cwKP@XYaM`3}X5tW);|=fO+R46Zvh&#aLegGXh`9cSu2fkhLR7=_Fc&5i2v*!1ZJ*=b%lj"
    "f2p@Is?hft>4Q9gU#9SgrBWa#}tRSQzuyS=J3p9nEahe({Lr0JxQ{&=ZlJjSZS7;wo^}uMFHXh5&oW3p+s>{uY68"
    "9Sy{RmU;7P5N!SH2?`o|?{HT4Os<{HuHo}X0(_YmjmbWEvoe8Dw0yRtL#OoKBIY^8VR=(t#faP@tQr$mR1&UPzD+"
    "MoCC<mrf3pVLK#`+^kz!PtC(gg|~@atMZ$l1bFl(HPBWaxW*`gC9mOw4aPtZNRiR0TC#=qz46`10jm|No@@e{T1G"
    "7!AJczy1RG1i+?)U+s2Dz@Z6%gwqKj)oqq01N(v)XK6UwA~n6B3cQ)|rBMULA}Mc3GUpOS0@U$!it#o<XWy=ME@z"
    "9;<utt_A9m93>-P8g9#`Dgi&}AChsx$}NkpcX^`QY=jlag0Qo8LaV3#bczkC5$`uf%Uu_)Nc1D~$Ow_B(>yi0G(K"
    "CEYn4cAC&x}t=lJu3J*&z3k}cmN}t22!jNkpJZ=LlA~NyS>SVWsW+<t2z2TU=$-v4d3HNflzq>J9UaC?BRR6X&?A"
    "&2_;`3kl@*b-+1r95MQ6a%@+m5I-xy9hW+xrLJA%^$;d=LEarXGpI2z$mTRZkc(}6-Q|0<mr_167FZ}?QUOY?7H^"
    "c4i@4kCAKPry0ch493TNv;P1cBuOdxhf#!1c4)jky3?(|CDGhi!<JOEA>+1az?9k^JI`FU`6EREK<m093yNo606X"
    "-wuIxq^s%D0@MkAK)NFM+2I<jT1)05NKDiijylb4IRLV~gtIkcEd@4u^{1z(#5#G%`SFN@k?^155L`4o0XDoabZM"
    "5(VZ#?e-(P&!R?w0mxt@kv$9rU|G`H6DP!WTGLQ%e9zDoQUmulsELPK*#2l8V4v;}Y3hc%{hc`Yiy;JP3?F^c~Ks"
    ")TX%rGVl~gbR%SCCw&;29@43t92F&=w4w7Fh*xXDp9T54VSH6PLvKVAoM{CDVHNul#~BA*)HImtXKKi`#}7JPce?"
    "B$2TFuM#2t_y8Z3;c0||HP=;Z^V_np@7(diM+zKMN0~;DmwfYRcYK*;z$YRi_bdZV^P^~fg+A1^#EyZqQ`g{5XP)"
    "Hg@g2JDauQ^XxH{D>dns=})2qD*aDW|93XDp)<G-);~P?5z#DpVPROceqPDXceqxP900Ie1^MV<YnioO8PFOCU`6"
    "c<-F^-8<~cX6Vw!esM{b$AZjdZ!=T<*`<Bn&TY0Yd3i^`86;V2n*EfBVtNDwAF*F={v3$eX2pC@3zZM|$bNH*=9}"
    "K>>Vp38t8svINnUT5(G5(Brs%dr__KP=%)b|FspEMvm2K<_dIpZOe94|gkxzT^x+DxAprI4^td=?Ro&}6X-F1>Vk"
    "Mnf!;wHsKNI7xi4IZx+q(w-s0UjC9YqLpTQ~$NH2nv2glF{9XZZ}D5yg}xjNfG+Hg&bg|B=ciNApq+m5b7cd(0V?"
    "eVvWL5ziKZ1OM(ufk5l0uGa<K{4@-3a+sv^8@BcHK>j9-}+>$^=vI(J-7Z&YY;dB${g0*Gs;x3B=0{6~&i0lFGxS"
    "V%IS2V^})X!;VuVTiK3Z*>LL&Bq{7_ZOdv08Id+aH)TFY-Q6aE<LDU}B=+DNyHWS1`6kLQ}Jwyad!Agi`S+oBv~#"
    "t<?PIn2n50bl=vi)ci2a*KxZYxlrv@z^XMMuqWdskiexM#%rNN*>tAVkO|pxf!eDDchI*<B<GO<kT1da>UT$O1qo"
    "Earvf8`U(*()KL(Q5*pd7-nvNA*XK*Po^KN7DM!B64{pxhFgyYV)qp!aIy7DA{muftIvt79>pO+Glm09>jh+)IJl"
    "}DULNU{lJP^eDXWFr2E=_cN7N-VSLiGbbF%e{iWGG4GB-V)Zhz2-oI0erWnrlk<?io?q|#fHfpJ8L-6K)e%DuuO8"
    "?nmTCc$9J1bCF?^d0SGEp3Vk`xNG@H|BXSBGpD{<V@hMGFZ5*kXW{nSod5hk&B#qkoE4(_Br0Zk=nEc`%XG4akxr"
    "LCb$%GmVzQ`z$3N2<<?+^_lRR1c(hvWnoRDQQg+xa6puL_SK35PLNritA%7a6WN(0N1(Lz&9?J~adChAoB#jsuR1"
    "M1}%iGEqfjGoMcksk#C7(3z!w(jUVea?0j`y!WNJIfD9eU$P%0yvkJy%vGTQSUO{f>qLT-eEDVa{kLD6P?t5wR3q"
    "+BlEYt)FpVIrmx#G;ULgy&Zeo{?u0c1hbKy3v)aZb=<U&h*^vs)KF<8pIlW;WtuGDOXj|(Y@&ByhPGo+UKv93h&s"
    "U*;Tt_`u4;SBu-qG7|ThFC~0?9Vd|yQ!Lm=3F(KR%7~x#)BvM1i&H<gntBz0gS@wbj{YAZ??%OqhNu@9&S6mh-Dy"
    "lR)nHN$ZaAARL&)+Q|TDVNA~A+y266Zj~qFU``jBT%^LCC9@BC6wMkiPyPxKTU_UDfgu&i=I9JQR+MUDT^l>Pu`y"
    "3W3jHZM4B>Xm#uHq$k^kk2FO8<%c`yUB-K`+cC>{e^1R{yUXbpG{&7U~Lt%G&&kIuHM<tUK%!@h-Tc@TfFy66l0("
    "8p~=tJU9J`;iOioo?>xyYDLC%pU``UI_@Lw#cB=_(FST5LQH2_0BS9YcZ1l+FdQxTj(g5S7|C`9%cRZqd)x$@8^V"
    "RRn0VEtJrmrMxL1vN4vVYv3U)uP)Gbqa^r<VA8tNZk{G=FM4)u>OeyVR76y*_NmEV9N_ypBdXkms@TO`))R&RO(d"
    "dg#@7w;Wa#m_4$WAfh|H$HUn=H3;^K&#s%wnE!Y_#mlb$vqPUH+G(o^qn7K|FHz4dhVQ7%o-y)S`>))?soJ7=!X%"
    "gRfXA6ER8&0Q7o&ic#XUt(?MmRn7Yd?5i}{RMjsA#(NvWiI><&Mf;4aR(%YKbM1`JuBW61ud-2!nD&pOu%x6JyPs"
    "?$hcUn9+0jyirJ&4r71I>1Qf2mgbH+R5fi!$}Zbb4#v6lG2^g=Yc)Cym?F&y62f3O8@s0DL3DL(VfS*6LfV!t$<U"
    "ApAID6c5clpwPt^XZz-pB?fbrsDV<Bfo4de69oyP7cG)fQ17(3!fqssS=fzad3%#pD%Eziw@5yxJ*rf){cExlHxn"
    "6Ht-%6*Ji!XMNU&I!)ZOQBU%(P$Mz0uWe3F3?FV!wu4sD?FvP>6r-gGJKi$WJGRw_+a<07>f9pl!@S1Ura^n8^}s"
    "huK9&OXla*Rucke-#(Tl3`1-S#7=w?Ko}kRjAxO_I;_ON>e^*Q!^_^4Owf@iImV+QsQL%dpSdoFw{cgKg}bT)YR9"
    "B%nN0+`{?6<E3e9Ys2v4rQFu&yT4g$tbsdo+ql(U&MiCYBTV)QGe8y~Z-MBCTR81*+V*{u-!NmBZguQ1#UB+)p;}"
    "5~YPfMem0Da`n%{jKFfd?z89v-#a5W73~)5?qRo&vt5eBXAv{uiv8Yl9O1I(92r*S*do9jo~p5FY1--+D|KAs1}Z"
    "=e>Gy9Wy=i*=k1o5=#hgZ__qn3o8tv(&(u<HSTF4(|TH2)oX#olenCGH)L-3=MJ{^ll%B5*2HQV10@i_(8<y!7Ha"
    "L~Ve<0zzi&@0R#RTS_Hwm9vW1mHr%l00Qb?5;{dE{qVUJH9ADm#qwxL-7+^2`n4$qQ^7K@sU?w3bGc)jU#>T*vfa"
    "|hQKK%v^{x?**!QRLPtbY=pt9spQ&DU)a`pKQJUAb;y(w_g8c@gdrIPZQ`HTs;0`bzYu&tC9bHtqaJ1ticOp(JL^"
    "OhQT5=`5BTm8QM`}n<SPPnvoWmfC<$E@q2xM!nT&MK?6GkwY7vx6f;7@lb7T}w|xRd^ucvRUT5HDRxp)!X3?veMd"
    "UIPg?zpWngR=T8ZbcW`?r^Ga(H4wxfeiIftmyDBhO1;prQzj9;D<;1yvve_<h8@_7}tIILQ}DM?d^UveN|;Q#<TE"
    "vsj(Q@TZ>Hhzh$k1j<E>Zmrn2rU6y7302sL3UO0N&?HJFqR85F3N-lr@>F4@G7<vo5An91!XpG@EYqh9)&^Uj36a"
    "^%inm-&_4Y9NVcY7<T(hy5Pj92K8^ZkRA7}{f2xQ7Y9jE9and?VMCLHlq8(a~;1Mo0+wn6QN-zcAa_3b+OM3p_-C"
    "y=dxFQT6PQLQ>1=WzR$-js&eUSes{OHp2XX=|a>2sVCW`%N1`z8k~?I^Jn{r2*oUSlc^44Y#(?wfqjtH2u`Ask!l"
    "P5ES$OtGpl4?|uT(Lj$Tubf;VQC_T&Ln`B9HvvzTWj&oI0DEVQ@$owl#7#@sHV#gcU_u<B>p>@06H7nB6Xmy&tq("
    "~MTI-~Wbh6vjlz!u*8w1i2|lr_;=Ii=Ym6RevNyRyipCUe5S2kLNnr+?K=j-QV00pF5+_%Vt%Mlv>-kUTy(-S5Fq"
    "af2czzBA(TPE}lvSe&cQvPNI!+M98)(k08inBCi<>#YY}_txg4MpTH#7Ori`jnjNUg&emZI*K_vALltVxz{0P^_*"
    "nvpCkkOCzF2tlP}N&sA6B0)+?mR!(@T;_!3WVeXq?>#-;#j;z&luRskiDTm07XPS^NqG}FE#iTFASj~2UAt4lTOP"
    "!4bcw+=z7B8>{0A|6;fmAXJV(X%Qsa*pMtwwZi1V3&+uy9}>-`%Gw3+fdkPmJ#kM6;ux;I#ibRAun1vL?g{dXk7~"
    "IArD{2>L3X)M_Mp~YE9_Zx=s1(Zl;V^|L6ELRA6fxI-7EYk~j}QUd@fnfKhi8A#X`d)psiJ^PC3}k;%XBEn0G~eB"
    "||>r-N)CcHgdZMMPS!W3_`>rsgaP6(K|VB)PcN@*_r2Z6UtZI}e;VH6(k?dYLZL-!8$djb95`w_(~`XO)k!jOR#("
    "epCiimN2+2C_;k3%_qIc)fk<j8?}_F%IxU*cnA&MiiigMN;D9Gb7I11gjisnji-w;(mE`W3`f>u80Lg4<?05c#BA"
    "a_1<eZBEpKDULp26_Ic%a0Oa`X^#Ld8V&<aB~oyHO|QT0EL4=Z))Z?Nv`4XO>3fo&MV$T~5yZ-iAVp0tu)+y)lfg"
    "<h|b@r)OTf(huCNZ<Vj%#tJ8R?W^?V7U9nOH7>c9EMh0P^x<1P4M`Zw1C|WQm#X`wTWI^%SQRr_u<2{jdV>_War4"
    "rnu1uvZvB>54n3uT4JNDEP1%vrdPzE+7Vk#$bUx%n^%`dz*U66Aa(u`$6*Lh&7-1<t8r(hApa#r}XI;%^ePDHM&i"
    "W9Ql7ASjvUN)O60vZePT9d%a!O<O-cf?q+r3}${U?c9bwqWlloVJD*2wRX%2XT@8`G00NVa;T(?Z&d*qXmcI#?U`"
    "+wETRF!}s*DwJEn({H<72Lp&mn=O~f%Wh;!ah}*l2MH@xR>D~W^@LAD#Lns{E9MxwnW)-TJ5*i*5PXOqLZ*jz@pd"
    "0;l^)KbeWaw7DX&zpk}`v-t%OPxNy6gE;&NwSw{*YVgqxb0?&WGG-E+IW?YfG*bpfx2H@cPXxAvYtKRN!-gU4Hqk"
    "qdQ9(<^p-mt$>u=yHiKx9{!H|Jna3tl1`wEaRa>cbQDI+E`72H8!0Y==G#aam*y81=#-c7_EO{3AUj8{7>!o!CCB"
    "eI=@<<Ow-mf;xub8f(JmNu4r9##X_35_T96Eh9&^;<Wco7m0S>1?d_&`R?gL4w`y@ZK<5_q!&@VHH5TMEta8fo7-"
    "kV)r*E@F=^tebQ%lKYnU-&och^`~gsP{)UlUm$Lb@EdBU11UMb_|h8hW$Hu-Jwz^@?BJDs3mM<YHBV09uxsqhCda"
    "FFq0*T5NsWeHn5?5YAXK!?F>6bNjNdHpG+!hGoP!e8jg{ebA;$syrBVVi*)LsI{Z>U@dX)<B$)P65iO|25)KHglL"
    "rq&xmNh{-NuYL`JGRvV8;B%evZ8kSR5X_HHKTGB-WRsK}XVP>7mPQDR^kE{_6m@ia)kxUG^*Ak>6-T9nHH47lWC$"
    "8QTPX){i5q5-`Abi)WI|8DZvWP5Fj^qla~jm0JCnW3tV#QO@@Y85v;#v;wtIW%U6enRmdF!724h1yF%VQNyed3du"
    "G>tiee+QoG!bB`ul8%tlMP(e#e%cj<r=EyRaN>)V-XZ2waWDEfDAWe6|01~IuD1zh1Va-i$u~~^n5o2j^{QlyN5^"
    "G&rH;~z?7_zWSDfrfp?x(zlsNpBNwFUJX4LeK~$k2c^*=Dv@>bc%EILF;}Wa19khtA($?5q_#YmdmC1GPUYm0Zr7"
    "o4OrLWYdU+&+n2!UN?<8Lt@m^Ry`;ZOlTU~RuGP3GB-nJ8Pqhj^#<*>HA@mu+9B@8C3M!JYDB?^L17rX_{C4j2I-"
    "zY2u<mWy7bPEwq<R;j5sF?=Pk#!#)2uJ7?Hg%Y<=ORdQ}Vcd`!cnrtoU<O~XJhPO)>`!DCwkiXyCAx0@1KyhY)Kh"
    "U9!FAX*9pOH%``IKra+s|R7EbT_u5vlk~vBlC);_L9!K!Lm%iPF?U%ObdF4t`dr{GC@Isw@yQ|T%bg%NCf_=TJW%"
    "STEdCgCM^*~uO57Gp~cgV!fWsMy~8tjeRS~mvxAePy{C@wUilQ+T=+w3iMhj<_*MHQ^Tgrfrw5Jg^`s6swnK{4xm"
    "(TC<x;vXk{E0{h>%zM8YIBwMTSW=^GSle)6;2xvdB@>dA<Vi|Kn=h(JSKu^giweGe|GS`rbZPili2?ZR^-!J{!58"
    "?S1;qH5h?dFVjz5*Djx!+0C<e=o3m-%khYGq*wE2dw(CDo&lX5{rtxnNDf1j>$h)eh78TZU%vGHVH8F4J0=uA&c?"
    "Y2^sna6(_5@N>AgtrVB$@W6t&l_it2eo@0vu}jDv6aw{g|k%6LeiL)Tra@9i1PJYmn1+br7o!eTWaNs+%$L+6%Dp"
    "pQ@e%6!HIPvLYUdj{&mo9#3Du^`O3DLMFt_rQIPbL4RD>2lYiW-LN_mi{@C`rD<ZP<_>ly4mW|cHJTH+B(;oB;n$"
    "r_FmPx?wYt;@wAayl-&|uQf9mEcb+HJ0K6s@GLm>mf%**7o&vq+t)MvzGO_>PZR2S3kRld88O|`Gp16pBTP5_M)9"
    "nuOvcM|hSTZ{Drmip6g2OJj*?(pXX7Kq*QzegD^f%Rvb`wLSLacx8&D6^as6~82?ivyxF$0qk<qSyVf`OYGJk(k^"
    "QANrkoe@@H7#6hr-R|Bo-sQLKmd}|tTu^9+)g(@)^TRKMv4@a>Q~UVv6dUqBwzjcV#t>2<50EEWq;m3b82xNkOMd"
    "AA`L8lot9cw9pN$^BI55+=!7@7qx_&ui0a2mssA4B5GZM%BYtyE2?9{j7XF58P#@bO*{6xp)PzZgIgiY#$w^)weW"
    "Vc$QT;!KxQ4TwJ#7XeN;&yiBn0GH@dx`e}dv2zow#pYxgaQLIAjmu40z*$OZkL%&9i=Vco#HL~!b&6^Ass_1%Tb$"
    "b<>QY50`cdt<zf47K^yqwd^OQIM&yR^w;4~AN38Ke66@n+eKGWiNh-Ck#ce*aeLV%OM@j>-okY(;Z+%{M(-9awIX"
    "pSwx>pKBt+BPY@0?K<UXY5BILI-Q=5C~4=O(+?ZqMRO7eP_Rw}hB)07FJx6RXuTu@RE705`ieI~-%b*7p?S#5LU5"
    "!m}nlbsf95)-4|yY92MGOc!0#`^gg0H5|RcLU{*!YH$X_<94@$b#Ld%3zs&nQW2lCAaWXx5bEHEe|vF2)vZCqL>X"
    "8+x>9#WEN|-8%wB&)PY#|x-TQ;X&R9hlzIuT9spS+Ma1t_OG8f`)S^e`V@=ix^u!0{Ota+U{-BzTo!3a9sPPhc@_"
    "tulWLyDH*xl~Xjjt&;^Jvn-7aQGu8%@Y>6b~Kg<B>(pp$xg3^%!o9FdawzDsl1GKY)|2dk44ci+Z+X><JvHoCV&M"
    "pWIeFzF_$+A@m3Md?!vulW3ABU^u_-E!NFq;3aSXqGQYU1<BoG^5GX>_*lykrD<t@mcx?&9$X=M$+wJJq+uEq@N9"
    "ozw&aN}*P`${m@)8LOk|T;~#d6K#geq^+!f99OVKU9Z*1pRYCH2nRlR<K9KVHr-@bCHU610-!ay92pFchGJRISwJ"
    "mdn(3bLHMrJmM=Xm~OtCq!$>>Bdv#~lR>gi%Sf<m?6klLvt)IH`5P$?bz$l$Nx7f`hJ_J9Ju9#sbOt0957y`JuJi"
    "Hr=g#s>&p4U|?F*OIqb6!3JzZr>Y@v)g>li-5YnsCJ7X*c~cZIa(!>;Vyc#yiV(X8AOX|wbyAE(n@d}{|gKO692*"
    "Ay=EjVr#S`1vJ>H7SO0l!N4)d+}k-+!SrsBLL6aY_!P8N;|qKfH$2J+Ayrm<AdbzGU4i}lVqHh*VIm%TW4_xY+QR"
    "X%|S?9J>1^;nwHNi%&BWJJWc~=f8wmLT)}wBi<}iy#|o=}W60Gqo7?U>0GWq&g^bFjSsZr9HWEtmIehYSn8YN%!g"
    "8+#$Eq-b$kYePnM^|-n&O_MV`OIJR)Y=0xD}eXL7D_-0tklT%yd;|lg@eLU}I>Blb}l_?j}&!s~iv#pTULbLBZ+C"
    "ZV8UVVgPwdz%#}ffM%91Fb5x-y%ONFIW_ansLmlg0LN@ay^Q}I?9^B}E1OO}eRCB&pkr$Xr;Qwm_|mSy3T~fgld<"
    "HC&#zP_tRZGSCT0|HxVphv;8F=89ghL4pzx)U-^67##->QF-B~d>G&G}tfFSKid%zKX0-Gq}V_nBWJz^kK8@SY&j"
    "|l=#e0#j(f+X9z1Etqa&lO8SKvE8Umv2_qZ)w*8MCG+#kzzyp7O*y;dy}4tPsYC-lEvmPGc$EGzF8S&4u9Xp<`pJ"
    "N$J#>4<S3rcSUJqa*M4&+U4Z|zy;^zPS!MW+cgwu^!PzS$)T1#rBl0$iwkJ28>Nebr{o}o-2dDc7ozoZ3Bm$hj0a"
    "EmKH`zOd!?R4MyeBjRxVgwPqmNfNK&v#H#nA|CPcw^=gU*MA{T<%%>eZg3z>O|GLS_I5f8X0bd-_N6jwya#3USx5"
    "26~kDlJXiG6&v>QmU_Mk%JpxitFn))?epqIwm?y(SY2HcJ46l#WLTzgPOvduipz6)<{SemGYn~-QHttY`Y%;<JZC"
    "15gYXD6a0MTcMpKArOm1LAa)h(wa-*UBz!O(kvX%oOo$+}DW8@Ja6sjSz$d{DqI|-*IhU$R<=+7+%Uz5;;m2&%i_"
    "ND?5FldyCwVKqa1}s3~5-|RHkep;lJTV4+o;*KEK707J0XJ`thF{Y}wo1%_O&kOuNr}Y)nGRJ_&~?ariemt~pZ@^"
    "#Jvu!*d9i<Xczm>rase@Jbox%%E6f3|*@VEQ-=y-t1l4@0u%<+n$s)g8;(cW3<ubcr-rx;l7v%~F9FUF$KCq-P?*"
    "kqtOrA-Tn<>WOo{I)y@a$anO4>175k<q+olTcf2QJoo=gnuGo%dKwjSm!(8vPqpQaufLUH^7NmRpFGO{}7hA^(1S"
    "^7L`?1WxF#h>#2F-U_%60L<oevN-17WSu38{OWp%ewCjT?9W90P0ocZV>vJ(x#vByIqRMCL2`PNE*1ztnl>UQDG-"
    "cN&bdesXc?VYGZaIVk`Y7!$Pgc?t=W{X@6uKY<L8eKzn2=<JP!QibCyjnaoODdiREdPJ&}}LEwXG*A=fA`itdaJL"
    "Q7{(K#V8ptza&_h8hRpnw(R-Nb)-+Q(R=E=a3+8F$}|$c|cKOr|FzjPZC10p%BA|7rQK0SYwijDK>YFAfVz_!i{a"
    "$B99Y{gyjuG8HCVj9W!r`vapZWn-v!qTYacj?6Bw=7D?4J;$L<Pd4-=3E;q3?%h5tp%jK&FM};Hs;EGY}dyn%Uc*"
    "X*4_hZh7#yVON#Z<?0<Pcvn4Pn$vKL31yH7#E~aEd+b+OK@@2u&P-5#ZTbnvl#;&goTm)VZ`dI4_+g4By)YBpp4S"
    "T>4UeZN2{BP!GAMKjEhDCxF4<mcFcNUU9C$duGqP97|6?_*e=$DTG(vtQNpU%8ca&l+Lhz2Z&^JHzFVyN_vK8n#?"
    "qQf*2FgG@*_v^YLv0L})ok&QU}h3<h1?&h2wUX>M-v>8+IsklpE&#L&ZuYDxbG_=hmN%>SfTz9s4PSvp95MP|Sg="
    "BFJQ2o-jkKAB~*e~_?*h6i99{mwstoPN%Yfib|dYYZkp{5UC2vDbk_&ro0im>7JPc}7Y(O}s>V2nGROQE%%A<r9%"
    "@n7p)DQgJM+GV9f8bYkk-SszVID2GsyJT9MS1?dfll4u3>);Tah!1{1f%B#%&5h-=&gdx5TSH}Vk&YAzp=Xw*n1("
    "IBHm7x)mh)Nd&JK?X_o+FrFaNC_DCnPPnj8gFc01myXgfFW~vYWhoUAb%ZCVaqGqg>jt39FD%?QyP{bYon@S#b--"
    "5&MQaM7Ofiqarx(WO<X<lk^HZd@Z94<z`U;nVi8!m_pc2^=Qj%>246~=R6p}kliq&U?X&QHPW8YhN>=FHTToEUkU"
    "^(Woz$HWlaI3RckMKY#|@-`FmUPIb<tn3gw>Nv1F8>EN=GajdqLcQ<Ef}|39#{(`&3k>s~d)PSM%~T@c$Pky=vGm"
    ";||C^y!d*G!hTPcl@V~9xYIK??8Ari2$8CVBbhD$X&C}9yj1#p;=&8Cj7zQ3R%TphUTSv*pc`0sh)-Sd;RM}wRI)"
    "}KnR;<1GF+~|LGFG@cuna7Kx_iM*%B0lWjpiYiOrOHzU$lJIdKcHf*Mr(t3p8<BD<tnQToWE5z~DgGcc1;J-jP>*"
    "$9lK~#URM+f%gJ@qYCx$=V*WDruF+b?sB+B9E_NqO}KpN|&!fV`yveK*=OAfc6a6k#}H?<dbz!1*8l7l<pZBovZ!"
    "Ig|9IYE4cCiIDUL+2ph2kh`hC&a&A1UnmKtY`ob`mhTFkwy^|}Sjdju3+(ElEqHZA>BP%kNjk*(l7qT~h)TBpz;e"
    "_V+(10l#rvqvv!$2G8W%PWLkGQuNl3Q3?7_DW7+T(yW_|V+@g}qqce~YYMY0Pz$7*r5c66iuIC&%ly)RLCv{WkBe"
    "5g*<eD7_;hnTX}DZ5se4++M}!7qoWXD5HS<XfCM4he5*Q`v1}x9jPU6}q__DyZhsCeZ9Q5Ih)w+!vo2FTL-*Pu`Y"
    "lleTw&B%?!Zko;W0ahqYr4*&(v8P?NfqupnmvA`q*-KooLI!$hi6^&pjd8nTsZH<c+8m@?(j*y&=_J2Lt{|(bx*r"
    "iwkM(b__vFJ3#fOrS43b%!0BRxN0YAN1G3O|_eV1wj^bh$wh8UtvMtO<B9%qH1?dVG5D*m^MLiT1^@0RyDAvl1??"
    "08^^4zl-)`g$L=jCio%c;Y^6dytpVZ+6;>|vG??v9UY(^XLBkRL4K4_6~09Oqk@nqgaXN$^5B_Tx7@v$<=#6LZFh"
    "BBK2-Fiin0c0`Gp6}t@rxX4_kI?WIHgp2a7<I)S{`XO!=XBvcAhXn>Kk)Yvb>bk<5#Kq`S$_PY<@YcalE206fHCs"
    "|DL>ssGQv8%*~lvLZ)E(R;2HLa||k(FnsOMxzhAApGK`4<R6H1DB?GC2FZ$PNw-qpC|-M5P<xBtS2erHBr91X8)y"
    "&tDAIzJ~X+&{{!C}%(7*Qu?gnZ%`{zND5|+o-j>Y~8K>CIm|SY~)J?j)hLQCyR{yPww9scpBe+cY$D_I!3xuzU@H"
    "ojXS0(6O@n<Pf+&p{C6r-1%6w|s0j$(0@&hvk=2T$K~uGPGzMQ6;nFlY4e@xjSy%?ng<^=)2aLr8uDh%=rSq|x@K"
    "7u$etOAvw;LY)Z)>*T;pYErVJ2e-Q4gV=cfiH^MVJRQG*r5qK@C)gdMQj}S<WVv*XOx?{894(4s*}-#D-lSL-NKt"
    "ceq}OOYh&8*ObVXI9&5HvZ2kS+XeYfLz<qHf+>5OW}ws8hG6w@~<-T>JLG*_4a)GrI_bpKT^+4)+{OSh90J<7Th?"
    "Dh$_2Xqg4&L-^Wi}an!Fn<}w0LBV83ag-pC>kYubZU5oPC^hq2Ba#HAjMszczTWpvPhjEh<NX}j(^)C-O;R{l*g^"
    "3gM-JX$ybA&ufN#(5Fei>4-_w4hc~QUFc19f@bnbZ55hrl%%g0{g`=eHZ%~oQK6qBV0U>8MS^MB~J(s)I!hjuMQX"
    "~U<Xi>5^=XH=TL522WddKMvYMJ^t9`=h$Nnm)!_9F?*B^pWVx^t5E^ooKE6#7M_Q>i=l1S>~P0KFDjT2>NL>{q^^"
    "&pb->`Ca)?@csu^A<d@kzT&K7v*F7+h`f3rR>g!hZ}!!>mc->Ebu6+jL;ze_I)it2!<J|8j#{R$sHi*^8pLQ2tdB"
    "R#nn&PgqOViQJ|rijGMh$Ig@Uon3w`s#nt{n{-gZ!WI67bnQp82^=Nee?L*4*fpPzR<>Ac_SC0lHJqIbI=x}1x}J"
    "0+P|?iM;U185PRyxs4CLJ$Ugc+aDMaLbNcQbcWZEd|*P2gQ<bm`kEAvf}cR<RmOLmXBA{)ohSF#XxM!p3#U=kZ_t"
    "#2Y&JiGnZ}8%VXve`SuYkc~?ob28AQVr0`>miba-wZa9BGbx8KP{djOZ{NdqE`9L@h1x=Lb)yxvD{y)iZBgx4Ln#"
    "m2ieo0jHL)msbsUmN~ge{{40SR!y6^RBkM!?RyA;&m`hw^lq{}W|r7=+mNhj41!&eSNLlzL5k<OJ*HcFA>qsV6tS"
    "HlQCslQG-WX8~pz@9CKVC{nLpE#(ImZgYMrvNbMFwAax*d-sV}r~2`bmL~NW<f7Z<!pKp}d^rWKXfI5%GN)_-dx="
    "#>_lowlF&9+3ntT2@1fP6-V07pRTtHTte=30R4(D$%ztdTO>_qM@l(tgCn90%SKoKrPAs8R392YmDba+zhH0xHr="
    "s}H62@*K^ApC0~#{l&%MSuxp;S<5q8d?Q1e8E{C%q<2|_X6(q-1Nz;3`rS|=3>W*TnNBG$I`tE?8W)boTlk8a?aI"
    "8pyDq#gG#D0aTl8ymUQc<1XLgWcpJ@08g^d;x)L|MkxIPjR1#8B3N!_QcTNfO%OXivlbnj8?(Od%JU=^l%z2sCEQ"
    "j|cU{eW7c}<Zrj^s_BJC`m==gCDrPceyPfhC!e)4x5nnJo!C4+LbHF3_7loj8dkFvcxSEW<if!cHl7$}}PX8A9S*"
    "P*r}Tp)lMy-lS@br#S@+;CNVx87GPXgpyuG9m}sX$ymbP9j<mydNWQ8Z!H%qOr_w!B=i(w?N-bXqVyfOeo+)}9$}"
    "awrB9`7$n8sIB=(Jy`6gmuY=73QdrA(5=d$uO9%D22)E)6&w5*Ya5p<g-HP~Pr^z>BQ7D+hlFsXRfj(@9Uf^xN;Y"
    "O+GouSrWl1-&8pAP{+3M16YvyXTXu2w<W04}@1<1yC1?l7ppU^Xa5E6fa!YrJZ#(Hm_wMUL@3ocOKmxwz>g@`43f"
    "?U4sL<^HhAm39X1krkH)ECW(Gv0Dtt0c7Ym>R`6b*-*O+{dfnaq5fp{QY4jblL%gTY@}tA#BAupG@-l#qx)JYsD9"
    "|_4OZJan9G!JOr#zCm=LG)c<oLyNgbO)w43zL!>^MYMKc^*sWtNODR~HQ@g6VMU)#w`Ba3g07HwxK^jAANf1L}y{"
    "!Jy!S-t~rf8h2SL>@oMXzmYrJUnFxj^5%RF3h72{5WyjJ0T3i>cyus6ODV{kqtd^O%QA-k&?TDR;#6o_)?!8Vp*h"
    "=jqvi3NYk9{26Fj3*s?8L1MSU))&?v8^H#fjqL7pih!aYx`Q5*&RHNU+5L=q`CIw*xj@~4ctvx&4%r-=rmxC<UOi"
    "VV_Q!MN6v_gl~RPEWTu+KTQ;LZ#R~(eb&3?1OJ|0%R3UC|+zj?F3^)C)VYmcRalrZT6oC%%3GkMbht^L|1mq!W9r"
    "U(Z-xD2FadHS?1&=6qG=aG8s%9$+N2?rU{nv<plC2WvwEbk#z&zl%P)+m+B9`$c(?x_-&mOX;DT32(;LBv#UlBes"
    "70$-v8iJ-Hc*o#zbhG=M_97c%Y2aeFMSwW{_qb#rTBopc9;3hh0-J2k-_1Z$M%$d#0z{3=k6NA!`9}{W@Jt`fs@T"
    "3O52as=;}`594L$36-u$6SU$^!>nZZm(6(DpSYd<zGms&U?LAUaZ?l|LBCuks}idL{4P0Tn5;yyHcl9m{yD+ESY%"
    "fVbUgKva++l~CMf43TfRfjK7JV%fQB(@)db9n)e@@lwOFx(y0gDrO#A!Em&xhjFNa5GSoY~nY4YemLn^Ta3(R|*t"
    "}u0L3=qRk$b^j3?2>|##|sR8q=>2%+t-Y<ixguZ;HX^Z6WC_$sF2MHJz%@MDt32Yq5-!jKaeZ{f|q<=5NxXd^lUa"
    "IKPt9CIxCovm13ksa(tSgSeXMNCXaC+=Ag2e!v_=Z-)77H#VVi5GSB7l^Q6OGWEi49g)dC?3tzG-`{imb7Z9>JcA"
    "G&JJDYUv%;-bsH&~(;#(Oab?dS)|R5TTC;YgDuChNb$auTx)69@Bh)=Qi*bw8&d+F-<tq^pr%Ol#!ARAFz*Oe>NW"
    "<MK=9MCF$cw;z7n-~PV;@P|G|82`|xIra0huk+~RD1Cdf1~XVlzpu$<d8zuF>VU4-mLT5KJT*?3<S9xi5H8azCM2"
    "rgi%5=j<M1Xe{E8DW@2(pPq41r?9e5vAxN$agNFeqyaSd%axjT>&-q78o#1vX|H}lFcr`|jHW_@_e>^IjQw`t20M"
    "NfkQf<JaO=e@|z=S!QxsHrvFgaht$03T?(u8?BQ@ham)cVMZO>?j3UFg?Nz=dlo75<DMRYqnZyfg+M9mv<tyoEZq"
    "E82*vg88iash?I-!#KYeurXQ}R`qMe8kr*Ugso@SI*HD4pFCc*f+QkupSz|Qt`Us&<mz|L+F$?&v6MeuaF)Ap2V{"
    "=5LVTQp#jEcdW90rQcR+eV)b@sjPu2sd~<8uQBItyNWfoRl+Q0@LAA&y%X?}qXmnI=Ct7Du_^g^zw<zn1fc^=@^e"
    "U+C~CBGz@C_xV!v{ZRfO*zG11iz!f%LVw*4TJo>w-6o*HO@;?XXlCfaNZtMm5rM)K7{3D%IAK+?U|MI6!4#{Lqo|"
    "6*0uRUhaXe(tc}+Zfl;eCd@R%!z>`Hz(oj6nZoB%^yFrU!z<y}7C<xdkw%ie+9pzOkX4{u{)42Ttg{1}$1C4lyx9"
    "{%S6@oY?Gr(vgApUUW*Cw@`k)3d#&2Qcz&!L+_Z1(rB=umrGAxvc`12ThmP-5#ub^z*^q*(q8YF$hs>bWkX%y9^P"
    "GwwVxtL_+i85(xmXf&wVtVNC9%xS}w|Rq13u6RE*kTR=%cLR(I6>*z>$`fYuAp#bK>NDEA|if<8RbE=SEx_%HYFY"
    "b7jwbWVoX{8#e_G#66tV3@!y<4$B@F)ZH1$Vj;7q4xaI#8^j-VFqMZJoMf5sH(+LvY9}i`Bwow0t2v)&yxy28H7-"
    "!#0Zih9{PHIif3;lrc9ooxDZ0WMa8&X=^grFc=Js;Ru|;hHOYbFq(jb>#`KR@oMhd3hbRK>eK@SN8ij8w0`R0jLU"
    "qsniB1tM{sxvpUjpVs73Dswn#h?KY#`^xork|5b^FG<8eQykJC6HFVL&zEs{_iX_z%1xQ4<>G;%=@7QIcvhs|Kk-"
    "g$h(9$2daa|KYm8(Dzb3hl$Su@u#?1Cp&qw0DaR?U~uiy~WiESNxoAYG(p2>oTl+tdmagU`}tOIk)@P0Pm&pI840"
    "XM_6!iQsO(qR}V1mIidGqVQvH{Q&rcW!jo>8iVLdZ3I+@UH@wK;Nya)bwpV%e;3>Mo67=7xeO8NJuq2C|U+o5cR5"
    "7J@h#$7ijVjCGOJ4u$o^P$ASyq7}foKQ_E^7(%ajwaqLm*}U3$xCyx@Gd-3D&iQtXy62Hz;P#e$8I*z^W1IqFhmd"
    "M+xE6T4+X{TT;PP`z{T26GHmc0}X`iRhO?D%R%ODoDjJlDfn#uHV0nInTQ^^c-%pmVe>RXm$HNYooTihN_?rAF2="
    "#Mnk+upJXMmEEYBV+i`mrV_)iP0WyD6bV7fSR&1CVGnQGb$*azuaXo`cHN3y;MHlM?=ctQwt@xABg8ZdxbLFOfA5"
    "t~u{S>pZJB)!*bz+t0{3{b55qYf<cL?d9}Q;1s?FOAv|88+$wi9<)9T>~Fc2kdZF5P@vm03jJ5qz2@bF%z$OsNa|"
    "PAfJ??sZnOL<yEhgY!g(T2kXb?r*Y^xTrXkZ&)x65Y?(#&dfj&@e92z{a=3ARlTQ2BD}es`xte?uca+oLUi8ln_9"
    "}13vC#<!Gw6unF%7tgw;K^rW1ZaC4T`6jU`)j+`oFvOOX0Gt$-D6}SUrX}`D@=}*^X(geEH&*^*XfKQk^wbwbu-f"
    "aZ>*19M|GTy+>a(B+R}!OMxIP`eTsL%gld0zAbpm><313)^7(j7iiiIS}>an=h!`-z>#|4`!Ghh4z@frJrP~0!YE"
    "v8^nHoLY6B~LKMpFmV<2S|yE<U;1C8*?{^TliF6IKxCeqEM>wc*P?#hGlx${kjj}vPf8BwndN2gaBS5#h;k6x%&P"
    "bwALGKmh$2za_Xom%cr4>@Z<H%|Qy(EY~gB5Op_LD&VPT#Y>9BC-+NI8wL+>O~Kps+&99bB>w#x9d;Zl%n|?<}RT"
    "Qsc978ATal3CMGOC>o^aVC)coGDVIHT2KHym4>a&#ejo&(I_Yi8q3C1Cgmr~Y6Kq8^>KI?`5bUV!FAE-r1v?ug;8"
    "P?w%q|$Yj-ogr5(Yikv)}4a@}td&0yNz>?$b+H8I$Je9Yh|l1+j@g7=<~1u1aWSh&W{UrX$Q#l3p8-@S|(Q1{%!~"
    "8jxczOlkYu04n@^3#h~&+z1n8h&qWLZ(8LMZ7cOu1aGfJ4Rujt5fZF+kA$flXCRX1g@eSiq9Q?)oe0MtUl;i}D^X"
    "$<gVF1T_t*;L1H&;S`NP@k-jFrAD%`RR+WIluV2zWH+QQI){}2o-v7WYgR~UDo@dn5SP0WEOCYF)*f*q<OB)Ko>W"
    "qU+M_$_g%Z6z;_J3F~9sA6v~GK}ZRevS6LE%L&^`(#0_@mi~IxH=&Dta~*MYuKv?)%qdbaAN|@aCU<|nBGnpiw%3"
    ")us(!$-$&_KphgfK;>B(BC<*|*EybleGu_%B!NhB1Ec1ZDLhFKPgM`)%@zFTA)^(M1)O989IPvx1mFjjY24Vr*6b"
    "DMs2bIhUF3l*FXEC;k8ZTdyn=6|%;elCfMBg&9b9~pW4CJ}TtY=JI9)^C^eak~R$35o3AFCh?Z6y{aLQJ`S*xs(9"
    "pmtBa$S{P5%~`3=j_LuG7{$Uaw+@BW=#sl3&NTsvoG?Be9Zz!yJnyaxw6NprNYy*0tS5ly0<O)v5dg(e+uXORJGs"
    "`|U^PAYaTtBaNDMV8M-ZC(QLA;ydWwh{;}A0t5IPetv}ns7@U2IYF}Ij(ZPf%`7;1W*mQ4qP;B~LG(G!|CqM5wQr"
    "ale%Vmo$i+<gPRZUxG(4`K_cF*~f-iJ_gJjnl3TM%l+VLMvV=&K|aKDur|1Xcejh_9xJ@%M2I=URf3ehor$2XP5;"
    "t1p-Q%h__NZp_=8|2f7UoYumma4zlssQOBBWVU$Pt-LY`$i66*!EAmb2kpiuD-T{<)yK9P=aM@6=dv4%fH2yTqJn"
    "&pv1NsATw4_m+O6j0K%{mYTDpg;dl;V(rC$ZLc?3&|}G?AE7PAKO3Gz01(C}-|NjW?0X##|;^+=hMRqY;8?G%{XI"
    "?$COA3t*oez!B&WD7#HHZy1xjx;G~rUP`=J?juRKNu3s38eJ!K<eA1N)QOFxbng9n*UC7OzIJadhyFE<?N3**g^T"
    ">3WTy(+&@WdY)q<k1&+t5i4MGMu7!0tZJz{sejzso#b;oA^pl&Z(gmLP&rI&hv&K2ok!Ci%5n&hLkM~|a^Mv^@d)"
    "|G^-Zfk7&wVwB-;AB#z#3|P+%;!|}i4c^V60b_OcCuBLhzkEiiuUTk2Xg5bOY?!ocEt1@Ex65vwU8vryBPb^rz*#"
    "lhVE^e%>tyQ^6i}(mWEw%p6&g;14D8%9Bj%+$w2l5=_Y7ZOAGymMwRB7YSTrhF!9Un*U@ad4SUx|<{HrpojD;08E"
    "Xu}_^or>_NZSsrs<OIc5tn4d5BfZZ83eObR&dRZKL?`D5&D41OPaf`8#L`4<g$VO~@drPUz&qc0LU5aWgJ^k4_Yx"
    "*j=yIYU@@^RTyoCZ9#!|2aqw8tJYKnzDMSb))QV{Q#Yg$O`O4cD=Cmn0Y{c0`PK;SrCqlGXXNBD9K_`5;j_atV6i"
    "P%<y7__uNGyou=5)xRA8GHi@mArm#FO)l|R*+@($J4=6b;hAqsZuZ1>#2)NGnx<)$A;`O?LoWEcCmcHB!nD&UGwV"
    "veqhP~F$Y{X)^fD0Voo4~}|XExeldG_^6CgSd}&n2ye_S}BQB6M6B;j{+?2U&Bd@<V<`mEFn((tEvb7gVLWb)^ZH"
    "tVp(4UQuv$*Z`2wId%~Ei^ek`;PKtM7(Rp36Ec7LPX!o_>y4f2uvPjGO;tLH*0bKx1d=r+BZwa$XRUv0)T3rbK?)"
    "t!hMU#l)V1XlRbZi>2ht1xsylX;c7{6UGy3_IYaCI}0018V+J912V<~=Or;8f<4(m?WV<F?CjOonExm;IgB1ET%)"
    "UFh$pHMPq%BLu9W0lmxVVnp#y{8M_IqX65X&@Y?_5S-}edJ5eZY_;0+To2YEEcyyP79^_GHb9R6XW-5$cHg73ZTh"
    "4;ND*{8mj_VfcI&RA&t)W^j>!a7$@ueL4WEy*c;WD@Bsk}e6@>qv>HixseZ#A3SxrbsKN&5xB-A-~!e?1gsS!ntD"
    "V2)DSLPFb<K?c4%ddqiQx;9D*ZUsM>-%rmPwJw3M<uF&x77$vJvyk}Dz7HS?kR435WL1E4Z!z^65<SuNzlUnQv8!"
    "vPhXs7wOZ{5eL=;afF8XeA1t$E7YxqHjbnkYzA(Z`O@17+2P5AOr|Ilsl2X#sE=Z=7JB0(2EW(*kWDDpv+lM$=m("
    "H$hBzm@*F7rNm^7PcHF;aUJqMX!MpANRX#+2m*%dUe8zPV|rg^@=e$;bs{@6&H!Xz*W{2Ub;%w4OdHP6(X$)_;0~"
    "+Q^Pw$+0J`wFaFs;P=sCyBk$Z<)mXjAMZ*&AE%4CBtIML5qp435G8gE*jzry_S_K;7iEL)cZdhVLhMMs`4+ZgIn`"
    "VuWrW{u3nz3at4s>Dt2svFFRU1Xb`!Iy9nDK}#tp1Ib;4Y`z9X&>$1BD3>-jh<Nm3&>`pAONm&UpgH_Y)7Hkp0$^"
    "l<+SM(8Gwj}vL*dU|kH8IJ|m>1vWq2I0M`g+#VOz4~Q?A;cIy5RFEHK%1IAysh^Q9!*FftR#jBYW9dAIRl<_*Z?C"
    "!l+h<Q!;?1FK%#uZq<z1d%6^h$<0DeF+nDZSn-*g<B@s&yDhl%f(j2@0_!#h$Dh>hb{;L65z^br%1~4^H7w9YWGB"
    "tR-z_O;)S%SkyJpTn2f8`>A?cww}14hw|qfzQ?puW)70!E?jmM7?6B$F}Uc**PKmSdPproSpL)BxL#6NUGW3T|iW"
    "HITo=FwnOdA5$vDFu$U@f$#MjTQWJfP*p(e0c<-|hFCU6v4spzC>z=NI&W2EBIk9KCTIu*LL9p7fLM>PYhY+`wjP"
    "Ks2)-C<dfpvu-acBcF?Smx?0kM5qY#~a4ZE;k3E75xb7**lnM8F&XIl38#<>nRU_t94v7R?MQm)Gf+K=t<w@Zu_x"
    "Nc%_e71JIPxYj_Q!AmSD!r^@RhzJ!e*rb+ildv*)PE5l<cV#<4db*(DPD#4xXiP1vyPB?v#Pw+NKftS?u}1J5t|2"
    "WCljn^Qfy6}W<+Sm*qDy=os<g*IAVfP8a=RdKb|g|&)<zW^I`X`s?~wo`Y9{N$;X&p6$9)H+W0W7YDVP{RLLG;8!"
    "6DGqcqMuWE3@N3q^lRlQb^o;A(+^Q~&-|Dl;#&5JdLRnR`zy2_uSB>?K+$!&^a*A{s|*4Q}4ZonEqaa&UI?$Ck2S"
    "m*F~J4moahRBvWO@7EVQI@6)~T6F+#4VLHDF3n-f=2{gSJ?MF3xhN+15M#uxnk?G}%kH|F%2)PS2)#2iL-F0&cI&"
    "LMPS}=IxphCva;vTL$uvsZshCI=HT-xnRAR63*Ciy2C2$@sXw$}HdtEaRB{@6ba40SZT$wu`*j^3*X>V|h%$jHKq"
    ">zr$!l)=szO>Ehh12mgK|x@W>yI^tgMC#jZYd1dnFRw0%X?c7Yn^RS4Pu^M@bqs_^QB+#TuMwPzwbTmQMEu!zI>D"
    "2mh|f#rWjC^<}ANjNIj4{#*JZ3+|7c>4wsU^s!x#>)@hU`S7zxv#T5UBW@-_e;@<qW)|_Z#C^52gmYTk8`@bGM+e"
    "0Dn+2JoIduNBoN28q|?lj2g;kG}{!|gkcvlEQ7bH{NW2ID-u<2YXh<9v0;alQ`5`TCCId=rfG%^k=2HW=qyL>~ta"
    "s7(_ejprwOzW|=dc^ct)!b(s_k6z99PY#edo$dYn^dLEWLbbmR{(g9RcAD^KNA{?xoUDD%fEOS9a&VG7KRJB1ck)"
    "N_+rb|~2{@Y&MWW>F;O}QN>d}j*PeC?l{-&hWc1*xx>?6tEi?idyBN*n{!O@v~64-(FspQ4c;on{y$X%-M?cLWfD"
    "rpTJyJrQv4Od~xz;yh%RRJYwEt?HrqIY-g&^DDXy13Yv<<^tqlY_%wj&Q7w)Ab@bId}pb_-OyYuCPOequ0z=Dtmq"
    "+XmBNyPc^uc$hP(iQf@E?QcbM>n62R9y=_2Dz%7V8Z3?2h&oiuAZJP1-w*hQ^L%loNE^a;8Z!mK!oqnoqHC3F~4)"
    "4{LyuSs)OId!i{B&o+)Nkp;slh#c2)1dm8aovG&FWsqx2;One?VGz7>-fdCuwpY-cKGMJlT8k^eovnPr5BrYKObS"
    "a=fM->v7L*>T#dbkh-_)X8>yA>Lx>hp^3TH&%w&(j-cWQCjEZ0M==d7)n{`MQO1<fOPQS|W@MtCV*Ke;hlXO=Bur"
    "8o4qc}gSY8hY)C%PA9@;R@Mg3;U;+&MJ^T(ie!ZoKA$T`=#r=tB-p<cmva`+2^Jtn4%=EaBu-d@drK0xBDy8iKzJ"
    "c2iVIXserq^#_o>>ZvSboPEeJ~`_pTUtnDkx51cfnrx8=~Q*xLTDZwJ$}@BVxSXe&ctNJ_b@RYI5IJU{((|fO|76"
    "EL)G#f<^qB4$vL@$WWSiMW^?`kNN3rfa5O>C<o-@6k9M(wlQPlj@&9oO(-&mJLaR=Zs&tOM<AF?Ez)&P?IE7VVAX"
    "jw3Vy$NYxxL4all|kTFP<GGho_2r6Rdc|is>$PhDt9MSXBb+O2RN&S|O*>;sOGE!$nfY%^3*ufYNY}Q55FEpXCE-"
    "IFK`okVe<YTaVU{q`fVIQ>%n-YpcZ=+P|WnTs|V0JkGB$PnmjaVP#uvw3W|qR!ix>#ogDlbTR-Pm$8pChAGK@&gF"
    "NPRFjUzqr$vPJxPE#PVJqtVCyiYe8UdRvb0>GiMV(xZNVsgn%mTJsrMFer^iPVUPlAO<m4S5JwEt*jX*IH{Nr;)r"
    ";TrP++G&CJ#{4VE1yK3117IP<$8;0u=&*$s9#iOG)<l#ob>mPkB;E<!h3ME(yLj@_(ug|Z?eVmR=0-rIX(|q?#-t"
    "RSF3bEiRTMW@#bXMWdTPY2&PzHk52$8pFiE(KR8bcKBCsZ$NNpH-&h*4b9;k=^5${-Z_)p?qQ=2#!iAzt|2f|L>u"
    "fs3-T?U=t(5$rwwk*vrqkjbUfaJ;ZxDZWsKo>IwPUowV$6UF+)oz=Ocmvj?!!QDIe8uJlWc5U%{jt*LVywA0B{)|"
    "+;MMS2xxbafbp@XmJI@ych^NpL?NA0<43L+jVY`JPr~7G*Tr-qJtFpw{*bPBgXF+redzw1VeM!sU7)@FT2lT+29O"
    "<F(yx1GN&oM^9vta=9U=ZZUSllBsdjYmJGTeAE5nfa#5jh*eHZ(`4g>^jh<iEBv5iNGgkC8PZR@nS($E691!rw`Q"
    "!J^IE(k5V$?{zxU1~}sHq2Cz(Np?2p9*+S${J)ju^&!U<rHapnA!u7Oo%4}TZB+!L>S8mCf6wiSX@Z;JV5g@y+u^"
    "Th*~_Wf*D#x0*lu+YABbN7zRtGnvO=n-qy)!pTN8k-<xmzy&<=^Iiw|(3&eHuCF0i?)8eAX6{E=7sUn*+T|ijl^`"
    "!t5A5XAGE)zH$td9?Ji|k}U1*CCTvDg9bDs2!^eLz&ZWk4ndf<6=pzR3D;uU}?M?V^c$&wK(a1&m1=p{v4q5HGN?"
    "O~62mUK}!zI?e&p3fXOJ3^u7#T}v9N^(&5hT;7ufm{(2@&TKV3j6X1mrDy~nQ48?#Qc7vSLH>WZHQH9U$Fo=Sf;+"
    "`wQunU&RPd%3pxFGneRQiH8NUq)#R9l3IX%)P(|RJ^A~6y^v7=*NToiB?afewv!Fo(y2P|Q2+AtNE95wLb4^VmM6"
    "lfW<ODZ4(Pf?jE4PwUjLj2NTjUj+X{21fXT1y{(0`%ENBa{}T``1~hHZ?8A<Sqf<=)yFOOGrnctL_(ARy|R(3uz}"
    "<A3=mx^h|QW0wA(n-TMLr`9k{Jikynn9rY0uN<+F}EIQ5sts0LWt$#_Xa5l)H@zd~~e;Ql{0z%aU;BtT{iOS5Jvg"
    "a^6j+*LXWiOI5u#&yhy~0jC#BoH-bNV><&<N8VB&W;8YP=-zdn~$~^wL6#Ta-wz@eIHbM?1<4>)9xo7ncU`NY)EU"
    "e=5c@B|4>^7GBFC%lk0l)v`}15qxi~Rc^EOq~lGjC(TOblTnkd`@C=0wPtF2Y=kJfl@XZ;{=9d#|0{}7Mj2u=9;>"
    "n_yI@Tj5wp0Wvu|3#+G0X$QQslS2TL=-no$ok!;%<`vJMw8w)#5C&opA~TS!r<1U7dnWa3I9GO}(Ix2?2Q;yDJ+-"
    "i$9U96sKCvL30@tE)wJRi{}RBr3~nvWao89fsXEo%MhhP+RX_Ky%kC-|pKn%ZERD)%>9h^dpCi)70;MntBRyBoe4"
    "CZ0EuCw6H`V9D!YBEq*g9En~(!1IOvnKqC}5%H_y^m?KoUw3lGM)Yu}~l*Eh`mx@W~>?k!gO*~N(m3liQA5S>THN"
    "_G%Fig2z&9XsqxHP7#j1sNh0U6e{@qjB(#3yvlv4tNgu5a^{m0~=Aiwq0E-J$?)Iuy!U2Vy=GrpXF0)h;*O%;5?!"
    "JJJrxwnM93WZc4yTe;(6!*i`%bM*G%-TWu5&v1+v%dP!i5B7gce43p6^+({)a+=(LHD|_@0HpG6zyVA_Ye&a+A2)"
    "J4&g1Y&$UwG8Vo%Gf(F!;`uu|ANAQ4lbLgU-5Ao<#irNMACaiSe)2hbVALLQ$SKMx>ZBgWu{z!}?n20L}^KHMn5$"
    "-6*Q?0Q&=<2wQ0t39kg_=gouTr49b7*wRph!C13dqV4rOzcW{MkVjx7@LL1^P%-k&e{P~Ud@-gZ2sbMyV8f|YLVV"
    "tn+71MuU_!Y8EB*;0G+=0`9E;WJL&wEbX*fhkX{sY4B?sIUT9k#5rDchm}5ED;fz!ZiVX^TDy%*Ejbt%CU<4X6oy"
    "$q{=(UJgcd}8d-$lm2@JHw6Kmk-hqXuZ7QjBD4l=JrCDILW%Hd+31z^&w}XOC@(U_0B*7zV_Ph1r~~gN|BJh%80*"
    "GjbG$CqRgwf{x0~FKIkO+_^z5z5tt+r&AujxZxnVfWQ)QHKf(hQ$<-rMH53?D}s_jstrB;_migANNS&2*uI;@<#Q"
    "2d2`&qP9XvWt&Kdm&g8|{atnM7-jwxF$0MpXNG>7A_ky+O%wPO^f%Je{n6Fd<v3t|FN!gLBEK&jr(gzV;|15Os~0"
    "AM+RISHrEF9Q>aVK{0`MYeGxroE(qrw9wV;QXSv<?L@a_><`vvujQTn0uCUjtv!km78c%VQ=<RS@Zo9%(ov;i;_H"
    "tQ~`Mx$FfY3j>Cei>`VvqQmy1~bF8A7xkx14uR*lJig6kZhe`L-QcRWe61xdtM@TY~Cx_3T9XvjS17<pKB|tpbPR"
    "$2%na*yoPKxpd;2BrgE83n_ZcWS<Cf0Re%_${BT%a9!jvf^n%+87STFAnU1Uxvc+u7{FvLB8KY92d_{;|rl<@A<K"
    "+T0p}n`HuIDEb0rPmm}<yMM!7y9d62V<|`{ugW#vVe1*L>KyEzbBe3-&*sC$MI8~FT+&Z6Q!7^FN=LF>UpTTCBRt"
    "d>jqIcUQvS6Q8caRgfNvASrY-rB^LF7|erI>D+XCTph7r@5bSR<*i1|2QvIBfU>1@=DK(L)%G7j}%Ll9sbjYjph5"
    "#=s2JmJ!8f#?ztG%_-gV}}!XGU_l>O8;5PBm$x`OXuVdqyblRRuh>5;Vc-v@f9lny|~C>_ZPQ;0OZ>YTqB)hjg44"
    "4H&mwKwW=a5#?)XpKTFPV5vBMXsn8E^9Lb|ZCD2u}M2{89sq$mMUQPJ_XD`kO9dNAEy(b4}e~80)g?^T$e4nLR(f"
    "7~|oN#dupaobwjFS?iMehvbl;5C5=QAPP$e3{FgnW>{>^7Mcyw^Ay(sl*9^`eoJXVD`ymfUrp2^nNQ0K4-UUOQyG"
    "r0Ty*_!ltXDL_^o-;=zgs<gMZhB1)JiFl<zZ!a$U7r-(}+x<)^?kr0&7=@q8m)xX&@8~f|+xg86zlvJ4CAMf}ov`"
    "<crz$?az7TikQRcGjQoaPbe9dWcQ-P!LU6*h28|Jk*-s%S4$J*v5dqUQKuBs7*0busa1(n!?5je?yH#xsqEk{8AL"
    "3+&piET+u{oPa*E0_Zg#?D#hs<<mw2Ltv{x=s;gNeimAj~3#Rp-D#y$1?T-4E2%b1LP9KTHPuOz{VEy9D5;ZzaOC"
    "r8U-ZQWZE0hxvtRI&^dP-7Z0!=IBIhvS<89Xz=rFjk8bj5vE(p`DG-eakkoPkk6>ZHT_tyfbyPTfiCc%%1_J=HMI"
    "Q;-c@Dr7i%bVOz8jcMUn5b91g_5#3~gAzN_V;>i=jkSWbCfK%$N4#TR^_v&DpThmmm+yCqb5k19i^d6P|u>JpAXB"
    "8G~&xlXLdcpZA2Y(ZGz@*W{hTF>n{&rW9RMTz1J0G)AKj7RXuh1UTyKONsH4?bJ{zK^TkHJKvgJCee*%)i2Y_Y<Z"
    "io=1MTe(a&i^MhHEY$Z;rZ2;#FV8KCFhA_W{snx8*EIC?xfK6?7chzRuQt`tI}ghVWWhP7X`pR$BIXnGP{VBYCm@"
    "kp+7U95`kyJEX^E>XSr5!UG+srRg`+-AYfyN-b13vSAp)Kf-L*M#|WXbWxr8T_x;E&5EY-)xcN{i0MUVr?F&Q(Kz"
    "TB3e@7F7;YtfQf^DwLy>a%=%andGNl%9-X<>#yx)g+-f5q+s>^%GRI7=Av8zlb(h$j+BLNc&iO2PW)D3jW&!a!{q"
    "+#e<KrIHZzr6?KL*JW9OSEPPQ=!*Y{b;dd{Hhn*E69iRgQTAIk-e=iS>h=^p_Iy<=EzkK(h_(JbrAt=hPhmXef(}"
    "%BP7j$sJ;5I23ZmV#!mXQuOHhLQW{`T#}hA-BThfEipLcQSzvXY2jMoJpPk~OES42Cj*V_Ed_fW%Hn6)40sX>7+g"
    "rkxL&1z8v-wRj`6Z{WTUgyQe?9fyDiPIZv!>p$k>oh$)3D8iD`550v~P>M-^w3_4>Yg!1cmjJ)m$GaY^aE?ah<Q`"
    "f=e#va8<#5O5@Kle;X4xb@`4*$cp5C#?+Md}I~nhsbWRWDTrilHu6+2<aVcl_+%)IjsnUvpFlp=j2o|Bg2|U0?AQ"
    "cj(C~p15YJ-*vLp6juBxN9-r`7L^n{p9*$0b-Fx^A2ds=QzWMt5uP(m&Ci^<O{O<e7ci--O_5IFd{6o5vj(_-id;"
    "HB07vFsM_2qZp?R-1l`ELCE*B6&RTzotJ{)exAc$j{Zh0&rrZbW@^jG$<gjdAO;;`QNvV%WW}$aOon`q1@{p4;6>"
    "xHGqg;7XlbQ*@=w>rUa7b!%!M;mynP_0T4y@G*bwRDHGio)Dxm+gdca-`U;`Wh%3+N1R}~4d~;~^e#xWX03NdC5^"
    "*sLL#M=YbU8L{d+<jXKuA9^YL@L8=5$Ct49zsxdznGdEEsmYS(mUbTH`<%^6a{E?Y#FOWer4Wl7oTHRMTs{Pga~Z"
    "k*{g1@r6t8pHWzYIh6huUprqA^kf+M$3yi2)W<yYGs-J6a?d`76nF=6ZSyoi*87Nhu-^otK<kl>Q*JbDmK{kWk76"
    "z@<u!r^453oR(U%ggBmd}R_F?)F}fTs4l?g*rb%9`Iny{OOH<e^`Ot0qRduwG2@4IcR2zro@=5ZBGz;|TQmA>VQN"
    "cayX*U&+m&KcmY>8<9z)xT26M(w<lsJmshQ_DK{@y7Ea+BCE&K`+zPtvOtEixSR!+Q?PT#@O}_^hQX-V6Y-L`$5u"
    "<YFhEKQU4((8y>i8pGd<^c|0e_L_lTj)9C2DG1%5M`7<>G*JWS;lG?ThasM1Ou8a(jg#!hpXZ9{^b&Vj2ly|FvNT"
    "&F<)X_#3zJ==j3>s5U?pTFzfRv~<P0Q-q`fXyI|U<YRvb$q>qMI^c~sH2`Ic<dVi<h4NZ}pIa^zS=m|+g+lr@JVa"
    "{V-#f{8}t3AL?wQR`=4lh}a^I-auw)ZjPU`xVAg7YI$I63@aBY+U(ni6>Mnx_n}=z1ut^J)OhsgOm$JOKX<w*~m>"
    "a*XN*O&NIz+o2It|IpOCfbB1!R<ai1b>A;7DAflF;vW0<nA17GmDCczfv`TkJ@^GX31RxE2wVG>{IZj})nOo>$lF"
    "%A1RZdIBh&PFrWyOEW5wq`530(}<SE?9s4`tnQKPJB#ZlSfekP<KR<!yAs1<^-7{!~g(aKp^HFp40Lq#BEg4QX2i"
    "oVF@GQAI<s3$b6?cSQ`6b0_LzAm}+V0X8Gva~|P5=};%<E;o7<GO@tc&N&)(ZhMHN<Q}i0IQt(d*NCvjein!rcM("
    "mel1L+g0X~@_gO)tUu>-d39a7VCSez5+c8<|r?EY{%v6jkLZmE7$S7I&bP?hF1rORT5eiVvgqnHac2Ib2o`{$ct="
    "MCUYb+vm4{AtWD`L{vjhM3Ne2)1wX<5z&~$2!_v;mU)N&_{~m!3ZAgUOlB_A_oU4uYl$eaQ%!7q!gb5M~Yq{hj1f"
    ";G6I!ES$&*TjFc>2Gq3I-IpK^yiIlMc%iK>9kyd=F=*@HHOL2U9f;3Z{Bq8Q19MMS9lH}}B7>wjqF-OAAjY!Pt*)"
    "+MzNn(&M%+gzlwn4uoo$A2B@7DKmGwUh@Mu(In9vb#pnolo^KNDL>^vgxIT-<&+!K_8Ng5nL%kF;(s62O?vMG;ad"
    "yvKl}z7rU>gur^zfAMvHXV);vKAkPCh}xHAOW3&awMvS@Fj-X0*!#9X35)>PA?GGYCCHvxoS~_7-6ruR(7|PYoG-"
    ">JvX}E%f<aQ4wMW4e#0yMf^14LD=&`O>7QT)t3%%qypb-cP=r^W<QfL%@ev*9l6~M`CRNuH4w8O}>5W%A3I4-~H0"
    "w$L4vIRxk1D+&jzaE|%`G*+*UBa{}mkID@d5qxI1H{@l*)3b4gJ`8(V>8n74YuVLTs5)#`bp|SFLB6(=O}Ec&z!d"
    "(`O$PbNY36B$>YN(M3K&z^*MN~#T`#E*oa@`KpPUrg+$;J`GS1D26iqJ<<2ZCjGDv19SpZAGdRh}_hibLBE6cYWR"
    "{eb&u=MFCx}V|?T^DwIR-_j!_Pn~PyH}iM+7j7rm+~`i6JALsJOjEInN4qfAAe8fIU-}Kme%19BNNaXKn}irxJ_{"
    "PtPb60rg!RddUGd^mVT4K_hV{Hz#_M)Mz6Rk0g+no!ZJ|c72ejTubpK0KX2APCnBd4!~NGgmx-rhWp7m<<c=#IXB"
    "_}3^z@t1qN2AwpL1H2JS0>X(B<FwVo{#J`D4KvnnbX^@yNsFeX)!<wxJs@(qJy8^}q0+s;)y^4w#@A6TVw%XwNAK"
    "(oKBy9~B6U19P+?Rr=t27a;Z)<q4y65i^Z`R@^^3wIB;9WdV@?k$iUa&!jFjkMSa1p@8HBqq{yh<WA)O=v7qK2fZ"
    "dwa^l6fQ@C>SJ`|8YsT1CSX<9$+Qo`B8H%6LYTF?@VwSB{w6TR$g<;>a4^%8As9<M(niwJj&N5wUq*21nkV<svN^"
    "Hhz%pOmJt;hpZ>EpBn4Nj%6;n5Rvr{O&usYWq3V?iVYa`q08V$5sosOd+}Z)By&M{kOo6=hE0ala}gB@of5E0H@$"
    "E6`ZLv8rUAQVL)<9_JZmI#jpU&EvI-+h{{x_DzJB{ji%n$Jh`F`Zb}`l&W#UyzSxk!*BcB-}fK>&|kqjKlCY@q@S"
    "04iMM20+(!#c-)?%3>Wk0-yXb0UO8}i%+^RIWUd;dpId@Iwq?MYe<lY7MD05<l*|J4qT8iPeyu11QGFu?7YWf%Bm"
    ")&HH&YQ2&X<rt}L7+_6-KX}S9%6-`f_)N$h+-(#DK1(jE_`eX$?=XE7I;agt5gKS)W(-Fo#DVU4bT~AJ^+t8;!#P"
    "&7h&|(1EhC0NDbqAOkgy5s4J5K0l+O?MqHFxFehsGRm)Q0`&!h5ZLd<R1VnVH$t<hF+LmI0u(_@b3bb5$JlhFm-&"
    "Lyw9A>sEN<AiCzMF!_J=oGoX+zz~n88*%B1X;HWGoJ<O+f>H1ELk?yx>}{TIXa>$PmOPYE$?cl1@0(y|tlC$ThEb"
    "@OdKa3E_vgPmh29(utnLFb+O`1IfjqkYUP%_Ec#!IPz*SumizpQH@6cVsw@~J3c*2RF=(wCSYO^23zBZxW#(S&D5"
    "tuqOp0m8vkDwQ_Kq}%!U(Iv1Wb=!bYj5d`9?@aO<lCvk%%}f`q)a$g!peM^!_nt#`mgWJ_M~aLnv@6;)yw>EI3D_"
    "R+?xXc#ND5yMEoxJAT5Qhg_$J+vmpNz#Oz2Ba?Nh6YT26@QFgu0Tvu=jdht2f5{R);K<?|0KQK_KOO`ny<?3uBju"
    "x#-!rmnG-?pU3Vks5z(K!(U7}r3<=L0wH;3!mZkRKoTH6U?$^jCu3X7nry7~+M)YbREtw^IT-(bUOfpaWWr>@F+("
    "`Il)gL2*8K*grVcC&zA9mN34U3(WMz3u&=yk0tG0GzD$(viZgj&z;r(D9_Jic{HsQ3QH*U;bzj!yqBUg36dxO3=$"
    "f3(+QqQ+OZoVW6(d&NCxwrh~?zGnL|Xy(jz9e$bVu0<@J>D@7^cCAgQL=|U|Vxl0V+KxCwh)$yNgt}q&CugJuT8R"
    "+zo=MC^1)bHC0cZk81#4>k3P%8u-%q=e3K%4WAtSkD-HN`3wjE#ssCUju>n0sfI0UGwld`LU%HjieRCHJwmkY?2H"
    "c!JWM>#)D4NVo;?8wasC9`s9HBnoa0OM`hcPmC<d^aj1WXqHESSi>f;Ref00vyowy&MFRPtIwvmpO$qlc&{nl{zx"
    "A7)UO~%qMw=k0gtb>`ed676sXDcuSEx`(qGXNVuR}S+*yz#b2QWA2x6*=C0To1QDQyYfV%ZioDeYmYXcIu2*c~YK"
    "1*<xQ^EN+MO8MMRH17b2DXq5d{sz-gH=IsieF*J;q+m`4I7e+PWzgS0i5BXe;^e|0ban&Yeh(fdK8_)RUaTB|glq"
    "9a#fN(t(J>Lw#L@zF$s`Up!BK{=+`;>)wAJ9{rN+AHO&{>wMnD7NRE8-P9YwF~l^~|M5qv`Da$Q)iZs>I$QJN9i`"
    "oJTS9oP-pulC#LTiI{u9c#=~K{%HT@=?@Hi&FisfXdtBKVVPSB?Y+HUBMPQLF})sUn$R`G%ANG=<d5fQVn%6*~C="
    "Ll@y*b=fO3Vb=$;X#W&(IM6Lt+`0EsuY9KzD<=+wZ5+4p4@h#so1AXKDUl2Kc;Z7Z<^@V2W#-ojYK9W7H@>8MTG>"
    "JiW|_eMxCTgDb;(-QtJ!~xWn~26cTzocvrO^Y8oGeZXjH>)g)|e9>x~jAYZy;2+p39WLUzNF<QZtOPCAVx?u{G*Q"
    "hZJY*Rq1dS6N%6mzafynp=cIh+cFrejSa;D|pF#+7XnAktVEX4H{5sGY8qMo`_qXZ=lPAGXYnV#6$SK^pq2<bImJ"
    "$)dP0s91j#D=0rw8G<${APWXptZK5Xp$n>ly<%^}%CTB1dYa#0nhVA;aXsrlwAvl{%nYJ2b0;UNl!;(AMO46;MaX"
    "F0DYJ}$%$cV`db}ez2adpLkR?~UmipMHZVc5gCqs=u)W&)cDeA6`FEW|0reB6EB^FdBRt0^;A=ds;Yp{UuYC8J7i"
    "S20?L+Pbv*`9MG*Ncu!A?{0>HxsKD<3Dm@cNE;s#2O@anOT#l&XZ!cZuXcZb!Q9jy0$ffx`&=z2Hh14DL{)uE7gN"
    "N%`e4EpyrNUTS3$iPZy;<WHtvYbtqxaBavE+g}V3oM{bb(f>xVsVws;?jRtGxVSUHNgn~td#e)Wr&8a^+rmiiodo"
    "I(bsIvmC(L#Q)y&H)fjxfMcXBryd*ng<%PHAXTtjI&pfl7g)C(Cu_A9qh052X9xdJOTl8a;C#e>TIJ)}GLNoN0Zl"
    "leiQbl;Hgpc5VsV2h2drLZhE|&QZ)LLzc`puJyse%`FI@u$IH04`9%Pr+;Wu42`{z;90=}tp0&9h0?=j*Y(@!7g)"
    "{(H2(eg<hPy)m{09omO03PiceC~niwUlo~ulnC>5+Bwwj~Vtj@!<M8O!H$DlR7dQcX#jEl88mF|=0B*ly~^{!uJ*"
    "PwtFYrF{tEJg&xAER|Hr8^5`Zmp@@L_fr=upAR$0H{@xj1O81HnKcDlZsQX9$?E|*h@;bz09#HGseH9SK4$yBXy)"
    "$$GX#N0n~VQ?PKfDg9Gz>VygZmYme=HP4aFG)}LfOlXYj=z-XNbGP700WnL@4x*xaFjgxSKYPNfmw^-E$_!|niWD"
    "YWcQB<*~%ry21Q)vthGhGvuFN#N!dKc3ZBAj1R#7il1t?d!yb}oK|LHLt2n_-X%1syGzM!n4_a?LhX!r&H=!zU}O"
    "9EFw5a;7JeF9Recmq2P{$P`N8(Ux?zEN1JgKckoqgM(7S;(eq663NY-%5y(D0`xrlb2*|&?@$><QDj0k!ck+}i1#"
    "118cQQ4*&Jcr(9Uj0gQ_|7N}6jobiV^){-^eXZ|M8<RiwY11um|-8Nkl76%E66l-m7r9Ho<07J;b*oraeiLcOVRI"
    "o;{iJ6}!hNO#6PnM16FCPjmd7x~R{6!y>5*4Y)ig4gBNUR_|d!6+x=iUt+5s$GNw{EnRkeK)oS>0&*i$nEj^^HRS"
    "sDcLcKE>NR0W;3h+)_O4-z2mII`Alp*K{NHHrD&90JbxfEcF>i*W~;q#@N^pR-k-07`uTjEF6L|~tm4bv=WgN$KV"
    "kvB=GmDJoY+LA<z7doj77XqCaYwvm>Y3|!%x=`z^1eileNM@a7->xh#Oh@5t-)<*x9-kpLORLOTuswI{KP)+|JAV"
    "BA-^%W~3=NRcC`4JiUY1d&)wyC0(T!WXrMs$|A>}seDeUuJIz{lvx`Gfw7!6Lx~efSW@9?b0nsaO9b=GAK8d_Q<D"
    "zeS}eR*McxQawbE`<5q2Ywi<lLcuwo>8z2uujWy+eVibz@!c|XZiEAnrAsu3#R1l7pnjZ=*=k-I{*W~y~i{UlSZg"
    "KAXLHpPVazf|3>TrOUR73w-WJ-0xU()rFq_wqw?`78JGSLX89?&Yt|<!{`}-<Zqax|hE-m%no_e`hX#?_U1iT>in"
    "m{DZl?v+b>C+uraOYFE1BEp^APb;n!mj$Q4Jx7;1O-W_kjJ9fo8-ja9hns>ZK@7PuEc+1|g>)!DezGGMZ&|CRKyY"
    "h$L${#v=>~FGN`9p8z5ADhydMkfuSN_mj`9r(%hu+E`(#qY6gXIlyl#!_=P<EKO*m8qdnG!2vz4nR@p5kk0)^M^$"
    "mW71STMdy@b@5~Ps%i6S+GqL>eOu80Lut=>u&mQP=lEJ%EzAnR=voO##YHhe9z}rzq@PiaRyNDW#T30xHeE$)2ix"
    "VJOIH0HjcV`63YI$4vbh?_*LqHaQpvw#5`$IRfiw(UfH}?g2q?Q2us{r6z82fo>hh8r>)BV(4o*g4SSjz#fc6Zpl"
    "^s;daW2EN#VYIW+DOd&l4BL%sR5Q-xao8g8EG(;3j;2qfRO6EdeHB`df<+rAkOEj%wCllelbkg|Lc6O9;_XHD;(W"
    "H5hk?qoKCpwKOO-NeAR!&`9jOlrT(RTnhys#{FBG(a`^wadl!Z_t}I>juhclVuO+uAgX3h<a;tj?#tA1FJV27^F<"
    "hyTN}#r+QdddFG{b*C>$zXGO9(red*<{x6Qg?U+K;u@Ue9l-hzUEUkKsN2ehb}p9ZJpaChsso8+XeQ(6Q48yqjXe"
    "_I#PnK8z-1i+}KXbvX8=Uwq)@bNY@&i}7U`E&dbSvGsN_9>YvKU2I9Rc*6^6S={u;l+1GLynIsJ*jJY2ZMag{NVQ"
    "s^oC%Vo!1Pru=R5f)ga}Ud=^q3rjI`i%gB;eaV1>(UQsvEESB7x@<lV{c4{vt~hhuUz=K^;<IQR*5Z{yeh4$>xio"
    "@ZF|u9y6}d+5FYm4Q1qwU-r#7WRIu95PD?m8o`6y0Z$+9i07m{PUTnqb%d~mpc{;8a+uFgG*aMBz?pP>qVpM(?ni"
    "nN<~L?(}7Mz|Jk32iy-U4Pf4y|bp7c8e%D)9*rJVJ2xuf<6hox=#$kC+GjA1fj$bw5OZ*+ePA;Y<9<jHhd_(`&PR"
    "IFjh!Ezj_WSJ%|0XCqABf+W-IyY)>{q}>CZAxEQ$dYb-g*-n-$InSfM*gnn_@w1XyX&SiuO^{ROB_vE71EXsY@*y"
    "aPU#tf^UF)_9>{2k}x9;!kS1gFy5k%qAwf||I*`WZ0G@P&g7SDy2`0t=)CEa&edpA5mdJvziPO}F;b4$5#_G9%oM"
    "ha@X{Fn{K_z>J<IYi+;D<P;G{KD#EUuSL9l>Bn~(%<htoVwjtRcUwl^(qq_*c`N$24VF-31O#)f4C8L9-E{IJIbq"
    "KA~8e4VW*0%Mw2ZgT`qgiJyzMin;{P3Gd1Cr;&OOq-W%d}kYb?EETUVsMunFXvC}O#~Rl`SCZG^RhVBf;ns^$A{R"
    "l(337f6E)#1xr<uk5@vO)e2JRSh0_R2N59CUW}?`6h6UsIXN%=62ar0BO_nhYjyj#A$t{+Dbfhhrq1O77Xanvd9v"
    "z;D{w%AIKA5Xu2w3kaN5)#lRz0OGm*wpOZc<;Lu46s^9au)*C+5*$(p6;1m+q_N2wuZHbufyt%R>q~4qKHwmQAN#"
    ">-^0-SiTl~={+IgRz#H-ZQ3^!n?T{w)xAp&5wYQ6f;i#&G^;s^;&Y8WmY{An7`>7t7=xux+g?1=bo^p1+ZsV=zI*"
    "TCBTc{FGcRo=cRGDIWcj<3^l{`?_rLe(PnzjQWF42Cj?bT%gdQ%IkXNP)c7eLdT}9K`l6&oY@AQ#{k{-JYp@(x5v"
    "*nrEg%@v=g#ZOO2+fl@7jQ|FH%{9AD>37ZpeK|+Zq+^f-#)@km`0w0ME&|OdG-v{xo6MF;-}L|^)iyHLN%#^U;sF"
    "D+cq}{N-GTrq$@MX0(dk~RE{ErO9gNWw$;PP5m`C%h@s4HvdUCd<GN1RS4zXDNC3>6=bS8~D|kaCTnn$ngdt5NWm"
    "RYS=mDh`-K!RfeDC<}+k-RbY@yE)xP$}EBO74CbvZO}J&+eUTFGS$(<uYE@P5$>%a{NTG+1>#J)ThwC5s}{ILG<O"
    "Q4f$8Ut>ze{9`tqjBwH{U^~CSo>;^NF;%WA1jWM<QP!I!*DXY|m~aX*t|P}f=bWGK(`Mr?Uay*XTb1YE-><>eggv"
    "WdIWapaetrhpF@s#?A18<|S?9%*3JZ&&SA}pb*Plzl!x%Q;oeIt8dh-zwY@eFgs<UNH9xwq-HllWepxE?z(?I(XU"
    "5<X?{E3@QQ9Hw0X0esKXUldH9>BJ-Uv>|F-ak!Re~ly_Jn6*WGT&%N9(c?EpYYDFEyV-+^!heWjt>vN-`)Euy1;>"
    "!%{t&5A>=mrMx82^-jx^rn;6q)*&KtK?D;d9&GOO2JAn?cG7iY$@9ScIT_)I7Og@Jpa$!RXQ^(U}S^%5-`5nqOM`"
    "s>)uALkT6J$bDuuk5HTM#yjEZFV@;<%2!TPRqAbhy&QDDsS<7|RGS9ljtX^$?w6D1ZTs-C&eMiuOug=K`wX43mQ2"
    "`Dl7tX^mxqm_V>7bB^I;tl^nxq6=4Z-giC|l~7I)V{WD@UqnVW%W9<-DwV_ngwy;h!7n1HP(71udR9$9cFbs>!BQ"
    "W~P8*C~pT$ORDXcGef78Yz{6KiY_aMDA@q@=M?q~9kfOo|C#&5_ooB=$VYt&ctGuDMk_6rQ1O6d)@PCHak{6*=v1"
    ")n;c$3=f36pT{_YBU)xxl+mQ{PrUGuTYvtb!<~p+Sn+ElGYZfHFO6(xrY}2ie`uwb*AGCDXz|nGJ>@=Q;c7$M0e!"
    "Rkm?A8OBuC@g%p~H7OQ2;-_Z^Oc4o=_AU52ejN&)C!L)VUWd&O6#-~-7g*+WM^mBY3m$XhttJwm^DiOfKuzR?49y"
    "?}PxrZJNtqXkqr2F~iikeB91th2Du-1`pCyvnb*SMI|GLvcQtA@gkyDg^vfG}6{_ZW=et+>Tv<Ad60T(s)%&Gf@H"
    "AWB>sy1`V*kB1L^&YUp*@$WG!23h&PZ-JdO;o6A4q97wpKaIFbx-7X_>1w<}2xEy8s>0StL?iHK<8hT-Td$lV6|O"
    "~aLQ;z<e<5ti=*lSZEt5eb%IiRC1|Vdh;Jt-yEef(bw;gO<#ct6$sT|&0WgiJMbyR4fP`@F??ixG@fj6MC-hU^wH"
    "pY()+Y&5Hg^YAUw5hgOnhcJQlD*@jH-`tn@6lEL`dET|24PzqoFm|cRJKzloe_RBEXq1ut_Xd)xS3Z{x(!V(VN>f"
    "Sx@L!A*vM?DFrkxLbDF7wU^`UxOJ{CzJCMYx5lE#&vSOfpg5>R6-!LMyB{;5Iu3q7y5hDe=;p%CC-kpB&ZsslQfk"
    "HVb0D`F9$mMW<N44AYC(cr15UE}2<aGZmF)95Z(!j3DN2z`@-kQPocHRhk=ibv({sGow_$z;%c&oXl?SCf6H8FL+"
    "1JBjAmFrFx6Ur^rW1seZhXIoNK#`c$7@t^v#K!G*e6-})irb}+&u@bd&&>L|`z)aGaB$w^&uWeIbuFbR)WX6tLNX"
    "5jX#ZDGDmHn^nYTy1VW)A!*Sxuozq6L&s{FHQYurd&B7FZYC%PwKI|ZU6>PH|?Yok^TiTV)Up>2#RsxZc|1=1ua3"
    "`HWP0+kF4UvbsVnzzQ$9CSPp3Ho1pRECl3Vb`{L#BuG9n=%@6E=t2tZeDq6X|)|H6DsUCb#F5=XruuBvO@wjNEi-"
    "SuXBITrrF3SBGl6Hw}$sQ+#2<Ly^_M0aMvX@kIdM+yx#A0!uZDOfw4aNN|0))_OwCjc0Km!>YoggANFeG(-?IGU4"
    "BxP!{-&@6=Mt$aPs-`Ife9BAVoasCQqck@+~5BQ^X>|P>@_Q=X%RIxj=x71VI(AVAdBIx-x|QF`>Z}{?PA_SBMnT"
    "?@P>nLKZ^}YP4}@C;YXRF*3BfD5}hhe}Ia5^xe*PZfN1I2LWPW3>m9jL5mQbN}W0tm%ynir+>BZwnn{Mb9zG!`{w"
    "#Kyd?R6;pMr6TH_Rj_ntX_qx5!Fl-z8>dz|LQP;!i%Xw9{g@RrFz_Md?6!cF*Za9#Qg_vEkOzT0_t$o(18opc|b("
    "m$Oi5U#n^PR)G7QE;bSy-)6-WB18{&flH%<vvZ0;Dd0%{VqFz8%dh-!ab69vz#vx9UBvvxl#v3Ol^S{#yFG|zWYc"
    "bmaE7c<rVd+=nMK=ZZB`CMw{H+o!^>o1!GBV1u}m2wejgjAD@gWeR7_&WD6>Me~w{!(HAfhaBAM!$MK)A5Njf)rP"
    "49YO>1=7HGSw%F|P2O^zBJRuTHaGOf{@_UUL06sS&-0#h`%!N_jF4jk(NiT{ur=p&HAwm|7;f3TnN}x#e^Lj+Y)!"
    "4wgFpNJ0~hq{m=nGLoJ-AZ$8OPbSmE(vB|IpLf2(!{kNBoA)UX=`4BZAd^EZwWnmttvQA`)39raGmqP^p`{w8ah5"
    "Gx<}hn(I@~{C-Wmp|)>C{)N-h$5ngfK$W>eeOwikNnP7ySs>viFUD9g)4qp<ZCw=LGsdw23u7vtbsm;!n4g!nU>k"
    "VmREu-_pBHJwWw(Zg$L2i-~5E}}Ol$0x7%PY^K98D~A@NQUf8j7zeixc!fw4NIbCFG8soh@5Y;Ch6ZKYUkVdXZ-="
    ")12sFg+=zV8*(J*D(lG^GD1~tg_PL;uAb!D#t}?XI%<&nY%;Wj5b2fA!qU4Y*8&yY&sB?)BTPRduLq3=DUuV_zba"
    "ELx2@qgM+@)!DIn;KL(Q)YvdJ1E0p#i=7jw0ZHFCMQIRL|#oWbWyi-n@T*(Mf{*tqZ%cc<%Z_11-by3j&+x__n16"
    "ObS8E)TGsQ_TrmwTMzIkGy;`N3UB8OffTw5-3RWJUgw{<11)c_lR!_a+ll~^t5eAyR7MOl=_r(yfU?CXCx!yFQ7$"
    "K#GEi}%gYOhMpUa`f^Js=UZb)%K=sInWL1^?>%1EGI5(l=U=^M~c%RH<{j-lC}>4L)^{IhN3EBD^a^nA_AK4p(WH"
    "(-}0ttbC_UL={ip83oDqV&Fx!|LIGoSVFwy@0CjlqYqCzrsMYWxQxVL$Sgvhr)Fg4c!69PHz8{6DqZ3g&SFnT5&x"
    "d^F5lRDO32;fCbsI9w546q#z}JRoD4+N;Li3G8y?%ROZj>8u~f4KTX%{rQ2#VIrTAYi4{qGRP?#uhlNq4<PQIHlH"
    ";Wge2|O>o#a$KEXU(ti}FK5gIM+jmfUt2^!vVEeCVCjmB<gwdY|O~SOFz>oJ*&m!#K=pHSeQV)58CBgU31qsaUK)"
    "*BQe`Kz;1n?R1;Ds#(KyPjbyh@A!h7shE{lACn$2A^Jp$7upUNg4@i1u0FoszF>p#;&yqBLCsxv*UEjQl+>m;3Im"
    "$sNAAc^U%>78$4Obt-^1?<I=JBvJh(9@XZP$!ww3&N{C1!JI@~?_LH^q7?;c_T6824AvhCfQkEJa1@rdBJ49E|5&"
    "<JrmQ83wo*o4Chg_fI9kJ^Plf?M#V9{gYU0v<HIH`%nx!#ml6b}_8rVzuOSw;*;AG}!2MUiY5-8{B2a5q?n*14&r"
    "0!#?9i_NBfrmJABA7$nL`3@-l1R3Gob;t91=_eSV^PHXlw(Aj%vj=5fObS=NoV1lub8})`n+eZ<grRxz2n63e!hv"
    "qY&0IkYa%}8-{fvAg8TQxn~DzlqpT3l824ye5X8kS}79+7Mwno4ixk59?M`diGiDML+QXgGhAts0+ooGAKW`!|@$"
    ")!g#BE#h?;G)=V^fm`S-a#=g&1#73!o`DP*vSu!oH&W#~$H~}-y_ZjmMedf2?h5NSu-@H}-D}Al_>*voS%A~ZJG3"
    ">;r9fbn6P+sP3Hl#X%{*T%%WP^f1sy8GR3NsXvmbo#@W3-b=qQk5x~im#IOqfPW<GW5KY!fuCQf{V_}X1){hm;3b"
    "U&>*yUw2=u9AHKf7KpwQv38dbIE!5-s_MFXXqDWLc+$EVO>OLM?t@Ze*#UAxZ;VU=TFQsMaNVY#gf_sQZ`SvYaLf"
    "KWS-8~1#KOQMlA$j5Eq{8N1P8ZsTM^=ZWPn1PtE=qnQEdv908A|#A59xKU?o(Nm7dcRPx;1Nb8d$i=xth<W;LiG`"
    ";Qkg2R{Xt#;$Uxo@0QiZu&kGr*zfc*~K-Wc9IIeSV-7n-VQ(ExCT(B<hR)o<oyS?4Qi9I<lh9ypegITv_aF^RZ|7"
    "`)Pw8`sXTF3?xnB2U+wIW1@2^QXmN$X(Dt*9#@;2>XfOSX2TC;K$tiAQFV6jJ=Md}h?QB?v|!67ji0VB-ZGlymi*"
    "|d7#!EdTY;OAzZ$06Q3D%oU}nN@Ps5uG;0k&{uZ25|<i4)oh-3EoBd!v`%gl{OjfYK(u0w8x+&y`4_;#wuH?eLG+"
    "x|O(Nnk!HyOt-3r=<P@rz~@tvHK0gF9g*Uh!f^>wH$AKXS}Se1(Wy2GnlxqHyK07D%{*dX$lh^oqnPFg`2piIbYK"
    "^YmPNt373YqiCLk>Gdq4TdwwsuCLGGWWNRICt>zjFSgo6Y1VlqmeW4SdKH!-Xum7Pzr5r9l$DpDSJ-x{BtU`v|)1"
    "Q$i`if6i=Hs3-R_~b=PFe5H<>PkJdQPnfU|69=gvFIn++1S8;($7MvArYHz-_7!sDDJBk(KSqzwYO?&IFl@-e35("
    "ng%9dwttjbEuo@N`XTSrME`z5Wpv4u2tSKf*&b``8_qc^?Wy{2`q$?(5}V;UbL1>a@bjfRV<50{ih^JSPYS+z_@x"
    "b%9?%rIE<0xpNp2GsDF!toBYYM))@!=O`ICKxrDkQx5y7>Ruk`YtERJ*Q;y0uc;;VU+yk|%uCOYrZm#9P*^`10N?"
    "kA~7!3;aN73(x4^He|d@H+%=M%(kuQ!SrYw3UZ0GD1s4EU2D&7azMcDGCCT<UX5bA7Nio1k|R>XjInK+`JrHvo~-"
    "rv4vV@6RsI$mB=dgR$LX0Ja~Yqw$mwB06IJ%o`&)qh0$OE+BNfUIi?*#YzZ9wtyx3O*dAG7ytc!ZL>Umi0y?z2EZ"
    "mw^l$}YgSrmf>FrVnzkNHoC4T}E>U3(N;)HAYXDE@IAHlmi!@?M0Kxu6>jFITWHE{jjWna1Wu6kt99gQ2bCvyq0@"
    "f~R`x--`4&Hj4U6w>=pHmdYG~9*%3zx8fQy8Aad8ZfdflYm8Fa@vG#7V%x{XiV9HY2pO8R_I40xVnuOQSVjXyQ7L"
    "KVhG)YY$7et8pJ-dm`sit5R<?{)K1M?hM_hr{G)V1U16mmXyVN%P`0vGSw;RNOcawoTl0mW(I|y)Mal4rQu#N5#@"
    "g)ZjgMtk-%b#!-v=PwS;aI<{{4#+GCySY(bO+;jaQUj+*>p1hz=e}0QgcPyqN!=+=~e0v0#DmNgKW&mI?&I*4BAd"
    "n2uk+H%C_D_SCX87Rc=8EB!d|hU?JZR7j%J+Q0fHC`jQL1zzj`Ov-tU}z*JnWNv==?W1zTescvXePBDJT<mz(C3C"
    "$I-IRb=CC;$VtkiFxh!b<W6cm~eJ7HsHilkf4BXa3@3lhLZwxK*yiDBdtw@fFpH3a9}q!*E-YN&#yyUd@L>uh_<A"
    "QG7r&KSE|G7ON@48!frZHw+9`XfrY|6ah&Sqbo}j%cl%)jcddZCz_iVCM$%NO4}mnhWyh452iN4Ba+7~EoVzuT%h"
    "DRt{9a86Dk4{1K|URjL?#B?JQY}1xToTe#_8I6O?l-acThYxWN2n%8Aj&?LHY^qv;VJlac}09awIvgTo}k=pQ)$L"
    "Zra0Kr}EVa?AqKs%QcTzYR}w=`9#>u$cwYge7FH`Nb{iqT#F{=?n}v8)zjjr6jS13XNz3W|&(9?X6a2d%4DiG*sh"
    "qWp->pT?jCRy+jhhnuAo5nyj(J?i^a9gmFCgD@%`tiaCN6s6aj`(=Y|X<eF<-o|KTA^HCL!jKJ%>vX~|8Pr~Bi9H"
    "b#RC7xEP5x$`J>7?q*;C;tG++xbp(}Pkw@Un&pP>0F7$;}`qZF}Hha*Rwq=FTy)ztc9r5X4V7hPaqx$T}89G{=P7"
    "6l!8iiU=Q1KGjo#qDO+qgWblCPhKA!Aqe=J1FV~{j4UWujnN%PwPP<!Aen2oo$L}w#X&%V-N&<H8n7EOpU}cch1F"
    "u}bqL5qD^kD+eGbrqF0}`|h<fbuDG(%s16>{!%iY)dNpb}Z&@)Nq#{$|NVy&wZw~C<OFp|Md=7Fl|4syeqX`xy+?"
    "~GkWm8djcCxUf=rAKx}R!m|TgD``8U<;_*W`@}c>jx4#Tv;qJ+H#;UbL_?$s8ZdLyf;R`@kV0xvNyZ1)~^!`_^X!"
    "FTPpSoTN)WCPU9#Uh<+ja8KH4KAUqJSCgaJl!=Slv{9Vn(Zm;M^wCOTLsRfQZlo5DnbvojXY<8SYkU-Ia|Fp8D$Q"
    "GEIR}!EcEE@&8dmwtVY>Gi_C|JNc<Zg{!cmJc5V7p^J?zY5X{+9%4gq0hIL)vI*azc~B)ssxlW_3e=7Z`g^6`j5L"
    "p=!UfYQpGA%+uvI(+gc2O^Ls+BO4+WvS-^h<Zs|~S7ssU9E3yNZHo^XqK&7O(X%i_0=9V$yqsKqHF8=Oej{kF$N3"
    "cr0oNhjOeq~ZTB*1>Xy4wU-vjsazm)e{csb55VybAsrHrM4L;#1?wD3^F0@};Al<}KXkrWk`?U2U!Ko^lu#UEVcD"
    "jUNof&~URmze4a)!V{PGC0?~m~Yvx%z?VwRyeY}3rm|u+6jdpp5!QBbCOZANK3;h2wNR*pNeh6o}-j^#3fCgEmUu"
    "dn4f&_5w;o@q`9aym0?<PQV30h$jxQ0{23?6@wTor7r-3%%^XXJ2U+@~HiPioc5^jq2aVAb*W)Eq0EpW9QCE3R?$"
    ";5<Wh_4}1LHm`t0~`)yaoGj?OSl)gd0*2vkhmotgx3*rz9Lll8U=1*H6QUNw>ZV5P)L{addKiY{gc8n?`_eiReDR"
    "ufv0T!*o%?-6^LLcowm&#G)~j&w!8n33mK=GqR}Uh)9TL-Kp5<S=0aB#V`AD0h|vJL}At+H{sa&#`+vpaSQ7T+Sk"
    "mbwJ+)#+xPTJ!)x~q*Vj4?BY0x%5U1FZp#2-@A{a~!Fv#z81RzLnVNy3eOL=4YsIe`_!XHH)c8T7hDOQ1;d;ikgq"
    "``3TID0XaBX^^P&HB&NZqo=mev~eSJVU4+iF+Dw4e{Bge+NaJo2c5!9}~T9=Bi?F5l);`n;JzJh>x`?EW&-Z8}Wb"
    "mT!y{*Xlhh9RZtMl%dv~J5V)xtp96lfEQZAtxHG23!OGgD!R2+f|Gcn4XsLpR^_AF|6(Rk$4E+JE;NsTk#GxVIZi"
    "z<@b3^uj4GH4CmuyoLZeLa8kcme{5)Vs{XV#IV#N*M6#CZ@VCd{h3(QVTfY&+sxZr5ug(89^uOR8-R^GhCc7dWz0"
    "!qrX;8cm<u&qrGHFBTaR(17g~Ouh^C5h_-(IKam42%2bLoo?}0Y!__IpDSn779Cj0cDLCxOP>aL=AU_4^f9W&{c!"
    "!Gu3(K%196kJK-S@|_WgSk14I2Yd*Y!knFsq7293_vcgyY746U9cS4Ufxt04yCj)bT9U3zdQlB($JMBH(>8y^~ik"
    "!d98k@EGOc1T=OE-2oxzCBbo9PentMj#wFT1+^zdAyq1Z!sf({GbC)6G%VtWs_mZtDijze%->^`@QFozV$q7ZTIS"
    "XY@^<-^-=#%4IXj+ZjK>Y1)&-BFp^l<2>nqH8R37{g^Pq2)`X1&zklEGk&Y~Pb7GLiVX|pK?=%;PY^clIj>ZTg;D"
    "hEKstaXH!LYO?3|Ju~u9vhZ<UFU?RSL34w1XV0vmqGkvk^JMhRbIHh$c>lks?Kqv@(MkyDUDLg0ax*Xf@0Y3>7wi"
    "Gnvm3>ihunuB)Qs0eg4Y)<=zTKUQ<<+Pq797j3D@Gx07xqhgs_@8ubqq@;J47>r<7bF5L1&Pei_W>*<%`W;ZvhZ#"
    "EX`3^Zr_!e3YW_d=&PvljC<;^)wz#eiceUV6wcbPhb4VxDXJMeK*p;e|Js}foS&gZQG;(|XfSD*o2RnK2+zxZ}*`"
    "@1dpFN|_aTHk_qw|H~SMgt(H#+OC}Vw+93R`B(gTe_6+R-4w7u+l+XT1|6y;buz{@NY)Ivp=tHFfWNbjS2bxF%b-"
    "iW_pv|B5ch0oT~Pf`4$X!3qjp<o&-unOy2DCom9igf)wUKnNPFZltNfrZN~<@BBEsH2idkpTL^i+C84$%B{`nYch"
    "cn9vtP+It6iUEx3t=_zW<Sw6SPLoi<`ebdzK6o)s2}LQU4%$PWucxgRIfNV?op0>o9>Y?5B0N;HTL<$=!6UmVZMF"
    "xSoX+q7J>`udczFgRjE`Ovbm7o6v=?xeJ1YB8YG@4;?8JP8+=}D&s05lpQQNW;%d_*7~Z-9tP~cX1`_$i(1Smzg$"
    "=3-NEY=|HCC3%N*@z)S6g*iv{je+!fVr1s#6HHQc7TStp`5j_IhxB~0kixt0gWC11$R-60bmY#J&GgBCv`DCh@BH"
    "VhP}beAR1U|c*x`^h{xK9$lkyf}o@k06NH&U`hyB*gpkImdR<(o_=`qe(}BHp#__KFJJd105P&KUrDBKraTcr4=W"
    "75Xm=8B{XBuMpGmWe2WGRQ-}%STGfGki3!^5#mZYvu?#%bddDnhs=e?bPloJW$+(<6c}>v$S#p@b-tH+@1VCIs8s"
    "W2NZ<r$(!}A;`M*?M1*ekmfCc=&c%Jod91#v!ECCX>QT5Z<#t~&wi^|(8MYWCp5!QTIXJs0mjyOgSBv8d3y;lu-w"
    "bsk^Frx=$iAyOAUMy^hD>Y;-k29SY#dy|)3C1Erflgzg4iZhiOkog3jUJwwYgj}Otn@Ll84_j4)tOD4p0|}Tm<0p"
    "0qR6GfeHD}YG&jpH(xqjw^;K2DwA^eUd+V!1;oRtp?@&0j%7IWYck3x3`Hkc--z^?gx__#yIz;|2r@X-@t>wJzW>"
    "qzi;IEc3(Cp6GRGc8iL#OkARgasek{9dil;PVI|0>qo4_@KOUa)SBN13AN-zzSl+GW)IIY<{bp;`8|)!QA&6m<K+"
    "wyLWbc(m&kaJ>Bo0ogKonW;iV%DlBBu`vzph{r}kC`<ZHs?w+0PzkPSs`1H;<P6k7a@3*2EXuEt6=$VomDbl)@Y_"
    "2&MOgF&)NA|?%h9^eTX+eHJkGZR#k@Pc#8z{#5Q22FtqrfNfIadWk+CcXd|2D=lPHwyg;kr%yjl)IKWPn8aav(7l"
    "bF47HXtL%ca6(%^8qc#O)#h2?R+$X*NYzo1;m4$s5Bqocg_J(iqST3dtzP_ZhPYcmRR7H9ns37~a%Af|%xyhPT_I"
    "f><|V|kQHTy_#>z)z^Cdyj)Q0s?7|91dDl-rn*j37v?de{H<zN$3W8sUl0#*fiX+nkFbzZ9LtmGx;JVZV-7~=$g*"
    "H4X!gg5AxlxaMXQx_w%D$5tCBe1FnPFY<~770<I)R1VuqPjUB@`9*uY<eXJtVpgij(}70=A121Kd#6rO1g?vH)2U"
    ";#3w~SKycXc(L$d`!-aHC@Ri9a&Rm>=9OW1ZsKkl?*RXr<J7$kMx4Z=-q!4XIGufQH6lVemnlrpZ(oOB14QO&-S8"
    "|sZ@(U;vL;r{{9Ve*F%=a92$^ij`lb?@{4vu~pF#HyYXx;S?!KT=VBuUJ3Q_2a0%s2wh7%U8yppn{ZB~LhVP75J("
    "xK^{gu7UrCenm41n{_M$RS^47(;<X5J?X{dGBE12(;OE{oQz3=GqIE|PDmxhss-Fw=FKRY()Nw&7aN}8s73vNe_T"
    "%HBhgB<bFzGn?ZF*1ikMj8HDi%gV6+8~T4@@^$2Cfm(z296$t2{Bgf59`y2iqEnQlV_1y^0e?Uw-)6eK2^Hs0490"
    "jtnlVpc%o$1F!Y)k!t0+#np8wmkbN&&~*D+ldlH;hfVC1Tr$mxIGE5qOM~_&nUz2SVcW^=eOl54NLu{oQI8Z(gRZ"
    "#9hoz?-;D963K;%Fq6@s!1n9@`T|$9!I@Qjuj-J$JwGF8&5}=$Nqk>60+}DHAWopYTwFlk~*C0F9$T$E+CewsF;m"
    "8dGlC+_v%_gdB5g&z%gH6lj<%+8u=`JFHU=`w`+z3_9MF@0H{T9O?H9-Rgpn&o)E)B>y&9HvFr+U)C6poZ7ES0*2"
    "RfVO89ABd+Rw48p!*n}&$#uI;Ku6xLAnLKwm0!$-3%P6z$M;OCAlol14xt0z&S-=NyS?qf@HsVHq&Q36`5|MkO@<"
    "vrD`K=~$U7I(-7~m*JV71~FZ(ZfoWKDA3?RKdU_FORtcT`v9t@hW++=+VE)==ffE9PvH(;N08gS?=Vo_`<6sJRN5"
    "k}lu!-?gdLh&w_)2`L$(C6*zhMKkPutS^sqiLHBy2DvUl{L9$099BQ^UmCu%?H}=0U0pZ>|hzAZ=vWIxC_T-+qDd"
    "m<5iUt1}5M%Ipz44it7!7@<pYII8O2wTNnF;$+&_0Pb+iALdQ{+(-WPnWEyxqq;p-YnrcWgoCUdV?qqmm>ui<NqS"
    "Aw(KWeoA(rx_+Z$3zpGY}C#MI_ORsRoV{l&`FO<eVtsLGntEQO`wESucgDPUlzBGa&0EPtGDo!<^WgaB7josCKy5"
    "0Y*$x{^y2f2m|*|?2bi;kyD@@+r1R1a+CZKN!)?S)5B^5>~^W9;c;6mtr6}k<hemr2fkDf_H@_|KjQWGuN{YZ;O|"
    "Hab#_nHsGWBH%tt@~2OlZz9SkE9n?_@aT!)Eu&)vuqwm%O%_;6F^Bzx@34bU5tu}TE6!AUx!;GxG>RewwsQ<sG7$"
    "y{FnpNKkZ%9=Wa<PwofK+Tm!)>J-ij)`2e43D6whbU`vvUoJ5J+A2uO22Lz5%o|&zZ2Ncr~!96*BHq^sT}>XTC(L"
    "Un{z-sX|;-n*SM<ZPmbrp*sV&+r;GJ!YSMp3W+oI`*0dKUt~;*htt|wzN)Qkc{w~O9+Styo*<@)P10q<f`d-M=vS"
    "hH5T37Z%&6|UA4u))@gJE5wn{m#u1<iwU5epoB^g5r$s~pvF^5}&TD;yCG!RiHVPXm$Hnp6<J?ZN;D&g9=tD6H$="
    "|Bce({wECPpaql4GC1`-{4{wdq6eF<JQ1sF+^4QF-&$!&`?c0w37sIziZ!Nk@(yala9@+|)B=~yudGl?dl>Gw#uo"
    "B{a6LJbm<+X)9*%7n7VFSR3x4Ya;l(xV#hh8%s=t~ZPPANct>K!#n^HVf9OSuxhj<fVK4$^a8VsN$^OV3cLyKj(S"
    "}dP~Rzf|19<x1bGbwJJnDrP5@3#=gek&0wCf3o7nq*g(t{M)Z+F6rqeE5Hn@bb`U11HTMTXTJKfiuiIdpr-ENkv6"
    "Eit|xSOX97#F#FZ)yjNa1$18NQORi|{97b{QIt(u`suz{}%W>ysw8v<V%NfFJWVa!ip0jD*)ZZ^Z$Z*VCB91v~o4"
    "5eb&pny1DAq0ZcEhQI0TH$5+yA}C4EN-LgC;m*8jhE!<(Ccv%%Jj)fgez!SuH%$lN?Dk8W>u2tq0_OU^fFd$J}nR"
    "^v$dz^#RN?7&gWa&ne4qy@$t%GLusjE18msfhH;9mBEVgk_-qO`?@HXTPUoJ%yw!SCyFU3ipV~gsya}F_fFumN)`"
    "@yt_9R-)3&l_A38K$cL+f|CJf3@N&;&|&<lg$03Vo$DVanEJ%dByF{%a6lP&e-xZ`oQPlo{{U`W1GlYti`6jD4f6"
    "^Bl>lWH<sQ2AEfP?cgBAdatsKTW5}$4O>=HI(R3i?D;ahhr^UbkLnV%qbH<w~VuqC<cIqimX~XA?90dkk*2*7g??"
    "lw%Tl^nod6CS}T^MbOq>QqUO-ZE~ABTZmeUxCzI?r`TiKV&Y&JWhrBOzr?YR(2okbOMVhdlX3yxPi%OAKmTOT2kq"
    "S;a-*Qm!6D<**${f@X%7cZ|J$Zfb<_#RGM`zBKg|ly~KHp4+C^=&}fs#&u6JkMRn4|eJcrWaplOnXzgHc36`MpGr"
    "*ZYV2AaZPCbZ-_R>H^4yLdBQJ3c}SRvcgW?!W`Q`|HDkpSgvn0!Dx~PT=has6oJzva2Wwh@CG6$jtJ3ow75>*ink"
    "RqQ?hZ3<rkSXXK=V~>_}X2Hl;`TD3(|vidaOeGH^Te;1PR}oJ{Fl#Bd$8Sv%(H{r`A(aI*iJSO9WPqbf)$3Ptx(7"
    "!&Qu1}-6*1DhMn@oQ&|&zTmZb!l~Mvc?m<j}riHA^=Qep;;Z9^K2~o+^&dHoa$ayBFexpF&b@}Enq_`b`|Ml$PFS"
    "v1gK69I_zUGK^Z*Og6s=8^T)p)nP5kI<~UI!$(e__BjFi0*XC@Q>U3GIO1%cDXKbc^QaM;%5kB9+hiFg2!D^$DIz"
    "q6Tqsu`@?N@xx+)P;G6B)$BAGzViR--I2#b2Q<N>OUDbcVysUGtI=sEF>BFjeuj;nifYB7g`=Q)Zdn?1wXkJE#O$"
    "qC>cf#bm^qFA7zc`SON@dlFN!k<r#^I-bjZ;?@|6j5K;aY5l?iv^fio%{7hA1f7OHHgPoZOhDthJVoSkAz`CszOL"
    "KNnN~vOF-O#DUYSWy$f%?#g~7FXl?bTFz!44*iQ$MOxsP415E&&#gm*$Af-=HaG5j&*l)Ur|HAE3JFh3yw&@AJUf"
    "G)KFULeeqgaKR|sSZDz74xm3S!NM~SSlhOG>2_E%q6sj(6^$&0wrS<Y^c3VjKjT_RIcG}^;5T3Kagtf?WPmYlx(_"
    "OudVCX1Cu?Ray|Gz@l5sTzfGre%)|T8XJhS0f)i>QNf@F~3+UlWUqQs(7ey=NSt<bwwcen2(i4o%pZu@mgCnO~$R"
    "Up5LUU|9Cj4h9kf+~{h-RsYg>WKA{LDk+bN}+t2<=lJPtiWSi#~o?majl#!qybBP>!?N<f`O-w(~N{Mx)$dHceME"
    "j6Wqj5TF3iUM35wV(}G$?lGss7C8bA(au;8RBStDJd*lWDJbR9&xeO9Wnf#HxYZc)rU6Ny)FGgn@@zu=yFzvCniA"
    "ZwExL|jd4dKeEKgZonr!Vjj_mv&z%Cg*YK7TaY!$$E+@K&~NxMIv0cGr+?7zhmNusGZN0UzRWwJxufaE2Pp8g8Dh"
    "_OVKjeSBp5T(=QQymsVJz;zTb_e-Jkm+Lvrt*d_v%|6DfS9RvI4m=g1j(j$l?_D@0#?{RI_|q7@An5suMduX04m*"
    "bWb!B_bv4@|Jpr<cZPc#8*6BNQh{^m;SWYCu?)-^ghcs;0&+v_x+4Qq}<%524@r%y?xe-GRGt+B@)rOX>V0znvbL"
    "dUEdrDFMd&9arhr<710Zfv~EG1W^4E#lNfKvUeY1SR<1uH*Ksazj2`jF-sO>Egn|6CiA{{4h}E3q&jOvs(vSZb6%"
    "MH3o$b28|tjzB#8(v3~`gPRp|(^t|qik}7&J=y>FpII>keNvJMyEdp9b0cM<AB=5|m}8UH$4RzE0e&7yiS6*W$-V"
    "6>@xq`2RX)T9roBd>hKvVWv)L+k%N8q<)f9FqB~brsf|;wA(@BNzp#o}=U;B#!PFFF!8RMElce3`^Hu3(Ps;ka`Y"
    "#zx)A+37KYNSyW5JOq*QI!e$Mk{~~{O-ovb2S)=?j}noO9{?Xq|2GKs^g=5SM`3vmIvcO-J}WoddJ4lV{C#=SZAX"
    "qjxC}atD0Ri?RRWybqYN@yEb_-zH^oG8n-AhVq%l)JzuK33#tJghBD`KTO{jr_wD}L;A%eFWpbD0s$$6MVkSFeZl"
    "IS*U8DqO3H8@$Q9E~bRWSMpnWU}%ic;NK2_^2~sIo09CzXWZz;ed*?E=?_VQUd0$>uXpbEDC+tC*)t%)}G|&YPi5"
    "PH7^MB11QSg5{!xq)Jwfqm5G-xaI+^sSpdPv+S;}?6z?=L~=u;NGVIz!TkU7RxnS8<ei{xfi~bJ;X}EZ0AiQtI2G"
    "T)L!Dm;#xo1X9oL3rGE`7o?}TCY!nM}L+H)pF@v)XiLpZpaKqRPQmQi=s+a^2g8>LCpX#2K}TGiwtSRFD0247}T+"
    "^EXB4y{35eUe#}Ck{?YF&MO!Va`!Tm~u$#n0z>rtiPHIEI8W9Jj1@W<$H_hbWnSfP>qD>x^7m@5w^)JI1BkT<zfV"
    "*M2p#Uay5}`{Y!?poN&=iVvh{8DTk=7VmMqHh=J9z02v6k&Gh!S4_W2&mKEg<91Nl<2Q~yJqG7{$|0d6Y=NYJ}xw"
    "gaBEE9Cmj;H20HX<w40a&)oqcMpaZ8FVDNUc4$El*BO{u0JiG=)hpCv*n)_!tQ<7Q#d>_-C1K6(>@OqO6%Y5zcwn"
    "I2-p(u?oAf^B)Bc;tH2}{1j$(6%VW6BS8(jl{u1qPrIKrN6FE##$3KEyd-jB(^(Rn*75fA4ELy?bFJ**&9?FItd2"
    "JrIl-O7yOS&m;53)ybph^FArvOD?*GQv5m&?yKzx6*_z`Q#+nhnIGsA(Dsx}KWS>{>l@GrRO{HnV`nZJ2fzr+1EX"
    "9-yYA2u6GbnB5=0()yay2diN#yZ%{IvcHo8?u*Od$8Sk-}cxIx%M8#6420AV+mtZ&tokRbOU9J5g_{T9!ZDcNj-X"
    ")UhR+tx^5lC+GQnbGL|jxbZ~7Lz}fo5nR~T&8ypGd{+cs}U^%tv-+0N5s+}2MYh6HZSOgQ`9X}z65|*mIVT`Tq2n"
    "Bq{IVL%VNuLpKtre+JuA90}S(ZSTCQ6STvhjAQ#1Lq>i~}S>?<I(KIYXM?nw<T~Z$qQeICCd=d^`ai``S7zA0*xy"
    "cZmmzw>L><GHt~Ynk;L@NLxi%gsY}x=|*W!4|94qU!~4?Tgcjd2D#QvRcF$HjN2&wbdtv07GC0Zr17Rqmo+Ibkqz"
    "n`W09K;sFfS!$9zGiRr56cu31^YsrJqee%Y@RbUZBi5#ig;S&z}%>)d;Pl_R_Hz;?tAZ+RlUw5_glk%sMf5wyO^t"
    "?mSr`_@3-gxT?+FSg^5qIJ&JVr2<7doms;1N->EqHWYbJ_5$)1Z7N&U1v-PNJDd`M`22^L{ebBFh*L0Zv+*a73_|"
    "UgGOKhQY!Y2-@e1Jy4TOYKRn+13I65sIRpvmCr4x>ty?rz?;MA`bOq>4LF8OTCc@&a`&w%X&ch<Aasp6EAPGa13b"
    "6P*Mj8=k#$!&aKn$<vJj!G_A)RJ{5u^jV$tVa_fPqLLn^EnZiVtD&QSiz7%rk4X)`2%tVW>70Z07?_Y|>Wy80EtW"
    "d17@WK|+*0`{%`F0TKsBjL`TQleLC1^*FzO^Omkj`;Gg1V^_UKbNChtMwlT1v9UL3O+c%J**f*$_a@JlWBM_k**$"
    "Y@qaEGx!AWR?J8A?^{~pHII@=OQHOtBmuC4Vct(_qEtg6r&>X<jl(CveU_*>i}f(*%lKc8r1GS;&~87)$O3(fa2U"
    "~YjD@%<)-Mk$$`4S)LNkPF;SDqG6a&NEAenFfBTsLZ2N+g`EA{oq1puvq($CWsj1;QYz!<0ClU1~%$YIm((baTQj"
    "4`(%Ib_!pqv*U*H!ot<5Hot2}l4+MLpFo=<OsP#Ka4$X)njAy5*Bj6vD6pHMTV10WDOm*_$5!!xCKOb|rQ#eX2bj"
    "KOQTH#<Fzi`GCLDmozr5_P<{gXNF3#Pp~1+Dyj=ncN`Ga8AyS&%Zq!wunvM)4&41PVk81$vy4_Ah&S*JIw*b#YT+"
    "t_%hBh-(0qvOzPRsGA{U>a_A1+QNW+=}sG&=PP!l*nNuJh*<``nmebDS4fReR~HL-ngVc)2Y~!JGs3tvjl%YwEs7"
    "<3Pn4x&a>-`4`4Xq#e@ILk+%Y@$DU-!h25<08W;OhVRw&p9ry=mo;52+6k4NKjyktS8iMyz82@k2@aht)0=Et^yY"
    "uO|@3GK}y8v*TWr4oZs>m(A9!*jSK*Phu5`-->dMAB!8#4*@7S;;73Y+l8BW3w_PAgSM#JYV^zWkbL|o=u%(n*DQ"
    "vGOZ$|P@<0o=VfqQU!*3q8aBo`HGnxO%z<kM`H``WGqO)FuMs5oR-^ti!m2AS8GPmU*%-~xQgzadP-GK9N=vq;T0"
    "nxK>#U*LSQL8dQ2}LN6j=5p0cO8pWY=#N;q{w9RYlAI?RrXSwWgc|f_HnxBz8%0O#avD@zL|M<F|*;--56^LTOBJ"
    "D>-=`)X3We?@K|PGW|>pX)3y<KZ?ryiM$hgC(U8@-*j@RZN0-{h<s;xOZ!=F?9OkU=r|fnpE};c7P|BzK{01tMh="
    "fl{lAhJjwy~=vYJo+u|hU>IVInrbgYw%tK0dogM15g@j+3gz(Fx7budU>&Ojw(xXxnwa6NiW`otQlBt|V~i$0Zz?"
    "U_$QGYF-;Xf4WoJo(fcpFc_O=ri~rIWK{=jqw)Tfd=pbe7NUyMFIL|3uqW;rKKOzJ0AKD=>sst@dzdSmearUCpVY"
    "pPskn5R|AtgU*r;QEu8pv^!|9ds;*nXJ^Ua}a!Vh=O0`AHE#_sX9S6{%*_PX+lTF=Gz!O^aT1)lVc~`38$}Ln~%%"
    "vHrb?DZL1Y)}g>8D0>9k)iK7EC0B(D}0Wq7zmp)JVV{nw(N9WR}x_{q9kjGq-zq!P=?TcL<-2HI01)20ELzK4#NZ"
    "-o*iGS1G6yNM$>UBEFB>BOUpZ8CC5f)*wY;haa#EjIfW<s-%$1!DK#J20I0_{3^u^=00?2y4}J{RC2S+@(y(4!a3"
    "Mzi3w^|?k`u1X>Qjld^Rh~?6%+Z21``<TKFDsJF4M?695M{&yf=b4M!}jDdv#nGdyT%Fe(1>%Z1j5Jt2_vaMb?r^"
    "n`AiZ-n*jsE$>wRUDF3FvoszX=3l0_uCiEA9v?#lP*+wq!C&_eH$Y@csB3vprL!Lyh;?<N3x|JweLMQOKoOl`GS}"
    "k_ajsTa?O3G6Yr@P7wez5+k|J87r))qNy-iLNG&M6NAX-d7<&GpNsEkU?U-ax7Z1Pn@W@22@Tzc9xgWPsF13kd%@"
    ">>^k}1NlFIrf!Q=0H|c<j>ChVh;==^4R8#HIqS6y+pnyl;0SW5L))uAfIwYR$Yc)f-;Nn_3v?o{WK|!z9&P#uAD="
    "jx}x2o{aVCJ+c-YKc2BX)8W?eXvdkc<igamtA^Qvph|3K%z=qh3;&}Joa_Id*!xe<pFHg(Px<mwU$Pv-%{yrwcIe"
    "Y)8IwZ*sc?iVXdZ+1iV9&TIX(Ur{wrm!7@ge+ouCx_UF(Q^k%ZYSMK)CU;JP;2vU7-b90{T09LEjqP}Ne?kc_&Mo"
    "X~J#@Cf#fmW|`_uSNO60~y2mcm==El7G9lg$i{yKBa<2SSYb_3JJQtMc{;5*I-<k;^fzZvmcLtJ~PF69O#x`D8}0"
    ")x#i;J0sST)ve5ufcf=FPXIS4p`SqHDlkqc(H!sbKPR$mx+$73&48p#<=!@F{6<1ppYMmBQf?O(S4x2kVy4GT^&@"
    "zX?4+#E)Mg~(8zCsalNmv$(jLd!%tPbB*G<su2UQo_2Ic}&1W3rn_q`SDh@)254FrWuh*H)mlPRVqSfbNx(1235*"
    "#zaDHw1^FXdfu%q93G0Uxj;xk>J@jJYk=#6TsRk>XyKor6!5F<4dqZb3QNUCzIJcR29SlpIRek0e3jg34?v%v=J%"
    "dS4Ts(Fp5G$C`+<3BGP)nspP}jq?qhTz_9VexxlhuCz8WWjSAzaP_rFSB=M|9%ePZ3y`h^j{(B~2k>l@!4B4}l1e"
    "7)7W4?KHML%O3UVIqn1vpW{3TR)$fCi~pv#RGjn*4y;i3ukRp20dtTF9vrN8lPYWlNSBSOVI8n{A&n0qV}<3LlZK"
    "q3v`&i#|Gx;w|{cccE3;;ZfHBaM@P@@>$m&qY((zDy6jPfR2h8Nt=dXWjxluUrh2Z!ke!5$H|`~T^!M9GdZ=Re>="
    "7V*(3}An(5m|0<IjTzBZnX>0GnoHEaAK<5PwN%SzOfkl?~oUmpvY{A9GK88%+?-f=l62=q;93w}(H_!*k2LD*9=6"
    "x?MOlL9m5}43yDRkO$k2=u@7yYkTzEX|`RTV3s|+`Q=MpIj6%3bZi7}L~qleMYf?|D4FwbhxGJ&%_J5nE=x9mFZ}"
    "#$6mEaJxBLz8J{nE=l}u3LG};gV7`+TlJbU)<Qg*ca?3rmtOOZalwbxj)HfU`m*B;;6uF(4}HaF1rbQjz<ohBPKE"
    "jj(m!f%@|gRH-Kn-}#|=6170ehpI7@-okso<RPiUhUeHFbaX46KW8pnfZl!lL}-g{8LIg0;?f;aW^qvP*hhJ*`AU"
    "`yDEFSDEyMird5IP<L7gAj&kxt$A%`D@_tmu2)k?JmyV9l_|t--G{2?pons+k5YH-t3s|bi_--|2j3B&Pl=g<3Er"
    "#+NjBC2QzD>?oFSZdg0_zAlh;p`87@n4u#RTH2yf=k2eL8t4<tkhtrfE_oEex-zmQ)9KD=`Zr>7QKvX;M;*1RtC1"
    "-&%4OAC8@$Nd{DHy&8a^Jh~EOSqA&WpQikg$<o3(dL@z1wCv;-GYdnYa}ri8DUP5$JL@<x@KV5RW?6*^h1m#W;3!"
    ";pmP=_+>U9phM_`4is3}R_Tn+X*`#7QMbz8=bhodrU#gL`w&%>BHXpUmKR;0EY+C!CoPEHgI5F0Z_bZu2{GO7=Da"
    "CW+X_=eOBDuF{&k+cS3jPn9N&ne=Rx%{WL;8B@0VH4nn3Iv|8d<;>f62{n2^OP!xpXel7Zcu0n-Hf=HFu#lkv|3X"
    "5&<$Y(V{zLLXmig5zRDfTQ<{CPy+qrGDvP7O^H-P+)WNP0{m5o~;=>@l%@zdeinXArXdA*272|~KTfrn^#nqu~bJ"
    "C!s3gjs!x6Wnd(HBbqSR@XP-0I<)!RWFCLu2Gyg~d00c}4TL`~B(u(cV7wBl0>^JXJ_8_y*32k}7nLr<27Sx>#x)"
    "{wPNi{0;5_-tUr@G+beB)tHCra+s7epkbj<H<Oagg5t&XT`44WMx^u^^;EyK7;2*tV_lij=nFo>6Ptx|jnH(uARB"
    "V+<I1c*77(v6`ja4YIQR?e`ZCpW?QZ;5iL|_sL~WR=*ZXgFe?C0xpPua=?)P`!ob8|VPxtqZk6xdG9Qmj1?TyM58"
    "WwrV5q%*rAr_vpYz!Z?RREdSCc3zjCF}m>(zQ4ddI3K{UX~>J3L~@+DL^~<FLdJL0OaqZ<No*iyJx2=8sq2t_*D%"
    "(6i}z|u3J9Md!#B><uI?m3X9g4x$d$Z(tiz?r%!mY_Ax^K6_Gt>8*K~i6M+d;Q*97LOy{Q8^9C0t->BtMpXlP&w4"
    "wmTS}_!yo41Z<iW{pVq2BaicWP?ErSya|1M&P=;wWhU4BLxm7m+|w%fSVCjYx>baThcN!XL8?zQJon*+lp(e*V6m"
    "je&A_IDl@tJ-#P|0{7|m1M49~7+;LB+y~VhM|O|d@Ul_0yLB45ui9=0q^bIp5XqaCu@(oK1=lpo<-sktu4j>08IT"
    "2-8&Iq<49=|(AO{dfB>^vAUI)`<O{myd-%wx+xLem_XB}^6;cO146ZB@sDkU$ZSbI$>mSUQww&28qE$$*tHH<}4d"
    "#FZX7LnR$tE?8d`N>Kx;G$YKP@)3h5^A$>BC<FrAdW1Vvu$y<qV^IY*eb_p7!YmqX)J0<=-16_V#&$PyN6jzN0+G"
    "+$?K`ya{H0_P1jbaZAmY5F`BCo#<PfOugeh^uS{4El9Bf<-I%`hIasizAmMfDX<)E0p=%B{i8Wt$Xq})0#u0Hhxi"
    "TIkv%jxEYGEt0&E7V9I<4mnY_dEi1Z~YqaB!MkHb1;FIX-#4f0BIvH}{R#`=@&yLu2?WIXrlKaF*=o%a6zX3nnd~"
    "F=327_mm3A#pT;Q{6^#bY#F=HGZVYxaPsJjbz12~uGPtudPHf3Ya_($#9PxU2428nW;R5!rLDEoB}kxC{i;sCIaw"
    "U(1`%nYoj><tLr`a_h(F%OYF;Z{C;8VIv$s4vFx{VCJfOEsGA0*5jaNwk)aXY5f#)n1{Ox0}Jz|~KnOT<0MW47Px"
    "sSXky9Bcmh*AG|cJ_{H8l%FWaBvoBDB?{YPFEv1^j;sGc5wHNs7x)E-@vRPjBVw7lQV*Lu{92c*AqD62~n6~td}G"
    "7eB#&AWUtIA$TFLP_FNVfrmKx6b4lVjpJ(OO$4)XWt|~4efLII@LTr$ZlBGILews|DS}G9^PVBj0)eq79hyat96z"
    "fh|y-T!8PLq6$5WgmWgnbr@n1NvYWE&|zPFTnMJf|;;B;US_M;Xq@?OP8y#2E8zYJZyQ+4j<{{d-ly*e_ElKILDO"
    "S=wB&YY1Xw<)(jBoaAgoX!##VaXIzgj}#+U>e0*s5Z~&{^Ue-LAq|nZ{~=Lc<AwZzZV5*4OJ>!TBuWY4$s=$kK|h"
    "?T(xG|Wrs5x;?&ActP_pB`2+;M2WQ<KzI~rnzLFIl$X;YUPcY;K7y0V`;V(o5#O6bIair$1-F_v|kVOy6hg`CBnM"
    "ZR3Dma2yJlAZ02)|dJ;nXP7h`i_bRFTQ=z-+B2`K@Dr86!;vZRghvZH4sj0`9@pwBDa%2>vxeo@4VR7J3IK>`!GG"
    "w-+uXB|C@jPwzf{cU}T6c6+s@tjwIG(SP=`kz{}=ja^(nVBPmK`W0MQ*E|djfMbLK*|8&AzxGH$>NWlEr7gRZTp1"
    "ix$%Wq$b8;$%rXyqrLTCGJ{M{HDCDAxG9uRSP}i4V%3wMS%9@eyx+od@qdXAQ<d<#P@Fv~icoUp8G2_lPiR7Jf9j"
    "L}}29>-qfSq%1fG$O{@ki&d8+g#0Kqc8hV?6FQQsRN7GAnh)rNameu4-T4#5)#1pY)4!db?Y})aK0cG*r-%JLkn{"
    "U*4p1WSF1<cDIe&8R#SZv!#(Zq`9OjE4ZJ{d)_^?Q^Y{fi(vNhu`h4w7}46dgXOy-N#vKOvRO>!FnCG?y=y*qffU"
    "w;VB0{5ZU`@dj@UGSXHRWD9oF#A1c6@$Cv)bG8)xU}ZlWH^H<OJKyyV+bjhEfZ~-2_iUUXIHZ@hipkVHqKMk;*ap"
    "W)T09}Gi&k79*C{8du79#hme!gBMKDCM}et$N5wv!89w7V+~>@x+9uN5?obUolXR<BDsqTxS#89t`G+~;_CJv#lh"
    "&vt-Due@+DU%OZ|Sf0ngeh$XT--~#(D6;cTB9~43xYUoe&cPpXTqFYLO59#lhV{@?3hk0$Hm}QP65Buld)uPcPS5"
    "l`WTLi=`jhL!e_Ib(YYnfO-%}9VJ)Fv@acbPjDs4{Gn5vChv|<5B`HGvLurq#$tsnT3?ogN^D88FN2FuvyviFQ4Y"
    "on5<s3Idh89I{b{ffX&@>8Bg0Bh4}RD^JbBwm?C<{TH;3)`t2zPt_QBb~TTtIm_ID3EG~w@^qvFD8gpR77KEemtg"
    "%>7tRbcOXfuxw8)C4qL+xK!-hMWr<?`?8K$|j>DGH62%!=8M{9FrGu?ouzvO!lE*D3>c(?mni8zy!LoYrCCN@<`p"
    "It@F|`+_5&vLBdl)z8)4vBJt9s9QDi-yaS|7bAVqYe@u2FA<=SRV;gu#*Iw(!tJezidr`k@Qcg{Ogd5I-I>xR}L9"
    "#vZj;5l<nm{gt<+pIH;gYoial!&7$~l7Bon^>f(~k1rZe6~jxiIq8eIz1$+h>l#_Bu(7OLmg8+eKbym}UY=kpu2W"
    "`ADGZk)69w%&^}#rfz1ir?;33wf||dYzenqqmoZXezw~waNxiZy2WAhXo4mk8N)p*>k`D^?NL^a5?H7*LqLJgQ&6"
    "jX2;*&X?EQSU*YUi2d>n|8mV0QkeL_n$7jsJn1AegoH~8P%x36EH{rKbCx2LE7+5aT}3-WK&RbtFcXfuMYP}u$$u"
    "fQ?(hrj({_J`5gAAbD9+drKCr;!w_zd`d3`r`Y5e|fAa<!1J;^G~gpnAKs&<=ChGV!j?55}M9$u&M?0$}X92_2eg"
    "OV^~8JRWW5eV3#!B6svM;G0iB0K^{mLio8RD@JKv|99!TP0oj`7AM>e;wM-_OHyGVV)QT$flD+>!Yl;z<WKm#Ej1"
    "uDp#^iWkCK?P?!I7FVP9KI)YO#@-#1Uc80fuPcV<sR&yH1(VlHqMJj5*YU!eTCSTY~wOTPM-MUFFgr<D5`MK(<LL"
    "=<;Rp>2rPF!vbUpiS*7chx#Z2r{K2S+<>xaekdMCNFR+JbK;`rGlC?zk)h%p(u!xwC+cy`B(4Lw2FLAO9ig10_LU"
    "93e4vZ`j>xm?gwGPW^)WQC0+p7M+c>^4O;e0_RRvGzv5&B`=t`!qfE=VLcIJY<VCj~a^hWQ%suq~TOeDU^3-E=>r"
    "q5femqC^4BUQB6(q(fciOJv}uG$G?ObQ3IN=dSCQWw*YA=YR%zYU}p=S7ZswQnYrX761A)!DotdAL<Q8H@rsDTrb"
    "^PQz7n1hcru*L=XY^`(8by8#EcO;VyT$o}S?cEdypdB|KMk;G0P#zDY%D}wau(SvEpIbY{bQp_zv3MfOff_(>#3u"
    "t+WzvbmS(WpGSy^qg`ZN70zq#2wKY^h-uEX~VAuvsNO5stZpK33Z>@xB`T`$#Ws6WCG7)sA%(H_RitICtpH-Ewpu"
    "d4uqG@Z(2-??wBLu<<Co$z~NO2s)Lz<~Vr&{(`gk0t+k+O_-Zy!p!5k^@2tPVyAeYsD+kzcU$d;Vj-Nbwqo(R>PL"
    "u2^X?zLIX>A#94lNAr<dYd`}t`1m)(QI-R}?g8(PqSNviq|J#KhRYe`<_TuEZo^u>9>qJVqEuf?I<WD-2Kvwj|;w"
    "}p+4wkSS=C=rt#n&PYPP=|}M(DpsC*jCm4;xSpi@fibh%f>xoI3CM`vEaWjmRB4ALQ9rYh%*H=-$2V>JWNMO{k);"
    "FynI;}AE;W=#$yecCJYz<rQwp%8iiWa`b?NCOXy!bVq$QgH&*EI?G}Q9Y|%iQdQ=;$$6MR?DlK&q`-sb&;R*iv5I"
    "`*d_&3z@Cqs<DE=HE`Dr>f)Dnf@`{yiUhXc8<m*NmHSpl#9tvAuh8?s9eCJ82c;;y6&3ynuC-XPDoNu*!_iB{~@8"
    "SYg-{=WiM+LK`ksyX9iCI4RjQ)@~TeZ3{cuuhm||0J_o0>%P<i(@H0a4we@>p~SobA?C<&PVhJ*0f4AA@8DURKJc"
    "1k{<dACzfPyKKAex9_WEwMRIGXE+|>#LI4HMB>blSfBz_)EuY<REa{%XcHcMf_6~1v=6bn*b_c+Kt`8q-S9yX4*B"
    "tQ%cBdsuqjf+&^XP&_Uf(f+MGeiplf)xNS3ag$}3yupupR1*x%Ni4e2)4@sa-!1`>Xz_Hs#)}*;_xTdm~%!qv(Ui"
    "Vh$sDgZVE|?rz}kf3XQW6CUZ~AT})S36-Ve%v?29lHJ38KsjZtVxh$-L6A0fJ3mL<EsEOyCr9#;kGN~{X%K}tgwj"
    "vEU#|t+UW;b_rha%qRG+*W|;BytM7YH!4(L2dAV;Ak9&#})Lc~Ya3b|`)5O!QjJGci4p1e5AIqv~k1mV?tF$ks4H"
    "(m%@gzlF($2CC$?SS2?lx<oJozr#praDrAa{Rl2BcCn#mRdA`Tat_47DbFB15cmcBD`yo8k_F84Jio<>s&I8FTgS"
    "nT`;cqqjFd3F<Vee|1o4xV1A|jc%r>gU9jS#S6vp}3ScjCO+&WJ2YKqmYaHNAuW?;Y*at!_1JurJfDUV8<=yI5bq"
    "!>7SumT#MWyIeyI)|yw7?S_;x}=iw6uUV<vv)Qb7@HyI;2?d8`(&9*YCA&mXP_A_gQ`u0_Ski8HiXSj2}oI@qoHd"
    "UBkCHDh}bLLd1KAxkcO$dzyvU^kSUk?b5oqpZ4;C<mCZQP=!cv^q%&M)+(di8w=U1;Thadk=fK-q9czI8<pJ3ylq"
    "jle9z0@5ugZ@TE<VM@!iR9HN|RGIAiC*KdhL9^z~h06d{KXf435qCgJX>98ReH*$*?!%;k(QTSeqAUa8(pn(>xtw"
    "FH<;@N+c>Ii-8E7{cJJ8gkG#c25a`Ak~Ao)xghXg@<C05a{chKLkcVs_W`r4rX5~3RwaNIQR{$Bm@!T?up2sEF_R"
    "?OO$4Wk$HoakCG3xkKzrw-i7kf)5`xw)Jf9niV8ka84cSHT*+Eq`thNWOa~+O6UKiyQwZhSV`?N#k=|lLNeah=_H"
    "u@Zc&9A)WjoV@FD4qwyDLcs-Y++O_xkB{dYO&n<CfymB^bdnfL5u}^E+zJcTq}~;HE~y97_>pLb{|0|qnBXGtdr_"
    "I)AV98Gbz&;wGC^&MSf^LU%cf9Ub3Q%t2|^<E^l!Sb3CytCu`>I@$3CV3f7d~<EozS@15+Q^^Xv18gCuH+dtYp=<"
    "mKe=>N3;H`nf7qj0ei_%+8v3p_JRqDWc^+Tg%KbeH0Wlr+lR-3oLK+cnYgM`XVySyGG*l0QkPxU!`qVe9CKMAtEW"
    "x$Ck6`8Z{SQo<rnVRVx_eq$Pk`;gL*K1+`3+XY6(UCn{NzUTW}uqaz&441k%cO2UsPt!ARryzQ?oMrLNUyNcDHKs"
    "VAJ%XI%x!LyVAHV$FS1`qxLdcV2)`5L6l%>WTr6ZwYL{loLvwaPpD#qi~ZFFE*qpnwlYXwZr9us0aXH0w)fU$yM^"
    "@`*}PJ<Y5#|Li9ghEWKJwu_Mga{01b7X;=oS<3tl$`MqNtr`VCl{F(Jk|Mk!kOYLAk)mkxnfiKI8c5-hVoTnG%ZA"
    "zQPVvmQ#|1sx(dYLV+F>{R)!x%pc!n51S40V6a?}EsYrx}150(s`ZYRk(=<gtZdP8c2rU_}-ND8Cdk>V}b8VtLg#"
    "~dW6|19wm$(jcuHj%`S>e3Q|4t_jDi{>X$#HI`O`s+UDyH*|xL_!8ijP%R-DGsL0dY0ElvL$l*8{)qvy9hcbDyJl"
    "Dl>0QMu>-rk8Rzd#`x)84URnmxYNnHfMZyvsfX*-lH1PYU4)=+)S$(l%b~B7%-!N{!4s`EWrdW>-j0viAf#s2TZy"
    "M9{|e#s>?k{lC+PRk3?L7;ljqNqmlD6=dx+F2<jr`%$x+<5l*M9*BeAMt80s&EhJMxZr@S3lEg<#8Du9APUQu>6u"
    ">@mMCK0?URK!(rTe2OH!i<=GlijKw%gqW$z&InCPfA7)h(*W+hb8;a%=$$=pAz@2ZfhmAK8W%Q4b>$PrlmM8+jzP"
    "xN{+(wY}iJD3!<#;PgNzfSNzkBwh>oX)9q*Oqh7$V`;|~dbPKXanU0%VNJCHl(o2lhi6*ka+f*lKPPi4q5UGE>Dp"
    "|V0mI$8Kr|j?CAR}y6(lzP%_qJOXp<{vi`o~_f<GS+V48mg)k)R)98%hUTceefetvkw9W@HoQBy@peXDt(<i<Wyd"
    "Av7y5VfH<z0jJdKXwX${qWO7`X>^c*^9^reW9EWJhT-VIwft$?i^>K%q(NDrc-oyHU3x5}rt{iLM{<fFrf<ciw~Y"
    "pG6l%{(WO7BWFs6)*MI^AjyB%d{i4gk<>mZsk>Dg-ND1|+WVe}}_Yh`OOO>qtiD4IXxhD#Wzp43EJW1Juf-W{mDY"
    "=ut_l7g&)&e_PgTVaA8Tc7A^^?^%4MtPP9;!Z38aXn>Jp4hL)N(NdPA{pmnXw{KLrp1j3><QQ%#e68M<HQ|G?Wi$"
    "3%5P}mcm4jNsFr=4(|+HkkOz4nv{FnA22g+~%@vq=w@C_7PabH=Z1mVX&OiPy;<zU1%a;=U$bCsVz}G2O52u2D)e"
    "{qBsFazYCd5egLf>kx(p$-8@Q1K2Hv%4R?WEhrEZMFnKgf|1cNEjO-y1Ir4w?U~oklhCZfYugkXwzmu`Zz1B)IfE"
    "noLpUPaTy%<!sB=vuDzUb}g9GC}%c)Q^K8-)7L+#7Bxd?BGfD^hlw!0X06OOJ6)_bqa}o4v+^{0pLd<)rrGR)Vlv"
    "F#FjT%$z2CkFQV8&J%Js<lJbfA@K#`-xg0n5Gta|?BzWYFD$J$$bAMs|e>NBvqyV&P^&}Ti<T@Y?Z951t*KpN*3W"
    "f!E5;y@C5D9hBzfYGkyB=@81;bW;9hU{sN4WB=`yRTs^3o%dwSXALP#nA3Y!g)2mo+k<QT`a8t1Tw`mUzjg^odW>"
    "LmN#20v7Fsv!5Nr*^m66)r4^_B*6o=J=a?R!nr(rk%Bme6R9XcFPkwanC`K2lezCUM4MPElaCVN&{B=Cr*09Vf#("
    "?sb_t>o8-PeH9i3%--vUz$PXnAvkL=*9HC!tP>%RU)V$v-|nYucqp7@=%v{%SYMg(Ok9yOS;i)oXDS7|Tuf(i$fx"
    ";{`Y5hP03&V|$HE8EhDlE}5utfGkYYbhSm;d`Q5=^O;ith@e~RlrnO0p-mt!xvDco`M^*aLUWxAM5n8#>n8uzD9X"
    "N@NJ&j{B=ANPmx|igKLQKoh;RemCjK!;;q`=B*M<ZXRhAfwJ7AG(K+-6z1Cc!X(_j#JHY3GoStM60w7RhxTxN4L_"
    "(Ml5_6FL)JZW01DbNm}JRA_qK*1vB#*U4`>1fB*w<9JM9Ca^Z2BxSWvmCA=O!grPsq4>CniOXtj5J5SL7Mv>{kMK"
    "{9*~$A<B^=mKt<P4;gChEfZ<&>&Qv8wmrU5J8jIFO(Rnub!(y_~hZAi}QeV-mBUWkj<KR>y<eVWWiX06i;yv1Iu`"
    "X?Q2pvt!64BN<5v=QYI&5LDz>eq>bv7x5`I0iD!T?qs;5;+d#f_EC^`{38_uL}aAOeZz_B*X`b;#`2=jzLHRW0>t"
    "u7wHOB9<%@H=_w3(E2u)6Dbnr_QrH^hKCf@2{C847S<I%8)$G#M}x!7Qzvhy?aCLch10!vWEh5+=(=aUeM&f@pY1"
    "g!=(vj6?OZbXQX_}EB9(GhG(!WW71JE0G|Fq?n8+X}HX}jJt=QH4q3#A>`WO5XrVfy2s)iPY!<)>*<Y2W>_)WS(U"
    "ne3>8LwH78o=6Ia;28WrS;U%RPQVmeMsJA+z%8xiw~%JaPljbY$xFsu3)%g-+3KG(RL``!E`ns^LO!lE?t?f8IfZ"
    "<ZhFPDygQWSG~wBo8yn)$nu2pY^?0%Q70o2Ay~cUpXm#_i%M`X_P2NZ46ya$!p-$bP9GSyqjD>jsThU}C^q8V`hr"
    "m!Ssj?x$0|GPdvW-U71!YwoQ;rX#NS5~Jkodco3={{Z*Dl@hANL`2ydq58$wuSfwZ~LYCwGrN%Wi18c%L|7DfS7w"
    "f9?|x0c}j^*b6+!#E77;cmmdah|o@O=V%B;V`6xoct~B6;@lJZA}t$Ue9u2G_?#ajB~fr5&Jf;y%1u4JxOcl7)|*"
    "N6Y3u_$+q!cS4*8BD{ihf0d!h!#=C_Jd*es)x+PUE0>a2U;eYw4TaUUMK5sdie__^6|p=!R5(W5vRoW_hP)kX4l@"
    "0~#1fW$hP&$&W`iST-{{o<Rg?QgeszN3TmkitGdjDW+kD7Ro#7*;(Zd>cZcn4>rEsGy|qPI8QB(;d{g794L-VlXD"
    "3yj%hW5m(ElR#?VHk}Ej-mI9${K@lh%N!+@xsMlc?iqB8odZ2V42ci_^V~}+?&rzd9Y$VAUk}}G*m@&EuSPqPIj}"
    "4lxp;By=jb=iue+|>&loe|*0ZLGAn2h8+;RHN{xG{~UMA(tvbef@bV4D^dsg!(<Bd+i#*l>dNBAsslh52?nYMAi="
    "Mu@;WHlkH1OB-gmDc%d(at#2+m~l<qGP`i(`JRnuRzsryT7_ckoW!2(;1<Iez>OfSnQm&#m4_MfE|O?2qp8DoOy&"
    "<md;E6zAL|i5NXO`7o^J(Bz=q>05>d~P3zVwgq~Ta}ROhlc<3LV8x_vrsvGPH^4O45elc?o?)0g2(Yu>Kwinuwgt"
    "u}^(z|sOaZzz!~8cg3P#wm0D1yRhnWX|~$%c!G`N!~-Y07YdApJlUp4Kc`hl$g%Jd|c3jHYB4KcfPg6;3R@bM>+P"
    "R+p8TAUSq;oXRS98>x#kLh*Vb_u1Npxh}p6u=65yi+BW77O+}|>N{gA^BAjM5m*_fRC3q;w=m~^dA{|9$lF!}N)O"
    "GR9Hp%i-Fr!=yWtr1$o@PmW5Pe~3YFSm!Em@7?vrlQ{vjW~K=93{6uZx5C)QSeOP8^ap&--)9fPtB;FQ>dJD~eUI"
    "_f}<9L=agT=2>Y>VflnzW6X|n^3{BiT_Ma$J`(4vgCym7$6y40hlFtI{f?|iV6l~ucB*<80Rky0?0^$olD{Nhzlc"
    "6l&%<r_no~?O{=A%-!7DoNw1p|4r+najFS029sGGYQc)J=1Xv4KuU?JR3w2(|$v28P&R1|JUD)*h=>v{+4=69QkW"
    "EvvwVi#?*E{|56`j*>HQ+}7%Yq17?pH+D*(9sNPI4!C?EFDZA^}N-@F*v_e3>u;5wPT=*L@tX7V~wY-n;doH<KWF"
    "S5}9P6{ydwDJKZ3T0Z+N~Dq=bl4$To*+_(L&zkB(xFo!d`30R!L?=kXvDzN^ia-l^qJ9@i5^>i8d+7)JDl@TGL<F"
    "vj1%l^?>|JRd)v;APHZmuyK07lu3V)5YqG9TkVTIhh@w#$`l)tGd<-2~GUg9gJnyWw(J>gA4Xczj}+&pEJ^OH8d5"
    "oN(b{+X1Nq{`+3q>t1+tiWNI1A9Eqq7)AP2`!H>KqTlbj<P-xKt~u6ki!QlRyqAE^!v6oiq+p@|)=|UodNu#Rw7~"
    "FEYsKJ~+kg7D-8d*}Yi%zrjvRJS6|^apsN^b?(q(oqMSJWjQ-AF7M;qa<oQIM7eEUuRo3FoJ+r<bMl0H4ehH&9Jf"
    "v)HLN-H=mmMtZhj)&T%c5%;1j}3+x6W%FrI5NowiM6lG(QkxE6`oO!e3<vTN3k9K{9`@^g)~4_Btw;a)nx!GC+bm"
    "!a;s$KV!^0c6)i#E$P;|!+K-hqo2C~Na$Wts7I;|~zZ*sG$;d=5X+M04od3dWJV3n0f8E=S9aVu*Wa*Pj1e0i5JK"
    "u&|-BO1lQ807WR_h|H?U>{GAdQ+wqRdKVhzMg4X9@;Id@sVbr*dd5MM+WA-HgOpuC&<k2BI}u%340%Vh`JG>-1bk"
    "<!9^HBKHcCxVJpfmyEx_8U1I;tFC~;5laM>quYh^2r#x~Qjs4pFfw_Vq=C0fu{0lw8!5#((G)cD@YarfNA|Sboj)"
    ")IE=g#BX#VLsTUASgj;%ZaK3;stLr51Vz~pf}AqM;+=D9>bfqhUIBkf07;5*E6Cu@~EK7Hxd;)ju`T0(BMf?|v-M"
    ")l<(BU!k%6^^nAyl&>ly&o$|T0&El>Q@xXAsZ1PV6U@}RMh^1V(FLp{1s*&kROV3^>YIh10yR|3mjRx&e8><0-~o"
    "b0-M=_DsC8H(w`0v59?}>Ai5nE#fQ}*8XIBKpF-PcdXvHNsnD9=L<$Od5^PMz@<rU5#uaG?t=sDxLIKIN<XY=<%&"
    "B)nibJD*Kq|r8pVr&@KTOcjkG%7}r&?-uRwu~g>-rjK{i&t;D7wD8!hbSGb-w!A=%3<>>K<<8jKCCtpJDlR_V?Ug"
    "mcX9)VVs?El0*Ml-e)u)sx^&)lC&#fcb0kjTljvwf(m8+JJz!^B|qhb*Yf+}@^_}}oq<-q!P1v4ETZc;r^TPs^}L"
    "CQFjrE5>IkC&;Rt-6ZT)k5>rWSdq~JJcLJ`W`=(ql`sIa&irlBkN9!HK7G8O2DldJ349qJ1G?s^dN*pM-|(97-h!"
    "0}jq^Oj^6A&Y!=nU656r%GmU&M=n)K1*qwz%(mb<|h_@uOJ}i0tIp^MNs{t1j%$YYkf+dC7($C`h+A!U2DU&wNG)"
    "MeOIl!QRNMWmH5<22wvcylLd`j=8LRxSUZNH!IS4+HxUcj`N@Hg>9)zl+W<}RElt&r6EpcECmqAh_>|lV+(x3Eiw"
    "7Gx5FA$CwJM)ctgUyEYI6>6y=zRaxKq8$A3=Cwd${{)X|%5a3kEQf^@tfRcPSG*y6l?mqJ=4Fq*{}(Qy*baE+gm!"
    "#Stu4m($76_)u!yM|WPT&P_vp5u%S4)vz^<YK0N)gCS@2g=O}Ta`e_bFxzgUgVc7JN@REswd|0^^(w;^@QS4~Cn@"
    ">vD&)7Y^)j}fJW*<YV5iFdG-+M7S<2IP`ba=XB#Qb~+SXKPdwg{^>JE8&!lzm&tC((@1@(JuyX)6-t2Y-^cQgM#Q"
    "s%8Lr5pO~#FenRdlmoZ*kw)kDdN@qs>e=8%6a)4B68E2g!;{vQzi}8*8m`UCF>r-^)gisZAxQgm(w8-+s<0L+`UH"
    "+_aD7ar;{1Rb8^0{?sgc^L%Wt+Hyqc4Y!t1HD%nu`_{WRw_4|k?L!PH2?lL!MGCQ<uUu>I=!=oiZpu<7GGyo9|=x"
    "!UqQ*OL964s=csT$5+8%yNGSNY=N!inxIiyL&YZCs@98JFVIJ(~Rb<NnFMLyOSXQ|41olj9Tr#$UaCMe$X{sP0T1"
    "dw9T<kmu@rNL_6kYDVSx=G1FvEojQ@+#zs<l%=zgv#!OD&&Y&j8D|}m|CgJ>gPqdMI<+VMOP*-E_2h;uq;R5wo32"
    "BhFa%BAkRpS@b>8VMFPw$X*Lpoz743rU(vgnP7lL~=j8W;cFQU}zxtN)PgH!@Lq*t&VwjYe+#X5H*jQ^){P{A;^F"
    "^8|EgunHe`hvCkfv{fRefx5IyCI30Bps}Z1<$p1l0VvxJx{*gc86?_x*m%V@_l>5kHRJajKqwuzk$7>^Yur)VCD<"
    "2?BEscjk(}7ZJNENpcJ#6!^#|nzR7}CuNV{||L_^?cUXLqJ0CtNBj|+YCK^Uq(WvRzv3;@}A=0h$Co6*t`xp|&*b"
    "b9D6G9i(;Id7GXX_eSB{$UZH#R$X^3j*_S9UhpPfL;<xj5(p?c~pLg1I@#c(%@FB?Sa@lSh5sIosp`N2xUJPbm0>"
    "*s{0(jbS^O7cs~2oJ71kFoJ1Ta)S-w5z4p7^S^Up_*MLAj4R>?oz;#%N+f`DN-w;)Xkg=dH^RajVj#kq!p6++jx*"
    "@EEw;6t{wwWp{P6Zg%FJR5Cj13nc3RRWCZo1X1vePuTgrzm4RtX0s0ZZXM?ccqC~5Pnaq7pSQy+8x5e=@VBTTvzB"
    "Azq7K4dFmgjmmF#FUH9-Hh_3im?K+NFE#-JsG0YWZ|$!`~y{{0a3F|id9KROet6}n$`17LOEz9rL`e*sNe=@fX{A"
    "=6-;tIL}XqH(?47dZ=WI@CBX=>f?5e6-oR+A?E!xqaTE|N(Sme-$n%B7Ive;_1M-kn0$xQyt0PUJ*6JAq-&QOntp"
    "JnLVT{jc^xDbluQ;qD$=6d5i7XkH*(^sSOU%$C`!!vbEKf;*Sa40~%qG^dL4lW$kSjG6l{zSxSnqC^*U&SQI*Pf2"
    "z8Q;x`ZaYoeQk1h2hgM-V^|nf|05j9kA3CET+OGHaY#Rm4W1BrF)(jBp$xEJb4GT;>2%Y)K=DN1j6|OkKrG`6UV_"
    "-YBE(zgGS87*z~Hhu5FMrxbLt}y;Z0f0uT1$`3QR*_L$CHoncY+_h$khZPnJ?3-EeFRJoN75_}%d-D3zHgh@4oDB"
    "q!h<+=tf(r}!N11i?k!HQ=2Y6*iM;LUQ3kD@!;gKI8%tJ6e@wcoo%sVz-jMe?DIn6YMS<Tn!4Km}b-Dv`7rm_s2l"
    "Ql;JkZ$m=jb96@r(VR$0hs0OlA0(cZ%Wat#;Fiw`b%vmzH%z^y!K8<8R8=SI4vA4@5f#l%^Lo<_m?}kWUHY-mT=j"
    "Bd#VkAvmsme~ifj4n_$`6|wG>F2AP^AuCRTw6LJGi<LHx$FNvNlIEQw>!5@TMg-d5V|^nvL^U)Fmsd@aUjlx3d{c"
    "JgyV>>gKvY^2*Wg^7%Os8(F)36wC*W-V0?P@&LLYD0{S%>=pCz<mxq+%ue2Lo4=o>JPxxd)ULzlBSVRJV{A@~y%~"
    "MM<9KEWhT=QPzDab!mKecPw=W}4GncvsmO|)xsw^W^{9KUcgrImx9XUJZuw@aL;O*{z^o5nY?!VoCdwg=Rf7*Yyf"
    "6@oua(Zxl1Tw*Q@YumofA9F{3<I6^e%w9T-8<VqIYmY11-B8sNqQ3Z`iP28&XZ9wM8o#{s_HriNU@mYhOxk2G*2-"
    "P5-e%dcl(T;1ZQUln`MTH>2q|MH#Pjc$q9H#;fxXiTcVt;Dj7H@*&u~kMrd>PP|tEq97QlwIL;6wnlP4-xghr^#3"
    "Vr|u-ssQ`W*q_q6>lxYr^)W<g(=QR-u}(RTHp`kdzkb7n3&VFe8Sf9k;Upn(Hl#9MI-?HKkA@SiP?Z($(=4a1!?*"
    "|FkT#-uV;g)Fo<e1LzWh^yBD!{>^6{9san{$P7!!(9~P11V=qMf3lta3DhNKnVDF<u~kA{CYJYi65A5y#TdOzYcS"
    "<B$4VLSS3ak7VV`qCX+&a$Q%v)$4W4`U5d~6thybW2bgZ)YaUy$jR|SKXVJK}c6Vp5(8HGj3n76q(>W~p9*`=Zq;"
    "E|g809KOzNNft|-14hdIP<<A$AX}Hh?>OXTC>T#4+q56^12t_ZVW@Ak3=_JyV*a%AGykVWDc3;y`6Ns?V@Wn_U3E"
    "v!!NeC+ioh_2{FgVPJ5n}yb)-->NFb&G$S<s#uIO?#Ib?yL<U;3`@Zp@!(eV;N_Tidw;NICST7;GQ?Gg%kIs`onD"
    "A&?ke5)DKp=%Vz#Ql&!D<lG$t8gTp?;85GdZp#-<aXs0`TSGl%zcbD&}$ldVCWB5$q{SrpN<P7DDG6l@Cs)#cIS@"
    "@VI2Cs^q0xqCo08U>l@VDfctzb!CAy@en1aU;yZR7KMq!wO|EF=l;RdieJ3UFk`j9xGoWySFIWpBBTZ3FbJ=zP(A"
    "19y&)F6WK32qv0b7NHnQ=Ic=U3UWT~OECeJ=3B&d_f-8H-w<ND<De&dSl`SRt99f~nq7TUTz=Rbhc3ew^SgnY(tE"
    ">SgGU=;`|f`)Q}fcT>>DzNq__Yr>jRq9U&zZaj865~!Uu?z6e`T6U+?aqsr_vh!MyRSQBE?0L7Mu_BY5^0F1N0Ka"
    "7`*6}<Os=(K<T4qf54PaMHzD7b2r-w7#xcEXxZaL(g=z$K$^do~QKMj<dXU{-s(lzbtkcm(EK!E7vs4?k)to<}yM"
    "2y@>pQV<j^6;oEGT<L#+V|;^tgZ-3c^B#AWc}gPx-X~LHoFTA{G^>{giKmn7r(p5-(mVKXyQ9G3<eoTCEFZo-+4u"
    "$uk}8OZ$sDLz1zG?OM76SCp){a;)fTvF<Iorz#-N^Bdq``0mh!>_VhC0g5s~gt`YB7TDzd^owy{AMSJK+%C%t7JH"
    "iy3pyrFc*C&AF9<#KPu3HpQmiRRm7xMvn3=DyMl<YxrctRV6ie*{#w|0)%OLnJDW^D}1zVLkhkal*mXsqn2WG;BL"
    "uriw!wFz%5JC!zV94>WinUk!!RevG!B7OF1&_nJ6b3CHzJgKB$UbwzPcg+s)OIFts6ar-0ti(Z#B2}f%9$$fSRGH"
    "1T(JxSAC}NHv~!?((hS?8>qMHK;uhWyTvHMz%H$^OPQ-0G9btwTj*Bb*7y;@g_joE*n!#!)Vgx{097BktCI{ixj`"
    "GE{xTQ)Rc%rQ4*ZFjTio}FP0XdaB>PkQ%pX`RrET5^Q&gKLzlanR`UhY1cQU%daHH|Xg63dGNdM%unu~JthS*c(~"
    "u87rI*<^}krRFOCQ%#?dvn(}Pp;_QUt@*O}R0H~z^i0-0sGODRT+$tboI6~usLswcS-TzJsGmUY(Jo5O9hx=YkVv"
    ";p`2keFcpiN@NCF(KiI3n2MQXE~1I?F9kOeq>m|gj>%V;+$GF8SnUkJXjn#~xIxGq__@e`3K390MLf@)oM6(W(eu"
    "E)s%Y_-TDr2}}l7aO(G*ARXc47a`DSj;5uo3I%hVC&Zf)fA6t&WKa!TUfpX4s1=QGlQG0z1aSGbnncr6P8B37kR}"
    "AW3OqeK_t6OMoo;d6%*vBbF2_tiEKo+LMjVF^{=xr>JP!Gpi%H>5e|eKGi@fBB$!Rl-|fZT`8l|koiIlw9>!#5C5"
    ";C&SzXq`OQwzPU+cc_N$)Y1CGMuB2ZW5(n`L?#KF7Pk9R2!yt@rgXf5ws}5RtAxJ!&Lee<GPd^5fnoA!`<nQIMy0"
    "Z%99<BzY<Eu<L|6d5tQJafvyB)XWsAESetbR2r{FIMhPVkJDOD5q9oKEf4R?j_Y~*FN`EW&D^02`W-w26!0)Dn6("
    "O`1aMTiMDHJg#KDUSZ>G_hJwck#hzWkOIgn4PI-*AxA1={-gy972<n1Ico1xMq&cc69gD=*-C@~lQiyDZr?nO6x#"
    "eLTem3+AY5SwZj7@jg=>hvGD@SwO|-VbYR7`{3k>IoH2n45jsWs<660x@@u9Y2=cV5N#HtaqNAV3iQZ2I@I-MA%k"
    "CF-vf%>xmCUP;@ni-cOcP91o}i!5=tfFK?X$j%&{JMH4EzCAka^maP%uBk_I#a<+<zB-@ZEiIRSrki&EfI|C<2fP"
    "Fvk>p~2Sgu4`(lo|tBm*sP(f(oj);dHu~RjhmUq=dWs5y^)$Td6~Dx7+pnAeK3lY2R7UL#M=rw<qP#dyhr1;?naD"
    "1U5+2nLlZ|#t^3QMA2L!Zb(bTYQ&I<K9*+MG3bYguBeaK6YE?#UGwe@rFA{WQ-tgvK3opyD}q(ZL{6~7D(Z`<t!f"
    "K~7ggD<;sT?w(Ui+|HpH*!2%!nQLESu1XP*k_JQ3%VYJ8#bG$$ea8GQMf{Vcm=ouEOpLR`yB^+X8}k?;yd1ZUMn>"
    "WqoDYLo^l4C;bdlkqi84p>oEWpRjdy|@BrIjdQ$sSX?sae&5FtQum<vO0anNi{Pf#l!G>A25FR=K6NcR+7<Hec?_"
    "{@f)88J))Ns)KRG9(_Rb-o>X)P#S>|tlwvm;t@gR(FJa$;ebVpf0`y{g=Ka>rMbs2?hlL9`2iAd7<)%7`L4fRE-U"
    "^1*b+PDcCF}dzp`j_Sc4OIDv4@V`M|JUiB6>9kB`s^l87)g?c>XG8fK<4mKq&FO#W9c~1aQYuJGw~}7tHaG_F7G6"
    "{V7#oZE&24O9Lx03<(OH+&i?DZYEQ#j)4CSGbP*#9%vh6^Mpi{07Jwuwv5Ave&)Cx_Tgo0Gbd4*O<7|Gb#H>5o#$"
    "J{cuXZO;WMdVackzuHm=N08hbw`yIB1DnpzimT705C$^LbgBA(w71jjs3@qKd|u*$_3kN~R_5BTt`G=!L@g7_)dn"
    "%77@v(~1!RG*kpL~>$#^Uqo-PknlLo$Pe2S%Hfh%dpI#6@eTJRUTH$dj@qAC6#cuCgI}Uha+{xSnj@iC7xchWJ_7"
    "g%dEkA1)gDu%POsqJHpLpY9V5QfOD7|?QVOxediOhwNUL?1cAWai=ISdBEwS1barA<Jz_2#>fmrdE$iTZ3Hsdw{n"
    "9FpMqiZUzgM$jwY2~3cCG9w(8(5FxL6Z{>XtXR0jP-cEa4SYP0}+Vj|g54cqEAcnOIZz4i6YI5rK{112Yncf3t+;"
    "O|z71c(WZ$#wU!$-!^9JdbwQmDT$BTFfmpXz>06ZHE1S2w1ev-F-1#i;Rc@{BH*OKJ|;TA098Ev($4mFM?GdekVX"
    "DK%&^|w`p@mHKlQgR{`mhGUvnek#b&v^K^d+{0awlLBFt`f$Lf??m#}oEUh^xoxC%ar17%QyjZx5aySR6X+u!v;`"
    "vXmk(E~&st79?n5MsC~?|l&ei@LzHffYw;A~qU4kW26gEc}1t!}bvmJJr)c>(Kg~#OZ6k?fcDbC|cjZPFJHTr6%W"
    "k6xC9G(C+@vE>4F8UK<^2&nK%nej?&#-9;$z$r1rn6T`ip8%T6f@j4WqOJM1%T5*!p^;z+!5fQy!!AJ3wxUkF?&M"
    "~lh6tNjWoeZ_d(`h!#G_-L_80_LRF$(m)lx%;_{1D`Hjk_~w?I1s~(_jS33D!Fv($RvXbA$0H8ORe^soE_F^CM%*"
    "f<uT4VIc}TG?CpH#G}h%G?5%6>)ibL<d6$76EKwWrp1K3*U%hS)|OO=Y{Xd?23E(Et0@^j+63D$o=x1*wGl2l3Oo"
    "H(IkgND&#3P6=QOD}xbDu*%dcNP|G4ArZm(j~`ICR#<nyn+2i!uJS~)vT*YN_Eu_WZi>XR@4kTudpR@!)IxJIaQu"
    "0e$!K=+Q`Z0YzzZQk#asg1|wG7(^So-@0qBAo2j7)@3){`-1zbuCx3PfeBc8ex;o*D<GJNx7?`U9Z-`Uea6yv5qa"
    "q^N>>HaPAcT!?l+Yz+aVT2yjpp<aMMDV&vg7avuxIue)!;&>HTOYozJsxXi9*9^^xNbM!t_X_`2MH@z4s<jqb`H-"
    "&_x1%r2H&ukf<!kriYnr_4Y+u@(x?#@P(fQe(vJm_Mj_|C)yPwNDsblh<Zh>91JYk=t!X~~r}&YwId=wo?Q_Ur#u"
    "<*r+yH;fg7bW%~B;9L>2+BG6`$}4QwZxSkBtt?(6CLiMQg-<7cc3tipkS-=XENxLc^7d8nA`Oe|7E|MWf%EsOH>+"
    "r*8UYlmNBgKEwE2vV=N0?>wjWh^R!06=g=g3JDr~c#RfP5$bb7P}k=(z>;{4W`k)E>!hs=SP@SK#`ClOh{@7LpVn"
    "uT8Na+=>ir8*YHk$@ycvP*Jj-uH_8tgVdOT%^HcHpw5M`_Ivm_}r<S8@H)}*4A@mdYMv-j~z2Z@c?`BZv${YBHnF"
    ";*3VXYuDr>3N{ILa%4{P%WURI=@?R=d<Tg~V3X9_n%F601ARpd7h+m2CKZITxtrIM3sK_W7Qb;DMS`Vc(aL1G-#Z"
    "RhRk?+y@;9TZ<rJq$5z<h0eT>yEEFwl0QPsQ7PzkShI1)%=Y%0UL+6yVgG?82D3q~tP17ScRw!a<BN+iAz}=16H~"
    "AJA56w({I79_z!<r#|xWW2i?z{V?&^8q(Z5ue38dzaCpmdNZn?%vB~~K-n=PS=2ihqQ;VkZC!1R=vO7il)@a@L{|"
    "rdrR$_qtSh4U{$c~fT4_r5h;9jUoy%oxHUcdHBKBD5%NE%}va+PuizecfGTe+%u%=#e1DwRDClx2SsAEF~4|K!Gb"
    "57yN2Wu}{4~D~4qwPFYfUzC77AXpYneny9iH*+i7v9+K8lOj9*o&>N93Zk9k{wIYQW<*XeJH36y#a49krk9TqOxp"
    "VfTde1sBUjYbG~xrRG4l&HslL`iHOH~9#BE*M-aZKO$0|zs<iG;7MPn5ot;A2Iz#m7j`XpP*4q-i{*EK2caop-Tl"
    "wn%Ylq9<DC>D<wFd`aeFQFWwo}Gk1M1ss0O34}wGJB}E#$TMtxeDr$B0yz5)2>bIApe1vCm+X&mZlXc_15}Zs+sW"
    "OmCeo%YFE{t_t91`C?A@e%yb%+y7<%MAEu<Ts!0`c|zFC`voo)#WbR9?7oz}QkQ|>1NZC4Tpsfja|~ND0s3*x_am"
    "O6KU9d{-8(z@WuI>GpLzp6a`5JF2S-29J@d=Hx3~Z9Od8c6dh`3k<Gr8cp8T!1-s~P6${qewZ|v<J?d>1xJ@d;q_"
    "MR3u(ILJ**?n_HFVbJyCRP;r8(pTqOy5sW4}Lh(zRM5u&d*0jI%@gbHuuZH{;$#={<+a)?EScV^uzvX|78E)f8Ia"
    "Ur)pO}ckb-u>>B51bdL654$Rm2?+wPhcl`F<;l6oKe?7RI4a0C;27KH_yLu$Ed}OD1?{N3vt@eWd)?2TSkC=e*4|"
    "5R~oG-#3{+CaRX=INa9PJ&v+dX8k+7d*>$s#lLw%_i;5}nY!S%wvD^v3bK{gd6ZW4@&x0lj(pZujIw=e1g3z7|?x"
    "o&9wnjlw>k0Iw|lj>@KPIjp7TQQ@TGv8Jp}M-=XXa=;Cid*vX!>*5BT4HRjqAU2eqR;@@d9~jsNO0B`im}&2RfBf"
    "@Va=v=8z4Mapo&NN5N3(JyW$Iu_C=3{DVDoRZgs_{;bD1L}MsqE>D=^NG$1SxDkexeuUDm-04cuH~_c5(Wl6AS`<"
    "{F1U@&tffQ)xyJ%E34TgtEgqImk*m34x-OD4p#(&!)@kTMXMD{crZ(g}aR-I~V;c9qFzmz^eh-PHt8LU)(t~1=-B"
    "dq(V~GnI-80O`=J51OgZ|Ac|4=-*4?#)viZ3KuY!`XPn8h2%x*Vx@y;c?{9A|)(i#5wXQRQsscw@d9z%n3xd4DpL"
    "MmKmFhE~#~4K!@t5`Y{1$Oo+0AcEyP=AN_*NQ9^L>sr-_Fa`I8lw1!btW7nE4X8$MNNx#p**F)oq{?CscTu>jsEx"
    "1r$K4fTVP;v26!rpcfxu-}xG@nGpgM8v?vtW+lb|x<7%W*4pNvVp{3x@f)%yDWB7kyCrbml_R5hQhnj9ysFGQ!uf"
    "!^F$%_%%n()ey1F9L<2pZ1`nZ$HI@o0J;@&Ji5}*nIen9mHk5(_hD!nebPYZ&eSkQEYL95O;SZm9XK`tqWL|?O#n"
    "36Zc^j$nm4mki198A45JG8RM2GpRLfs=e{!D>#U44HMmD1)L+D1PISrrSxF53o-{^~GYT={L1|iMX4<{LaT$3s<="
    "2Xjzd@;zu*U7_y+%b1BWIEXR=kFhGXxoeEJCb|d_g4K&eC(YyEZ0uSe%>x-)Pzp_|qsUwI!+Ojs*iC)xzDQuhW9`"
    ";|`HrsGzpdhK5LY8_w)o9J(TpM|1xQ`^L34;Q^`eIeA?#2}CR`u8lTplk4fboxIlZvwS1?t6Zuy~0AOQXgw*W*=r"
    "slIfR_s64U4wR(bmYj8!?Q<^Y{t}xds+LvI2tYA?<^+rBop;l6b|*BFI!XL0T7$CSAl1znB|ziNbWQnb32k0m0P_"
    "V@<2CU5e;H&qR1|w@?z(0Ui#HWCB5DwZ^L6G7ibDNwra&*w6Y&%ab1`FL)!WQz7E(gj?YmfE_c?Ul8k@hNePEXus"
    "#Mn4a_FaAZh+P`;R1p@rwNfd!I~9(zqo<2EyE6bJ?ipPXD@rN|Ec%gv)Unu&{SB-SQl8U4S)|K1Xks|yhSmvHvD{"
    "A0^iHd?~K&=z*RXi7L5cx0~~;g(GA5KxrxQmB;5#1j8>KMGG>CEa9J89bbdv?3^D-|I|ML+cVh&CQJG~sF_I-Pg<"
    "`gqtX)>6f25e}Ppd1#83VC^?vnBOUBE4;?#tB6cnp<PaYA#k`F300X~ggoZQ2VLo5McLS=$vQ<s9JOSDSgx#AK9a"
    "0s&o9MQ80a7bhSmc{agO;BSHt^Hywvsx%fRRC7klzb@}w>5ea7P*5X2Vs4>~K)K}y=1?p^xKHq{)j9gv8zNDwgx`"
    "RkBX}HUM}g4(1I`inWo9LVlERU3`U9sfts#jRX_ZM4T`;h2R6MX%eP|*Xt0744(#=gVpGE=*dow{qZ{XtK#cEwa!"
    "Zxt`gqjc%VVY7Bs*Rt|$@>`pQg1GxA=Ln?OQ3`e<J3IhIVMm|!&K7{>&h15tB?`$a(#rnhl*V|j+4j(n-=;NF4xU"
    "u{|3Pr=1NDMR?~h&scXbOTuWgM$Wr)7wG8Kq{+ZaQqyXJ~s6pc_DD?uln%Tex>T>FR=`Ao|YREWPo&x8PVe({J?j"
    "^{kv^C|Apv%DSO6eG(;v`><8UMyAAk5N&t2SW|X+zSx(#9%H!k}lQzV{mZ;K8M&$}wQ+TrF13oL4TiEw3A{6Mhn("
    "LPw0hDyzPkA|cV=#^C^u)-YrU71HT^=8P3~NLbR_%g&)7L=bln>?B)OU_PCg>6N$0XKR5j@_Y=23#QB_)%JcxL$F"
    "JQ^ow(X7bV*a?;^xQDE^gKI%WS34<Hap{5sb5a~QS=C7HNS)L)}pmBq@~3oFOO@w7ojb6vql5?4lDG}^8)FCS~wM"
    "VfY#yl>ol%{t>>imR(tc@^+EJ|yT3BBr{dwT8>oBHzKbpO&CEU_}U=Dj+6~RQv~kh9Qg(%YS2FTgv)Tgg$P)m7G|"
    "-I<=k+^0~>kBxtFCoKTuFX!&SWAxJtwj2a`-osW$-d*$$PYRLUn-ZBeFgL_`+^UeHrfh?8kTeok0w983aka?x7*J"
    "ZhaMjz_Oc&qhcdkFC)Ea^xx?rI*(sZP;I@}(`nn5V9`PuSkm=uX2vom+M5Kq0CZo7F|aMHtw)H5gfblQ&U95>kP)"
    "q~f(SLS(ZnZGv^=WjH~P*mk7?>A~Wn0XTR++nau@YQ#%MHqE|<0(=5ai5AEm_HwfJY`--edA%@91^kUQsnHxSIqh"
    "dJjNDk=K*q?rx?nFpLF6+=KWoV1&X$7)qJ}H$MFFv(N{9{)Odxrld=CeSZ>dFk_vHWUh%}r2*ZIs$+x$!M)vhZ`O"
    "21m8amS~7EV^&)gGIgE4J^v8?*s1^i{zpAAvAN7ruu4Q^ycGLbyc<K24{quyt#nDu(5+MhnwBYPy!1)8bUF1=tyZ"
    "u4+kY<22B;<vww0MM;^ZH`)|Mh=iayf()<2jd$8>P+FQbHy{hijCE$&&%O2d_tIeTUA@%0j^<svV(ULe&m}3Tf7n"
    "Bt!JVay&;&-|2@;h4qq%%5|X^y=GzA*K~VsFFI@fn#|f*uR5-d>OIFoX)2+H>l(jPKWD)(00sj6+N%^6@%UL8A>p"
    ")2*Bw({u=c%@Br&dHfU9V`jJDWiN~lHrw3How&-m>o_{eFUu9au3c4ZJOGG>78<op-N3gVM?7w;zL7c$x5Wn|4p>"
    "gKU5L9nE<bJ=Z@VkO3~++Jf)t_F<zE|*Z5xNTY1Ls_V)LAu5;7L6t76Xix_%9<`oYk`VvTStm(^9Co&NhF0H6^{A"
    "Dx&h7?!ZA3_UWkc>PrhXzOcZrp@d}m9Sx`ZYEDE&TS3`q=jK)sGYJtpnJ%yI+tHxxpLj+tdFpm7TK4r=|wq7U**)"
    "ag0nUn?q=a4vxiS7jF(~L46)s>(|LhPyX4{%&P8mvmLTb6pAZj+4n*S(qU-WHqoy!sT5Y<?TH$;bgX9$4iwakdB;"
    "c4W`i7C%(<$S0M8!g*Y;2!123dQvT<8ApMioV`EUTsHZk5)O=38dUW5TqxEjU*tUYNK^en~yI+wcbJ4Dn0wQ5svf"
    "oqVxEVYI!!2FxAG@3-{Xz@)5T<(v+j%NMXE{I}({(H9s>;BUo~${#t{z@Iqm?0?dCD=E+4u!qWi0y{%KEM{^aNox"
    "KeVLbWh2dMd5?QCe-AjU8XC~W!>W9$p>b>v$kr}9WZ-4u6Ya1h%*m*>@V4Dr};!S3|ji;a$Ll&x)Y>?*SR2)NerW"
    "y3v&N{c&p^3E5ZppX^69plGc?t_4pyA3`y++~Am!3w@PKKb+MtG)fv_|@^@!T!PMG*_l{vAR2*7t8v3A^Se9E-%N"
    "@z{#!@#iZ;r_<N`W9{<O}Qc%r9>HED#3R>(966LEDHYa%qvLaOoN<}#45~yD>|Csfv7qgFLCoGjCu`tF4Rzr2$bA"
    "-O<)$6YpH#1!?!#IxS4q^8l|JJ^}*Qjamg(u{v=W)#*3$Un{AGK=jgDyh$8KN2EUx;3A1zjh6pYDB>V;f%5r?y(1"
    "AP*LEk3Zih0U`ljoZ?)<iNT|gP4uo!GRZLpB^T5xxP&=)?t$?=br>k9dy_{DeQp{Y1sbQ2r$miJ{k`;9gjFN1A8*"
    "YW%D%{69lRL8e<(Ey8=paU8t1{Lp(qOs>u&FA!?4&XB(?#Aq17AcghSY}GwMR*K{P&9PHq-I6a)op8d68jX}XBm2"
    "HnP>%zS0uq6aAKuyWyIK}y?HY!~wy^alOxv@9jmXq3ojuA`bX&*{f#fSy1#q?eMj4jJ-);qe3zqQz80>Q(E`1tdo"
    "YCo*x2DWk!h#+wh*YGsV>6=v`g7A%tDc=17_kO;oAsBfLdv#D-y7dde*r;sEBWPU$r4lP5H9`FStTBm>;qUk0>&`"
    "=39tMP8VMu3B0e(krkg_ptx%RAtDVjcaDkhNZSQWY^g>T<J|HuqS=B3w!SqSR<8=j*_q!*@@pdND+6)LLwZ#?C|b"
    "2qaY%GeDStM<G7Sf-q39sjYeS`xvs;4590laasZwti7N}oTy3p^}%wHR4Xa2oJ`bb-91tL^6ts&IpT5g+9Ycr-#y"
    "u*jo`$3%Q34dL$^WhPu+Ag?IL(<8ZLN^Sc4EL@ihTFFY&1HdaCz-D{J%K$7O~9&A4%)b3f>otE8&7t#EBBhiVQTe"
    "X}>&KtFRA_F(~YI--8XiPp<H@`w7p;P&GB@%(1tRaWUUmpfHYmHAMqKvZ(=R|WuWe-~9h%V`^AZ=o8g`HY2aV=u#"
    "V{XShd?XlKSH^_PRA)8^{r<KphHNOehXsAPuV71jE+FddxCaRsuL-U!hW;Aj)Sx6=4tu^1v-|#&FW@_5J-+HhL>Z"
    "`1v70_KI=1Vicg3CwFb>HeRDOvp(`HR(WF278DnMh^U&<boK4qA^24%~i3ab@LrnkA^__VvAkD*5L_n?xU|w9_Hk"
    "*BzAYP@){v%E6Ki@xQ?@-ntFFPhlr@T9YCF8^N;&Io?D1Cu+wIkI^P}Uut-QEB(yxA%1KI_qX`nX*l;so>-?R<+2"
    "BO5ajPiom)AjP7mwzi?EcZW9Ocyg?!+LapqR;GOE0~TG0xas(j6$`dC<G^B|U((iV0i`-FtpM)?pLXrOn(X^^*db"
    "Br(UQy_a7x<NYL@44sR=~IzdH);Y>e2nhb`z1@~7YjEp^_W>3#0m`2HF7p1Y3Afc9w6dk^pbXW)DSj{UBgG&eCa&"
    "cti%XA=Djx{)Ql-W!NCzuaP;%aH0uo9UP<75?t(sXt2U8vJp`g4;F;|mVi8>L_|@o0t(ylGj#Zo3gA`HiX*8NHcx"
    "NWubnpg(iezhZ>uc$IaeDklgM%2hQIUC$fcw^sMvREspmSCQN63seTcSe?dU3g-pu}rpxJ-2e09kMbyOT_@rL6O^"
    "_xfKUl{dSrK6SlJ3TWn{!G^#L1uGq~v9LI3aRc4Z*ak<W^<K~Q1SFGvR$gKw>*7PHWhjNCIKAW!XrRE7d0dhC!u8"
    "U^UC1zIYV7y*ZCTDW8N&?fzQ2PV>;%;n?0*<5TW-#0)rDDh;E*e9v<<JKk!ko9)aq~<GVU0yICDkzKmbe~fyOCBG"
    "_Hl`auW^;;<F$k$ukyoKHdcwT0zq({#(P6o}2Zs$*@$}1zkI-Fu6LfX5c0zcM1LKtq?bB#@q)8`%17Ak?#=bwpc;"
    "yDLo-9zwv;3Tulr^;nSOB9U(*)17r70weE8Fs=5Um{x@Rmh7h3eH>k~OR%U7dk_(j?#|ww<&rr_*t~kX1<}o@&$n"
    "A!yFQc|GT<|k|`|e5kX^E-E^4{Dd^h33@i8e82gDwmjB5o4+p8KRSL>Eh;kWnfmctKtiLQyQx`3&)F&I-dsI-9MB"
    "ot#1yP1hL>SBpp&B;`!v6n!i{ezeM^cry9^LZ$WHlY^tv(aBkMe3G4vULEf3kFtZKvtuBVx}(%Aj2ls!dpG-a@9_"
    "2LH0%5+&*1;N(OY)R?Wm-|zj_4t>=ZWa9^UCTOuk8BgE3Od8a?yg^ACS{nuZ?j`vD>>a_)pmz#y7<_k{ha$k)z^C"
    "!s}AQAyyX9*k2>3=b=9t_Ieb7BFOvRVOUcxbTF=#SDs-#sGOr>4A)3P1z)n%tY5w3%m}ab(SjxY`MKIJ!f@;0uhO"
    "89-?-ItP}ikq6fp26s{=b4wzJNx%bP+mmx-amA6Fl)HVlS#BcNo6HiYsf-z~Qb&uYW$n~g9?Jm6}=d6-kowsfh8~"
    "@!Cbs(P6zCLR@8hAV`Z(V|?5WuYTfpO348&U*~t62M|q`^xDA>GK7H{UiZyMH$-?LIDsAvj3R#fDC#_F$CfYE}{o"
    "5%`)0tdX$N6X>2oxrT*d{xP@UBxM=;j{0WCBp0<NS8Y+X64P%`ib1&w2g|5Ja;CyC+;1{h+Dx@(A3W--JT*RtPNz"
    "w}7jUDxWc8HPS_*6>LpPzm(rn;EKv>yTfy6_7gp9r>8!bNMiDT8Mh*)|~e4h}(ny6EEY}Zp733!<Nw)te!s)P=Qq"
    "zRs80%PpTAq#4IbbA&q+%pJI(8CNxD7)w9#AYJ52I;}M^36mU4FZNCu|tNv5FI*|Z?EsTM>GH=v&@b9@Bjl1y%4|"
    "x?f1D&0_mgm`uCtN;rb_(CHx(WefcZ=aKDwR>}E4Xsr<_=bfXVDZ;MaHUxA5uB(FFIT(tFU`CG94e33$*@F?we%;"
    "Go|rsxKB5)_=WzK&Lh4NYk0Q>cjP;AsEw^$S3k5rdx&(#l9jENL~;L3#~Rqe28OHh&r87mG`@R!Z9F+*TanZDwVi"
    "KIjQlWuiM-rW++zHJRmw80GEGn}k%=dylQIP~En@o!>nX=zO<2Qes8G8<0~ZRal-|>atov@H4yG7-M3-25!dN@GS"
    "D2NSgE&3~5Y6V190-ufzqJihSpRBdu2V0bgbv2=peN3jvn1-0qsFF8N9v3!qcsboo#l4-IF7@(JnUH&8&=7|ymwr"
    "N+uV)O#e^fMeV2;lX09N5-{ZRpwCGVioPNM75$dvVE~xtq65dfktk(Xr7#o4oCZEQgQzB<oFj9KG~a}M<=5UTSWf"
    "}IK9Y}Znu9~nx!)D+UdScy#);w8a4|2)!Zgh;=N~FBNiHPCP6=Aw;~<~$uasN0avE((F0aBz89FoeIkOAjbkceVt"
    "&LxhlXCkWqi8;zEtROrVFFn8>Tgb{po-)k3CmHhQMc&ozH-i3k4r~pxVp^y}1CrI3@2dq@XW0GxkaXTf4ZX6M0#z"
    "1jvdQf*J`6D!VS<%GPXNh+)FdayeS~Ebxv8?$4)?f#%!DLQDc~>uWd^dRKZ+v>|lG{R0o<B|sklvBM*I3irY>&&4"
    "?pm4xZza}2%E(W@Z+Xi-98nTst;y)LF)vs7mai8gJAUmzwJrrwoPYKFq0^K6e`E!uV<J?55uq0LCYpiOk)USd@3)"
    "o}5ff?Fq|2bnk2J~b!2O={#K>w#3(%4M3gVT*dm59%uFkdvu(OBZGtYQ&vpr*TnV#ZF^V_J+)v8d{4tSY8g82k%d"
    "ra~h_{&E&RDOaz^*L6&--Aq@XMy{)@X9%t+*f+jA$GS}m;VfyJeP+j(9;{Isrf-g0Q)KDomaxj%d9?sbJGD|9rz0"
    "V8q<F~Pyv`z1C*7&ztP*1ZnTT+gCJB3PJ=+p*CIWk&b7u+<~s?&P3P)!Gk>!`){rou=Z6yH0<$9{uZr?mD~qt0VH"
    "^v2HI3Jg0<EJ;Q8apqonS3M_=;n^*wuqs<&zh}~VbET=7iYtIPU?tVnaW++_Mn=HOQHojg9S%6_mW_^Zuc|VS$|v"
    ")nPBNC#MnYR|O1i7(f@dX`9><Do%<5od@5n0!*Kd}ErLn2LDZII^yi$FstBE7|W`=A4kyi`GSE+B}%UKlCmGmLL$"
    "Xjo|*848LCEOFfEBrdX(ce10&HpmKNy$d3qwE7cy08U}<E{&U@4KEa<<_V`c$_p<*mDIIz_>;X1fNUJ@1i61YGKU"
    "#^Ho(|8U;_3s@rOg2^v<N2Lr$zHYj~4=6<95L+s$k6|RaAa$5AfT%-Pa@cein!K`cQoN6rgvex4DBLf2%p;*leql"
    "sP=a)gV_sE3<sUShh(#uzOyRdao(FiTw2*T&$T_S9fD)divS{*?_GSZBD{+&qPtJw5?*JuuURD^cd<CuN!7N)y|L"
    "8Vhjc(5qb;O|wRjJM{Imxcy;y$Jm|#w9?M^-$r2a5931~FS?x?DQt|z%;z(UzVIaTr*|ZXt)x|XHvRNjsLd0ye>*"
    "-n3OLbn(J;pG5j-+HPv-q-ro_R_-yyvw*%`Am|H?dy^d%Xb&*T~!MCk^i;J6HX6!q&_SuQ)@_P_l;mVLw?qky6_p"
    "qt~0I+B+>x$*B_b$vq`@K=sp20B+BI=H?{P;*sscLei%sYQSL_I-|_%=gXhB&h*#<StgVw%!Oyt-`2gi#b44s#F5"
    ")BqjEdqkd7d$MYY%9tS*O^vX$xlq~gl=IY%E=qBuMog(lzhG)_5tk=OTw_z@~n2+GnwQT43TePh!#u&~fA>F7P1M"
    "w-7A`A6-1-2-6K%1CrKy_n)c3124J(amv%`r!h3*73ydtF%5iTP6Ux+ZQP@fEKsL$40`i{Ee|xe?$qAZ2?F6?O9~"
    "?8pk8D>@Kh<up9AZsup<BZ_Fuw`Na(6MHKz*?$Z@<!#hA-%!3H1Rho^1S;=;^9|Ypz^&X<#7AkQhNTKvNUvBB9Y@"
    "ctW6v26b(nZbpILw@M>etvtnwXO(!etwC1?T*GiJ4<%{@3#6IT%+13wx)n)7*hJI{0>B|Q!xc>q)g=FU*v#!f7!J"
    "mWw-OrD`=CDT`j-EE8{ni?YYP`IB+oL;sANSz<6DI&2XqwmG6>t$=^I;xs$7|;n7*Zg&(%IO)RbDrI<piO76zS#B"
    "n65oCj2G%BhqeJOsVg<!D3FholGle@e|1eP!ysx<ClDOFs;JIfy>5~9TGa*iMNDvX<tcqka3?T2~;@ZztHQEaLAJ"
    "v-6BAX*gX_S;-#rd*4i!LiWA=#@8+c2>fxR>4V;di`j_3&Ht%M8Co{#Jk?#GM2}Di3HTlKsdqCa;eU{{8g`Hj-VD"
    "7vM%#XP*7ZJq8;_S2GSV$R=ssc*0<o^F>JIqS11#Ir&1Tw4{cNeEmffV$EVXu>`i7ee#g}L9PXy6oSewR!hTB&4|"
    "m=sQYttsOFNX?A6gNj5SbQP|H!YV#U+X8nby0(L-iYOzI89`R>9rVt6jCm1AUD{UzE+h`#d;xTVb+mR=eTmCQNNC"
    "03jrIaBf3ib8*8Q!toBLMzZ142gWK3gLarI|$!R##ovS{F*7Eo*?J12tdEc%waRT<yF>f=76O;WmO(XwZar*!-o!"
    "Dj8ZQr33o8LwYOe2e#q;zTvZSaK_cr=n*sLh73T|koV{3XeK)_jUM=Q}O$~>yPGy8c?H@Sn4f~m2VL+Imr2ty8xG"
    "Z(&UBIt458BKn^x_G!m(?e#f-k9b0Cpe@i$0%tQGWPrJwtF@J(ft}-9$mdbH!huhk<j$!3ls{D+~=mbbVT+48T0N"
    "TEQZ5?{wP<E=WL!nth|Ztglx_FIdhtwdGws%iG9Nmotc@$WZ?!{b~nu6Ua^NB;ypYlsHw^-S+hXcuGTS(4PhThUT"
    ">hQVw>SlIj%UsN^~`ZGbDMuF;V83V}0_`vuh@Z~+3JG0DO6iUgy|0F{QfK%$%=0bG|e7zwi37R+zdd8lfVH)c~d^"
    "YvnLaa|G=CuIdlVFS|*H*moo9GDU36-G!je_Bm{u<99VxR(%(0E`Dp76wHUVJ<YOYo#(LJrniHJY${KOzik*f0WC"
    "?$BkZ=O<9RrL=vY<5ECDE(wcM?%gVbAP}!)k1-O*iJm|DS0^qM^J*&$*W{srhDma0%Zv@4=CyWgT4SslKQR_8~-)"
    "I<6zS<=T$jKE<L{TA&SJ(kGppaAjBoYV6NH|Ur0tTh3fqM3XO==<7T~DQe3t$7T0TNZ(7~PmPtb<u$QVfFb0=6#g"
    "J)^PSEYNLTVPkJIa19a23ED*yZ7~3_7Z|5Zar$il^b+&7aa=BpGIE-BF0L5Lo2>?68s;xz6J5K2Q5n<9vw)ykD}c"
    "rtxl9Q?*@OealYwvOcyDEN4q8|%DMW>|yC<uRSf2B&q>hA$h#Z&Jz>_fb-pV_i4KI$7*Z@mu_#WrRK$5?q76ms??"
    "b?kP{ABITvNn3gtg7Mki;K|ezMFZE05`==hkaJ2x_9=<g1hjQD0&wQ?}YC!<`iNDq^O$uO#r5<eKT{m=*SHkl*wj"
    "C3ThXuRWd?NQBm7YI&Vn00LRF4SbO0<=L1byF@UCrkf#hpOj@#kkva}=I@nK2^;h(kxXG_yfm812464tNxqumAll"
    "~D;r|bEHD8z(kzP3zx&K*F@6=@G7Us<Zl{sG6<(g#-R2g0s_sUX9zX0+Ad4|rAwwOwS%K&&0kq#FtjBinRkg`R{<"
    "LIl4?XiQ5~YpS-G*|uUvgDJILZ`5SNp`gWSj-5u?FJ00TfdS&Nw}Qd-)P$gn(nRLUK<nl_9pM&SkO$31Q8XIE1!{"
    "TS)4bw7@XQ+a#GpYzwhQRMA1$zubyf3PDNP!gnE2c~O%@Arp3qgJrlWi|hUA=ffd^hor_1V7&V%)APZkYPq*(za`"
    "na<Qy~N<hDf4{r08u{|tj52Zw>Z3gzptz1FyDJwa&z>1+oS<)E6mLg0I1`Ib))3a6cDH%7t-B_ny!9S=oNd?i*BM"
    "F%v6vGkTCY=Bv#K;=$jTut{-rzComy)lkm>$AOG^p!5QfXsiACX=$!iH`C1R*(SpD(RyQ+1#3H}iDm|DvIpl*}8A"
    "`=EQX~{kSL-)A(EU{<-Ro}9R-@1o;AkpF@|k}s1VMLl(nSzC2z}|fc+Yd&DyON4DJo=Y%h#}~o)?vQ$aLnA;n7^&"
    "lxS<sS$`wzlme|3l!n9>80vlZge0TU4Fp7Kz3Si%#Dd;EF|6UQn=*C5ApN?5L&`tSs(Xu<|2j_$_K=d7tz|D;OJC"
    "Z`Us_6F{0K?Yy^pNqZLg9+yt19zqdy|=|8dzRHX_uJUXqBTL(W}0SWr7)kP2FQe)PuU_&NPGjT5H;47{Lf+r8n*r"
    "gD+swC^y0%On^?yrKQ}+p$--(spr>ArWSDrEk68jog?{1Hn&4oL^ZIG?t~tX*bRMJWR-G$mH2p;Tl)D;=s?elxP<"
    "e%^PNyv}IFwqf_;=?KHBLh<2eE*6_3xYq)ZYaC@4)a`+I83PCQ3y_e7;I`j*Wm-ZU9x5!ib6m=i;j$bZjQ*4WjuC"
    "YABiV4IHLASdq4@JimrLI>VU?_p$c3Of`)+>nKJkG9D4iztvP&}gE>p)|+y4w$`MiQxjn5yV!J0!1Be7*6~2h>&i"
    "wO@V&oK$!pkk(n-VSt<BziTfpZ@qG4!TQV3ftC&%x1`XBhnL8}*lcTSXG%>A*lj*iE7OVf!i26f*28}s{d91Y9sK"
    "gk=*7X_*(eSRKV^b}@C7)p>k2?)$&B)?3p3(o38)J5bw&Zv^UXR0w&gdv5O?ec(C`XcBXZ_R=V9^`gw2e6$|sn9a"
    "o`m*NT_gB&p5u*5@Jg>WP~9^3!7gD<eCXCG&JU{>WgYQGp291pC$6kpdT`t)-e>Y3dupEP`4X*z(E9PI9fk^4R9~"
    "6{b^O9(*mnes7p0C+TdBv&(Sm~FA+q$8*=4QM0cX5cyc5rUJgQ}Glb9~aGR+md-uKz-^o#V^SRv2W_|7XtEgF(=w"
    "=Pcfq@p{XMoy48qz>_DRA+k$hg;hEI)_O{)g2)Jct?yHWa9EP|WPS6dMYPvw5HEHA<9*0+Eqw%aULq&}1Cn7h$Qn"
    "V5*^{h5cgM?hQDJq-bdBX&4PNAg2sg%Z?`LjRvMZmqfw-86GgbD%ZQ#%TRJ+ueTudZiFd*wRiS&!|f*+RY$-jo7I"
    "?pl=LnfhH9~mgtryd2vv=vRQxsLC7U00T5mUXvuA#EQsT*75sn-t0e8Ea+7lNLUYIWj{rVmdDXF<7MJUTdy%2fo?"
    "p&;5RXV<zdoM;<y-~ZK=K6<5vkk}dMoLyNNxubMbArz3ZS}40@<getk^Iy<WJ*<8xwtC#j{b7?Gt^X<M9VI)DDJZ"
    "!UD?xv74tiEp7kU=S<e<$GKRg>DRy~xV`S3@hFR5PVS`?Dt#QEss}tmDy}(qAqJ}E<0z=Xza!i2965iyoY1UJ?=="
    "FvM2VWY2*Sn}ABozY2x7)9J9h8=UB%>G1y?CW<4PjftV?dOcC%r|O$njK=);OJ`&KNBT8puq&Qg!ttYFZ@zMQ;Z("
    ">IG+(2d52<ZJWBsOi0<V=z5_3dbs<D_7DA+DJ&|i-?Cm<YgkxADW@2qS7j$pC|AVO!3t27?pTiarW{94UL3r9IXW"
    "2~o#{#~*?K(|CQ8y?k|KeMJin?^*=4S{_E72rP47_g^Y*|z+NR(!^^|q)#qs#--pnk^4BUFNO<jEiknO8tsRx@t="
    "E`#hwSR6gy#Cq9gR&>17-83A0O#o>qpTS(H{vO_R7WH%4A*f?fU`0xZl8eN+XuE@w0FHA#X)!RVPzT)n4@%v!?-2"
    "xp`UGP3zZQn4%4vUgi-VDzW0u8Er%wM0#j0$34@2K)dcr|uRKgK%tz4122!OL*d+W(?o<kisXeoi^8o0~;h@0@hM"
    "sg*OQDM`Wa1X;LPkf_Ra~r$y@2uHS#^oEB3&wK1<p|;?7bpeI1^q0HBW%vtJ15eRt#KjEeq<(02!A6PfV%<MOSB7"
    "gkJuq*{{sIr;TjBjS+^Uar@UtGsAE^*j7D+=8DV^A?4GY14zQVnnQPkjAaRM@o+pt!1ZQz33EWpo->IiQJbAfF8L"
    "4WVt`gam@29bD@MP=WCQ5G!r>dt3-V{HjR@d2mq;x@bnz?iAOG^|a0J)_m{S*-9ulc1evkQ_uoSS3;ZOuMzoEYL8"
    "CJ$i$v@_LppLwe@^5n3P@#-5fO0>nMkG^#sUYrJ(yr%hV=g{?C`;@vXR%X7p!O3H1*F_a3E&_`+&(E)60*hcVdF<"
    "yM>rDXXMJv%gt377jXppyrucr!E|R7&Wz7LarbVwSO|v`F7VxV~LypZqW6FF6=k6IS?lTW8ws}yXFNE%)>}u%!VL"
    "1Mxx+rUxw{9s^!9BJN$)xLmLNIljm%_m%u%~0%GL(bXZCqu?*qXPsF1feoAp86*&z|+6C6+Oz0UNRt6HvF?va8CJ"
    "+7P*nL#jyL+ij@bk2+sP?W-VP1pibTxi0;WSOW`ef-C|}?TRso>?WvO3v#QaG)z|WLq?Axj^sN)xpC4Uz<uD%C}W"
    "zHbHE3R=EvMLVy47vFR6+A7jac+6=gK089);`O998U<u*&~bQwg-nT^iMfHT`9F7gJhuUCuB6}Gb%9zF0i0oGgXt"
    "BGID%h53BVpd9sIHRsx0G^8en{X^p>gd$--0)4!x*&>+()LlgGkklqgd7)s`EWGM2LSF~V=7-JI8amejhjgseQ>?"
    "O2WGf@k+#G0vO-#?P=eAAx*z7@&t{NmUZSo}Y+ecLaJ&X9s!3bWIzlAaV*9DpXu>32r?^L23lKV_CiPHxp0aAmMy"
    "jwk%#lPB6i-$^9OVq9a&ak+22@t()T{UcadE`3g)~dM4Om2>vwFoD>3TQypk2jOtq3w<+`K_Hact!|FqG$$=vWd+"
    "IXK_-#gi^O_}t8HQ8!qx3h1B{Efe}#sSy{fBLbMg`_jsAL4o-N+OFY2pEz|YJtuJ(y9@hOJ?7a!?jB+h0ehl1wwH"
    "y&A!!TtPJxN<CAk>IT5yY3Gn_s7`iqDd=;^52TtFl8RX=-E%sxosmU^jdvxH^nKs+F!ESXoEZ6FJvn@y}FnAqAI{"
    "iW;?!A`Z&0^3Y-^roaoy0-(jykrAcRPh`$FWBYt(p~71e)UloPbkuS6Hze0ym$Pbf$R9XFfSJ%3tV1OT11rDCDqr"
    "m<e>j`T+hqw8%H4A39g$?mazj2mGo^$f;l%!_kqB%L`*a6S7O2aWo~`8N5c)z)=MGxwahZoo!$LgN~U+ykKZ`G7)"
    "}z_b3hGL_8I)O_F-KP3|NlQt|Pmn<m4qTGxV=qxlz<vYf%{;jSIQqp@)z#3`kTR3HVk8d5h@JiMk>A*j8yZkB1zT"
    "i$V&VQsK0NMj_ukUYqzoYtY?6LvK;KIRU!Rb`WvS_|yxBX&bNhk3}UQ(y0F27Us2~YUt4#{<|j!^N)rsDkzM}uAv"
    "SPSbRXDz)o+pd~KdEs|r}OXt|wIzIU+=EX^wGfI2B|w+%>1u(W<_#0snFs80t@@$BUI_zWo&^Bkx^AXIGeZg2t~7"
    "<~%rcWmM`YH3Q~_sr)>zD>ULoMbtEefH}0S+@W4!Ql(=Vo@CgC^_Kn{03db-?VGTa@tX$+t|h=tTN!_+0M8rAw9a"
    "PZ}Mn%OMp-XFYJN}xBH$925pK^KhUO7BZC5`OWx#VStAGpr!g}O(T3|KHVu_HMrwyLMhe9i2R;vCwTeob;Y~abJt"
    "HBO)Q?_v8^WQp0(`yF%G7cwYPn+`lRHk86w8)zf%A1W_g>e$kmXfL*BjI&lK)Y68o>%;vdF;8!M%j*ZT44XbS=mq"
    "B-IgK5)5_l>JtCPT2tH8dP4ZTSnpfME+|<Ajy_o{5b$TPZ(MELrdVZ9AYmeEurR)5%BU;2^u~HWZ?On;UDc4IADI"
    "9r96KTdeR~8++dkn~e77cbrzGxR0EOd)_K;n_M#D!R(r`3{DG1WCM6az_3%B)FKEn=&0%#y}_BgTyz9r_5T`t?;+"
    "Dkt$m@T4{t(S`PeKOV-#W_`%k=+(^Xjor*b&Q3fO@oO&M!Z2t3nMBsZh$q2SVp{&zL6zmDkoU*e!19CpC@RwlXdp"
    "Ld%pjDmy6+3MmaN4jIUi*pW??yOlfXeUPP{=y+h3bp}eB7uZI;%P+*Okj8oNcv!YF$sQyC}Vl5~yHf!PeN}aD4fx"
    "QrOLnzCqBFAEm;U6j)7Dl!P<m|Ol+AsnW8FQD#`Klsmrnnf8+$FrCxP^pEtQid4hpBbxVo<Yqe&@|SCn9a)mX7sD"
    "k{el-&tX7)v!D{@ouI!u`GM(RYSlX$j}*(8955qN%<V;ZvQc)%ToxV7vbwIenE8e>^t@awK8M>|c28Je1s1Vcont"
    "v+Ptq<DVT?7VVAM*a0V5VFN1s={S$)LvE-Vt0?P`)93vvB9S<#RBf-N_Ync>8O@5Jq4Z`9Z_!~3?`fDg9IcsjV7W"
    "R&3-kG9BoJUq-zwv$y>TCJ#^^%l>vY92ARxe}Juye+t}{V+N-5v5JE1ZY4`21-V=j$xIG<^dn4ZnO;8W)NDlZ|+e"
    "Ms%*_8HkpH{=lZ(yZ)4#_cHP&t-e@Vt&ZV=H{l-Gu5S0?zqwl(;@%tJI+h)EuI98<QJ>{F^o|=#&fl8CvGsqJFhr"
    "gEV$0siij`mLe5{~xr;BbVV*T{_F@^m}JPACV=>91oitsD@lG_#|)8MAzXn(F#`V-$z}>SnzvV~?*(-r{peEO2tP"
    "Sns`%;YLuj2AEj1!3#5}SpxI4gabLZEkmh=8>63NHKu~r5xH#!D?csGW59b-tYb|9MnZqHQ-`ZLg8_pFA96KtLlz"
    "eon-we|;=CgVk3^O$oA!5t0C|{_c{~cHbJ+cdNSfLtt^V?EFdz+{QbC|>-sQfb3jq|Ri7<lRs<@38jyoH``-1$&?"
    "Za1EAqH2*@9yAciaWJ=(TRX)^oF!_h+?NV6eW~l1~tTWimN%Om=~1(9_eh9`NX0zSIBrS;~g4bp!0KykQ8&Wqwvm"
    "nJke(nuxSPrF4oX71Tm{FDzHg)Mm`8aEDtyO??q%s*|N=PSzoNGbKFNrU%R-#&FE)8A)=n(MF{svCRAt}YSPS{CI"
    "e+vaih*CaDlJHO1x|5IjcUDXuyDJ)&WQ{1aL+Dmw_-4M=4UlH*>_*E9z^3h=shOVpX994<P*I3O*XrgIN=YDB>uh"
    "OvG3#fYP)oZx#f4fOFuUfm}>ND<w%PbrG>kay?-6HsV-#aEJhH!fHHV%a~tK-RLpNm0t2;P-$@k?{-y^%@?SGMv1"
    "U`9c99Gy0&`U>3g8`{^q)#YT6oe<2L0l5($#<8cK5v6k_wubr%)3b~VN^UgYhVB6uU22cR7;H#2l~tPwqHb_cSor"
    "Ot0=log^*m<O>ZUr5dkBk0<oKdw^qcqD58^ClEfZ31Yk3)$0?MboNt4WCt@yMC3ohBS~c(I#a{!cb2YcM&0flHyZ"
    "n>$EA)`1~)?$WHtNN(=3jvM<-cp!c!eO<{HDHQYf33o8|L?t9ojKq@46?xOel=;+|+r>2CS{&u@I%CWJ|^h;xLT*"
    "uAEI0tvYZNa>PF~cxpak0wKMbwX}8Mm!=mQz)yG{J=Uju-&Rkolc+0a+8=sOW5##VQnElwFa!yTYWRHwR}wAHO~W"
    "L1);5=Lr*E5<7`vQyr>BOYtWHKpL7^9jC++5F5MbwN{Egx7eu!D>NmvG@gJsMi_>XgWM#LyWGnS6bK_4qP@Vw5xo"
    "rBsTAm|EW9JTPtKF+CpfQSX14UEs=?Q3$Z6683)XxK3yR)xvw;gp*#`zy4+(#ZH0fwO<ZF-trt%_eaw1h$-IZ$vH"
    "zKXxW1lMFNBZU}tY1y=Wb^X2`vYm7KAU1b6uH|kv--9WN}zYi&R2_KI+N!1NQ(f%32T9nL7<qY@)!tDiSv1VMSy&"
    "+Hm}l1v&HIMK;Q5fCo@}#NX5Ml+7=$$oUmH*<T=p*W-cmZ>-FSNfIu)O3JTwC-HH_;;P3!nON+M(vm?;)AU)r(Zj"
    "_p7SgnV!FsV*0KAtxic_4GIgBk>RRarF=9caLk-Sa0zTKZ1zg$5@=<hzW03IM8SQSjEtYA+bpY5zOaE%v{M`A2Ca"
    "Pcsx>tMbYaC1U89r6)>{Jlm`?J6B>9(dnGUW<BT5`btTd9)?(eO=*Z$t}|>b%wj@|HxVeo1u#ZO(M8flu5uz)F7k"
    "^D?YZr>GC@^c(bO`fd9^5BdL(Kn2In?%AKzs*IEcX1SVo_iGrSq9n%3Wucu#3d2PMmWStzTCOU)ATu@x7yV7$plj"
    "QC7J-!V+QkQ+S7Vf9yNr6d8Vr8x?ps+%GKFZGl+G3Ce63H(rI2vPy8X^wjn!%8rsIUzs8kq@Q3nI4pHFQ^U+mbAE"
    "yJO&Akw%=`XRLe4^z<u<y#Ry4x$=fv}lvR;xgoyqBGLR~lY4+pj<?#stgR&x~z}j<il5)CFL&oZgx7Gsbx0kwcIF"
    "-wGqLZ&2L0tm4^Hg0|pM?8>m2Xm7reS;Kc)1RY0;=GEnk7U=MMho|5DHu-!!yR<s!Gz-@33eZxG|@K>>ZvS+k&Ky"
    "#+3^}>1{q{h-Z;%itFrUT-Q3l%TzfjxMUNr1Ptv}X(15Qf#LE|;ZwYLf<D7S4odSb!B<rWUYp`0Uvh)U&MxO5&Mx"
    "O<lUmv{q6ZElN3gI+xuh%bj+z<56Uiz0Ar31Ewkl3fsx3VTR5XsMq?O_MP_kK1U`krUrr2_z=RY1E@Bi8S`*QE#a"
    "P)#4WBAZS^-?1KFG||^T~iLeNNW@vTsTa*CW|_ihf&d@U^rFP*)beCyAaiZ0W}}#Cn&ZQ3hW{glU4`W^3VXtXhm4"
    "Oh~0}oHq`{bF=xNC#qT#nksupY`@;B|w=g<)NAe(7_~^Q_yNF{XTsO7~FX*+mttc<orG9-Tt#N|1x=nqwtWg{Geq"
    "`R6D#NdhN|G)yXJ|05rJz((hImls!7FZjmc)cacb4Iu8$g2n(Y&7nx@*97#57lqfsF-2_DBBWbY3j$>&3e5<}`fa"
    ")9kD;3#tW+tNTE24f`E!{Ed{jM({?syR0=tGUnJ}E{0>~tTy0JQ^R!ED9uyjks~(fE)MR;?4@!GQ;%$r3}Eb1c%x"
    "G#2Slw5G#IRC<`Zvd?V4RuWK!RC2R3xVW_oHc1ivm=2E+M1K1E#u1Lu|(gMp4p$yG_#2+yHm{gwqr`Vin3P(nCAW"
    "a5J<Oj0>bsSy#-TPHTDd?+xt3@<9HCABBuKu|7%A~jyXv8aWaA#_nlIwJH9x2r3Zh%D{E7EFz3B%T!r-0GV2^M1B"
    "pY+wmEp@eoAvGbwpFCYddF+Z`sLC6?45wWHLqLoue6y!U!+A`Zhu)C?@WP#AA@n8Zxa4n9G&9$6p*kPPO0(&$#;N"
    "ZmLlRcK7IyMnMckYI63dR&ColDZ>Fw;@7PMhOXVIXoY5^1mja>vl)=er_$iXGj8=PBzwBzZKZof-4j?Ne|5kzOVJ"
    "JI<tUgHR37jTSJp@d9D`4HvX5-)?|}b}wW0t|W^29q0LYMAhGa2QZ}t)7Hp)*XSd^eD~z@`fgeBR`kbXh-i+-_k-"
    "*+es$k?>;H)T@s)v=J$jn`xx71H81Z!gE4SJ(hdRA8Eca&g3D6<TvyP6>+@7Ppe%B^`%K8GVa`M5~AFAb2q?mAXP"
    "-aQMlD>}U{?4pO^J6P5h`m$Rr3egm5jsArd^1$SpBl6-Ft~zvRETr{jP+_&*0n8iGdhM*6Tn1;E*d$t#j0_Al7bS"
    "I<ftxH5^S-ZyQ@|=JP}j~cKgnz33mQ8^_0Ci`PN{=u-i-R76szaD3e8^5WNV1Kx(*?QSW5$;B@q2pu6mmfdHX3?5"
    "l#Z-9A-vq8oNgS2`$T6a&emvEB0evi1wX%4^80_+TEI8nJVF@YA0U4iCGk{{4akJ2hf~nhf%FxJFSh*^0~cl0;=e"
    "LU>Tt9K=eWmehaczYTArrT_b5Y`n7Az8%2N{9IA~6?3AE8Px~Xal9@+tviO(ED#_1-ILAwviC2Mk@Ylt0f`e+g`B"
    "F;L~6tsB!>?~$!VW63|XP8JC(rf5?9dG3dd%R_7VB34NIDq>r%OXmMg?a#Lvr=qq5AdfS$3goxqI-)~_R>NZSraR"
    "a!LG3!x@!BURizZb5Bz>h)r|#E9Lt4E~?Q?!n_&fH8T_MeI~|H{p$;yd>7>1?VqmwPNE|Hm>ThPbiRL%X|rm#j!s"
    "ZU~7&8h{OQTV6mjO>4ue0nCd27U_hMZMqwVXHls+G*vL@l4rZG(C?e|(4PL@&liOm)2}U3)ZO<v=O+4$fJ8fj*yJ"
    "9Dhbg04;3>pyUyu|jB#4^b<&Iy7g=G~<+Xw+z2q3h@yg)ZYUtjZb2-!Wxqoua-8#Y<(yG0Kq5jd-DogUCPT38Q2H"
    "7W$$ZT96f|ejf@OM<t2CP?4^ad}A976Z&(Mfkr&KykQY@PZ5a&Dr6JR?t0XK!a6^g586IWc>LrTLDoh%(8p>q<35"
    "93lDZTW#P}`|4X_8wB8M;vx10ku1D0kC&t%D73Jq@1IeG&VD<Fjo(#g3`*=;os&CIkL$Qqv1_vH)hPCJqn84V&8l"
    "HT0tf#sVAGwP<tFb63#nHxc%U7}iqjoEIo3W-9RxvWZ)QXap3`blQuA~VxqwSG0MD_Ns(Oue~)W<Q30ItwcEkZ>b"
    "bKSRf%l+t9qEz}fJX5fJ9me5QVvVRX*p?AbqteeK;=E-=#Zhhb^$8Ja=wfwP8njcPPGUvdgijenE4lfxW4m77}G-"
    "~Ip7F(!e((m_Wi;;7k8!>PJy>r%}GB+9dPq_#-Z4wpPWqFI-vBtXY7;7>div=c^hE1?OS%e}1df;Z)wb5o;+<Z2~"
    "Vq+nc!VMERujYlcP8R#1EGaNr)hVABjmCtZ3yvC=PV;OKrn2^s9;a1Z82hV?G4Fza2rslIJA(1g-moPoeqPzOW@2"
    "HArDmB{#1qRWbmH0%IPs8;D~Jbd;PwNusw!g?qpW}p(VM>9td?=yM;Sh<`!{C(a*7-4qGF0z2~ekPGK(9ChN-0n)"
    "tMGDnanglSa-ehS+<=s-dWr?QVDE`jwwZB8O4>MX*OlyKx(cRz_L<FG@7nRNu}u2ply1x|B`qN77^OBE{m#RLNb-"
    "{e1Q?5mX;L8@KR7$OT<OXK69xiS^fV2##i$Q*ggyS{Z{E?C6In7wP{8V2#nJ7sBTkJo<|t8TaDUX4BH1e?i8k7&p"
    ")U?sNj-Cw?8jcBj&cnH`~|w>}~gb*O<&`O8fHKx2gT#?px=*)xdtZk-dFGzrMk|qc!ha^!_>pX}`MJoY!T(wcVUf"
    "Zh0mFa*QkH(w2|SlhW2#GWy4$$!qoc)eGohq11u`+fkvIgL*2|130ZDk?E3jF{08^LIYfuA`pCbsWUDwN75P=%QR"
    "S@q*Q=y1U3n04v@!D!s?{QNn(J6ylm1y+GUBkdh6P$i1p@N9AnsELoj?h1xt{|WB)X>Jw61cKoUA!=hov>qX*2V="
    "ZjC6g<slh`pdg+I(k+hDuB&Slk%I1rt=BQM<;jJ4Xpu}2+QY+WU0{zX*%XNc3YxSB?^G;pD<{OXTJG)bTR^K4qp0"
    "(aSZ;mTwz;5qmNCHMyC_@RbsYQo))f>>_&vk1s~3@0&!|^t4pG-=dish*Ga+=pXNk|e;4GT#^xIKxG-eKnzBJ=cd"
    "d(PHt^M*A<22hS^luctP-qO8#5#oS_B$++8Y}vzVt%0o9yKZasbQ=P~X@jDKcK1>#&Wcb{%OgsW7T=B&eA%kEn_n"
    "U><iY@2nwjQTQn2pJ+<#CapD9!tH4WfT8y8iFkXU0a==#ivBrnN=dPL`?mWRx8qc;Mp&SsOEyNo9*o{3-V1BC+TZ"
    "M)x*b3My*MN5>%SEboP4dieC%{7aqR2Qmd0)EOK7u|{7ELeASSMrIVJj^Y*vtSHhf5jIu@rI@cndjCNY*FG=OWAr"
    "Jz{6K0Iuxz#?{wef9p7?H#>n9qUhz=`#l>xj)_9ENfl;bhDV2voU<#fj^C+#99<|K|)=02;E4&La#XKFWap9?G|_"
    ";xdVHLo7gwv$mqz>4P}^pm*iB4I)rz%^)=m6YRQVVYB&7ZdvmC=#1=vzne3)nO{Ps>_sRpUSx(ZD)PM4s^tV_NXP"
    "$E8WmNh($l|YQL2}@-N}Flx&^-!iNiJ%Xk%)m()IO-EGFkInDQT+$lWJLnP#X7U+o-VYY+ge?sOrV!<l{4?_{hO+"
    "9~C<<6;QCG+Le%-X4`Ir&74SvY`N)RUIW-bkWmKn=In1|Xg?T*f+6D}xv$t53!zw3ZgBgwR6S+HSjIrUjdv$QpqP"
    "l7NO{%WV}l%5#Rkm#zBoP_C5<wruynAK6ly5kxDx+u<XkC{^5t!4)oW0eacdviC#lY>Kom>_gGkEl*nkx}EON{-&"
    ";GhGYBh!*F~s2)FB$5aRu}65R}|01l`>B{N!g8V(P_K{*XwdNb&ZdLoYk~hN9q4S#ZaenQF3WB9O<G0$L1sm44_|"
    ";9Ouat46t9(VlA9glP8U03OxT9L8HV<gaV{^R%$pZ;4u#r&3}+_ZKSRZfD0&G<t|98=4>2N1svrAJI9S-l<%G}_>"
    "lX3g_w28*%6K#>-uo(s?>-G#RZr0+svE#8ZJq?E}-gT0sG(ot&fERHnm&Jb;^%x*!-wI#(XO`D1*DUm9AS!M{xvq"
    "hsA1ZyHDr`?h8X*%JtbUDWGRkVt_vW;Dz+!kzMqz^%+9OsmIvg+T6J<Q`q9y-8whU^?<3(jAZSE^W)#UztQ~OzHg"
    "_c>9l2T_-l8kJ=*ddt}}0Z694|Yufh(3DcwyNgmhy^Va7D}`Yc-27rG*&$H*0;M?SNt@Xopm@{A2(=o_AqnTcvEm"
    "e~rhlTBZ;i8!fDC6%UzlgSl|Eg&!R=oa6i2c`r-o+m(FDKQC3NNn*lzFLszFQ@{pKsv5_R;31@iL-GHqTXg5(%4*"
    "Yy>0OzTeS>vq#=KFhUry7#s6xB@kLeXn83$H)!Qs@V-jN?#K1~)7d7E3K5Z%lbxu@F8#OPA3X#(nFly%f>9p}NtR"
    "|<)+(I|tf-E_@3ynjg2(Wo*v7Ljq<Wa39MZX1xN@l1xWZGqKB3I(ER)UEt30$R`dr#Obt=P)MJts0}=u;uC6fSA$"
    "_ay{thyN67gKceSB(kDfQJQIJG=WbW{r*%H2ENw?mfMw6#_pgpRtZwlRtz+ZKvS|CcZI?jCsl?U?ROSMIiq4m-3O"
    "wfTUT7%oJfjjL}p5c&ABmbs@2^kY{M*WrFTMB)YR-QtP$46!i71o!O^dylfMufw3xWM<TA=3qgZ$@rD1orx}+1ne"
    "J&t;4al$(eWJ5qX{{%;25pkNIalMGxl<~h&2LFIx)Cr52Ejx2GL@7uu_x*5f_lus(PXP90-+}QBv*+c`P?YW#H=t"
    "+b+suaTW){v@Gv_)I2*;N!2%F(&OJUYBBeas8eIw-<;%{^u?BYqb)dl%&=S|VQrUPXdq)G7aTCw`l)!<Oj`d#kwC"
    "dAEEp)(31OYh0yqBz^ILUBZ2_2@NQ#c0G3Iv@rrnQ&7lka=qVL6q64?{inKsSnz=6nlFW=+_Kh+ROg8lVAXa+bD2"
    "AACG#XNT119as!Ou*OI#<VWHJNDfG1|3Q~OT7If9?H~CxGO0u@wIr<0$}6+#H{f6<m&?L$)~O_N1IU1CGt2OKl=s"
    "4O?L(N4yeVPvJHVeM8dBnBqrkw`U{U9)<3)?G`JdP~G%+*oql$-e{-L(H)J6(IqBmBXuJD}Ng3xN+9Y_Q&wQ9x~?"
    "HM}i)g||KLP#@-JL9&H;v#XyACJ#|PM&h-fRq|&bdFOd9ijwd{JC=b%y>W;*o}o0zijP*C+CaaVrg|Q$Ai2LP5{~"
    "8_C!c3I0CH84kbdgt!)+IU$X>UH62ID*^Z!j-%6M9oPQ}-Uf=N({8C9OBk#@suujLKBQQ1708V$b)lZmCC+YE|+V"
    "<b6Ngge-33S{7WKDQHV2-a0Q#UMlAi0F}BjioolnO*bg<vW*qI2{@LIH=7NewO}>&*@do^pbML}C)Z*QL%XkRUU6"
    "bOyGIQqg;lGh|Q*>3d#bui}M2B+ftp7}s(w^#8S>KtY&_%c3hufB?2d)&NrXgb<wo!{!3LR3BW9n(U|79hf*kUEK"
    "D0Yb)0u7iz2QRC$&PtcNnH-Ds!BZ#b82GBw)K82HXav_FT6PN#5*E)*z9V}%gj$WMI-psQ5@j!MdyTwH8!s67wzW"
    "E{Z*SRzt%pXG)9X7!4!yZTzfXec&jI!0%>v)A=k3vj4{6%Jo`F6j_EFaiuzgv!^sZ^q`|W=oi}C+!&kn~U3!x*<Z"
    "09rN{hH66o)#<W1itfNQqPIXt)>eK94RQ&zdCn&nkfd%)oQz#ijbTw2;3@e7tOc*CIuMg?VYs{_;Yr<Jigj9}tF}"
    "L7U;7=0X02799nKQHfNbS8aw&fATQ=6%R<}|&?)ax0JSrd*B38O|l(q>s`^z%FBbUwt}9lVep4SdA5J2zs$(oXgH"
    "WK*#Mw!sCYD{1pldgJN=4SO;)Y}l!u7{g2gca5+o;Y)RPr|@p-h4nB(VEQ{Cu{2>;m;TMoW{oo?`2jY6Ln(*N4@x"
    "{t-t8ZTvJHmTy$gKAB6FRc#N*L$f7P%1*?U|yy>~@okc31~;eeO_nP#}9cQagB)j?AtDrMtdb)*zA)uVGBHP_~-|"
    "H9!t+m|Q52;wk8=9xK@&+4{`g>goF>ARL2aE-XK&7GocXsqpIl_05YJNE90(Ez-05JU~uE<FPZaMO&f?1@EBLdI3"
    ";J%AV}N#!M%9u`gjaRQx4Cy9_XP3-}@@b1aU#^+iLt|vVv(o3M_7@$oKZH#nl63e?Mr=!Es{#k}<rML)C=@_%F@$"
    "%&O7pDhMHj2k&C^aRTRVO%Oxz6I~8U@uz7o&Y>q@QVdQPsq1AW{RW47j}`27)bSrji>@L4X>2bd)vTp(Pt1x*>0V"
    "umII`I4|LdW5(M@NN<Voej!Df{>dY-I+ol+Uy$darq4oGqEAC5hJ$OJxR#6SVJ!HAn}YoqhD#r?blyy}Vwd&sEns"
    "yRx_L{X>Gy=u4YJ+HG@z%@{V%&Tls_UHXixOig>+nb^YiiHh%}1awy&5Kzz4<dvt_XB1n`#8h%tB5YWf)9?ngjUm"
    "TdhcD6Xy`1a5x~X3X8s#>IMQ@uxJgdJN7HsZXVY$Kyor?eC9Xop~v5m~QCCFf!0yz5nyx(NCk(@yY1lUyn}BZC~)"
    "@Xe>`DY))ApB*Hwq@Egu$IXWGkoMi_`XUAeCb~t$hl~i?*;<J<69mpGaw?s>VQ_;<S-8(dInRWh@XW{>MlMK1jR1"
    "<)VxG`{?WEOgR8aYM8F&eLEIDEkrT`COb>vo#b_1YOmDhIEdrhqTc%Gr{7(42OFkL`=9oZL^Fej^ieX5B#TDj@0e"
    "^1=_A-p=pbrMNx!&Fk_g8xG8hJ7dDlrycA1(bSOn_MCfx9JE}mvu2-V;`hGQ`mmTR2QK(A%5_AF46&BjeZ=^nzWW"
    "Hm*Kha1NIJoUy!VkOX+&&;On&M7#W0sIk58~);?dqOqZh31xn?$@=S7MIu(q2#{|tXa9s~g}?E}h;6jAQIIxzBQ@"
    "gZofL{QWyhQKkitkB9UNb$e1AmwC2lHEw%<*?L1BS~883GS`YaOS>L08w7d$#wvk0a$MEkhOR?w)!7j{v9f5kFei"
    "3NmdW^Ky5yxN<bT#FVsRf3v<nC|6P+V?HO?uWkl8z2!I4n2&%v?vqpiiJG(08Eoqpfm@B(TyY)dsZ&%9c1kpZ`a2"
    "jid8=O5S2h|zMy)E!uxXB^?<E}+2pU>9A>OT#?LY-WQZ^y%Y!kACtLB=ygrNMLpKre*7!{2*^%&fxwZrlk?dqM#E"
    "=za2BGd=1hjROw2<p}1DVEGIilU}O{N?hO&9>+rF2<eABgWLhs4Qe5cV>ggtz@*l4JY`jV2c%l-RtKb9=NyomIi="
    "0j9y=GtB#b~9dXkwFOgB2QtYUdgUywrh>*(a*<zFCWBx&kIie>OLQ83eEb{j!tgJIxxWkli#0vxO8no-AwZf^9AU"
    "R0n<Fo@sU#dA^>Khk-l8_DvKudx{bkL(&H@D9A!Gl{KXYf=JilHM$9dKt40re!ZIKTe@T0Zd?wh8Io!y^=v<!<hT"
    "XiWx79yV;_c22A&Jihnt30R5#^|3p1%Z1tq3i=re<%`%dA2=NZGw~g7+vihiH_1zYkNs)Jgez2|z=y?Qd!Cxy2@V"
    ">zn2*#~i;rOc1-n~y+f`RY#*EI8G7ehcH&2EIojcId3_{G#M+u$4inifNYnAg1?zIB@rEnT=!p_M(fUxYpm^xq7@"
    "`(w8<8om2qBmMU^T=kxk`I-{p-rh7BtdV<yS17lMf9~bG4P&&}whuLL+o?6C+D7yjsB9CsCpe%WYhV5u(J{bJ>5|"
    "k5--=|q(KVn#y3MkNfI&;{@D@hok!r*5-?jQ2zum=(ew69$_PJlx3$XKwb@I+F<~STCb-r_pFEY>mvs!k%qb~<8$"
    "ECkmQ|GZnq{TXchuMh#Y@i@uL+a_f04?sYX}j>rkb@2B<(wpGpetfRa<zz;hIw!;wp`s(n;p3&<pJwPXQccebavW"
    "t_K-Yhbn4;T$fyCg0BRu=a>#=5-Z!40YCVSjsKalmpj)pBoq_L~c9pwyvvaxW3A`Mwip-AnAWFFq#bKvadz5k%C`"
    "w8v*@be{BbjE@ty{<_P(`-Y$7%z_#T~Ro*+5I}Vz#9K?Lo3MmZ!onItj5UVo*z1ts6E*KvNN|hdG8Dh3A%{YHPl>"
    "lF<%a&l_x4p|X>Jb|%s;N9DUFPVnva;dfwnyDJ_$kmDEnki|7fp-0v^INIo$rB$P!dHz*6%}%BmmvQ-P2Y2kYoZ{"
    "2o&Kll5AteZBv$^Sf7wPOsB0?SA#$ncrbuq(orgGkCQ1?>>Mk%i}8*!0a+WNKA?ypC>r#VA{QklT5iES9I55~G3i"
    "<E!U^JMLT*U8XlMK!R9Yv?@clbisFq<j=p0nz!?gQrf*n+0b7Df9Kbgx2M*!lJklG<rxHf*xL!K6SM*dsEif0GwB"
    "$<{nqGS!+5SF7+T@C2(ZCkKA5Hgv{^EPgor52SsWMF5@u-O8`^hl)q{a<qz%Tx$eie`d%QHAa$qi@rWTI*Mj!}?K"
    "qL~Lcd4BoYbS`O|9Fpqvcw9GVF(Md)9V8JbUG`&+d8ull}3<{84`Rtvo|*%-7`}$MYtCwDw^?t0RZ>5vKjQ;a*@f"
    "*b8odX1S5M)3YYCJ^q#)>>5SgcY<q&w)EVY{@`J+JHRZio1q*{tfLU;sPd{;H-!1c&6aKMrE2LLBwtcg&Z5o3%#}"
    "k2dNLUHhS1xPH`2<ewuD>sX@gtCGw0|+uJofdDVjhE@Q%ZOg{f~e&$9;wE4cR;ZDT)rQy2exP_h>ToM2{`=f>jnF"
    "86L0bSTuZ-#y`2VRmpz2u8;zqA3ltXV1Fk^NZ0*_Tyh*mgdVBqtpHD@Zgt&v+TP#S4?e5B2CP!V<$exzsBB{kHxH"
    "-f~twiF(oz8%VJ^^XeVaL6smQ=0;0UxFs}WeZN6Dxp67+l`jdN(U)oIN-)&Fr-u&rdJHu+{-)L7LTfAO>?enKMM&"
    "uszNQ-46dj8+&%>55LJ?*Ml@)VhWlba*OQz?Z+ADnE2?qXDo=A~0El$d-4tv5jYEXx(@&7w|IZZ<YPcfa<tQ<PGo"
    "HUqSi?o;k2q?`7->HunnB&NWuq}X<lrjp08KopZD0<eOFonxB?>>9$L<__Qm-T~DGZh(I6d&-ZB6_AUW|GOtNdn{"
    ">q0*ajd#-btcZyyNMz2qL#QL`hny=G=Vs7tVd7Q_1ROPeJAl5AvsV02yAF6`_fY%It@4w7GEQv}iSz*|{IN2GsMV"
    ")h%>Y(42hhG-*nW#62(O|piZ)t4YEH#)$y)G7^xd8Ow%Yhjj+=Pz;YSmkpHs^Xvsjj;M)i&NZ~C*`_;t{mKXfJ=6"
    "L<p;IM9#sn20|C`?hKhe6&DJZY!edN_<7$Cw=?aT|Cv6;P(n6!k9>rRuy5z*WlwsQa(&CtYsyL>#x`j5|Ub2-8ur"
    "Q%I-vAqP_-l0=A(U&#0dLC1LmB0FjFA>h&J2jfrbh>bYV<8o4d(s{alLwyL2V}Ky4Bq9x|MVWr_u31{T6)Y#lh*>"
    "!O{L%&?B8n9vBikC7vHDf&(7ITld+W@gSKZPQt&}drVUsLxc!P)a^5Gvp3uhJN<@CLNL^f2@iUKB5mGn2tpfC*OM"
    "AIhg3s_=74*sHho0hz)7rCH3L<ysU4NyPKsNaqls73SX2?HhU0;x{UMfzb5SBe1Q{dJ>?Ctn5gynU(X9SukHLLZ`"
    "Vb{5p@4;Th9=C|Bx1@<&9AAaw@xaB{ubEM3~Ifg8#wj0xI^z6R**plNl4*>l59X!>}O}U;9D`1fyN@bK9Z{Z$y1F"
    "jLLdT8Z%KADF9rSteH}J(`sBeu#|p$=Cq{ea+7?#aqMWRBbo6-{ufXywi`QlL)5Qga;w=gX)MqZv7hLZw{nHA5Ai"
    "v5Fy5(A~Q)s8shO%C$@+oE`hA9|YvbcicdB6-W24?W4hNW1wQQ7|e5D`8A=L2vApr%VucnUY*0IHc`dySHYrO|kU"
    "n%(NWG80<e5ihwwP(%U`L7Ao6Jc9HqZxE&kTU0Hw`R3*vTgNM15s%6mE8IEhk;{~DUS<azkzhHqg>-ULQQi-HI|R"
    "+Cv4b`i5H6^(riC0bs(Wz`wf{zsE&Y9bPTlnC=ih(({Xh4<{g<BkZ~fPq8MVhN*8^k*TDhJXtS1ZCznT7~j2C!Vi"
    "r@MSnOJVt@xy}MZG2V;cc@t{Jh6ep2F_N#eUJq5;UBemAg2Q=e1+BG>a0Tr6W_X;H$1^Gxp|4%Cc}p&DDm>{AHP0"
    "2>wMGA_D-`4so+EHIh{O2YmT!LhYzT^My?o2yjXWQKA&gL{x82c_gS|Qbh-mgI$_BBzUBR}iQfHD8cWh-<vpq9>y"
    "BL;vy&INjsI`<Z9;F-y8%#q9hgZhg7sxu!iHt~4_BVv7)E0iCNS3o1|v5X%E`g6#{R;6Is18(9gbf7G&=DyFbH*|"
    "{@^tSG-Jxl+<+mZg@&(4KoK;<TeP8Cw3HxlF6(DfZ+8mt+&_qg2vyshI?iT@Y(r@AvZKCbVL)R>z>bc$F8KPMDtG"
    "WrJw<GctXMS=YE{2Z=%R>J4i^RWleB&4*%RYX7wW}k1)c(>b79-#Ix2gCBJ@uvtY@2>d2NA29K7|vogN>dpj_W#7"
    "bT;In3=LON((VUb$}Lkpuxjr91I+ej!q6xTc*`oz)e{5DrRsu#EUiFhy)5$UpT{Jup^SomPcBiIvp{9K-tXcN|dd"
    "nHc1gC2Pi(wfr!m~&P&GHGU{A0r#{i^#bVYj(Z0aF7bUK6dk7>{X{ga6;R2%~5O-T$d+ecmT-E(0mfcH(EKDG@Cc"
    "tNZvgWsE9q_t!pWQb_^Y)a{8g(s4NH6GJ=ON)#03G)=%XP|D>;jKLNLw&IqdR#-{e|ZnBkYP6Bk$om#M6V8YtoVB"
    "-Zm4gX`QPzK~a1{sE94#4ey?uCAok-{FJ4O<D<1SXKcW#&Y1aqoee8{tB38GT5~>S-eSwszS=5dA>n8Lk@vY;ZN9"
    ")f{TcoG?EU@U#iF<M^x0vx&#pCKsT@V=hQyBv7ryEJ-`H#ec9Bc}R}g^oamhV=oY-)&o-be1m(?e)9V7J$dgwaaR"
    "0VwLpxTcr#SDL{%)CGh)-J>{$k5LW4>-PjgX*SpFIr6`85tvwjjh#F;}ba|B#L-%?4ZURwwR7Vf|HRDJ+kVc11C6"
    "c#RisURXiAa`E0)rRx&)?<tZjSTy3iQn!+1AJIv8TPoZJV1e?!mN7;L%<VsGm*C=gi?t)&D{p{a1(<}5#Y%Z?3sF"
    "xecIfplZURhI~I`^Ijz2^Z+?m2q(Gcz!HsHAClwN@Qm39dWG!%jHvn`MheqJm9Q;^)5V*nRLT0+$<mHTnsl8Dihk"
    "v00htR8UyspGl~GbA0@}dS=|vbFPVmbmYFK%RvGeMBM=^mQmX|hQ73*Nup1k6E<AMtz{xCU;vmEDLv)d5x^Op^!v"
    "{zW&y5n;RuShIIoS`3LvO}nhP`?gEwL`P1SI`tS=zs6+j~K3QsG*D_elSYLNuV1^v`K8!^vd!$F02N-(nWI>7BEp"
    "Fj$694W|k%(w&%x|rYH64WO`%2H$7epqc~-gby+q2vEpRG9x_W}oY=h>dMn3b?ah8SBkFs?&Wu_LkItqa2lU2k->"
    ";`S*Bog=KRFQr6bk2ynN&w55u$p9t?1q_U<oI(>O-52ANZ?5$cH0W)cV*#eD&7PL9vH7RF--BpA-U)JG-8<Dn>wx"
    "@#dT!9GKo8Psebt-!ke)r5xZNjm!X*C>YRrb?Zixy<*f$B4G5@Y&-3}D03@RER>7?82A!8ja9N4WNKbpky`5Hr-5"
    "&G&~X)Tu;k&n}DO#rR@cH>Fp6{{`oX78!IRMl5N!0Y1|cTS4r8BZc(A3KZRNd=8Og3x(W<{k98I38SJmBy+MF=qG"
    "dTSvDPm-BKh4)s*Ve2RM-I53??3C$C4@$-z%QpGE1LZg7V?NKX9%vq)j(t!WJ2HHbx#JgU~t-KRkewBJwMcuPrtA"
    "+d&`917m`vjcOoAuJ2RXQ1cyT_4odsUdR86-{{@a~9Vg+O7iG{lN|QeP847N!C31W<>v1m;YlwI~nyx|9iCm`fPN"
    "{clhuTmM{5@&)gXQq#lWC+tl}RrUuO^QbbOy0Bl+#zo4vEfQ%T`vR&rYB>rzT35t6A6M-671iGFY4P#Z2F3I6;Li"
    "hFNpn8)Sy`w{ZE(Te{Qwh7!^@!|rT`seK6crc_GE&J6p!BcmzAfS8;Ayb|CmyqY#HKHTX-It<r2;{V>0-VveFd)m"
    "AVP%GgRk%$lgILA;rBgQTq<p-Zzc?VG3o`X)&y<uWldsclsh>$#fQ+xO}#QMZNp*dCIEoNb(@OK9KZ+aP<){ks6("
    "!^KpphO!$CS8fE)a~Vg~yQ;ragU{8C>2D4<ry2~|5`fdb25`zev1+2nqdH5cb2YKiOxxGp{d@RA`hYe;?->wh1sC"
    "z1ASVkuda+shLwC?&e<q(T1#BG3!3nC0v~cs&t!%GjWAFpcrq=x<d*YAXl-N}dyFo@k+=cUiAFg}~DoEe~E9DUV^"
    "%fMqd?;n1i%5ZrumzL?%&5&<{-BceMmshDDl{YRPxa=EF`dBCOxdqMLA_6nf?ly4eNIJtxxl=_r$bSlP^w~1O7i$"
    "~mMGek=}2F33d)-qN2@szt8!n;Bb_mWCPvg@*#t*`Icv|FnH5Ok&|UkF-WkkqSKwFvoPSW_t#tAnf++9$vUO28!S"
    "g!kiaQ=v;jJckl#;DW?xFE|2v;e#&lYxIq+cT;z|Lc4xBtxwx-^oP4t_sLHMLtu8v+H*TpH{~Gv{4CF&_0b>Tp47"
    "c>=LTjs9pqL@>BYu6M%@*V*TVKHNGBdpBOZzfVh2w;FbWIyAr)h-(8}VNWoIKID$@;0vmk|K>zO%`P#zpV=YD!MI"
    "w9|2IRgYSXjWHTTr0nkE(O(2F~$}1Q)9&wPD_G47beLhU)_21scXq*Y|S1)*i-ct>QI!f4jp1p4p0*ZNXO#rFs=J"
    "O1?AK{#`a@^Krpr%^lG<OB|fF}v*oC*6GORKcM6)3D17&ikIqgGetZq7(Af#5C+VA2qLOeyOi{9xadi*GAAUyw>#"
    "7VFG+I3vX4RwlW9#a0%LTZ-@H|PBd+5#!)I6`&-nyJGr2Ak4d>r?pk*rF$0Jb5L-x+}1<o)aoSMS1Ic~IaOCjJ|~"
    "V)g)-=EVvk@OnHd8~}`Y(5cZchG@%1s|H&~RTZ;K`?5mMxzSia2vatNeqkf^|NPn}6R@2k{k|om6SWp}RdRA6s$3"
    "ZCt=1mNkE|kL25|?-EYQK(X?FbPh>TZ0roJm4AMQ&WS$~vaJ!|B?0$pOUsg0qIy*6D+E6S+a*cCP6-Ix9WQgs~Mn"
    "j%$rVGh`|4B+@fNWef_`%TivJ$%lu<fLsb3U2fB(5`x3S$E(`+eWy@JIS{Fs!pRuh|aB!BL_wp=JTY&D<K8DS9~!"
    "#Jot5V0{ul@+zY&A3v;wzVh|!6rkZ~=hqStqQsB~aTOXWKYa%!cgFrl>M<E$?fX-ldmP*NGAg$&K4y>X~X0Jj&wG"
    "!ztBZ#%8HDSPVzHvHMS?UcbDU_%;Usv-@^d`^*s#4_)ixuTE_51w^Iv=J~GE1Og7u>uH;SSDDjsevCEF2O{S|Cj5"
    "we2xwb*M2z(L4s9;2h$%g=N$&lf13jO))LQYYk(&*ckR>dueiffqgKbOf9@hW`TXOR&`6F|6_K6>MQ6Gk7pDkmlk"
    "9vaqk41VH=;DW^msad@<v4@;k6h&5K&S)_mNZ$tEf%9!d%~JBdY&dz#}vPLM!H)7ujI=q+Y03Q6K!q$jw?@l27@&"
    "^)YDFM*&iTt^XPlG~w*fFxDO<eP`BT#f_y!oDgmeX($D+t%SCV+3HjS}eJ#4YyHb9i}~U#mv*G#pI;69h5v>Kpfe"
    "AAE&57@u}BX=bRA=u(cY5F1Wtlwwm>vicycLg*9(k>t&~hZNUT?c4WOO9g*}PVy%7?EW3=hVHFY~Rr-@2rNJOHRs"
    "yHtma{ug!Ds8jNYP0OpI`z?J=CF`-}z_?Y*p3dyU`<h$O@{dGx~sw###|++8S64=+2&!hTV4W6g=3|os}iXzkNu<"
    "tE8?J4bm)_9<Mg7N!PIe^KCI~?PLT0CaRN8?SioJ;xY-M-OLzVki!S6{`#Gw#Jt^2!!m0xJs}@k*dS>kE()B8@P>"
    "HIoPA4AZYzLLMa5vh?b78)ii@Sq#W?7s(NN$JB&hp7a#^LXtIIQw<Yu|zSM#RZDrr@%0O!!5cBuj~%zeGGf}({^Z"
    "G%SLh>{Uc9OEEd28>Oy7vgIQx}QOBFI=xk>A~sjh}~DJ8+t!lm^|zOn7O`agIBja+jG)d_DxQ)^D=<LL1cW2z;q|"
    "F<>;3Q94l0LcR>}FLKEaA!nU`nyq6L1asAW2-Zg<mLg^;Gs}Vi61BP?E7>16qDKhEV#n#%zFbTZV6k7T!E$0+2k9"
    "O$n&d`lw%-m2QlSF+1?)!swVsItj!ooqNnNkuYD&cJUD(+F+`_Ijinx2>1RpYvDwkEH_MdL>o2Y!7fFARP=$jTR1"
    "c3Z!`U4h5MJNa00Fn;HdXCh!%q#83D{(<fatJ2y^vab=wYvib9T2tuyj&<6=v|k-N+mrsBfY4(5-*baTU~3w@Ou)"
    "4w=Wn&<gCE-%Sm$d`Hn2NfYATKryAEO`e0|VA{we%cZ`eQJd(q|cgbQ(H0V*2ED)e4qMxePap#j?pi9{OQd&Df!2"
    "~m$$K|ZTd?lq}Zwd)4Xfze`SJ?>j_J~{I(NCg3E5n~BxyhazJ`eLzqP@_~jCXk7>O>+QRdS!P1qFNSGW51(vU{xe"
    "0$%t0_*^6>!Oerz}(WIl&I6acxMweS8Nc+OZHTiqGprvKc{w1<jfh#`1o^g&Rk*e>e0ySEZ@1{r?<_ze<E_6wGqk"
    ";lDGHZNbVY@2V*VQsS&15pl-N3^7VRjOU0|iw7IcyQxgsKx9wa1w!7CU5@cBzRDsg0cR-s@=kC9q1m>rE0II|YRz"
    "ud}0E-PL$X;GQN8r$$-(d0X$&dq|>5k;bF+!z~7oCCwW)S?V2(28f#QsNj+(_>nc}?p!DmXJkC9cUBuoh2s!B=@a"
    "CEsst1?<OmI8dNMD`90V7}d9C15uIWjSYRhD>DIcZ$`PL)H8~tX`z(mpmS|@b1&RK7#QN0n1EzP7KTti2CS`*Jje"
    "{K`PY^y$Wen;sigA9;T-eR^gqRj5i@7@En#OM26fy-p5eqFAo<z=y%tvl}UxaJA(yLtBZeYa0wF04XCnUYiN?!Zv"
    "c40E9&pm)xz31)c#1|vuq;BR)yfU4%$*4y`K$BDc5@k-0l!cP?rGLN5qmTBBS+a3oEZWi<wmtt!q7_}mY-rk$}S9"
    "Q=ujl{X(xs5~Y<gzoX1z||&pJpgaQJC5&2Ik<ZTWN?vn&fxylUqf@enA9E?~&dVTQ$nnmu*Y>60@FEp1r?01z7QB"
    "mN#N0K!47vuDAHf9=uVKmb5Vz+G>XQNc>>rRg$5UPD@H`+0#bzDA~$T6^Bxpou_j>#ENoD2Zm5Xi$ogmRk`Z9c$X"
    "-L5Cof>9lfC7PmnsrZYZ<>1J;H)Mnk5mh~WH2;2ttZyqGdpLxZ~D6!7uztg0Bv&76bgIV9T8#b5EU!O0C8X12r0*"
    "?M}UR796T?2`Er=rL3Zz<3)F!LW)EI>y+U(U$%3sI%nxu{2VOcyPC6gd|QA)gLA4A)p0XG>;UtUM-Zsb9rk!FnPU"
    "E{HLUuCr;_N;GLMxRm{q5VE{_=5+^Zfm_#XBk>%VH@uxIL0VHAw?3Y~SsNfkE1?&nOkeK=Zb)_Q}WcuYLsj{M-Sg"
    "xlOiRJ`kfLryp_uYGIpLojDm-JMXdVp>cUT3IE%sLNmHfdkz{BEe+IIZ0zy+vpmueo=l!ajRg=SFHcuQKKBvg-8j"
    "U7OE8t{>#_Np>=Nb-1@bqE=RJCp-(6B&6SMs-ZEXzD?CgMeU!qv;q8rK1jaG-q`2pNz}>6HWbp7lR<}D=brM2<-W"
    "}!3|=o|Pp(Df@Bl+W85m&qNCY|LmeTN$2K>UP>Ae6X<T~H;wvLNRb^OZ}maaB4=`i%y4dn4L$cBquNR6pfi>b0xs"
    "@5E@7YZQ^^PmU%OoJ!VFH2KHZK+E$m#gm4v(hU`BS=Un*~UNOrnR;=vAHs?ke2ry${)e+vH(LH#R*8$zJ_g1r{64"
    "(2$>ga=>7~_<N{+HQdRggL9p8!$bKOcVvtSjvxt8p3*tX_<v`w%wFNCQc;CI8JAC$G_uZJ(=|@fJ^i85}nv?Jyc~"
    "|rE#V5hE<6R2+-TY##JtK5A{-G_yo)u;%?GX4LGKz;}U?bPfFpK!C0NKWS=I$th8dn~kyahBc(ENfqJP_z0;irjo"
    "yvIDDuN|;MwSsQ+_a1gcYj-qUan?5U8W$QGkD%_g+$D0<qXocG#qFb`^T)Qd4uTAuIR>CX08c4aSg!>&BXiJSxh}"
    "}Gt~n_z+=jh$?5Pg2{vGocJ`6{5hl^k*w=5;mqFY89?U74}89<L-EU#cSUYT|ZjaNv)6}94Hx?Txemdb<tbSvf~%"
    "YHT5bJKeGiqgFl@Z6Q~-1D~6Uh%O^pXGtjZ;t8SjnH3zV|_a<?O?`vq+Iq%kL2ze1K)hoqK|^%p|0mfNvxrFh!Yr"
    "Uzr`BLfmhjsUw7SPV6|qpxP1G1!KDd!0=H%@$9kZM`1f4WfH5eCkKa`I!S>+qv}4|c@BKAT*)D0(9js-G(bEV<kv"
    "F3{c(Ve?phr%NKsa7)&S%wyd0KJ9MT-n&?U3yuwm2(l#?B*;93qg|Liy-jUFZ|sP?&kJQ1~<q+(r?~Nxd<K<HxEt"
    "#?ta`@~rk?VHX(6K?pU4q#|(Ji)nF(X&U7f#zOch2wS{dRUbXz;_Gs@wE1+^yhkrZO$P7*+~)vFeQQBOTw!2>3y6"
    "GZX7#QW<cr@3a+8_&Rk51RphGhvxj-7HOnFsZT>-90)z-MFm4im=JIcH9y8N`>x!urCcqk9Y88HS7h7R9C0+M6+_"
    "qO9HGe7rvU9D&3z0skALHdA8J)Z)IedbcBKRf=(e*=EwJz!7<upFsAJ!a9KP{lny!dzo`$&-3@@!UEv$LPR(zFe3"
    "?ac9WU@PsFFP#}6I7BT@3!9;pD1$07T&j^)9TX?p2bM(cJ$Cn%ExHBFDr4ah5QDrF6#$FXnqF=67P;|$4u!hudEp"
    "kD7ZP4_}cU|V$3A!W%+610|{Q7MF_?OZ6>}2og^x*8^_~?{l-u|6{wy(dTAULhH=N^4^a{TJ}bo647eGc5FX9vHI"
    "0$=L>-qHT(@No1(JW+{T20D|-`~BC^$-&FN931_WH(c=J;qm^TM=zRwez|vW*!-)V8()x|qMbtXEX7}(H~x+b^Qb"
    "xcFPNMmU-Gc+$WxPm;V~P^*eD5l5vn3ukx!aVETqnhlf9Q`W=8gllhNLbzc|8F5A^QC%k+JFr>6%$9l?C<SF$8q2"
    "a|ifdAvS40(yGHaD1!(R=*yM-aKkBywiJAp(1zD{?B_*Y&$*~{rl_D>DeyhMz^FE)9&-h@!@E^_xkMT<CBB4zwkB"
    "_3z6m<YIorWbNUaC_77g|9m08dVq}!zk0N5oE)$3@`1=ZQ%*N#>OpE~4&ACfu>Nx5zduFaDIOK9x@O7lIu@yRBms"
    "cy)v<YSsU8C%Q3kExWH9FZlJEq}4?9)A{4U8=augqmGFf=?caM@tM(^q>ZC(HuWfDDG%^iIe*4*0F&=V0$-=eW0f"
    "@@`JbU+8iMXY+vhJ1Oi*^$(D)NO9qcWPgv}!Z4n9uf;}}s+b|*Vghi(XSn-5gA(f-M6CHt&q@>$-(KGteSOagRik"
    "MkF;D|A@|<Hp{EpH=U?Eo2G7L~`Q>nh(1bz6Rq~f_zHpWm)X{7oP(0dk(x+H{hxaam7KEiaM?bvKHEg3YJUd*3F*"
    "X>UjPvpqvAP~pqRhH%fQ~!JI1jH|RKLi_jm?eZZrE1Y*a%z0kwOcOFkxl239<k5j4dnE3$daU_gU|fzd(gx`%L5y"
    "CUdBO1iaa?+kGnOxa}Uuyi6_{0V+>0TQRA|H$ksO`5C+@+8umN`Q(~r+xeT#!XIN4}e*SQ~m69oM9Mb?eIPCsAA?"
    "fyBz9Y{ie~9)x5W2t9ej{}J7w$DS2H=h6(s~A(rv{?T#Qqag!l;|!*WO@Kiy;7GOPX@Z^F={n(eXF608i{69z1Nj"
    "x1u?=Vcz;2+C!3TtP0s_3n_1ei;p0=BZia(!;})ATk1e)Cyy3r3ijmv29ksvRE4m3@){^pOtfWZ6X<&cJ7Tx|xo!"
    "s7pPp~q^;izz`$@Gd<e{7cE}j&TfFZdoR;$!ex2M{9x^Id*qhZVm5_S-pUWt9BY!g0{pQHZ7M!Q8NF8qNt%lk(hU"
    "*$lrunQO3&wQYhiLcC_;e2w`md*(>Un+~AWAeR1AO^?|D2ttcTzp6sD1PbbwZV}&keYiXoLIeqddcj3w!j?tD7mf"
    "8)@Qw<em)*o^J+aF`*y#m9jo%n=nkv9ezw0^)r%D+tYT1sT}j2XSYo3A>;wvDX@6E2Z2<{*-&p9MLdU?DnEZC${c"
    "{B0p?eODn4Hb3D^M6^>RSi6plR|DH0j-Mh8MH<!iU{&<%Q&X@zc&X19oar0f8(HE5JNZ8t2&juy?R;zu$R8$b&<T"
    "l?OWI&u)Ns^H(RwzaG38ot)~LfYhQZo5)Z+j|4vEa8#QF@$L!j16<_qfxZM57_ZPPtuMe;#4Eh@RF^Dfv*M<JOPZ"
    "DnFIJX@Q8UJ}P95kP;)GV}-I+&OjeVHeWkZbxM)33$x-1S5?#kg_&#C8}9qKbYqS@jQ&Hfz0o<Qi?<Bd&G7(NreN"
    "6%U|uP#2o4-a8$Mr~cgPHfjYg*@_%C?LmDyH9yWn~6#aZQ2%XgvoR}tHVCEn$78L?Ac`$u!4TJN4%{z(+4;kz+bA"
    "b%^kxtg3^7MzmvbkNR510)b>xnRpUSF>&?2FxsRK3GdV1AphrQ|UF5!{Wt;Y^oO(a-mEhu^Rp)ZmEAx?mS>G)&|D"
    "Bzu^SeAVl)PSCEM^bo)bPLmWhc-vxBh22$2Bs0MN+~HeO`|IGB_rOLAKs3XC*O}e!tI7@$ypz6f-rat}=A>UwBIB"
    "r+Pg#Y7(BtY?!$qg8AOeF#aB=O<&V6=2sexe*a&f(w3|BoeMA6xI!X~z?28xeu@s@bnf51o7>OkiTCfG^uhF4b-K"
    "H*5R%`1u`JH3nK8p`O;Pc&sAgEalASLW+>VH$jpDZjrOs$*xN>m0`SB%y)9YU^XT?==LAYGC#Nyi2{iJm?|1#d;?"
    "s09l{e1B$b%nWoa{<Q)(li05IC)X?{jQZ1dbLuQY#?{6{a%Y*o4U-b{JAlJQa{;3Po`;)?1{R~4U+IR1;^bK-?XS"
    "@D<cZsN;wBc`J{C66SC<B!Rrk8+;a?U31`QBLrx`pxNH)6*b;;U&-E{AqX)VoCWv^Hop#+`+`Rj8$qrN?mo`#EJX"
    "R}Ji1;v9VZD|ZxnH!Qr$)*O6Ifk+RJD)HLT{&fE6e#ut)X#^B+7#h*I$k=Q3<#k!etNROrwu3Ny_F?@XJ%HZvAkm"
    "7UhL3Zq=nX#f+@*)FQ5-+-(llb>h%dOGSHQigUFuy*m7Eo`%MES|r(;Sn=zK+>|sCt1F5mgJe+R-{B#H2sXVkC20"
    "tUS53izg*Z`^6cfN$^gZXBYBn8LmW-rL7qf?$W3W=|kzF|GvRKunN+roLX6<xhk4$*!?I5I4&lZah<IS=Ytcb{1x"
    "ye)TNEhm3K7qPX`6aH48U<Phgf_dHtboI@VNLx}S7pC0i<KF2W#oA0Pyf$9ynXlXw)ejK+5AH7_c|~}_rCKd`3d3"
    "^@QALIthob??h*1dj?E#gRtri|`UXg$b`$Z>=0I!{_}~6wd6%&RcsZ*sD##R3PGGB^ek@klhPrko`etjWzRYO>(t"
    "Vt0aZ&eqT5sWt_n0|=jS#Cj{)-j0@`bxvSwm(?fI)|QmuKG?&QyD3(4^^s=ySL8oh%DyKziuyjfuorO>$MjXrlM^"
    "p576Iga6H)QaOa%bJ7frJ>s6_vMp^As<Dr}2Vr}Kq0-9x;0p8Yowqzwe#!TFqL0K7nOpb^E%TV<G&17g3LwGc_ti"
    "~l?izmgZCIj!CQJCRt}rKb40euBN3a&TeyuYaar9{kP^(=4Tq9Nb>nuAItKnZ=-=y;gQ(ZZNVcjooeC};QBf&T7D"
    "$>*C68hx-Y-Xc=HlO(CZXABZqEgv)vuWl8q*!zAxP6-Q@&(`UzgX-&C)UMZv8}DY($=(mpK!VoK*6%^I|E`jF9Vo"
    "1Jd=B2c2K5>KtQoKK=}*z4-Y7$1({4MsNi--i#uT&RqBJ0!I=PrT_odLkMb-;XK7ttffz@O4${sxbGsNRuR?Vxd("
    "}@UbObzvP-}Eepanl-brJI&@Ky@ZUI5SMmOC=Aj~8P6srq4PN2=~Z?>@OI&P#`^IIWC4LkN~8KTb3(069n>#6mmL"
    "sej+%l4h1fbWp3#=H7}O1xYUs<2ZYuNw8mF^n>jI9=lGs-p(}1iNf6%y1EK-klFZP!zVeKy@&_1pZ&4uZIUC<Pkc"
    "Llh~~it4hxW`iTGxan~?u~FgO`R=^^Pi8Hhglw_5{;(n4{Vpj;ssG%&AX5mxt`FYzCa`gn5Y_zTY)|8APKHOn3}b"
    "9-0&>{FM?MD9JFJ-$$U->=?B#Y$AJJnP(Ix<^jP1cq}X${rmA-K>c&vrLDsrcvKpc9nYiz<+d&{vk-Np+wNsE!{`"
    "(E)J#<zJfi;cGSB8sh{<IigN)HN_H`$Jb*$RtvA#|!szs?X%C^-Hgm?k0!%8Sk=GcjEDgI~R~PlJENwR_;ilfivq"
    "TLbV+dTiW3YlcJD5V&zf~=lr0AM^`uMri^<=hO6YE*dj`fvNw!h7wFr_~HU3X*{U(>@2{_XVm=tT+ZJ8D$u+#7V_"
    "W)WtwtRWw50ncXm`P~yxR<Q(x=Mt7(uZr8D{1*IV#}AcVf3d0>H4{)3`MEioS>506r9Kw34e~7h?km0&cFdrz%{("
    "|)pRMnfP)QBM@BaXI@DD~r6}Q%GNxn3`Ce?q)3)xlE)7~j`OmQ18qQbFXh|_^CC<nWrX{9)$l1Q*#q0LLy>p_(Qc"
    "{=_EY2@jcGSf$s!tOgu_{$axz}tr0%W%>jgWMVv@Xq{692~ir%4i_x;(GHqQ6BnmL1ut?puJdrpjCOTP*6xfHuPR"
    "4er)h!c|KmqaVmjXGZvzIS?~vFk-m3y_VdZ{tAqXV-m8Q0pGSZ34x#t^>$9K7XUBgY9R-){AMU+=F&gh5oBtdquI"
    "fJ2wdNIcife+}<T(FCQ<bNG0`<5ZYlV<u-t~~l(cHk<nI#g(OYl$-=e-_Z>=n$yVcP9(xYD~-VVHKjdE{!;ZrF^{"
    "Tn+NI(*fND^&I@8QV58E*Ji%!TT2}vGQaKSxNNBwl-kGkfMtzKX@iAM;EBj2hw?|_njlm&q)8H+!DLCJ7d`V^TlQ"
    "Jq=-#7$0#g#_z5gs%i#GH{=~y!K>44o8qAv97rFmg2=;?$z4<gsx?)IyXLReL+Nd(Y8e{_I)%dl3sQXW?OfSfN4>"
    "J3Y?!bn;8klja;>r=xyohMznyaOFL8c6ufc%6s*@7r(RGk_cJ1HVzM4p@V8s1{TaZSs1!NL?RciLl){Y+5EVVtb%"
    "M6tG-q8;F(LqZ&{+ckZ_E=zwY3IJ{W6-FRZY{n<DIOrcj_T$hk)6AlyDgu}#XZKMy2gV-adfV?^-HjbtMC~X7o-V"
    "y{yE#7?y<~F_%{~o_g8`tQ5^<a$hf*SkO_Gd={mcZljyp#IQJZT<>N-7z6LnrMU(|<ULPDDw2*3SXC5HqbLay}A<"
    "F>Ip%NexOS{)Em=3CBMaVS%Ua8L@W1skG7^t0vT5eie14YlDEG!q^PV`c6?NCM%ZusnA9>jo!Yxbj|A}N~XsyYP_"
    "(@pC&x*Q3w`1dFd8M3k)rGa7pk1c$RTJ8rw^VW1zeNXiv(Wmu1E25khD=l{3$dzQ6I3KFqNYYyyP(WBA8VgG;!r8"
    "YN3YcEtX6EkqLcTTYmJ_&yr%q}@#zP!PPO6g<C8^7uUD@9(?G=QVlil{a?VvqkddZG*lK_Q>K@yIXFS>sUkad3h("
    "E$`Jk&Zs_fFC<EJt3}w*V+?onI6LMRl;ni)DRga4CYx9Aux|Da!-#WUISxZK(<DjD3CWj<de!M8<^`g7^v%6pyZ;"
    "J0U@v}(qRrG!CC+`k)Pe&KCYg=76hqE_>a-Xr>5>Ny77Wh!!bq5)GnNSXy&pCf2{NJlmyNQ(t3@^f*mhOF|d#mf%"
    "x6bJ;k;<(R-qr77<eO>J$__ayGV@e@o2cU@-tZ)BhEKrU)IGtHZ_Z`6@Oc*O+xo>JyCc}&sve_h79*mFi`6C*#eA"
    "Eo=G_w>;^BTobg-q}hvIFEJ%&WX=C3H;Pb)T*Kk|m98`{6ZD=gIx6>a+z;eN>Jv|HONkI>n#%3(;!53M@ERBzvNh"
    "!qvTr_V5vMa|v*{%|Wk&aJ4cGa|1d)o`Spiu$*&;8T18{=fgNyoi4fZa=tYm%HH;=~f8`AV=&UG>p@o9_mwcQ_j6"
    "7Kf=TKcJTfCKfH{;i<c314Y}`7?O@<#<n83k^yWqv{`c_WCB{tn<LtXTcUSBhVlSKb(vBpJ7V|ATujJ%V0)Snx<Z"
    "$D}Z1g=_=Od{gEw|f5lE?uV-Wse!<Bg%q@4*9XJ2v5wNt{i8ALJ~t)IkcCJS$czqJxxdj?u8az;P0%KlWsY-pIe1"
    ")3Du=9z^<cWBQcU$d%!(%0C|0Jwo=0w<vJeHT&i`F%dJPX_LwC2FZ!oJEn&5sQ7=m1iuo&=<#!m*}<H(ZTKb<YPr"
    "Qam+;mF^(u*T?V6wZFt<_ypFg#Qbo|mA$mKS_#rE;oZA<*d@C{THc?9cTExpb!$*qI-aX|9_r|lpgXSP;4Dc57<>"
    "h}d&@`R@DV8)*yh{V+)x8Hu(ZyCEFqoO%<;dcIk;R|c2K00`@JHJi)g17sFiBwrhPyJ-I_SBEPv>RBtdqJQzOm&O"
    "RK_j<&j?+9R^4eOhA=e3^Jx*hN|0Qdl@a=|o{oxJ%d+-LYuL0LC<qWpJ{NW46=o_SantcI_bU7=jdr!vSk+J&j$$"
    "i>Q?A*!W2-;_y(xT1J;q&ELxpU#(4`MIv|EVrsN3i}r+wXrL2QPwn_#R#YzS+ZS9?#J`Lg=7s&RQH{l1Ph3se$B<"
    "qN{3pRsNAA`a5`YJtUpToy&+jNq;Ub*;n=FzCJlUKKVOy=>Cz%`9~h-A9<XA<Z*tzJkCT}TjLlC4ti&&ZZHJ4Rc>"
    "}Br_S-;MOG!%Gj~`YGMZ4g#Z5sMKhQ%FYFdwWRD7*uK=-cy-4X&@*EewZxtI7`qyt8xKg{_{rul_z<%?7H*v+92c"
    "uX=W(4;NS91njU`&oznj_x1%e}8y<{>cCPBmeJ@{J;O{`F|0PvF9lwgx;P;Iaau94hpMtjQFqsWzNWLT|^suJ)cz"
    "{N&&NH{22T2ocdS*<1yOO2ZF+#XFs2vy~-xfKYsW8;<{KrXCEXukS6cuhRrXHNoW+y0zs%|89y3zjg^`K;?@@PyD"
    "WYQy7t|?^YQTTmpnWCB|BYjrqv?nYdKum?Hg4Jd|-$=UewuSLO8gSNp@ad6dP*U0JC^E=V@@8IIv316X%!JmCOeh"
    "<fJU7wbho){o2D~0mTMzY0o_F^!V4&$>`vxqw(HPqocF2neO<<z0=Y7^~s@`#JkP+-+uQ`@&I%F^?JP=Jb(WGxA!"
    "*eZ5v6t=wHD&=Q$=Fl9m%Yo1w$WDi$-NGq&WF<Yebsz9d9K64n%;AxJA)um1b3x2pOJXi&7{o!!09d1juNB7sJuy"
    "Sl#XEuPY4N$U5LCr`h9N=rGN4%$b2b+|W<4)^z)toW9G=+Nzh!DRfe;}>sE4v*{&M0jb{-7vEHiQL#FH`$?GJIm5"
    "#wtSwPEz;sWGgp5%d97Z8bfjONA(<X8u7D!*B0kb42-PD{&UCs974V2V<V>f}arJ58BlZ*L=$nSB=z^D|7b%XO-y"
    "ZNRqv=2)z9sT3E*S1AM@NX_3j+;Tx7DYk1GhR+vG)`19Kx5{L|W-CLDYK&Fg5nL@K@XQa&znXKgS>2bh5H;1%ezO"
    "{IY*^c<_n{v;lYQlmPS`X+L<@M2KI;hvS}W!jp8*#eNMwy~y79=iN6iPwaG9Lbe`!kgK_aAI6vS!NyOMUv^*a?@f"
    "+Rc2C|MGxpPiTGU9J8pt1B^k{PYx8sxXtI6x5!&k3Q{I&aAx#r}aG&wf3q82c~om1Fo(ugC$mD;P1eHqV_Tlz+Tv"
    "*gs_h#J99CL-GsoVcQWH0b(e0_=6%4t9+MB5oghc<}OX!1AKgLmVGCD`b$Op(Vl@U?}2-;*lSgOU67;&LP%FF7XK"
    "3c9WZ9nI;j#Bv_#T@Yd%)-1792hdM9=o^h8J6zs0((#g8u5~^YH19uFQmQuxfvG`Lj@h&&5xQ5BwAK;`cF8%}}^@"
    "pZ)$YiAIlKF{Tb~i|V<}Ou8OUa77x|Bw|Iy4xid9uza`TH6uKC4?ZIAW^liuB#=x=Q4dlzUyIXYdu!3(&3Q5~81_"
    "@pI{ox`EMh=j#BzuPPTGvWdvZ4jVPzd5lAI9*Nr50jHU)my4c=f?kqd=MyOFE(OU0?DQB!fXvDKYUF`zs?K`P*-z"
    ">Rusc|a7y0zTV{2uIj0(((tsbNd!Bdvu58vj_I!_6Q2zY>n;P7;5tcG3Btsdz35^qlsvubPYP>)`jW)RWn)inuVc"
    "uMXIUwPQX8{=0s0r2^etFK5OCpNVL(-|rrBKpYOR>^q^VOs#2@0@X^h$p3_$BLH;>Y+F<={))4dDwn{4<!!7&41~"
    "ru8XA%uz<2C%5QQDkYsKEBe5On3Fx)-4R&)PiQmC_`%m-wGcXyRdzbuk@)7g6`|ND*_>VONgvdp8zjzbk=|9bXy|"
    "t(5ZOp-GC_bJGcGdF3Cv>Taj^O+F#B15@#j>}3=UP1>?f(Z(Gt<Z<?lUujYXYOR<0OcJFEh!T_G%)H%r9in)4o)S"
    "86P4~lRQ_r;9+vUD8UqJjV{j0`K>5s{0NA9J=XM%4$WWcXXj_ew-VZcJkPFJ`J$79?|0O(bV>FoocR2J7bU}+_D|"
    "T&162nJ*}4Od#cO@%dU)zv`q5Gz?5%C8DXPHf1IBefeee_8e91?>^UiU0GFb{1#x)zoJ2>Dg%uO0z0SF+J5RNzpe"
    "ZIcBt~yw?F3B)xb)9a{Ol_q9+EpLvzkx)QngX8EKoma&*T31>Y2eB#SaU}=M_d=i<TBb|C}(llJZwPwAf7QT-R!c"
    "=XV}trZ+G6kwR?=ep?dNTNlK-q$m}<6lOr@MO0I)oufd(88-PVXOqZk<X6$6c^@H=A;|U)TDwJ=Q`qiC%4e23A+U"
    "97K^q;WDm<IS^B;Phx=odIb!$n%Ml8#p))0?CCzItvS#OJVvq=-4amB$bHxnCzw-fe&(#0U`-!p$?gNh{5GicQPs"
    "5_6v|l-ZI54r}+~{vGPAyaRci<3#<%@Sw6Yy!Iygb0>!$&wN6hgWc_sDfMR>K}M^=j^oTL@VHx;*5R!dza}QRxSE"
    "Lm(eZnl^!$D%z2s4y(b2{}`DSOZ!>i-D83KiWw|dXoyl5?q+ABM?Dt&XtyS>NC7mIyZ9YpWiZo*TeXnk#bqp4FB3"
    "5j5w-|19N@N{bRE6;k+VCL2Pd5)k4s#_{iG?1NAttRDr)4ijikSPOfRm;Mit7as|HG1Nm1r2h&72`DHc97*c2M&c"
    "<@O>`(+U#@;ceI!MHM`YMmJ+&=Imkm}vsBhZ*Z2}L&_`=wcwUQ|W+V$!`!=E~p<i{mP$^a2R`mX2hgc3dOhOOEUG"
    "JVz)>XD_j1l&)<W(4Qe<^%Irz!fs$0gV)IdEOJ+tedH3h`*aWqeV_u9FL3f^Q}&rZnPbLpJKQ@Gcg&LjA&q*CDVE"
    "*rPY{Pg$p~l;b(vN5D;nLt@_C@*)YnO_~Yh`<v=!Ld<nC3R!LZJ!;ODZnx3*0nK*j6Otr3ddyU$k3S$IfL69#-VU"
    "y4GVj`EBWn~ZaD<5M%)0E(p5cY%(s$9aMUTD9Wj$69yDc?~{`@8>9rfjeoiP)2#7vlrDFLU|<pljwiy1MnD#$g6V"
    "Xv{i0#$1Cw*J=aV;XfNa_g<M`kuO)t7p4CpWP;E{$bVM4xjDpyxXGLg_Ajs75V&!AIDCCW;DlgaADWO#5uvGJF6&"
    "{>WzYL%4lZy5V?}I@7HS~N$I8`YsLSS7uUB{xtKiWC@o78tq&fspyEuej{d@nH-Q$*$DTy~oHl**O&6LPF>J|PTC"
    "dLg-`ivAKfqFS>>2KbAAYRU`%q@*Jch2p^n<&Hv8Tt}NK1_s7!Bcybi7WjWmeb@%BV-Ky;jAT+@$>zV;|t&w@-$g"
    "L&ym@CRc$^1->y82z3$YAb4;^YBa4Ke|o~B&>Qj(-Q-Wn6PJ$|zQ>Nd0$o0tz*{w$NYr~W!5d8`v|gsE7L>_j7D4"
    "0R@>Pf5xVu&G)#fSrG1t9J99*;j`7Alle}m1&J&%#0?;KCRukB@f*y(?IT&N}bV?2}f2Do*Pq-&P@C47>f>tXytP"
    "PzZ<X!{2_+8ovV_wcX%<NabHWz0I<YZ11XeV=Gm`GU_m!@7EZ?2gcA$w&e&#Ik8xR&fi=-11*%cip3qTdk#5yeDc"
    "5eU>ItGPDKiRWWEa9p7U3BvtvVWLIOlyDznJtYF+#uoF6_1EYv#Mu$SEP|Xs}_DGjOaHKqsXKIM8zvPq}d<$KS>`"
    "=JgY(u_fIj86Pqd_lT(rrYM*^(}z>G9<@-vg9PCRTz7%)l@6h^l_y^&*$3iSYY$c5mPo1zi+QC$o&U;G^vSUj_6N"
    "6;MAp=Xa{4ah~|!STkKOYK_@n;~QewDXBjQOVG9Hf<4NCmU@%E&you|)A)&YjkX`yfEO4aE?&`BY1X@Fg0{+w*($"
    "lpz*+L@N{-*OT3=_&54jYpPDxns=`@4jU@q4rb1qJc&U9YRs>kwY_4u2eZ@%mAeBb}(k9{1;Klb@G`gzr_GGZk(t"
    "3IC4_a1OHpMuLykKI@F8t*&Zsb-%D5mb_(NGD>AG=EJ@Nf*$A)zgRzP7ArBc5m&C2FVG%4XN%(ZDG%)rU5R?^~Gh"
    "9k{(?yvx_AnApDSw0ZsVL;krZ$7ZNX!?P6X`GiqSda>|PSep=WIxUJna>MC@0DTj^A^^z_(>?W5Xh)}M1mkx`pw!"
    "zm>ojX(5SiHIux-Xk{lX(dM7)g{#Zjdt)ZrSnSucw94l6&0KbSX;rY4HTHaYlL(wx2G*%5VGU+`URk?dB3IHA{=K"
    "$dMoDH^D<{O1_Xb?9&%V<K2_--V|@jF5@LJ)cy@|ukwcO2Nk<fX7nU}Wyvz)9$;t0zlxFv)4rQLuY~=^G$DR@_xP"
    "#SWzvFu!$JOeGG8y*PLQcf5aBjk(J<*^&FxZ=;3|h@p3QQ^%X>+=m~*r!Usf3G+d*Qz-b@6b+v`P+Q)Exs&1J^Yi"
    "|GQK2Wy55#iw-ls=UHWBF3^v^D7<7yt;X?6lP!@BtMsSs^_h)2oOI0lGHEcUM}+s+C(~d??DfTUdGQr22=XLiJu}"
    "h^x1%Qbj<>0l?;TTf*BJDM+sLMClc$N@^p&q25-gBnRENG^gN?)xM+lz-bei@zQevS&rFk`@1G`1oEA8dZdE1g7j"
    "z|k9ywE$hCetwVP#<Gvx5ZZeqMs7_S1`^<YUrF4c+Fnv#DEb3F%yF|HH1~9klEu{<4R9YK4RL6XeayvyT7zMu<E)"
    "BdznsWL@>bg9i#wmP?}ppp{rsx|&^*wnxJOr<MJ$(`9x}??Fd*;^%5rk1NcAcrM*tktHC{*8ZRm={?WZq;DZss1H"
    "o5<~gS?K@+3^$V>H6IG3N(s_<)-oN*1VL_|&J_^e{zwghmypOIY4SIx?WRyC1WKAV|KU(iFivuG>8to^Dd9mn;`^"
    "EHX4vUH9P=%RVL_#)5+H5ZCIYu@K$*c83wsD!{dW?E_)Ey%XhG5wRYRZIC_iTL-FNQ21RuZEYq;{jquO#`?OA1R@"
    "9Kt|9uXPmwdK4hDfzetzMJX5V+i2pv0K}S5B^GJ?08IT3T!I;*Hxi1Qo=X=u>0{!(byC+G9Tb5h<62}AYK;MZCWd"
    "7i!)T;YDnMUT7F3Q4=t)?VB;4d_nPHH)B&pVQ+k02IZk*J*+na~~hYi!m@Ck6Z?*Oy*mO<te+5{<q=7wYC(r0@ua"
    "H({>z?@T|)Np^ltM`ePVn4#C@-4+Mof1rmyV--jC4848p@0lUWcfEjibgXw<5gkjFebPz;ii6T2s|0o^OTxTgnGq"
    "k=Qgp%yHxE`+#;GkR+r{$cWCj{3@*jfUcFP~D_tFy*U(!kDPpXzoR2S31${RC!eHqzeu%x-=RqRS=y^wS<vdiV?Q"
    "DR<nyLArSD_pE*65s7M*z|YigdMH_n4I&0xE=A;)}i#zFv_y!>el&}bSa&4Gdh0Lqr{grF)ld$i)U(4E=+in0Z-&"
    "<HsO@BFV=-svL>p2b%;R@1CGdZV<S*$IH12VE1{tdCt7t;hB_ORx`y5+L+zRyIe%J72Otf^FQ>{+XsC%9s+!Wlz5"
    ")Ld9>n^Q@)tG_n2)4*3fQNe0qM2E;a=<>izX*UD#l20IunCr=<RZd<VkFsR#q#a831cRHQs)Ce$GWtg2gEV{h9RZ"
    "A%4`Q=DL&>@tpNyo@6;Iq)YSx4i!$LYa9PtT2vzUC3Sp3iZ<SJDW^Mi3uts*9=~Aa2WQf_`B{MkgecTPq8ll+W)B"
    "l<;|v4YbFQG-ddU!^spO_XllY)-#`+2~h^(RGK-yTSOO`Xx53!#Ne}6oFIeu}%jxIga^y+LcfY1gl1R*94EDo`6J"
    "W)opV)jj}2V;MvuI#gPmLX4?(~5JHAtdW-YaA|29>v*hGM!i**_@B5s+8uPSoL9DLJQQyTs%Yj^MmoPG8FHK*?%Z"
    "qp*#~obpvk+oxZ{RY(NHPAqtvS%x6KTNm;%G)#2=wBWl)9KOY^wvX)ep_`%xK2QSAzpCte1aQ`5QovGwzo*W*~e+"
    "MFbCsrK5hdapUwb6e4%lK%F%juW><K*DY%a_UF(cbtd`RQ+%T6vS(a8P$}o{?frvau7oWt69=`^@?{*@Z?UyoGNI"
    "F9ydjQ##H);HGK0-g%s?u}_=$m@ry-vKCD^Rq|bozimT}E!MUXqp(JPx4Es9$u{>V+1s$O(3CoPFILg&CNsUbSK?"
    "pc!q18{VjN<Gz_?X!Rmn>mxt~j#z90#8)fHjod(I}S-g3PAYV0k=bKI-GARY|Npn@x9o^T$3sSUN9Qy1MA2kfcA#"
    "p@#f`#Reo9R3rIn4SmuqDOpqAo>h{40IPVRK5Gs>Oya-ouG|}^^2WTZy@nQ5Z5a)0?j%YSw@fp9T~#CA$E@LPX_`"
    "qP<#0(o8SPLCT4Za;`EY-|BIEB>|vJ%dZ?%4VQqv@XDaUQT4?iLZWHV=m`*dkYp&$Ik6hwd$%f}Rny8K7qnd^6PJ"
    "Zy2p6)+8nrJRfme;GP^d?C@j8@7Sfm3G+kM+Q^o2lYlti-B?=TP-KI*M00mx;l`yMXt@DswN(Qbp>SfZJ3^5}5iF"
    "Fhvdp$>k!M`t~^dy$|(0r=HzaO=u}5a!Tp4sbvZ4%4io>^;FnwwSMJI*cZ%if&xqUl9?5lh!vd(cyA65#=f4VA)2"
    "Y|(gZ6r?b3pBvAwr0T3LAps|!A8u0XoVQq?qNj|_P(ME+mq*U%RfQVEPVFNhyiD4K_3Lv+o+k6@f#2JWsuTX5zW5"
    "*x9kNe3!dB^(X>i<L8bJ95sh*40Y2!341LF$Vdm{4rORgBPSq<ghCO#lw)A>c@KbU@v3~wv#)!f}vaH3_k{;)sFq"
    "KXHJf?0;1_ovu;36oSaq8uEd#g2x&gNn>$*N*Awfci5;yAwmpsMZ;skfc?=&lB&IhH*)lU6Vam?xvvl@8>OGN|$^"
    "OCd{@&OHDcH!lCc5>Fg3izVH-GFuAu2h~f9v8*kj25SMb^e4idhM@BA4ssOp^CpgGQ17j?&y-5b0el*W!N`fYVgf"
    "3@@gksstT<0|dZog*QXC(HLCHBRJV4%nU1R*0AeuPnb6E9(n(kNR~#!ns<JaX0V1c#i~V?0X~3Bo)=|lKhty(XD="
    "7|>^680br7&}Y3+Bo2Ht84DYs)a`lc!kzLr?DXs~`#13$!IHcdp;LTudRjJWk1FbYcJF+HQjDb{DVJ~y66i%%d(u"
    "L$K1a;_ty$eT?Fo>ad*4$2n1ZZq%4E^|xc2DcJzeGt@Z)!ODj+(#oM?OJzaz|d3*2-~+VH+k?jZ4l~R5r~uL#oEI"
    "gvVMzE^N0j1YA@VkDc!<(#Nw}pdvVW+t)O?Xd77X^&6$6d`21+7%&5+bkebpn{H?+Nj3%JjEH_1i>-CcS>MEP(Y_"
    "&H4NOCLTIfm>H1`e1rSK=+>*>Lg+I}EN$K9uQ2nRJ-hnfMN1aljnA;aG6AT&5o~lk;Qr9<bO38>D~M93D$&;vMnF"
    "0*`k*7+gHXT&)iVjb6O`9~8kDuFU=ye=KXVsvDTSFYJ~;IF*Yo#CK%!>0!)CMv=AWbm`&Na6*EohQ)>UbCts0LsA"
    "#)N-Hl<AFv8U=jJNo-3Y3-SBgg?ypKcDG$F($3(0-DA{`1#RKE~<Cw9ckFA6i$47liGrHV|0sv_*7ZbpqZ@s1kv8"
    "rA-e-Ld8p6f>&+3FO4c#V)8ZGKYHLzKe~HfQP$QLNk#x$#UUcg})*fx!{Pq)P2{|Fr{|A{i-EbvmR6NCKdTLI*8G"
    "pMjB=8)U`FZGA=|Lu>7(epHedoH3$g@CPybfgNJJL!{ecx`pDUd_w!8c$+cMhw-<;`F0g`B024*LVeEIWu{@Fi%r"
    "bCzFA+2T5~7i`o2Y7|&H5Ir-=jATTJ)xFXj^!yWyb|&s;(blX*^E#_xmjMo*ygoCo>Ep;5xA(TN+vfACD6rt&3WO"
    "!Ph^}l?|&0=6O;=`M*IXoaW6sn_7jD?^e%?{`n%mxLm1Y9q}JXnzLI-9*C$nNHYDNk%H(?>noTZy*W78Klt+$50@"
    "t)RIV9eymoWZ6GRxQ@ltnknppP&_-3%rO;VH&@4u@AMlMR5Lgw&y;i%AX1Y&soMV4}<>^!l)hd%rkSHv##O*9FMm"
    "&>G|agqt$WrE9io;p^~G8-br9ochzX$=YqddY_}pI0PVi#1}2>!lT+>~B~aK2@;hd<C6HOK(NI-&k)Ai+!tb7+_%"
    ")p3@hHuU@|#!yDa8ra!$reDPQQh;gSs@9w{Zo0|P42J%w#nq+e>))^8*)Ke><mI7&M_D&py1tLc!LQJ#3ehU2+&$"
    "<zJ1oA|kFHI<GG^%HP40MkpWgsma(YX)msPpxpb)8eGO;szGaC6-B8_$0rCFzal7LLLaLt)U0)X&8+GizbQj&KYE"
    "JGOp*Xs(~0`7g14CMVz@ws3AKpm#8gq*Cj1RwT*pP%aW%pjxSaTBy=kOQs9v8I~t(P#PMG=$0*16`F=R%O5F@^c0"
    "Au?5XWdY+t;<KZ9GVl++~Y=zcZ!iWX+j_@cQh8#1Kf_`D6v^)z81ozItGBy(h(FVMr0a|a{^lld?5SCD%*x1|hZT"
    "0ZSR`2vM}P$Jn#bJ=Z~;7H-W^K7w7TQk^@L*kd{?djiNAmL4Yspc(MahWJ~bo2&@3%$C?gqX-=m=xvBRw;F_qy(s"
    "EJ}_Hg=*RKy@fav4W;Hr=wALt|c=C*P`_=BhPL9SWNBd)7Ab#s<J%|lVcn>ElI($WEem1l#R4W`%dRV46wh1NQ<5"
    "CbfesERNX5y^F;4bKtCsU(LT;EPP5@59oiP@SoUW9Co4E%Nf;MgXB*g(E{?!ow10|=D3Opq06yI4OI)G(vIW>uGo"
    "vj<xrZUF(&&x_e&JvSyfBlP0&|F*7Hrma!MhAchwBI7ukyep9KIDO)R52(6bF|hou!4`FK<fhIIe|<oGEBwm&oWw"
    "@SXcAbletAQ7TYp`c3EP4+I$s@rcS%H-p^&Z13KT@lP%k!b!af__70x!1wFT^?jn0;fxs7&vt^r;Eex^4+-?xm^l"
    "?5x5-Z}%IQZ?n9LY6wBAV?D@qP_dW6}0c;qmyL+;N(z-6T+-IY-Jk0I7wIC<d@x-Z-}Bh50xVcAwTRTKZboSI(o)"
    "B-9cWJl1J0&x{!o%iDT79I@iG>rLE>5rpaZ?mMgs|aC+CkXZQ-wu&7|n_@}DJUu8=*7?xyJU*~Kuh&3Tz2mK4ENi"
    "HC$$6n8)IRGQ`y|<2J=nI1Z_D^*Y_C}HCkK`X;T9~Pf>?i*&;~kNIy<afN5&qK)tMsyw&!+}-X#KJ~yOATiJ>iD1"
    "d?AAIEYyzg+@_zd*T8CUm5~;Wj0Lm#>?*!GTq=C}Kr6TWHVYo}rtR|3#|D*XZ9r4W=N=T9^(E+9;V1jYU8n^Zlq*"
    "LTH(7Qq1Y6a}gO+0GNw|<eBBfon>?=%%0W-N3Yt48}wej86T6^3{LBMmh1aPFqr)k(MGo~PZV)n$kb30wARPIdsd"
    "X;2xY&`GU_vG~+DYh2kEjb>asC_z09unEmkro-n6T-W{dHJ%J)Z0aw6eO*7om3f${%ZLPJu@PG;|)q*V9;)4Pzja"
    "V`Mh|!yZ;Kg4+x9O2*S84qGbSu{TLYo*LHsNvaj5>B5K-ANIbteVhcELD&?I5t`3iNxx2lOL2^PP<3hLssy@zhLB"
    "y-rhVL{2ueEH3TAqU&0~dHB);piHA1GNgIsw<=xpITmPLKB#cY^YQSbpv3uy7A@DEM$F8?wM~;n}k~7#7xy&<!$?"
    "wvMv<$$gfbg@A$$XW|i(+CLx(rX*=-t0uq9r0P-hNJ7o4faiU7njSAsZE^K|JYQF1wL<}=k#zVVSsf#ltAc&33K@"
    "gfz_=PD45W+DQ!(#X*{Y`|VLI;vBvbk<Tcr^yl&CeFrm6_oQ)ME9!62!fSS;>T^c&8sNZD;|%tKdVgk-d2mu8wS7"
    "ocy`(v3q)%s7qyt)(JZxYvxd(DU3#%aUuY5@ZVuwII^a4p8d3Fp)-tJd{8A7IEnC-=y>9ba3W2?eCUZ@my=eMaPm"
    "KA3YKy3mg(@CWA+hc!4B=j0|3<VJ3{C-Jj*l*&5SRrCOa#WOG`9XB;?z_*UBFS%xQBx$?;K$T3xfvX4_K#reE0Y+"
    "O&PN`M&5WjEL*t4|VJuAE+(T&8oY#j37sGH?*XS{Zx6&%n{^)I&(~MNUm;8spo<0j6Dce*0j0T9V<-X((~FG<h`7"
    "IgPwnJyNqmLX0d;FB#k&p*kEl>93J842y2-A_xAvuV9OMy+Fe1Z|17DTHx>cbJV&m(*+j=KkXYE3#Gp<MH56JE=!"
    "{`Vx5?L847@HF6$x{?Upx#Bo%F!ID;Ruvxd%Ksm*d_EdKH7Lr%rG8HxcIP#7;0S8kfURqDmTeFRE>qzu{7=p!PpM"
    "~SnAIuft+OcFK&;h@a5@R}tN!oDWO!>1X|*kv2e8c5;$;+&cuz(}=+UZpn7u`|!k)~mjm)H$d77`HTO{iQWk4L{+"
    "&Flv`tG7&1lavBTDxDeU*tk{mo`^;C)puS0sVh!UG^G->?jF10Pdk<O|<h*ImM!Po5_ZArk24qo`{?cp7g;w?Nn("
    "oW=5TUzYro+<9O%2pWr#pQ-ouj)m{j~zHhI;gcnPB=HU(Zbg1M9>Nl8MqAk*NZ;=Bw|b3iz_d!>mnjA_2EeEdfs-"
    "C<ikMe9eY=X!IwJt%`l<5?@d!!MuvyV0|>&?}rtU$aQbHeGYMwg&i;20kL7n)r2=N2i+<d*6Y-XKa3pD-ELhjSvk"
    "w*3}$Zby)y}tB3n~CFX#CqQux?-{#|p8ZZ6S$a98V@CBlG`_w|{sSml6I<{9)v;d>TOLHcKGXjWO@16g1wd-t`je"
    "8F$Wtci^oZmekEaN|Z5oONlHMY|>5>TV@MJ*%@-mf9`dbg?JD8ZL=@@Ay-~ZbyE<rZ?O(Se+Q<`&dvWUnlHy{B*0"
    "(xLIPZ#~@vY>m>~WQBd_t9Fno(#<|K)7`Zejh;DcWJ6uQ+FgqC%3WW0TZY^h}Jm<eeiDcbWvsh?V#cQWG`s1A~(}"
    "I)_Ui+KG=?t1y#Ii;<luirYRNZvB^mJhk91N(MF?NzE6h6zk_yC_5olkPMyc8SjX8XN1x3NR&cK46Hrp3$OHZ5M2"
    "O%t=<KW=BQZ}<>1<*mUK8@*xS|Je1+e)kIQmtfNTb*n#svnj=n>jahtjQh?fcniACro6GP<i-Syz^B^r0v|#1LVl"
    "~>KXz;<$Rj`2-5TvEuC>mu;-aN9&mjg#z$c&FDcGn6$~k#&aC+PN_p;zHN`jiQLt?R#Q6NVXzV%AFngr2@!F_;x8"
    "&wG<yEI=(BRN)VdHmbK#(uuO3nsiO;*gOTXWhxEQBW(5=od>a^;3=?`NLfVs865KuH|g24~7m8Gj->=FskaOI4>B"
    "#J$RR)hiKpG+IW>N-<udqW<_nXcX&V|gmyMRzcqmw$v6SW4s>=0t>7|kS`-0;(V+BdXfR*h4mBsp${hChdYLF!={"
    "!@oxU3Jet^d7^;z|NdJmh05k&!dfcMhC(qm(AtH`ia2)g+1>Y*OoKfI~ztA+dvIZd_SYWLEhSCy1oNlJK1d8#A@B"
    "tE2J1yxBh*6aRgm-IB30??YyC{=aF`i5H#`BOp=Arjh3TsG@7^Z8bvQ=hwyv5|f$Nl|4phnky#;k)(F$enOC6m0Y"
    "hgvrU6F&E|Z)(Cq}yk5Ftjci=c8gC80Xp?W72E6|wiJ}#iC-dCrc-EgNTNfa+zf~Qtqsfby9TLC{O@<j7;8@%N)S"
    "Aw6ZCG_mgKy*+H!UFAZZ6V5v6bz&7pd`uwASN9hO&7<ep$Wk+$egXrAz~08?w^L|0w)Ir<!pSKnptm0+!hF+p&UG"
    "q&QW+uX5NhElp$6Ne6ot0{jws(K@9{t3PvF^jsjyZi}GdC{sP5IK<Vea;esS!lEXL=EQpdy#~+%gT6f?Pobxj0(~"
    "Hx1>!o8VX(|0n@OxfL>RsJ%zSmORmWSf~KHfErN}u(7x><=@9clck(zM{BJIj)tO&K$*UnxVHMdWPk`Zlv$%vXl3"
    "h#7I8Vyk>HeHs$s;pYde`1Y+-(VDu>--Vj^{&B5Hj}=Z|+B^9k-1(oXj)jg0cREC&0!4*#6ME1`H|akt@7h!>_Ck"
    "Q?Gg;8#W#dc=8A`*z)jX>uj?~v%26WMjnssKG1fJqp5qy0G2UiCzBfijHPxN+DypaYDIAO!^;H8!;ic=3_CqAS#e"
    "-+Ol<U(IU9YqO>h2JxbNz-0t?uEEBW@LaFk@f}9cWOH1gx`E?tv1wS@op_jY%QYr`*l#GZ~ETQsLDAs4{KE;k2u@"
    "bHanyl<;aU-8oS2E<=o$RZ@jygq~}OF@)EkN-uMgj&0Jk1Dm4K!VHT&RMLEsW*D%y!?Wq~MrO+a{y3)F(=Xo{DuY"
    "r=yt;0PR@z4;k!J6n_WmmM<b}myb*D{4*Rx&F^p*WS@<ElwWMYMyBA(_eZTal&*Ek+JB=ZQ$Mn4tC4RY&H1>agw{"
    "|I5pqi?z|3why4iB;&f2)h#_v^%7#m#2Qut7E`z=hDifbjU6%lsRSlizv|a40_8H?*vo?2h*wEFIaOw<q<k$bL<1"
    "Ha6MbbX+ItdU@N?QLNd|O~){NspgzzCR*MK0Bz@#aWlcZHviUDnsT;=mVr<Ki=-J}LR$a#VdT9isH;+8j;x2)2XA"
    "EKUqONQ1B6s5+9bDK`YM5M_NKD7^<u2%oruSxQ>I%AW1Hd6D-T9!%ROl@_fwq02FfQ46S>AhoK8q%_T830VY<-34"
    "0Y#7;NQEq&#;Wx<e(pK=^jH#Y6txz!aU#L<2_+>qlRt@`c8zpNSg{svPwjoQtg%a<*D(<9f>h<2zSR{oFg9on}D?"
    "(Hhi1rbpPQMyY_(wYZ&YjoqqnjfskAjm@&LKd>SX8&32qcwLaaD_D4~|%0j+ifpgqKPkdYjV48Nm?|Y26HhaSdCY"
    "5uu?N3O>n))+v<@Z-L-Uu+-(;4}KwLozbx9XX%5SlN9KK(!XF8^HY9dB~Y@1TT?Wxi12^(zHUp`Y^#%~=*^IE3@q"
    "KCT!PUma$;Aj{x!l>TqDQ*i~t0v*+RBC=rZ8}fMs#XeH|2J<oA_K=pSWeKQ1%-vC04_TrS;@ii2SPT%W0xVZTC-!"
    "g;=MgCH~2{;^)?^Gywx#KUxZ&E|k&0~fcL^;gLW@JfiXU+?cFPqftqD;V+y(k&&~^{lKnbS0D;uDlSG6KRO>m3W0"
    "nEhVBC(?iZBRMh7TgobbBNm_igQgzwx2j^x~?@^&Jd2x7fvb%pUKAP+w?4KB5(g${Rd+4Fi)3wkpv-Ejj3m}LGe="
    "W^Gy|xNyn=-r17r4Bv2Ir`B?{d2X{dSa?KD_jEm92o7Ponv)>9?Za=JU>X&z^nzH28Mi{1(#n13%w&dg>)_>tCEc"
    "=<kcM)Bj2SeViAx_~_<!u}rUs$ff1W4b3uZzUU8VQ>XJj2T;o;NZ+<?q>0?5`RbbH$FJGLH@_Yn{$=;z&&~~HZuH"
    "nXT{gkib#`!p4kN?T;p3mU2eRqOh~*RM_Qd4%W2%f4)7cs9;9EQ9lW(w4?yPy}HGEDx1`HR-0TL)s1Br~hA*fjYe"
    "$+F)HQ3;D7gnTP6I0X%L3(pJL*YFRd?cmY+{pE_m1`#Z%;W&2zvWkJQVq28P|gXZPrjRc`~B0N78Xa_A#2EjWv3>"
    "vDc-zs-f}`C+MLYuCG&{e=Sb7%<XZsW!6nC`vhq4-b>HP<?(sX(x{wqTi-ngWVQhbK2_7Cx_v@6T0o~&xa&jH6%_"
    "~FKDKjFS$M;AHS&-~21(QqK+akMad6tu`UET`sbktliZoZl$htjzvM^@iTxed0@v*`5nOAAzZ?z4FnuW}{I;YMnI"
    ";8tm6m6J3y?TnYCF_9_BrlY5OEKeW2P5b}8+y8%d`hR@qJ`NraC;fMi{<*qLzj^lEkbXGF1s}hAdWZAJ%NNPcXts"
    "8KeR#b8uVN&!>uQnD@>M8<bTd*G`GVcx_q*-{9e3#b{9wor0qSpeywfPJ`sADM@k>aG_~yH3lV{(47hI2?FLHovy"
    "K-*-!UVh33+4s;XA+gUk~rylxXP$CLk2tC6JPwKNDIYLvNN!+j@jt|n3c?b^K{HK<fx#$6o~F;x4?qI%)E(x$d{#"
    "%75ATiy?gQ(=pW$9@|${_<Ok-7#DE|~yRHwt&n^Om7(3zpIAHfe1#66kTo?Io!^e*w(`=$ja_~wkWUjDdLf_2-0;"
    "fQPNEhaOSNLZYw9ob4^6lw^Jrt)OP5yFtd_v*^S$=MU_H>_duQ97;po-TcK^jGFQr{aN1o<fGU-o#gwCan!BUgO<"
    "IM|=x-G$AiX6ZHSPC};j9)4)s)c8)?7muFxg7+MG?;4gW<=X#nAo?TdsX;{A3l?2`{X<DdL>D(JF5paC7q;c$bFb"
    "Lk)**KT$#Q=yIqAwh+A05OXQx|l25A+1LjrwbXkW8RH^6_McDjSh>^INQH@e{91QBDf-#sHOI;5AK=f90$fRu|qq"
    "pRf0k<OHz0#>s4q>O_EC4n19*(<u-ZHs^OtsL(3)<coBU<S(FgY=B#1H1^`E$H2lxz0?g9!>V62m@L)k{A8*S~>>"
    "_<TSKqpZ4|HvIS<9pFOKp%wV;!;yv3#oQChz{RIzvMXQ(8S!Tn(@Bl1MTafh(6O3G@ja6mTnob)XzP^6EI3u%5kL"
    "~L;uhK?nV1-qrkGH*Sc1dg1*(sksed>4amK053@zp&rmLD|2XX_&JZNI;~ihj-0ZsdoiS6S>@vD!w3#1xtrEXs>M"
    "d!Lu>*Ki37l&$Em#F%J=@_MxU@@4A<>3lvq9-olebKfO@Zn=bH`I<K77Q{?br8m3jU(>{CA0@)kyoL1is?XAF^zA"
    "p%E5uzf`tE6brDxacYV;)f(^{ou<ENqoY0;3w@1Dl*tZ1!whC9(8SHzZLFY}mw7%p#mMG0O?|5U`*dR6}K+35TD+"
    "^@t>Q2XVaxZ3S?Hadf&p<2#H9~+$Kt~ak%Be9!$*!tF;r-QM)<@NxhmQJlA;2L-C$A<eU_S#~-`KP~$kE~jrfBN4"
    "A-5xpC7|80^fcERuL7({KHXbmC4E68be?4wqlr5M3MaL(5hi^`rm*s_j)!z6Qt^o}%i_2)QBzd`-pe(wgA&9L<+W"
    "C5@CMfqdkCS)t^jgt)N38ax%Vm1&msy(TE|9PmyT_Md#w!u`80fBSLe67JY&|)HJ@LsO>lfmGH$M*}P;~kesVf)2"
    "dR=zH7wvgtkAE2-oJ@|!yL;UTtQ<Q7u#Cev5}B&0_JQ)DLzY8lr{~t8-&lu+7>{7@@dBUhzZy?o?!VeUiTOfpi8>"
    "v}_wk=+m+RuaxF~RDYfCX;aaW==^y$tYzw2&%O&)6aS+h`A7b&B_3*dp;p9QN$8!eO3Qu7FGw`_FAUcWi{tc8OeY"
    "(}b)c;w+t;&p6euC0H0z2VsD7|ojrWU%w(9vW#~p_&H1EdiNZ@nTQDd1}yiP7V)WP6TNywpI!;<gup{d5y6ok_=n"
    "7{2nSk&$&Kw*jRt_oa+)5z%4L@n^&L69UUgFFD$`dGpL^6>mBd^`LFvgUv^tx_O;vu2>JE(I0D1yG`^~;H*Jq>__"
    "J>OB0Ys%+}f#K(`rJtn3Aj&8k=#(MkSQLchQ#Sb>S74IB^-)er?RM+)jGc-u<g&3N?4i*@Uj~#s_$KI2zBXv-@H("
    "qZ=WRnY6C@>S4w1@If7p70n*u1{i(v(q30)X^CL%D(8!pTtLbbR(kaRe#Ad_|I6D4-*!KsAFI0gzMUDr$(;Y<i_Z"
    "X{x>%mcFBR?Pd!<|e23>bz(yZ<lFjnbok)_4@T7!q0??NiUdA_{z8~_@s`skw&h!>oFOxg(0|5a8Fzlw<_f4?1u0"
    "_0tHFy{{L0%wN)+l)Sj1Je!0Q`gL}p{&wwi}fT;uN&r(n)Q<_6OqFsVwMuJIe9VUB5x7&qPq<tI=?%y3(%c#$SCL"
    "w4CMHs#V|A3f0BpV)1ubpyw44Ja4V8-Y<pGr7?XmJ_7p~UcG<vM!KN@*8eTS+aj#XtB`1<u&#l+{JN@tarfURp=z"
    "CIf{cOl-jjl<MYiuz~*TD8<ECQ6|78l%WQZYTv+=pHwq28+r5GEy>=0bKl$-@)B{c~O!U`rWC*2Z*LV_AtB<Sf&-"
    "`0#{l)4%sRQ}<5`jr8{IO46Iz#)t5ryG|L>iwt>X+{{KhY@?I4gbdK;IpL>cpE$Ez(&EwLOZv#_w+>4}w&AphrPV"
    "L8`2{z5^U}GCTQ52<Z47O5XNkm0=_ykWq}@tOb0o)&Y5KYODyZN<!k@vv)z&DQvS}blae^zw9~Tfl0b*aI|DL1L0"
    "k@#MRb!AGs!|wlZAoFF*A&EjB1yLV9ByCy%^*1j9Ev6Tt1u0lTV9|9{9yd+l%}#QwU5=U2WZkUnS-u1Iyk!|h!Md"
    "|1y)cHTe_|*3~!=&nQK!;B}ZFvH@ea?`bdZ+Xl*3Tyqa*FNWIm!0j2Ba@}Q!_*{0znj2tSvyuKO4lxQF)bB&Z%O;"
    "wO)+=`uz1l*}?gNIwIjcvHIo`_<Az0z7J{%Ox9S~6TBk~y#)S{Fi;AkjHa^kksib2)ZMGf_fCKSi2Cb)9O*8ii_2"
    "rIWTa%0BQ8rX(AByejPPg~58ZHUvTVRDdgmsI(u3wqDf0fW?+}D8NYDSeR-K2D~T`_5+%2KGT5QaR!ZgY~obt-pE"
    "ks;?WJeE)y5wGq0`PRv9<r;atsC$@whn0$VJ{#y@*BlUsvR%qvEMCSRNu?tB1kMuGil!q1Vi3P2)<ETgAT311GL0"
    "1+V|g7OYU&nTe8_=P?<uF-&YahyMABmk!2%<t^01z||0s95mJ&)k;T)Fcb0pO+BI)S&$>L1-4>Svox8E&cW}bbyH"
    "`SAK&<A7xo{I&JF$?*MSGM#JA1+4)MAonPZKI$Fn)RBFj5!pOF7Y8{8oG}kNh`yT)d>bRKRWASn)%B&c|sVCcRzW"
    "dKX8Qt8tTb|rYj>{W`AN|bk-O<~WkC}y=e?Y&BN!2=Y7A)p+AJ?npddoEvn;9j!ksdqcyd*8js<VI5EmR+3t|#Y^"
    "xGj)<7*@ZRaG;JxVL5q&5;0D-HYaILDi)aqS$c(Lk46}Q@3O{1$Fy4EUpY<~2x?T91)Cl~Xs^;BjSRd6mtlfkAR9"
    "3=z%R<;5$iKKWyP<DM~Kh9JUo_!CQp#bNj}+s@z=@z!O8gOm))0><ME5bgFS%8elytFsweZ2VRDXghHX8g^?PAB^"
    "`_oJ^!Cdw9(`=3TcnE=_j_lYSMWoFu+Cg=o|E4Fda1yzlC1LL<^FSZ0_&HwCsnNrx9SYOPGy@<bN0<vZKoBHj$Sr"
    "^G5ySutdQ|=vF%Wi&Dpv)whvorUOb#|t^8$%UJ^swNZ<vxmay<C8;~oHvMO7tj%m~~13qRFOX(Oed|F%~b+y!+6i"
    "+_NM*C#^H>=RHTo?FF(_fIg*+|}!+Id_Bv*gLD-sSXPZX3i?;{$nV+uG}rnG)NuI}hr765>;9_&bl^yrx6Ae|&hv"
    "O@JC@*-n*N32iTK71&3(#JUj3F|PKIPAyvB#(QfjMWl<6wfuOgVwg?ZrLPX1$h+JLI%!~~d{p)tL}P;uFex<HMGY"
    "j%qh`M*D;VwfKt-LCN$r%PkHNS|5GO&zsfk=C<D3A^z?(qDs5GA_a}#_T27d!4H1`D#tp?<sLQid$A9%%(+*g<Dj"
    "0#mM*t|4$Yjr6eBj)>yQsAf1VE29uE8{q7T)1C+AkLR_4;isg`B9Rk0n1iesa;xMm5^ZbnO?3pIPDSu_^eXJigcC"
    "QS{t+~Bvwdrq$SjWmR)C-T3%$y7{>47v{s>qt4Cxq%<rTwSc4yBq)(!!-NM(_O=nQ#LeDI`7>)fucav!~z1&+_i3"
    "x3Bfoje`jJ3Dq*(_^Ly^i7$5m=W#;%N|(5^GCyzuUpep98-Gi$R4DTijWLMginDBJ8b7YmN#p{c5Tj>BL*=K#7}f"
    "@@iNN4u&mYKa}M6@ZaKPD9!;7EJdW{hkVW^ze(u6EN{dKkS<-LAr~x=XoP_=U<>%7L}NdPNXg?bIoVKq(Lhl*eZ&"
    "_`)_G-d6xktw^;6%cmQsQ(+U9{gqg0o99w2a>J|ICTD4u5w=cgO9@9<7)@D)R|GYqsC96oi_h|#TPyI^#x@jPiT{"
    ";74K3A(fo+7Z-vR$hyDW2Lic!Ipz{;~CAXDPg8`yvys#KU&(Ws4RR(p>!+s2%ycn$2h-QbIBNHmDP1v^6dUhmV&m"
    "O_FhLSo+|YW8Hh;K91U&3skXK^UG1gf)b~>KO!=+hN1WqqdNrAe88b;<4R8}u@oq}`&rBa9<EBb$t}}C``svtgG-"
    "deuU~#wz7+i8o`kwmXqI!}Z>tQ>;skNK6ZG?*C{%!6^(`RLQBmS=+T=K|u)KtaH%{tm3rapZ_L!xFwCkajNs+z6d"
    "&%#($f!_<qSS?kL6zW5%KacfPw9WdEscSYDr@n!MrQ@<pXIcqv3JdCkvGioc=mHlb*dTzmWDu-rKl>r|Z)|Ee5v9"
    "SY9(HZLh4U2$hajmsbI^E!CBVqIS8D2m-_2o(1n6#S6DG8H+}N&3K{iH~1mfUthWC%~T6|5Xen)p;1qA4&_AAKQ-"
    "dmYWiiXBuhA=r<=(ahE6sFqi*0BFZAMawxUD8g%%|zQ&rmIGt*jz59{b-IuIuZ&J8H}Ns23Qxro=2tSQngt}@q*N"
    "GLy`DsUI0)ioe=lL{ub0r-gTpepftKu0Nkzda^XAc36^RL;}4c2+6G->*_%1yF2o=QOZ5l`T7t%s#)`4O#0TFTn{"
    "2Ne<}C)bz;{UWRE^%gi>LYdzEvE|sMEEhe|Yyf@uPjnN}4Cq)dyDRCe7oqWGGks(L0p(?f~BPtK?T{g3Q6<98H+Z"
    "vL{d!_5OuAqlX$1%7I#8l{-xjnP?HmMDawpGHUvGD!|Os$1g~UM~LZL<;5gxmcU9&q~tgarZ&|la>6-O4YGC+X}0"
    "KLZbh&b63Vh6k6V8zPu&$9JL%FW1Enh${rxvVUK6#^(uzNn5y_&>(6T9-#>(hfL^q^Kpp_xhj4$8pga=$LQ6Xtkq"
    "S@Y0c{QnSuNHanzLCKZIO03@mR{1NmS%6nh9uSGG9M>_6-r1DbB_w0x02?Ooj!Qi$XCrP(<8dm7=c^Vbwe3uhV~r"
    "o9pl(<j?i^pf1R*@B)~mH28JDbTCpJ1mK#VP9})n|oH)<FXDsDp5!nnkq=z_$e=h5-XSary2;e1r7Yq<4927(^X;"
    "tgB1x*XzYNB#&80W`mWqeC(V6Vb0t@}pqMeAs37Q9JYu-90L6zd3lZ?e=D&<%2+UzVDYA-Y`2PENRHp*}pUEI(n0"
    "XBX;tSj7eth~-t*PnW26fD8>Rt8QR|urkRCW^630b4n``#MZgok`o_=)i7h`pHa1{y+fk&I!{b|P=KSY*y*|FNZ3"
    "|F&=SUuzN0bUoHxNXANBS%l1$f7X%o#ykd!v*yw;%Cjyqe8HkclbKrtA<IQ)f<!Cp`><>}5I8&lrq0i!hZ?@u_;@"
    "ZQx<a^j!J_<8eAmQJvCcp`fJX1d>izUw$&6iUN;lJUGmCvDL#YCGZQ0oG;D-9KQ`mKa-R-O|-OFk+go)q0zaAM!n"
    "$I8_Ay$G_)3z#F~2W&vP?b|Zk36x-Ka(=;bcn(kz{ukVra?!vlwXSFmVr3sGiiC*VjA7Jh8`aYe4JGP&%4E~zoxq"
    "-imEpeBTG^i_!VNiW)a)QJ+lNFqR?S;`)%X%!XnowP2hr0Uz{wc{S1WNc&JU1zH;=XJ;ko<9CzNEm7w2*e5Q7@AB"
    "dXnxp@@!j<8Z?Y!uZ~JwX9}n4Y5&Hjc5aZ5T@MFiFRx6I?Zno_`Z`Brp?yVEh7YYBIhO-D=CuPJaaF~D=r_0;yjc"
    "Fo7Y8nyKXS@k>zSy92P`wOK?3Yi=aZuV90S;xBXG1XI1UVKV^9#81>6fgNESvt0|WLjqzFSIa)hqD79b_(3N%hUb"
    "sCQ(2=6VaK*L*yiNkh}cmn+#gQyQMs7IGnHuJ=dLHpQP^;`lw(&q3C4b?IZ?__=Ly_JZ)<tSV~zqe>4TVljt_9Dd"
    "YPjO_wb@lrm>`RQO9`!h>XPBIoB`Ihe)*>Cn0kcFiW}>Z=5%z%(9Ed)}+|R~*@p3=mn$U!cLQU7R@*<!0(Di8z@R"
    "f_}bm?_|n_^m=0W-ziy;gW3{W3>WATHX5%H$eXWdEz8C9VaexERBk<OZp!SLte*|Hdtwdx~a@luZV_V3a209O^M}"
    "4e;m}$?8ik>AQtEubg{f6-&UU7gZ54@ZXMMJEnZPFaz_bso9081uC(sJiRSl%IThoNtLi}0KRXzmZJX7@&2D_-;&"
    "44i`8<`f5CXG)4yi7XJxva@0&}f64{-bUS1Ltg<1Pv`gD~XdBL)3`n@E4ft8!6ibOovL^i#uVc|&UG3(+W?n4v-m"
    ";oN~+C=ff5(7FG8L)axROK=Sz~|C*Fhg6o*-9YA^7)*WL_m3zpogJ{+z-$Iipqp+Ktm}*pt55IzBl&El#7LLNNEK"
    "?kp64Cu*GtKh4{FJ0Ch5z1*|~}M#+-qY~ZAWG6EY92vUN9*;*F;i;{?=8wtViZj2J99dlSXOMuBSn^fqGxaxpL*H"
    ";Xn)T6f~Eo7)^0epCC(S3TNAfaG%zs)f}3#ahZwo4GiRec<E1ud$Re>$$jj;OC!o0b8s3hMS04sB#5pC>IR*T`I("
    "$R!>5CyeAU`>L0EF-P{0eWa3?Yjx?BYL|NuTIS?6jr_=cEd`{TUT2F}3Ma4ih>c@AY`dXZRJn0`>f7n3okZq2Y>2"
    "r!HStZJ>^moO%^h$o$uRkd3SSv8ssVKG!tVB+s@4Drc<SC5F^NfIcp2~ajSv;5G(;?^oyIaFbzPX#+7akByJ66wk"
    "3fK>qM7)*$VZzk6R4thNyi3I+!a<Iz3U}jnnahq5^a@Zt89@0ie&;E<sxeWJf7$Uj$iAQLtCLi6HnhW6pt1<q*rJ"
    "8#hM7o;(LD)kjLfypo|UZ#U846SLLi+v?b8<(!#`Cv1wuMon*z|*V#JzLJ9Ssnk0X@T3t`r3gtWO3ecQ@E`|R|Ch"
    "$+?s$*W!`0djf`yOpNv5^X>lj=y<6Xa8trXi3({Dnw*lBrnS4v{qLKhO}w(33V3QovS|iH@yJg-j+|O}Z$4KmZ{J"
    ";5Z<{eLZxbag+YLui+Axh79)u9U15}ORx^kFEO>Zen<fNm&HZ)pt<djx{Bu4K}FP!l(`{h^`!fD=iTpK+hA4XHeQ"
    "?@wXP4y)_!x16hO_;tB7b6)A#hj-Tf;<it_sUnmeD@k8#dE^G&cD%Zw<ms1Pwxs*5@zL8sO5@#FM59~fSOf{4dw<"
    "%;38{B`~sU@iQuylcdNgykC5!0jBCUF3#KE+wc&AJJQaVVL?Sx7V2z(Q^LIj9>S$#0l;;)JlK*Y>QH8S(SmJeg0v"
    "Te6zE&MZvRO|9Lh8#KXL0BXsHE=uwbA;7(T^`{1*3e%qWE6OD6~$p}3ryRMXK;|tCCEZ@^3jR&R$&?G`mpY{ss(7"
    "<E45j#Ev4n9?8I=V(Z?_oWtIQj{j$sfDT)SK7)$(y5>mBlf$?Ft5(jYtwn5*YC)m;ih^+L@H|JgWw=BNW*)JpkIY"
    ";<@R~YMu3HOvcuMhaie2X3j29_*ReN?4F4crd7v&8YWWcQDv*QBvIehd8h8Oy`K9v6}KHAHFa2FvU>`g)YA0n`vE"
    "Ai=l0LsSzgsWV3by#lkJc#$!fJK0GE0r=PSh6G%v|e9btMhv4bIGRWI56!u2wV_Vwzz{tv)R9kB<|h$iqDdZgjuU"
    "Pp5{$=VVBwuJT<*VR_Y{>S5&;}<8%qvYqK!&iyZHFVy_W2Rlv<jWBPH%UkT>iPb{0RxW{RdhnbM`i>X<WFx+I!uL"
    "_)1Snu!GAh&N%|T_E+-|K&9`OyCAX178JC#TXXVN$7{RgMVlDO$j>ktQ3E3Kltt(;QTKWO#TXd6Oc3-|3A19q3d-"
    "T6pf2hNQ<i+8^&oB31oR~4X$=;zhp5rJP4>{saTk~u_2%g}+whLDW7&J4JaeDru61hFyiQxIyZCmDwaE33q-@%;g"
    "O3s2~(M!VTblXh1Pa)V(2rcf70dVwb<jRqMS>2FygI5UpWE!*5Erse*pUO6R&LEL;2Ru<#-zZD5$rH488`s1h-tz"
    "n$l}{~8qT_9Gf}5vTM;_^<NT+l;1QHifOMWeiAYj;tt+BG{Pc%>!-4CP`$rBhRg+D=&R8+!Y@?>WRf2#t7Z@%F#6"
    "v|_=He(I}f@3-bJa~84fP}3nAbMku7r$$;k@Z`~L=Lp3ZPa<WIXQ#D;1Hcz+@9x1<9C_HpV*~kDH+7uGxH~(e>(W"
    "Ha+p8;_(?n_U@vWQfT7vsm9}K`%HSn~+%*378T(~^aS82bWT>_8Hy(m!p<6?$h3@e1J_S{-Qw4Q9^|Os2#E;p~K;"
    ">5nHp^$#e(ev9TWFfBOxY{y3Y8o4hh*o|W^I;;>2-D6c=lJxUMbnxc{Kyt4Ki-h<$@@ijHRnHh~RVSw#;ZR$U<dI"
    "WG=7~hEX4pCV#!w_^}VPAt4aSjYJ`*c1MavxI+Q}RGt3SbE{E%71y~lSa-`kdafO3JAvrczmG!9Oj<~!k`9KhQI|"
    "}VYjk3>oLzDdEW8?DZuw5~JkKht{f?o2-wW#-iIZ;C_TDC}FIVe(M9u^7#TxI6FJmVsqD#~)le}0(LhQ&*Xc#8(>"
    "+nuy7Hr~o1(Xp0znonazm2Ei-OF#fc}DfCZEx!M<pTTZFJ?7i|3^1|y|?FB&U!jaFnRjj;Ht&1fMDgTT+4wUMnTV"
    "kHneLA7a|mcn)uTPA659=4L{nmeHRCT%(=J3miecF>?Crn2~YOAipZ4NsPZI#^dhevmwv8-#OMU5Sy4A`rwy0W<@"
    "o9IYAeCp&TT4Wt2}}1Y3e{27D|uIHZsL?%?IaoY{Qdvx{XO<abJB&!%L)xGrU;J2Nu`o+FLx5XK<-5+N$(Gz8E#t"
    "NS&{fC*5vqMXk3;exFi~I!r(ypXY{ft!lRQLwt(Pxs=kV&vzbZmTXkfG$EJK!`I`3z5RnfKa6eO(cbtd`RQ*sjts"
    "?`Jc;!6Yj>#hwo*p9&3Y8CRd+9J6KNs?<qdlb6`+F5jX_?`>1?9W9}9|Ar+H1+IL_V1{+`U*=30ZG-P-Pr5BA4<4"
    "{f%s@+_Gbr!w}s8=c9z0<gC+lD|b?CA*qMVKNf|n8jtLT$E~}Ydz`f!^@WZ<>chGw9u1+b=#wKOBS)^dmkH`+U~`"
    "pZV=Tg<QDj|1HZ6hKI0-7ir@aR;a}JL@z2}gx8LuO(TvE#>1pv+aSB(<b%C7K`E%>LO;OXoTya4W2V~!c$%X22ch"
    "+?nZsC4=PzyBowuK1Unr01Y$n(1k4@7O%H+k*rwIia&1b%$iq}eDmYO6HvwSIX2mUrwLZf}MWQ8?LjL}rT8gDu(*"
    "B5S4*8u-`2iAxb|_4oTv_U?Xj@YjRGUk}UxpTc29f0=($^VNGu|Hq!b?C3|2_-icJ9FJ|9q{^E{;!$4s>tncO!<L"
    "=A>q=INES+2VjlEC(BfD*5QV^J~tu1jGwlKEZ_Sw{#`Xq;+l$O6CC7(BMU=kUsEYjIxV^Y3B@s)+aUz6DCcr96^9"
    "izQv^VK>rce(vy(-SOn8Ab$S>wVi0&Gy;*1`tUqE!uNQ%5pK8=hZbkCITR0;$3s@|LMR)hV1IlG?^;VE@tZ$acP6"
    "4ca0M_2M9&+p_DN0?@J`%TV`;0F-Jf{lbN~i1#x`=PrVnC(IuO>A77=M#Q1ty{&vd&=ncI<rGMz{JO#j}`RylCP+"
    "y_CjX5Kre~go^HFHIZ;Y1IXK~89i@giG+$c=3mLEJWU{jxl8I$Hf18K1IY4##eJu{LOH^saC1+*OWi{@Bni>f5)-"
    "dywo0^}-{XO`@5jrIK+ROJuA_@4iYov7)`*V;mw}PQuD216ZVlCMb~mX&7S<g6NJPIh!HUMM--%xy7;<d*4OxL9?"
    "wZBr#X^7Q4E-!JyrVE<`;A!auQhewxmnp2KVzOVkNdUHo=%v=5cyd4%-g^NUB`XN?v3W)y-g-W7s|GU0kbpCk;i5"
    "?w$<y^A&n6(G6aCc?v{;Z~y)DD$dFt`RRMzY^NW*G_2XBR3=#f@j{u^@R9<Tmsy6*KO#fZpbdOKUh`ezqV*Sh@tV"
    "gO3*pZ0M(n*83-G#o!9zK2CvS(oypGA?<ddx>AUTdqWICO<VM;zx|}E3%%1$^ZyaiR%#Z%}wE-IQZN5VDcvtRxGv"
    "S4YJaWq4^3Itf!TTm%=gjU^ob}{w)_|sZRlzxi^+g#LuJT?vNaljPd7o)I7tDd7<*b{z&`Q{t_gtvz7=h_5+-8}q"
    "*vIU%HJ2Gv)}LQo%latUJVRleus#%mY?uhWO=zfz4srV6Zd1mWWjmblM(bPWLl%*aCu)CX?-wnI5iYd^pPDdhWbx"
    "E~=fME+io!IL*aQ3-H$8Ygt87L~(8%D~Vpa4=Oze2Uvgo!90`a6zte=RDu8oaz5I_Z0c2>6&mqoL2hoX}kM(frxX"
    "@B0Fv8lR?cFGDb=Y6gjqFl0U2t>tv$1Y{Jjy)4;SIZ+b`_(qy)N5B8bkmKpnOMAJi+#``=w&7gdWkGp|JWeXWc2i"
    "bRBqE3{Nb*d)#fL&*%UTEudNn{rK_}$lKiDsa`$tOy@B+(+pMlvftvmkYPF1_Zng74@^fCyx$H@Pks6qbWs1XiZQ"
    "ldMKa6^}`dlS_xt;Re1_f@rX6r5c=8M-_X=}4i$hMnT>|%0h%0_V+hBi-zY6sM&&*2r)4%eKkY+?cfFxRED=-NW-"
    "{#0H2eRj*!jbS8rwtM8B*-Cw`DtR*A3D^2zK=b1&T}(JHTQx1?!&9G{DC<E=ILk)5t-RQj749@k)+eK-N30!<RKT"
    "+=Y$#<-Um@RR)iG?DeDlS0GLytCilkPWii*Ks_OMH%MS68MPZMYwL;kPG{I0KAm{X;TLJvZZ8s1Y@$&70Ij8~_=a"
    "lU>CO^|P4m2l_&S9RvL+-Ba_loLK-G52IAc=MaI5I>rp8J@6Tpp+KxEdZbp6c?HC4*TeTCRfC2Xl_YsQ>R%jA!5E"
    "w9`Sza6#ln8wuUt7J@MqmuSe~7c_*y(=5Jaad-?#FSN~S#qVNyQxoouN%g<%7Xbc~#Wukc<zD!O=x3Q)J5Ll-{SD"
    "iloeY-<O6MuN}uCW&9TyksZ)LJ$Zryk1_-VT`%-ZksZEZuc@tR63j+|&*JNd6?mWNG}L;UvmulHc#h-uQK7I-*UR"
    "s{LV|Q8Vd#Y|;S>4D2{eIUXrAUSY*Ns!jGe*ehTDIS9~2exA*4XNzoTa0kq64WNSn$_umtijZf5X?=$EO0{R!E+d"
    "mW;DyfAVWy#7ZT0vEA0O>ckEdNhNe9G3>M&{1mugimxV0Ad1w&({Rb|@O&>T!>hM!n{Gx!l)eQIXJ2x94J0l%mq&"
    "ldG=K3VZ0Tc9G<bOf?M45r2ver~Sb&jdR1#A^{DxbC>kKGMf+hl%6$$zb(fkS-1U2Fdkww(99X)7>b9#FWFG7nxw"
    "md-%J822UdcPp&5=+A@;lJ2Cbz3e^h9%TKk^R#1t7p3_Ma4?m8jLf3TQIW2JVIR=5+-J5j5Q1bMdG3T)xD2w2pHb"
    "8~bo@vW0Gc_YQL)Ya<iiEu<i{@2CFBY^0xISHG9#{w}*N50Qubl_#-`&;-NVE1JulRPZ`>CWC8`=Y*Vgb`k6BKUr"
    "P77>smd-9+akK%+#mMK3xsA8HMb_Dz%rA?*&;3c^m?inLxc&UjIG1U4xya94bDYaHcP-l1`HJnN`_0d<>CaZpu%("
    "@iYkmUo<(pEJ`3;l@#OF&ohl6BS4I#7tEAAcl71Twp5<!C_-T=S98bdy#`ujYU(sF~BoErEgi&wxs=fBbd_e8WTh"
    "19P%j#{pHtd2x10<;xU>MUwj&acaO2lP~A#R>OdFOarzTB3V8Q|BVRMPWR5Nzk-I9FmIyn00ERW4s$s4cV?4<5zQ"
    "sE$->mg60B_F@E*>@M!nwZ<D?Kqw$NA!=t}x?M4*+<@NaJc>nlhd>}nst^qYxtvLE29bS%K@4gsMULU^Pf3ZJC_Z"
    "I7h?0k$oiwrM2;$2s0;7H+<O;0dD`7$G`vY~-+U>HZ!?QEW3Af$|)pQTH4V0pu|Bvvpb(!#?|CyCj!j8&UzyF(1J"
    "s`OzTR4N3BO?a1OEgpa3WR^?8a${U?({PaiGSEgrI4ee)(sy3n@zmmDbXuS@dx^o$>?JkuOm6n002keX;G$yrNIy"
    "DXW-0+V7YdM_Iq|-VmxMGg*c6E(_9&JmM)TzY+O;a2DG|%$_0jlveDuqBZ*q7rp)Evf6xcl$HE|$ieZ3_7W<Y%Nn"
    "`hs#D(ekGWDpMIqnP$N1>WobLY#lX`W_hikXa1yeW0p*$eiJb?*r_U{{u-eH5X3lK%uE12ATE*<4G_?ZoQT!asWv"
    "^C3epiv_U?>=!40GtgmdIf0#_Ba?BWOB0VF;X}uzv0YGX0lHF9y=PSkvM;u5&2+-4%WxO_816qeeE|r;zhy;@pK&"
    "Tn1c4rJ#@gY|zkLh3l9Fhxd;~`DqgdOxrD4gfNsZduuTS2y1mg|d4gL{R-HSG_R{|07~Aq2wUQqppU<Yl?Q31s?~"
    "b+ZQt`+<g~^f&t=9TbbwvCyCU9TG6-E;mTv=JZ%OW&98k)`so!)U~jnGFbBVN_0oBZi)Gb@DUQ$i)0DxuI+kz;}H"
    "VC+&MaCVO4q4KoA${a=rjGF!zvTIjjU>-i&9Usop`uP%FN<#}lJ0O0ddPy+rowP$ie;G9WR$dw_9dq!uEG3(`)mu"
    "w+DEmAG?wB%jcjLeNpjz~8K-^MMC1*0<!uche|m*@dc3N^#6tSbZu)`f7s5C9;`TerKNrvs_VLc&D6_sbFcu5@Ae"
    "XBW|>_gTAZ0AG!{@jq`@S=DsxSYW9V+*-+?RdRUYq(8f0e?(vaWY+1X$JZIg=XOq7xNe@B7!=^?mrw_(g*DHhP8|"
    "ju3TXQ+Wu<S!BXbymsRV-<@G4L^Ik0=&k=$VY?5RAXE7YZ%_Z~;I;(?zTo3au2FTE~pp1@y3?uugaTHZvFn02ne1"
    "I?V|Gl%^FSO7mwd;8|v0TLFj6DWK!iOsV8tJnp*Y__?0kfPx(p;C2_yfryNm98;MKbaGRwVepDT@s%Ta9=AD=N9>"
    "j9jgkAup`9oOI`gpVNYkYRqMFSqysGpc)mIGgv41C_7|uK4_ZubQ#LOW#jDD1K6q?)-R_)?2qGE3ym>VXmvJ*VC8"
    ">Fr|ZyE$_R*Po+*Ai=u(i-+?FWPMjM9$aal?3K;)j|}(viw)jgjnqo=kPuBb{jqaK7ffQOy_~__6Qg?oQv*fE8sV"
    ";_?-U?YwmeZ40L)V8f_Y!;DJMLlv?u>InFjXE#!Q$D9=cpeS{Rwrk9u5EwM)o-6-8|0voffcH7|xk+3m|_t;+hIW"
    "J^X(z;lQJFoqypYMMdzpn9RV#qLCjDMCr`6~HUG{xyp>KQsN((CFHirr)Pr6li8ol}Q>kbr4p%a@p?8Vm}m#n7eD"
    "R!azK^-%e;e1Ud_WMvbJoPNmRV}z?(f%eXLsH?mZG;AnN&R7HeX7eZkOY`Q7tRP!yK|0w@scM8YFEt8g;Ot_?#Eu"
    "zmy+lSLUN0{`<W&yaXjN*GHxy7Y>@b=@NkJJ4tbglrQ|v@EpkHYZC%)wM5=rnRw!fu?VYAe0QU6bU$4IQ3b8s~uw"
    "zsWO@1?H31#o?2y|MF0t|P(9_JEaibw`9mc<UwnQY=n&^(%OhF>l{D>M!14e)edZbew)lD|e&aBhGG}=LPDH_D#^"
    "Kdmz3dz-`|S+rvxu3)Z}CuEV2oY6&ZG<&FPhQZ{DxjlVgAeU4QlYk1cgb@VHmlwlh^{gcgn=QWDa3BvkIeoM4`P`"
    "p8b3S<|KuTYES0-v$HRt!J79?I;wod~I{KOMd~*n>CZ_{HJtF<Za>)0wqTWV-+j?c7PUJrltaHumv70mu3=56t6T"
    "<3AB_ctKm9y;lp%E)G<wfEqm!Vp`5I3M9X-x5o81#~ShK&E=d$*1YFBk}(~vmV(F_b&k)t@w@Nm9t6Z|JNMN{*gF"
    "ey){g(>CFn{~n$e&&LgNa3soj`;D(@Lzpia5gG+mt9l`~)(N1MjESn@#k9S$}<>)ps2!2$oU8@IM~2Gtpc2{!wO>"
    "v<SGkT+Hm!fX*`sZ19<J!r^qS99!`Hf<Ub`BZxZ=prG$Af4mq;v*3+Zt6)CqgF1Lx5mPUEyRv-$7i*Jwdh`^(=Hh"
    "di?ZanK)R}#hz3y=gkSl1U`RQz!Lw^++%9kix$33Ig*lc@Rm3?zCHYjI=QlOh<(!yR1}5diSCZ@o%NP9Emh)_G6^"
    "QUjOg~qG2>h}>K1JAP#-4CHO1+H2J~G;YsCMW`r}{!8^>3h#ezDQ2%d580?gC_u7J{MW{CL+?EPcJ#OvtLdqljXs"
    "jCf?CG3qvDZEnEb-Z;C8tU%x-$<=<1pH2@GGxs=4%FLkK#eC(`KtrrkRY!*#g=D*>tZ8e6F5$v?AaorQvslV#FR$"
    "}(tuv+3J(ifGjX86n4^Z;=#wX(!C*wWg5dcm_Sd}9~q)w#<_A|Mn&q%s+NXS;uJ%Rv-r<jG)^pTzEe7zJEBKIpFd"
    "g_`o$f_C}mgrwP7a#p4cZo_4kNXRn5orM=y3fkrY>UwpX)d!%{OgZq-4@&|-1mmcHw;W+{cJ(MR6sH_>h)so#v?5"
    "kPQ|lQJZ3CyG?$a{^UXVhmxI+xKL*>{Tmm3`uCz;Rl&xE~Jw?dUa<nNXH($??5>)BJH`RZQomXCaRkQxkt+>mlRm"
    "sKF$g-X>qE*$ZBsBJSGFMQ8JXaJIUdru^2_8)seD5dg=JtZU3NF2Lno;ZIoT{MmPV~~o>NV^J7Kv3<WVwZttS^JP"
    "cEMUqn31M>#(duTn*`U|er2NIzzQ*paL!(Fx4+jN(`mKNOSG#;^M^p51w6xSf3tak8tk<$jT~_fygVK&mU#_Wd?)"
    "`~CL@0}l7;kA!%MH(bMS|e-V=P?aFKPrC9h|udCyGx!Gv710_j>>yD^x1-=Dj=(GUd}%$AFe<*`ehN@?P#^KO*oY"
    "gYs_&d7Y|nOzv^zdadn#Q*J?K}T+IubXyTdk~}9m~<l(xI1$xMrxU`-+OAp{T!KhCsA5Drkr*&1Eh}Yz9I+6ij5<"
    "Uu3As7u?lbaZf(I!Trd*!o~oVA0WzfBo5Ly@0$o2oeQ%i6_DS><W;Ghzd#QJ`9_WsRsSS%kV2)%S2c4X*FI{Cg5W"
    "P)Y*}kR?FuS<2vS+&F$_r&H)03u+yu`GIE+t?Z^f<Qtp2gTAIf=M@nM(|Gc3GB{HjU0D)P$BgvI^_MgnSt1lOd51"
    "(+iRyG{>Sf+k;zl`$0i?iZli);U&q{ktBKv?Vka<ATy16&lseM5iZz|!-7omWCqJC*r<k++s+6>^RRw0ZFVF&%fx"
    "swv!EA08vp#}c)aJuBjxmiW(~Jtic7gvMM1`yc>VFKAm!*x8a#MPuWi?fw13VbmW~?EhTI}I?zx+~8BTmMqGTb~r"
    "LF0h2ALI;Ia{sr)tW87gx^{$!gN)elJt2iGz+E@4NM}}CLD`B%2pzPHCH5Em|@(jaC{w4Vrs*pLBd0etp#mWTeEc"
    "_W~lOmc5R{`k%L7``o0h>4NXT=E-;XIgZ73`lc|Ourq-a4Mj4WMM1S<lYcJ)@gn;2~l5nhTC2oAMmIl@ZXB$aF6h"
    "xFV<%l`>17jc!4C(D!3VS4-UPSeI#8iO|x}h`PX95Va+SUbbqMd?fY@RF`5b{%}OMK(zGM`-rY*2_<6(!rG<Ge(P"
    "31>LzP&pDYppULl?6K-`9}jqci6PVumU|3Bb(OY@C%VXj!7M@I09yiJDndAj6jv?en6iJd@SuqOccgv3STpKdEv}"
    "FA4gUl4yb6wz&eZSOGw}AuPw?sXYMBJkiLcWO&X}<6&U}xMJyC&3GY3MX0@#B3IwaA_dZ{2>S5lAUXyz)d*coDvi"
    "!xWURUjKWi%LDRV)4kj@>(s6Yo>i|ABKBV;3WBOZ5{ig{QHOHT|D}xmwDcIXBt_lppkcutGRT)X5`Jwl*@AFSP%8"
    "!J;aJUe4l044?!_3X|e7&EU_3~Sf@r(EuTS=kA?8;E7zSD=Mul;WQ_A+^6_CWc_@kQst`68yxYCo7|2`8GEeF$i7"
    "oj6rfuBF%g$>$x3oCi3y=Mt#>UY>@4ss^dHov1ag&eXx+Ch#d|F42QCdA2k$FyKMUl^N!k}?HE1}`;HUhZPXot_X"
    "l=(<@6Be>Lo>Lx}y$EkXQjXOU>N3(Y=;mNfJ}M(NTz9jWWcm1sG=BPEzaVk3%9#aTk}z#{@@Bz^geUusstXaREN`"
    "{-th7knK>BunJmQzMt5}AE+5!&xz!;K7a2n&&qfBDav}v7syqND8nA+xe>ZBQ1E~E3-njCp=_}8Di^n_~6WkXAGW"
    "Zgyuc&{P7>tZ`-{w`xN@W1?s^hMS`#oU36*ML@&;3ak#4rQs%vhr2Ao?Q;=m92O`qm+2%p9}xxd6(+dO}?7Z;Y)-"
    "2t9so~>`|^?q-?=y@!;<N1KFhUG5"
)
course_bytes = zlib.decompress(base64.b85decode(COURSE_ARCHIVE))
if (
    hashlib.sha256(course_bytes).hexdigest()
    != "c355aba9a744c3ef439e6b7841a29460084ab565ff65f528c7b2f84184113391"
):
    raise ValueError("Embedded course files failed their integrity check")
course_files = json.loads(course_bytes)
if "COURSE_START_DIRECTORY" not in globals():
    COURSE_START_DIRECTORY = Path.cwd().resolve()
if "COURSE_RUNTIME_DIRECTORY" not in globals():
    COURSE_RUNTIME_DIRECTORY = tempfile.TemporaryDirectory(prefix="lucy-practical-runtime-")
COURSE_ROOT = Path(COURSE_RUNTIME_DIRECTORY.name)
for course_relative, course_content in course_files.items():
    course_target = COURSE_ROOT / course_relative
    if not course_target.resolve().is_relative_to(COURSE_ROOT.resolve()):
        raise ValueError("Invalid embedded relative path")
    course_target.parent.mkdir(parents=True, exist_ok=True)
    course_target.write_text(course_content, encoding="utf-8")
for course_import_path in (COURSE_ROOT, COURSE_ROOT / "src"):
    if str(course_import_path) not in sys.path:
        sys.path.insert(0, str(course_import_path))
# Reviewed local subprocesses import the same supplied modules. Colab installs no
# package, so register the scratch runtime the way an editable install does: a .pth
# file in the interpreter's site-packages, or in the user site when that is read-only.
course_pth = str(COURSE_ROOT / "src") + "\n" + str(COURSE_ROOT) + "\n"
for course_site in (sysconfig.get_paths()["purelib"], site.getusersitepackages()):
    try:
        Path(course_site).mkdir(parents=True, exist_ok=True)
        (Path(course_site) / "lucy-course-runtime.pth").write_text(course_pth)
        break
    except OSError:
        continue
os.environ["PYTHONPATH"] = os.pathsep.join(
    [str(COURSE_ROOT / "src"), str(COURSE_ROOT)]
    + [entry for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep) if entry]
)
COURSE_WORK = COURSE_START_DIRECTORY / "practical-work" / "ch05-b"
COURSE_WORK.mkdir(parents=True, exist_ok=True)
os.chdir(COURSE_WORK)
ROOT = COURSE_ROOT
print("Python", sys.version.split()[0], "Pydantic", pydantic.__version__)
print("Offline teaching files ready:", len(course_files))
print("Save your work here:", COURSE_WORK)

</details>


## Commit to a prediction before the examples


In [ ]:
prediction_notes = {
    "prediction": "Write the expected behavior before running the worked example.",
    "reason": "Name the input and rule behind that prediction.",
    "falsifier": "Name an observation that would prove the explanation wrong.",
    "revision": "After execution, explain what changed in your understanding.",
}

### Reading the Python vocabulary used in this notebook

You need basic assignments, `if`, loops, functions, lists and dictionaries. The less familiar
features used by the supplied code are introduced here. A **library** is reusable code that
Python can import. The **standard library** ships with Python; Pydantic is an additional package.
An import makes a name available, but does not mean that you have completed the exercise.

**JSON** is text for exchanging structured values. A Python dictionary is an in-memory object;
the JSON representation is a string. Use `json.dumps` to encode and `json.loads` to decode.
Decoding proves that text has valid JSON syntax, not that its fields match our business contract.
Predict which of the following two decoded objects could describe a stock count.

In [ ]:
import json

intro_data = {"sku": "MANGO", "count": 3}
intro_text = json.dumps(intro_data, sort_keys=True)
print(type(intro_data).__name__, type(intro_text).__name__, intro_text)
print(json.loads(intro_text))
print("Also valid JSON:", json.loads('["not", "a", "stock", "record"]'))
assert json.loads(intro_text) == intro_data

The first result is a dictionary; the second is a list. Before indexing a decoded object,
check the shape that your function promises to accept. An **exception** interrupts the normal
path. `raise ValueError(...)` refuses an invalid value; `try`/`except` lets a caller inspect that
expected refusal. Catch the expected class, rather than turning every programming error into
apparent success. `finally` runs cleanup even when an earlier operation raises.

An **annotation**, such as `count: int`, documents the expected type. It does not by itself
enforce the type at runtime. A **class** defines a kind of object; an instance holds one object's
data. `@dataclass` asks Python to generate routine construction and comparison methods from
annotated fields. `frozen=True` prevents ordinary reassignment of the instance's fields; it does
not make every object nested inside those fields immutable. A **method** is a function attached
to a class; `self` refers to the instance receiving the call.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class IntroObservation:
    operation: str
    count: int


intro_observation = IntroObservation("count-mango", 3)
print(intro_observation.operation, intro_observation.count)
assert intro_observation == IntroObservation("count-mango", 3)

A **callback** is a function passed to another function. This is how the classroom harness
invokes *your* implementation. The argument `candidate` below is a function object; parentheses
perform the call. Predict the two answers before execution, then trace the result to the callback.

In [ ]:
def intro_apply(candidate, value):
    return {"input": value, "observed": candidate(value)}


def intro_double(value):
    return value * 2


print(intro_apply(intro_double, 3))
print(intro_apply(lambda value: value + 2, 3))
assert intro_apply(intro_double, 3)["observed"] == 6

`lambda value: value + 2` is a small anonymous function. A **closure** is a function that retains
access to values from its surrounding scope. It can bind a tool to a shop snapshot. A shallow
copy duplicates only the outer container; `copy.deepcopy` also copies nested containers used
in these fixtures. A **set** stores distinct values; `required <= allowed` asks whether every
required item is allowed. `frozenset` is the corresponding immutable set. A tuple groups ordered
values; `(value,)` is a one-item tuple, including the comma.

**Paths and cleanup.** `Path` represents a filesystem location. `path / "file.json"` constructs
a child path; `read_text` and `write_text` read and write text. A context manager, used with
`with`, manages entry and exit. A temporary-directory context removes its contents on exit.
Save your submission outside temporary runtime directories. Reopening a file is different from
reusing a Python variable: the former tests retained bytes, while the latter only tests this kernel.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as intro_folder:
    intro_path = Path(intro_folder) / "observation.json"
    intro_path.write_text(json.dumps(intro_data), encoding="utf-8")
    intro_reopened = json.loads(intro_path.read_text(encoding="utf-8"))
    assert intro_reopened == intro_data
    print("Read from a file:", intro_reopened)

**Retrieval check:** explain JSON versus a dictionary, annotation versus validation, class versus
instance, and defining a callback versus invoking it. Change the callback above so an incorrect
implementation visibly changes the observed output. This distinction will matter when grading
your connected work. Reference: Python's [JSON](https://docs.python.org/3/library/json.html),
[dataclasses](https://docs.python.org/3/library/dataclasses.html), and
[pathlib](https://docs.python.org/3/library/pathlib.html) documentation.

### Pydantic: turn an input dictionary into a checked object

Pydantic is an additional Python library for validating data. Its **model** is a class describing
fields, not a neural network. Inherit from `BaseModel`, declare annotated fields, then call
`model_validate` on incoming data. A field without a default is required. A field with a default
can be omitted. The result is an instance whose values you read with dot notation.

In [ ]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError


class IntroCourseRequest(BaseModel):
    model_config = ConfigDict(strict=True, extra="forbid")
    name: str = Field(min_length=1)
    quantity: int = Field(gt=0, le=1000)
    note: str = ""


intro_request = IntroCourseRequest.model_validate({"name": "mango", "quantity": 4})
print(intro_request.name, intro_request.quantity, repr(intro_request.note))
assert intro_request.note == ""

The annotation says the field's type. `Field` supplies constraints: `gt=0` means greater than
zero, `le=1000` means at most 1000, and `min_length=1` excludes an empty name. `ConfigDict` sets
model-wide behavior. `strict=True` rejects conversions for this integer field, including `"4"`,
`4.0` and `True`; `extra="forbid"` rejects undeclared keys. Pydantic can otherwise convert some
compatible inputs, so choose this boundary deliberately rather than assuming every accepted
input arrived in the expected type.

Predict which rule refuses each payload. `ValidationError` reports a failed contract. Its
`errors()` entries contain `loc`, the field location, and `type`, the failure category. Catching
that expected exception lets the notebook inspect the failure and continue.

In [ ]:
intro_bad_requests = [
    {"name": "mango", "quantity": "4"},
    {"name": "mango", "quantity": True},
    {"name": "", "quantity": 4},
    {"name": "mango", "quantity": 0},
    {"name": "mango", "quantity": 4, "approved": True},
    {"quantity": 4},
]
for intro_bad_request in intro_bad_requests:
    try:
        IntroCourseRequest.model_validate(intro_bad_request)
    except ValidationError as intro_error:
        print([(item["loc"], item["type"]) for item in intro_error.errors(include_input=False)])
    else:
        raise AssertionError("An invalid input crossed the declared contract")

Use `model_dump()` for a Python dictionary, `model_dump_json()` for JSON text, and
`model_validate_json()` to parse and validate JSON. `model_json_schema()` describes the contract;
it is neither an instance's current values nor an invocation of the business handler.

In [ ]:
intro_serialized = intro_request.model_dump_json()
intro_schema = IntroCourseRequest.model_json_schema()
assert IntroCourseRequest.model_validate_json(intro_serialized) == intro_request
print("Actual values:", intro_request.model_dump())
print("Quantity contract:", intro_schema["properties"]["quantity"])
assert intro_schema["properties"]["quantity"]["exclusiveMinimum"] == 0

Four is valid input to this schema even if the shop needs six. Pydantic checks the declared
shape and constraints; the handler still needs authoritative stock, price and permission.
Ordinary assignments to an existing instance are not automatically revalidated unless configured
for assignment validation. This lesson validates new input at the boundary and uses the resulting
values. Explain these limits before relying on a model object in a transaction or tool call.

Chapter 2's full introduction expands this pattern with a separate data-repair checkpoint.
This notebook contains the required pattern here so prior Pydantic experience is not needed.
References: [models](https://docs.pydantic.dev/latest/concepts/models/),
[fields](https://docs.pydantic.dev/latest/concepts/fields/), and
[strict mode](https://docs.pydantic.dev/latest/concepts/strict_mode/).

### SQLite from the first row to an atomic change

A dictionary disappears when its process ends. A database can retain records so a later process
can resume from evidence. **SQLite** is an embedded database: Python's `sqlite3` library opens
a local database file without starting a separate database server. **SQL** is the language used
to define, select and change its records. A **table** has named columns and rows. A **primary
key** identifies a row; a **query** asks for rows satisfying a condition.

Start with a deliberately small preference table. Read the SQL as instructions: create the
table, insert one named value, then select the value for one session. `?` is a parameter
placeholder; the values are passed separately so they are data, not SQL instructions.
`fetchone()` returns one row or `None`; it does not guarantee that a matching row exists.

In [ ]:
import sqlite3
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as intro_sql_folder:
    intro_db_path = Path(intro_sql_folder) / "example.sqlite"
    intro_db = sqlite3.connect(intro_db_path, autocommit=True)
    intro_db.execute("CREATE TABLE preference (id INTEGER PRIMARY KEY, session TEXT, value TEXT)")
    intro_db.execute("INSERT INTO preference (session,value) VALUES (?,?)", ("lucy", "09:00"))
    intro_row = intro_db.execute(
        "SELECT value FROM preference WHERE session=?", ("lucy",)
    ).fetchone()
    print("Matching row:", intro_row)
    assert intro_row == ("09:00",)
    assert (
        intro_db.execute(
            "SELECT value FROM preference WHERE session=?", ("another-session",)
        ).fetchone()
        is None
    )
    intro_db.close()
    intro_reopen = sqlite3.connect(intro_db_path, autocommit=True)
    assert intro_reopen.execute("SELECT count(*) FROM preference").fetchone()[0] == 1
    intro_reopen.close()
print("The row survived closing and reopening its connection.")

Index `[0]` selects the first column of a returned tuple. `sqlite3.Row` is an alternative row
factory that also permits named-column access. `dict(row)` then produces an ordinary dictionary.
The book's `Database` wrapper supplies that configuration and its schema; the wrapper is course
code, while `sqlite3` is the standard library. You will use the public connection and transaction
methods explained at the exercise boundary, rather than needing to reconstruct the wrapper.

Now consider a budget. Moving five pence from reserved to spent requires two values to change
together. A **transaction** makes a group of local changes commit together or roll back together.
The example explicitly controls SQL transactions with `autocommit=True` and SQL statements.
`BEGIN IMMEDIATE` starts a write transaction; `COMMIT` keeps its changes; `ROLLBACK` discards them.
Predict the row after the deliberately raised exception. Catching an error alone would not undo
the first update; the rollback is the operation that restores the prior state.

In [ ]:
intro_ledger = sqlite3.connect(":memory:", autocommit=True)
intro_ledger.execute(
    "CREATE TABLE budget (id INTEGER PRIMARY KEY, reserved INTEGER, spent INTEGER)"
)
intro_ledger.execute("INSERT INTO budget VALUES (1,5,0)")
intro_ledger.execute("BEGIN IMMEDIATE")
try:
    intro_ledger.execute("UPDATE budget SET reserved=0 WHERE id=1")
    raise ValueError("injected failure before the matching spend update")
except ValueError:
    intro_ledger.execute("ROLLBACK")
assert intro_ledger.execute("SELECT reserved,spent FROM budget").fetchone() == (5, 0)
intro_ledger.execute("BEGIN IMMEDIATE")
intro_ledger.execute("UPDATE budget SET reserved=0,spent=5 WHERE id=1")
intro_ledger.execute("COMMIT")
print(
    "After a complete change:", intro_ledger.execute("SELECT reserved,spent FROM budget").fetchone()
)
intro_ledger.close()

The literal `:memory:` creates a temporary database inside this connection; it is useful for the
small experiment, but the earlier file example establishes persistence. Neither example proves
that a remote supplier rolls back when the local transaction rolls back. An external operation
has its own state and evidence.

**SQL you will meet later.** `UPDATE ... SET ... WHERE ...` changes selected rows. `AND` combines
conditions. `ORDER BY` makes an ordering explicit; absent that clause, do not rely on row order.
`count(*)` counts rows; `sum(amount)` totals a column and can be `NULL` on an empty input;
`coalesce(sum(amount),0)` uses zero for that empty aggregate. A `UNIQUE` constraint rejects
duplicate identities. `GROUP BY status` computes one aggregate per status.

An **invariant** is a condition that must remain true across operations, such as nonnegative
reserved money. A **snapshot** is a consistent view at one point; two separate reads can describe
different moments unless their transaction contract binds them. In the book, `with db.immediate()`
groups related writes. It is a course-defined context manager with the commit/rollback purpose
you just observed. Do not assume that an arbitrary `with connection` has identical behavior under
every SQLite autocommit setting.

**Your prediction:** two workers both read ten remaining pence outside a transaction and each
approve seven. Why can both believe the next order fits? Explain what must be checked together
with the write. Then change the example's initial reserved amount and repeat the failure.
Reference: Python's [SQLite tutorial and transaction control](https://docs.python.org/3/library/sqlite3.html).

## Memory means selecting retained evidence for a new request

Lucy says deliveries should arrive at nine, then corrects herself to ten. A useful assistant
must retain the correction after a restart and avoid presenting both times as current guidance.
**Persistence** means the record survives the process. **Retrieval** means choosing which retained
records to include now. They are separate mechanisms: a database can preserve every revision
while retrieval returns only the currently active revision for this session.

A **session** identifies the conversation or work context that owns a memory. **Provenance**
records where a value came from. A **revision** is a new version of an earlier value. Here a
correction creates current guidance while old evidence can remain available for audit. Forgetting
excludes a value from future context; it does not erase already sent provider requests or backups.

### Derive a retrieval rule on visible rows

Our small table has two sessions and two revisions. First filter by session and active status.
Then rank the remaining rows for this query. Reversing those operations can waste a limited
context budget on a foreign or superseded row. Predict the returned identities for Lucy.

In [ ]:
intro_memories = [
    {"id": 1, "session": "lucy", "active": False, "name": "delivery", "value": "09:00"},
    {"id": 2, "session": "lucy", "active": True, "name": "delivery", "value": "10:00"},
    {"id": 3, "session": "another-shop", "active": True, "name": "delivery", "value": "06:00"},
    {"id": 4, "session": "lucy", "active": True, "name": "invoice", "value": "email"},
]
intro_eligible = [row for row in intro_memories if row["session"] == "lucy" and row["active"]]
print([(row["id"], row["value"]) for row in intro_eligible])
assert [row["id"] for row in intro_eligible] == [2, 4]

The database exercise performs the same filtering with a parameterized `WHERE` clause. The
bounded result must retain identity, name, value, source and creation evidence. Returning only
a friendly sentence would discard the provenance needed to investigate a bad answer.

### Understand the deliberately simple relevance score

We lowercase using `casefold`, split into whitespace-separated words, and count the intersection
with query words. A set intersection keeps words present in both sets. This lexical score is
easy to inspect; it is not a semantic embedding or a claim that similar meanings always match.
An **embedding** would represent text numerically for another similarity method; we do not need
that additional model or library for this lesson's explicit rule.

In [ ]:
intro_query_words = set("DELIVERY time".casefold().split())
intro_ranked = []
for intro_row in intro_eligible:
    intro_words = set((intro_row["name"] + " " + intro_row["value"]).casefold().split())
    intro_ranked.append((len(intro_query_words & intro_words), intro_row["id"], intro_row["value"]))
intro_ranked.sort(key=lambda item: (-item[0], -item[1]))
print(intro_ranked)
assert intro_ranked[0] == (1, 2, "10:00")

The negative signs turn ascending Python sorting into descending score and descending identity.
Identity breaks ties deterministically in favor of the newer row. Slicing `[:maximum]` then
limits the output. Validate the maximum first; accepting negative slicing would quietly turn
an invalid requested budget into a different selection rule.

### Follow the context all the way to the model seam

The course's `Database` supplies durable rows. `preferences` retrieves them. The context builder
places selected values in the next request. The replay model records the actual messages it
received. A successful insert proves only persistence; a successful query proves retrieval;
the recorded message proves that the retrieved data was connected to the model input.

In Unit A you implement retrieval, close and reopen the database, then inspect the context.
In Unit B a query loses its `active=1` condition. The old value can reappear even if the new value
ranks first. The repair must exclude stale guidance rather than merely move it lower in the list.

**Before the main task:** explain which records survive an empty query, why another session's
matching word cannot grant eligibility, and what happens when two rows have the same score.
For transfer, correct twice, forget one preference, and vary the retrieval limit. Keep actual
row identities beside the final context text so a plausible-looking answer cannot conceal the
wrong revision. Authoritative current stock still comes from the shop tool, not remembered prose.

## Choose an explicit starting point for this independent notebook

This Unit B runs without Unit A. By default it prepares a **supplied reference starting point**
and labels its provenance. It is not evidence that you built Unit A. To investigate your own
successful implementation, set `LEARNER_HANDOFF` to its saved path before running the cell.
An invalid selected file refuses; it is never silently replaced with the reference.

`SourceTask` supplies copied-source execution and handoff validation; `RuntimeLab` supplies the
controlled failure experiment. Their public operations are introduced beside the main exercise.
The artifact stores identity and observations; no variables from another kernel are required.


In [ ]:
LEARNER_HANDOFF = None

<details><summary>Prepare and validate the supplied starting artifact</summary>


In [ ]:
import json
import runpy
import shutil
import textwrap
from pathlib import Path

COURSE_INPUT = COURSE_WORK / "ch05-unit-a-handoff-v1.json"
if LEARNER_HANDOFF is not None:
    learner_input = Path(LEARNER_HANDOFF).expanduser().resolve()
    if not learner_input.is_file():
        raise FileNotFoundError("The selected learner handoff does not exist")
    if learner_input != COURSE_INPUT.resolve():
        shutil.copy2(learner_input, COURSE_INPUT)
    HANDOFF_ORIGIN = "LEARNER_SELECTED"
else:
    source_task_class = runpy.run_path(
        str(COURSE_ROOT / "book/always_on/exercises/source_tasks_v1.py")
    )["SourceTask"]
    reference_task = source_task_class(COURSE_ROOT, 4)
    try:
        reference_task.install(textwrap.dedent(reference_task.fragment))
        reference_observation = reference_task.visible("SUPPLIED_REFERENCE_START")
        if reference_observation["status"] != "PASS":
            raise RuntimeError("The supplied starting point did not pass its connection check")
        reference_task.save(COURSE_INPUT, reference_observation)
    finally:
        reference_task.close()
    HANDOFF_ORIGIN = "SUPPLIED_REFERENCE"
print("Starting evidence:", HANDOFF_ORIGIN)
print("The core task below validates the selected artifact before using it.")

</details>


## Understand the supplied execution interface

The course runtime is provided so your implementation can be connected to real callers and
storage. `SourceTask(ROOT, chapter)` makes a private copy. `install(source)` replaces only the
declared function; `visible()` invokes the real chapter probe; `save(path, result)` retains a
successful implementation and its evidence. `load(path)` checks the saved identities and hashes.
`inject_failure()` changes the declared boundary; `repair(fragment)` replaces that broken fragment.
`close()` removes the scratch copy after you retain evidence. These methods are supplied harness
operations, not additional packages you must discover or install.

`RuntimeLab` provides the same copied-source failure experiment without the complete-function
construction layer. Its `run` method records exit status, observations and the compared expectation.
A subprocess log from an unfinished learner implementation is feedback about that implementation;
it is not a successful connection. A syntax error in the notebook cell itself is a separate issue
to fix. The task below names which interface it uses.

For direct-function units, the visible driver calls your callback without installing a source
string. In either case, trace where your code is invoked. Supplied fixtures, database wrappers and
replay models are labelled infrastructure; your own implementation and changed-case explanation
are the evidence of learning.


## Main practical: construct, connect and challenge



Putting the new preference first does not remove contradictory old guidance. This time you begin with your Unit A implementation and its saved evidence. Lucy corrects delivery time, but the next request again contains the superseded preference.

## Verify the handoff

The starting-point cell has selected the Unit A artifact explicitly. A selected learner handoff must validate; the default reference start is labelled separately. Run the setup and keep the runtime and implementation hashes in your submission.

In [ ]:
import json
import os
import runpy
from pathlib import Path

ROOT = COURSE_ROOT

SourceTask = runpy.run_path(str(ROOT / "book/always_on/exercises/source_tasks_v1.py"))["SourceTask"]
REFERENCE_LESSON = 4
HANDOFF = Path("ch05-unit-a-handoff-v1.json")
handoff_status = "MISSING"
if HANDOFF.is_file():
    task = SourceTask(ROOT, REFERENCE_LESSON)
    try:
        handoff = task.load(HANDOFF)
        handoff_status = "VERIFIED"
        print("IMPLEMENTATION", handoff["implementation_sha256"])
    finally:
        task.close()
print("UNIT_A_HANDOFF", handoff_status)

## Reproduce and diagnose

Predict the consequence of this injected boundary before executing it:

```text
"WHERE session=?",
            (session,),
```

The controlled mutation changes the same implementation you submitted. It refuses if the declared mutation boundary no longer occurs exactly once; inspect an alternative implementation with the instructor before adapting the experiment.

In [ ]:
baseline = broken = None
if handoff_status == "VERIFIED":
    task = SourceTask(ROOT, REFERENCE_LESSON)
    try:
        task.load(HANDOFF)
        baseline = task.visible("YOUR_BASELINE")
        if baseline["status"] != "PASS":
            raise ValueError("Saved Unit A code no longer satisfies the visible contract")
        task.inject_failure()
        broken = task.run("INJECTED_FAILURE", expected=task.spec["expected_broken"])
        print("BEFORE", baseline["observation"])
        print("AFTER", broken["observation"])
    finally:
        task.close()
else:
    print("HANDOFF_REQUIRED: complete Unit A before performing Unit B")

State a diagnosis using those two observations. Name a test that would prove your diagnosis wrong. Remember, correct, close and reopen SQLite; invoke context and inspect the actual system message seen by the model.

## Repair the boundary

Return the complete replacement for the injected fragment. Do not edit the oracle or print a desired observation. Repair the actual source. The starter keeps the defect so the learner outcome remains incomplete.

In [ ]:
def repair_fragment():
    return '"WHERE session=?",\n            (session,),'

<details><summary>Hint 1 — the consequence</summary>

Lucy corrects delivery time, but the next request again contains the superseded preference.

</details>

<details><summary>Hint 2 — the evidence</summary>

Compare the two observations, then trace the changed field to `preferences` in `src/sovereign_agent/assistant_context.py`. Distinguish a schema refusal from a business-rule or authority refusal.

</details>

<details><summary>Hint 3 — the design</summary>

Use a parameterized query, set intersection for relevance, deterministic ties and a validated maximum.

</details>

In [ ]:
def connect_repair(fragment):
    task = SourceTask(ROOT, REFERENCE_LESSON)
    try:
        task.load(HANDOFF)
        task.inject_failure()
        task.repair(fragment)
        return task.visible("YOUR_REPAIR")
    finally:
        task.close()


repair_result = None
if handoff_status == "VERIFIED":
    repair_result = connect_repair(repair_fragment())
    print("REPAIR", repair_result["status"], repair_result["observation"])
else:
    print("REPAIR_NOT_ATTEMPTED: missing Unit A evidence")

## Transfer under a changed constraint

Correct twice, forget one key and introduce another session with the same preference name. Prove no stale or foreign value reaches context.

Create a fresh task, load your handoff, inject the defect and apply your repair. Then change only the copied probe to exercise the new condition. Keep the actual observation and a prediction written beforehand. Explain why a visible-case lookup or a blanket refusal could pass the original example but fail this transfer.

The instructor's holdout applies your repair to a new copied runtime and checks both the positive case and the missing protection. An exact exception or changed state must cause a failure; no broad error is accepted as successful refusal.

## Exit ticket

Submit the original handoff, baseline and broken observations, repair, transfer probe and results. State what Lucy would experience before and after the fix. Identify the guarantee that still requires separate evidence: Forgetting future context is not secure erasure of backups or past provider requests.

In [ ]:
passed = repair_result is not None and repair_result["status"] == "PASS"
exercise_report = {
    "unit": "ch05-b",
    "attempted": int(repair_result is not None),
    "completed": int(passed),
    "failed": int(repair_result is not None and not passed),
    "skipped": int(repair_result is None),
    "connection": "PASS" if passed else "NOT_READY",
    "handoff": handoff_status,
}
print("EXERCISE_REPORT=" + json.dumps(exercise_report, sort_keys=True))

## Changed-constraint construction: Retrieve the newest eligible memory under a limit

**Allow twenty minutes.** Spend three minutes predicting, ten implementing and tracing, five
on a new case of your own, and two explaining the surviving limitation. This is dedicated work,
not an invitation to run a supplied answer. Both units revisit the same invariant after different
core experiences; in Unit B, attempt this task from memory before consulting Unit A.

Implement transfer_check(rows, session, limit). Each row has unique integer id, session and boolean active. Return eligible row IDs newest first, at most limit. Require an exact integer limit in 1..100; otherwise raise ValueError. Empty eligible input returns an empty list. Do not mutate rows. This deliberately isolates eligibility and tie order from lexical scoring.

Write your expected values before running the table. Keep one accepted case and one refusal.
Your function is passed directly into the driver below. The driver copies inputs and checks
they remain unchanged; it does not replace your implementation with the reference answer.

<details><summary>Hint 1 — identify the authoritative inputs</summary>
Name the source field for each output value. Which input changes while the rule remains the same?
</details>
<details><summary>Hint 2 — choose the boundary cases</summary>
Start with exact empty, exact equality and one value on each side of the boundary where valid.
Do not add a special case for a visible product name or operation identity.
</details>


In [ ]:
def transfer_check(rows, session, limit):
    raise NotImplementedError("Filter before ordering and limiting")

In [ ]:
import copy
import json

TRANSFER_CASES = [
    (
        "current own row",
        [
            [
                {"id": 1, "session": "lucy", "active": False},
                {"id": 2, "session": "lucy", "active": True},
                {"id": 3, "session": "other", "active": True},
            ],
            "lucy",
            2,
        ],
        [2],
    ),
    (
        "newest first",
        [
            [
                {"id": 8, "session": "lucy", "active": True},
                {"id": 9, "session": "lucy", "active": True},
            ],
            "lucy",
            1,
        ],
        [9],
    ),
    ("empty", [[], "lucy", 3], []),
    ("zero limit", [[], "lucy", 0], {"raises": "ValueError"}),
]


def same_transfer_value(actual, expected):
    if type(actual) is not type(expected):
        return False
    if isinstance(expected, dict):
        return actual.keys() == expected.keys() and all(
            same_transfer_value(actual[key], value) for key, value in expected.items()
        )
    if isinstance(expected, list):
        return len(actual) == len(expected) and all(
            same_transfer_value(a, e) for a, e in zip(actual, expected, strict=True)
        )
    return actual == expected


def run_transfer(candidate, cases):
    observations = []
    for label, arguments, expected in cases:
        supplied = copy.deepcopy(arguments)
        before = copy.deepcopy(supplied)
        raised = None
        try:
            actual = candidate(*supplied)
        except NotImplementedError:
            raised = "NotImplementedError"
            actual = {"unfinished": True}
        except Exception as error:
            raised = type(error).__name__
            actual = {"raises": raised}
        expects_error = isinstance(expected, dict) and set(expected) == {"raises"}
        correct = (
            raised == expected["raises"]
            if expects_error
            else (raised is None and same_transfer_value(actual, expected))
        )
        passed = correct and same_transfer_value(supplied, before)
        observations.append(
            {"case": label, "expected": expected, "observed": actual, "passed": passed}
        )
        print("PASS" if passed else "NEEDS_WORK", label, "expected", expected, "observed", actual)
    return observations


transfer_observations = run_transfer(transfer_check, TRANSFER_CASES)
TRANSFER_PASSED = all(row["passed"] for row in transfer_observations)
print("TRANSFER_STATUS", "PASS" if TRANSFER_PASSED else "NEEDS_WORK")

### Design a counterexample and retrieve the mechanism

Add one new case with an independently calculated expected outcome to `TRANSFER_CASES` and rerun
the driver. Change one condition at a time. Then deliberately replace your candidate with a
constant answer in a temporary copy and show a case that rejects it. Restore your implementation.
Explain why that counterexample is stronger than repeating the original example with a new name.

Without viewing the worked example, write the invariant in words and trace one observed value
back to its input. Identify which part is a local fixture result and which claim would need a
live provider, host or external-system observation. Keep a first attempt even if you used a hint.


## Save your evidence and explain the result

Fill the prediction notes and your explanation before saving. Include the exact observed value,
the input or retained row that caused it, your code's invocation point, one failed hypothesis,
and the strongest claim the evidence still cannot support. A completed code cell alone does not
earn explanation credit. Do not label reference-start behavior as your own Unit A construction.

Keep this edited notebook, the Markdown if used for notes, saved handoff files, and the JSON record
below. Your work folder survives scratch cleanup and can be reopened in a new kernel. An instructor
can ask for an unseen case after the visible checks; keep your implementation general.


In [ ]:
explanation_notes = {
    "causal_trace": "Explain the input, learner invocation and observed result.",
    "failed_hypothesis": "Describe a prediction the evidence changed.",
    "remaining_limit": "Name the guarantee not established by this experiment.",
}
course_submission = {
    "unit": "ch05-b",
    "planned_minutes": 90,
    "starting_evidence": globals().get("HANDOFF_ORIGIN", "INDEPENDENT_UNIT_A"),
    "prediction": prediction_notes,
    "explanation": explanation_notes,
    "core_report": exercise_report,
    "transfer": transfer_observations,
    "explanation_review": "HUMAN_REVIEW_REQUIRED",
}
submission_path = COURSE_WORK / "ch05-b-submission-v1.json"
submission_path.write_text(
    json.dumps(course_submission, indent=2, sort_keys=True), encoding="utf-8"
)
print("Saved evidence:", submission_path)
print(
    "COURSE_REPORT="
    + json.dumps(
        {
            "unit": "ch05-b",
            "transfer_passed": TRANSFER_PASSED,
            "starting_evidence": course_submission["starting_evidence"],
            "edition": "student",
        },
        sort_keys=True,
    )
)

## Extension: let the model manage Lucy's memory

Everything above kept one rule fixed: *you* decided what the shop remembers, and your code
wrote it. The remaining question in this chapter is what changes when the **model** decides.
Lucy tells the assistant that mango sold out again on Saturday. Nobody calls `remember`.
The model reads the message, judges whether that fact will still matter next week, and asks
to store it.

The mechanism is the loop from Chapter 3, shipped in the embedded runtime as
`agent_loop.run_loop`: send the conversation and the tool schemas, read back an answer or tool
calls, run each call through the dispatcher, append the observation under its `tool_call_id`,
ask again, and count every attempt against `Limits`. The model only ever *proposes* a write.
`memory_tools` wires `remember_fact` and `recall_facts` to the same `Dispatcher` the book uses
for every tool. Writing is marked *consequential*, so the dispatcher refuses it unless a
`before_write` gate is attached, and that gate, `MemoryWritePolicy`, can refuse a write and
records why. Reading is not consequential. That asymmetry is the whole design.

`llm_lab` supplies the model seam: `ScriptedModel` replays turns written in this notebook and
runs everywhere; `OpenAIModel` sends the same messages to a hosted model when a key is present.
The loop cannot tell them apart; the record always says which one ran.

**Prediction:** the next cell seeds one memory and prints the two tool contracts. Before
running it, write which fields the model must supply to store a fact, and which shop rule
would refuse a correctly shaped fact.


In [ ]:
import json
from pathlib import Path

from sovereign_agent.agent_loop import Limits, run_loop
from sovereign_agent.database import Database
from sovereign_agent.llm_lab import ScriptedModel, describe, scripted_turn, tool_trace
from sovereign_agent.memory import remember
from sovereign_agent.memory_tools import toolbox
from sovereign_agent.model_turn import ToolCall

LLM_MEMORY_DB = COURSE_WORK / "ch05-llm-memory.sqlite"
SEED_MEMORY = "Deliveries from the dairy supplier arrive at ten on weekdays."


def fresh_memory_database(path):
    """Start from an empty shop memory so reruns and replays observe the same rows."""
    for stale in (path, Path(f"{path}-wal"), Path(f"{path}-shm"), path.with_suffix(".authority")):
        stale.unlink(missing_ok=True)
    return Database(path)


def seeded_toolbox(**policy):
    """A fresh memory holding only the seed row, behind the gated tools."""
    box = toolbox(fresh_memory_database(LLM_MEMORY_DB), actor_id="lucy", **policy)
    remember(box.db, "lucy-seed-001", SEED_MEMORY, importance=0.8)
    return box


memory_box = seeded_toolbox()
memory_dispatcher = memory_box.dispatcher()
for tool_schema in memory_dispatcher.schemas():
    function = tool_schema["function"]
    print(function["name"], "requires", function["parameters"]["required"])
    print("   ", function["description"])
print("Writes allowed per session:", memory_box.policy.max_writes)
print("Phrases the shop never stores:", memory_box.policy.banned_phrases)
print("Patterns the shop never stores:", memory_box.policy.banned_patterns)
MEMORY_MESSAGES = [
    {
        "role": "system",
        "content": (
            "You help Lucy run her ice cream shop. You have two tools: recall_facts to search "
            "what the shop already remembers, and remember_fact to store a durable fact. Store "
            "a fact only when it will still matter in a future conversation: a standing pattern, "
            "a supplier arrangement, a recurring problem. Search before you store. The shop may "
            "refuse a write; if it does, read the result and adapt rather than retrying the same "
            "thing. Finish with plain words about what you stored and why."
        ),
    },
    {
        "role": "user",
        "content": (
            "Mango sold out again on Saturday afternoon, so we should order extra on Fridays. "
            "Also remember the dairy account is paid by card, number ending 4421."
        ),
    },
]

The schema the model reads is the `model_json_schema()` you inspected earlier, wrapped in the
`{"type": "function", ...}` envelope the API expects. `required` lists the fields without
defaults; `strict=True` and `extra="forbid"` still apply, so a call with `importance` as the
string `"0.9"` is refused before any handler runs, exactly like the bad `IntroCourseRequest`
payloads above.

Hand-build three tool calls and dispatch them yourself, with no model involved. Predict each
result: one valid write, one write that names a card number, and one with a string where a
float belongs. Then read the `memories` table directly to see which rows SQLite actually
holds. `invoke_with_reason` adds the shop rule that refused a write. A Pydantic refusal stays
`invalid_arguments`, because the dispatcher never copies raw arguments into an error message.


In [ ]:
manual_calls = [
    ToolCall(
        id="manual-1",
        name="remember_fact",
        arguments={
            "content": "Mango sells out most Saturday afternoons; order extra on Friday.",
            "importance": 0.7,
            "reason": "recurring pattern",
        },
    ),
    ToolCall(
        id="manual-2",
        name="remember_fact",
        arguments={
            "content": "Lucy's card number ends in 4421 for the dairy account.",
            "importance": 0.3,
            "reason": "billing detail",
        },
    ),
    ToolCall(
        id="manual-3",
        name="remember_fact",
        arguments={
            "content": "Pistachio is the slowest seller in winter.",
            "importance": "0.9",
            "reason": "seasonal pattern",
        },
    ),
]
manual_results = [memory_box.invoke_with_reason(memory_dispatcher, call) for call in manual_calls]
for call, result in zip(manual_calls, manual_results, strict=True):
    print(call.id, result)
assert manual_results[0] == {"ok": True, "value": {"stored": True, "memory_id": "lucy-llm-001"}}
assert manual_results[1]["error"] == "content_not_permitted"
assert manual_results[2] == {"ok": False, "error": "invalid_arguments"}
stored_rows = memory_box.db.connection.execute(
    "SELECT id, content, importance FROM memories ORDER BY id"
).fetchall()
for stored_row in stored_rows:
    print(dict(stored_row))
assert [row["id"] for row in stored_rows] == ["lucy-llm-001", "lucy-seed-001"]
memory_box.db.connection.close()

### Run the loop against a recorded transcript

`ScriptedModel` replays what a model replied at each step. It is not a stand-in pretending to
be live: the record says `scripted`, while the tool calls, the refusals and the SQLite rows
are real. Read the script as a conversation. Turn one searches memory before storing anything.
Turn two proposes two facts, one of which the shop refuses. Turn three adapts and stores the
arrangement without the secret. Turn four asks for no tools, which ends the loop.

Inside the loop a refusal arrives as an ordinary failed observation, `tool_failed`, because
`Dispatcher.invoke` never copies a raw error into the transcript. The policy's own ledger,
`memory_box.policy.refused`, keeps the reason for you and for the classroom. The cell starts
from a fresh shop memory holding only the seed row, so your manual write above is gone.
**Prediction:** how many rows will `memories` hold after this run, and which turn produces the
refusal?


In [ ]:
RECORDED_TURNS = [
    scripted_turn(calls=[{"name": "recall_facts", "arguments": {"query": "mango saturday dairy"}}]),
    scripted_turn(
        calls=[
            {
                "name": "remember_fact",
                "arguments": {
                    "content": "Mango sells out most Saturday afternoons; order extra on Friday.",
                    "importance": 0.7,
                    "reason": "recurring stock pattern",
                },
            },
            {
                "name": "remember_fact",
                "arguments": {
                    "content": "The dairy account is paid by card number ending 4421.",
                    "importance": 0.3,
                    "reason": "billing arrangement",
                },
            },
        ]
    ),
    scripted_turn(
        calls=[
            {
                "name": "remember_fact",
                "arguments": {
                    "content": "The dairy account is paid by card; the details stay with Lucy.",
                    "importance": 0.3,
                    "reason": "billing arrangement without the secret",
                },
            }
        ]
    ),
    scripted_turn(
        "Stored the Saturday mango pattern and the dairy billing arrangement. The shop refused "
        "the card number, so I kept only that a card is used."
    ),
]
LOOP_LIMITS = Limits(model_calls=8, tool_calls=16, seconds=120, output_tokens=4096)

scripted_box = seeded_toolbox()
scripted_result = run_loop(
    ScriptedModel(RECORDED_TURNS), scripted_box.dispatcher(), MEMORY_MESSAGES, limits=LOOP_LIMITS
)
print(describe(scripted_result, "scripted"))
print("Refused by policy:", scripted_box.policy.refused)
scripted_trace = tool_trace(scripted_result)
assert scripted_result.status == "COMPLETED"
assert [step["ok"] for step in scripted_trace] == [True, True, False, True]
assert scripted_box.written == ["lucy-llm-001", "lucy-llm-002"]
assert [reason for _, reason in scripted_box.policy.refused] == ["content_not_permitted"]
scripted_rows = scripted_box.db.connection.execute(
    "SELECT id, content FROM memories ORDER BY id"
).fetchall()
for scripted_row in scripted_rows:
    print(dict(scripted_row))
assert len(scripted_rows) == 3
scripted_box.db.connection.close()

### Run the same loop against a live model

The loop, the tools and the shop rules do not change. Only the model does. `build_model` looks
for an `OPENAI_API_KEY` in Colab's **Secrets** pane (the key icon in the left sidebar: add a
secret with that name and switch on notebook access for it), then in the process environment
for a local kernel. Without a key it returns the scripted model again and the record says so.
A missing key is a normal condition, never a silent substitution.

The live model is `gpt-5.1` through the `openai` library, which Colab ships preinstalled; on a
local kernel run `%pip install openai` once. One run costs a few thousand tokens. The model
receives the same two schemas and the same system message, and the shop policy still decides
every write. Compare the live record with the scripted one: what did the model search for
first, what did it store, what did it decline to store on its own, and what did the shop
refuse for it?

One more thing to watch. The policy bans the phrase `card number`, and a live model tends to
paraphrase: "card ending 4421" carries the same secret in different words. A phrase list is
weak against that; so `MemoryWritePolicy` also refuses a *pattern*, the word card followed by
digits. Read both rules in the embedded `memory_tools.py` and decide which one the live run
will trip.

**Prediction:** write down which facts from Lucy's message a careful assistant should keep.
Then run the cell and compare that list with the model's judgment, or with the recorded
transcript when the run is scripted; the printed source says which you are reading.


In [ ]:
from sovereign_agent.llm_lab import DEFAULT_MODEL, build_model, resolve_api_key

chosen_model = build_model(RECORDED_TURNS, model=DEFAULT_MODEL)
print("Model source:", chosen_model.source, "| key found:", resolve_api_key() is not None)
live_box = seeded_toolbox()
live_result = run_loop(chosen_model, live_box.dispatcher(), MEMORY_MESSAGES, limits=LOOP_LIMITS)
print(describe(live_result, chosen_model.source))
print("Refused by policy:", live_box.policy.refused)
live_rows = live_box.db.connection.execute(
    "SELECT id, content, importance FROM memories ORDER BY id"
).fetchall()
for live_row in live_rows:
    print(dict(live_row))
assert live_result.status in {"COMPLETED", "MODEL_CALL_LIMIT", "TOOL_LIMIT", "MODEL_FAILED"}
# Whatever the model proposed, nothing the policy bans reached SQLite: re-run every rule.
for live_row in live_rows:
    assert live_box.policy.check(live_row["content"]) in {None, "duplicate_of_accepted_memory"}
assert len(live_rows) - 1 == len(live_box.written) <= live_box.policy.max_writes
live_box.db.connection.close()

### Challenge: tighten the shop's rules and save the evidence

The policy is data you can change. `seeded_toolbox(max_writes=1)` lets one write through;
`banned_phrases=("card number", "delivery")` refuses more. The next cell reruns the recorded
transcript under a one-write budget. **Predict before running:** the card-number write is still
refused, but is the reason still `content_not_permitted`? Read `MemoryWritePolicy.check` in the
embedded `memory_tools.py` and note the order of its rules. With a live key, rerun the live cell
under the same budget and explain what the model did after its second write was refused; a
model that keeps retrying the same write is a finding about the model, not a bug in your policy.

Explain in your notes which part of this run is a local fixture result and which claim needs
the live model: the scripted transcript proves the loop, the gate and the SQLite rows; only a
live run proves the model's judgment. The saved record keeps the two sources separate.


In [ ]:
tight_box = seeded_toolbox(max_writes=1)
tight_result = run_loop(
    ScriptedModel(RECORDED_TURNS), tight_box.dispatcher(), MEMORY_MESSAGES, limits=LOOP_LIMITS
)
print(describe(tight_result, "scripted"))
print("Refused by policy:", tight_box.policy.refused)
assert tight_box.written == ["lucy-llm-001"]
assert [reason for _, reason in tight_box.policy.refused] == [
    "write_budget_exhausted",
    "write_budget_exhausted",
]
tight_box.db.connection.close()

llm_lab_report = {
    "unit": "ch05-b",
    "scripted": {
        "status": scripted_result.status,
        "written": list(scripted_box.written),
        "refused": [reason for _, reason in scripted_box.policy.refused],
    },
    "session": {
        "source": chosen_model.source,
        "model": DEFAULT_MODEL if chosen_model.source == "live" else None,
        "status": live_result.status,
        "model_calls": live_result.model_calls,
        "written": list(live_box.written),
        "refused": [reason for _, reason in live_box.policy.refused],
        "answer": live_result.answer[:400],
    },
    "tightened": {"max_writes": 1, "refused": [reason for _, reason in tight_box.policy.refused]},
}
llm_report_path = COURSE_WORK / "ch05-b-llm-lab-report-v1.json"
llm_report_path.write_text(json.dumps(llm_lab_report, indent=2, sort_keys=True), encoding="utf-8")
print("Saved evidence:", llm_report_path)
print("LLM_LAB_REPORT=" + json.dumps(llm_lab_report["session"], sort_keys=True))

## Keep building with Prof Rod

Found this material through a colleague, classroom or shared download? [Get the complete book at profrod.ai/book](https://profrod.ai/book) and [join the Prof Rod learner community](https://profrod.ai/community). Bring one result, one question or one failure you learned from. Share this resource with another learner and keep its source links with it so they can find the full course and future updates.
